# 🍜 Cambodian Food Search (Khmer + English) → Telegram bot

An **Information Retrieval** project: search restaurants and online food sellers across Cambodia,
in **Khmer and English**, and answer through a **Telegram bot** and a **web app** (also a Telegram Mini App).

| Step | Technique | Why |
|---|---|---|
| 3 | BM25 + fuzzy matching | classic keyword search, robust to typos |
| 4 | Multilingual embeddings + ChromaDB | search by *meaning*, across languages |
| 5 | Hybrid ranking + geo + filters | combine both, add distance, price, opening hours, khans, cities and provinces |
| 6 | CLIP image search | find food by photo |
| 7 | RAG with an LLM + conversation | answers grounded in the results; follow-up questions answered as a sentence, list or table |
| 8 | Telegram bot | the user interface |
| 9 | Export | `backend/` (the deployed app: search, AI answer, Telegram bot, web API), `requirements.txt`, `Dockerfile` |
| 10 | Try the conversation | a chat played through the same functions the app uses |

**Run order:** first **`data_preprocessing.ipynb`** (collects and cleans the data into `data/restaurants.csv`), then this notebook.

**How to run (VS Code on a Mac):** open this folder in VS Code, select the kernel **venv (food-search)**
(or `./venv/bin/python`), then *Run All*. Steps that need a key you have not set (Telegram, Gemini)
are skipped with a message; they never stop the notebook.

## 0. Setup

**What:** find the project folder, install packages, import libraries, pick the compute device, load settings and secrets.

**Why:** everything later depends on it. The paths are relative to the project root (the folder that contains
this notebook), so the project works on any computer. On Google Colab the same cells adapt automatically.

> ⚠️ **Secrets:** never paste API keys or tokens into this notebook, and never commit `.env`.
> Keys live only in `.env` (VS Code / local) or in *Colab → Secrets*. The notebook prints only *set* / *missing*.

In [1]:
# Where are we running? VS Code / local Jupyter is the default, Colab is supported too.
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path.cwd() / "food-search"      # /content/food-search on Colab
    ROOT.mkdir(exist_ok=True)
    os.chdir(ROOT)
else:
    ROOT = Path.cwd()                      # VS Code starts the kernel in the notebook's folder
    if not (ROOT / "food_search.ipynb").exists():
        print("⚠️ food_search.ipynb not found in the working folder: open the project folder in VS Code.")
print("Running in Colab:", IN_COLAB)
print("Project root:", ROOT.name)

# Create .env.example and .gitignore once (never overwrite your own files)
ENV_EXAMPLE = """# Copy this file to .env and fill in your own values. Never commit .env!
TELEGRAM_TOKEN=
GEMINI_API_KEY=
GEMINI_MODEL=
OLLAMA_MODEL=
OLLAMA_HOST=http://localhost:11434
LLM_PROVIDER=gemini
EMBED_MODEL=paraphrase-multilingual-MiniLM-L12-v2
ADMIN_IDS=
# Optional: Ollama cloud instead of a local Ollama (set OLLAMA_HOST=https://ollama.com and an OLLAMA_MODEL from its list)
OLLAMA_API_KEY=
# Optional: Google Sheets shared as "Anyone with the link: Viewer" (paste the normal sheet link)
SHEET_CSV_URL=
SHEET_QUERIES_URL=
# Optional: how to read seller channels. preview = public web pages, no login (default); api = Telegram API + login
TG_MODE=preview
# Only for TG_MODE=api (https://my.telegram.org -> API development tools)
TG_API_ID=
TG_API_HASH=
"""
GITIGNORE = ".env\nvenv/\nchroma_db/\nexport/\n__pycache__/\n.ipynb_checkpoints/\n*.session\n"
for name, content in {".env.example": ENV_EXAMPLE, ".gitignore": GITIGNORE}.items():
    path = ROOT / name
    if path.exists():
        print(f"{name} already exists (kept)")
    else:
        path.write_text(content, encoding="utf-8")
        print(f"created {name}")

Running in Colab: False
Project root: food-search
.env.example already exists (kept)
.gitignore already exists (kept)


In [2]:
# Install into the *active kernel's* environment (quiet; fast if already installed)
%pip install -q rank-bm25 rapidfuzz sentence-transformers chromadb open_clip_torch "python-telegram-bot>=20" google-genai requests pandas numpy matplotlib scikit-learn python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # quieter Hugging Face tokenizers
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")     # no ChromaDB telemetry

import io, re, json, math, html, time, random, logging, asyncio, shutil, tempfile, unicodedata, warnings
from datetime import datetime
from zoneinfo import ZoneInfo
from functools import lru_cache
from collections import Counter, defaultdict, deque
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from IPython.display import display, HTML, Markdown
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from rapidfuzz import fuzz, process

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="IProgress not found")   # progress bars need ipywidgets; not required

In [4]:
# Reproducibility + logging (our own logs only; libraries stay quiet)
SEED = 42


random.seed(SEED)


np.random.seed(SEED)


torch.manual_seed(SEED)


logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s", force=True)


logging.getLogger("httpx").setLevel(logging.WARNING)   # httpx would log Telegram URLs, which contain the token


log = logging.getLogger("food_search.bot")


log.setLevel(logging.INFO)


In [5]:
# Python version check
print("Python", sys.version.split()[0])
if sys.version_info >= (3, 13):
    print("⚠️ Python 3.13 or newer: use 3.11 or 3.12 if a package fails to install.")

Python 3.13.5
⚠️ Python 3.13 or newer: use 3.11 or 3.12 if a package fails to install.


> **Note (this project):** the venv was built with **Python 3.13**. Python 3.15 failed (no chromadb wheels),
> and 3.11/3.12 were not installed on this Mac. Every package installed fine on 3.13, so the warning above is
> just a reminder for the future.

In [6]:
def pick_device() -> str:
    """Prefer an NVIDIA GPU, then Apple Silicon (MPS), then CPU."""
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device()


print("Device:", DEVICE)


Device: mps


In [7]:
# Folders (all relative to the project root)


DATA_DIR = ROOT / "data"
EVAL_DIR = DATA_DIR / "eval"
PHOTOS_DIR = DATA_DIR / "photos"
RESULTS_DIR = ROOT / "results"
CHROMA_DIR = ROOT / "chroma_db"
BACKEND_DIR = ROOT / "backend"


for folder in (DATA_DIR, EVAL_DIR, PHOTOS_DIR, RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Chart style: recessive grid/axes, fixed categorical colour order
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.8, "axes.axisbelow": True,
                     "axes.edgecolor": "#b5b4ae", "axes.labelcolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e", "figure.dpi": 110})

In [8]:
def get_secret(name, default=""):
    """Colab: Colab Secrets. Local / VS Code: environment variables loaded from .env."""
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                return value.strip()
        except Exception:
            pass
    return (os.getenv(name) or default).strip()


_ = load_dotenv(ROOT / ".env", override=False)   # reads .env into the environment (no values printed)

In [9]:
# --- secrets & provider settings (values are never printed) ---
TELEGRAM_TOKEN = get_secret("TELEGRAM_TOKEN")
GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
GEMINI_MODEL = get_secret("GEMINI_MODEL")          # no hardcoded model: set it in .env
OLLAMA_MODEL = get_secret("OLLAMA_MODEL")          # e.g. any model you pulled with `ollama pull`
OLLAMA_HOST = get_secret("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
OLLAMA_API_KEY = get_secret("OLLAMA_API_KEY")      # only for Ollama cloud: OLLAMA_HOST=https://ollama.com
LLM_PROVIDER = get_secret("LLM_PROVIDER", "gemini").lower()   # gemini | ollama | none
EMBED_MODEL = get_secret("EMBED_MODEL", "paraphrase-multilingual-MiniLM-L12-v2")
SHEET_CSV_URL = get_secret("SHEET_CSV_URL")          # optional: Google Sheet with the places
SHEET_QUERIES_URL = get_secret("SHEET_QUERIES_URL")  # optional: Google Sheet tab with eval queries
TG_API_ID = get_secret("TG_API_ID")                  # optional: Telegram API (my.telegram.org) for seller channels
TG_API_HASH = get_secret("TG_API_HASH")
ADMIN_IDS = {int(x) for x in re.split(r"[,\s]+", get_secret("ADMIN_IDS")) if x.isdigit()}

# --- search settings ---
TOP_K = 5                 # results shown per query
HYBRID_WEIGHT = 0.5       # 0 = only BM25, 1 = only embeddings
RATING_BOOST = 0.05       # small bonus for well-rated places
DISTANCE_WEIGHT = 0.15    # bonus for nearby places (only when a location is given)
NEAR_POOL = 40            # "near me": the closest of the (up to) 40 most relevant places are shown first
EMBED_CANDIDATES = 5000   # places scored by embeddings per query: all of them for datasets up to this size
MIN_EMBED_SIM = 0.45      # no keyword hit and cosine below this = "not a match" (chosen for MiniLM)
TYPO_CUTOFF = 75          # rapidfuzz score needed to auto-correct a query word
FUZZY_MIN = 0.70          # minimum fuzzy score for the typo fallback
CHEAP_MAX_USD = 3.0       # what "cheap" means
LATE_HOUR = 23            # "open late" = closes at 23:00 or later (or after midnight)
KHR_PER_USD = 4100        # riel per dollar (approximate, change if needed)
TZ = ZoneInfo("Asia/Phnom_Penh")

# --- LLM & bot settings ---
LLM_TIMEOUT_S = 20
OLLAMA_TIMEOUT_S = 60     # local models can be slow on the first call
RATE_LIMIT_PER_MIN = 10
TG_POST_LIMIT = 100       # recent posts read per seller channel

# --- added with the web app, conversation and Cambodia-wide search ---
# Models users can choose from (web app, demo, Telegram /model). Comma-separated provider:model, first = default,
# optional "=Label". Empty = just GEMINI_MODEL / OLLAMA_MODEL. e.g. gemini:gemini-3.5-flash-lite,gemini:gemini-3.5-flash
LLM_MODELS = get_secret("LLM_MODELS")
# --- website / Telegram Mini App / demo (python web.py) ---
FRONTEND_DIR = ROOT / "frontend"                    # the React app (website + Telegram Mini App + demo)
FRONTEND_DIST = FRONTEND_DIR / "dist"               # its production build (npm run build), served by web.py
WEBAPP_URL = get_secret("WEBAPP_URL").rstrip("/")    # public https:// address of web.py: enables the bot's "Open app" button
HOST = get_secret("HOST", "127.0.0.1")               # 0.0.0.0 inside Docker / on a server
PORT = int(get_secret("PORT", "8080"))
RUN_BOT = get_secret("RUN_BOT", "1") != "0"          # web.py also runs the bot (polling) when TELEGRAM_TOKEN is set
MAX_PHOTO_MB = 8
ENABLE_DEMO = get_secret("ENABLE_DEMO", "1") != "0"     # the "How it works" page (/demo) and its API (/api/demo/...)
HOME_CITY_BONUS = 0.05    # bonus for Phnom Penh places when the question names no city or khan


In [10]:
# Show what is configured, never the secret values themselves
for name, value in {"TELEGRAM_TOKEN": TELEGRAM_TOKEN, "GEMINI_API_KEY": GEMINI_API_KEY,
                    "OLLAMA_API_KEY": OLLAMA_API_KEY, "TG_API_HASH": TG_API_HASH}.items():
    print(f"{name:15s}: {'set' if value else 'missing'}")
print(f"{'ADMIN_IDS':15s}: {len(ADMIN_IDS)} admin(s)")
print(f"{'LLM_PROVIDER':15s}: {LLM_PROVIDER}")
print(f"{'GEMINI_MODEL':15s}: {GEMINI_MODEL or '(not set)'}")
print(f"{'OLLAMA_MODEL':15s}: {OLLAMA_MODEL or '(not set)'}  @ {OLLAMA_HOST}")
print(f"{'EMBED_MODEL':15s}: {EMBED_MODEL}")
print(f"{'Google Sheets':15s}: places {'set' if SHEET_CSV_URL else 'not set'}, queries {'set' if SHEET_QUERIES_URL else 'not set'}")

TELEGRAM_TOKEN : set
GEMINI_API_KEY : set
OLLAMA_API_KEY : set
TG_API_HASH    : set
ADMIN_IDS      : 0 admin(s)
LLM_PROVIDER   : gemini
GEMINI_MODEL   : gemini-3.1-flash-lite
OLLAMA_MODEL   : (not set)  @ http://localhost:11434
EMBED_MODEL    : paraphrase-multilingual-MiniLM-L12-v2
Google Sheets  : places set, queries not set


## 1. Load the clean data

**What:** read `data/restaurants.csv`, the file written by **`data_preprocessing.ipynb`**, and prepare the text the search engine uses
(one searchable text per place: name, dishes, description, tags).

**Why:** collecting and cleaning the data is a separate job (different sources, network access, Telegram). Here we only
need the result. If you change the data, re-run `data_preprocessing.ipynb` first.

In [11]:
KHMER_RE = re.compile(r"[ក-៿᧠-᧿]+")   # Khmer Unicode blocks


WORD_RE = re.compile(r"[a-z0-9]+")


ZERO_WIDTH_RE = re.compile("[\u200b\u2060\ufeff]")      # invisible characters common in Khmer text


def normalize_text(s) -> str:
    """NFC-normalize (important for Khmer), collapse spaces, strip, lowercase."""
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = ZERO_WIDTH_RE.sub("", unicodedata.normalize("NFC", str(s)))
    return re.sub(r"\s+", " ", s).strip().lower()


def detect_lang(text) -> str:
    """'kh' if the text contains Khmer script, otherwise 'en'."""
    return "kh" if KHMER_RE.search(str(text)) else "en"


def khr_to_usd(amount_khr, rate=None) -> float:
    """Convert Cambodian riel to US dollars."""
    return round(float(amount_khr) / (rate or KHR_PER_USD), 2)


def parse_price(value) -> float:
    """'$2', '2.5', '8000 KHR', '8000៛' -> price in USD. Unknown -> NaN."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return float("nan")
    s = normalize_text(value).replace(",", "")
    num = re.search(r"\d+(?:\.\d+)?", s)
    if not num:
        return float("nan")
    amount = float(num.group())
    if any(unit in s for unit in ("khr", "riel", "៛", "រៀល")):
        return khr_to_usd(amount)
    return amount


print("6000 KHR ->", parse_price("6000 KHR"), "| '$2.50' ->", parse_price("$2.50"), "| None ->", parse_price(None))


6000 KHR -> 1.46 | '$2.50' -> 2.5 | None -> nan


In [12]:
ALL_COLUMNS = ["id", "name", "dish_en", "dish_kh", "description", "tags", "price_usd",
               "area", "lat", "lng", "rating", "open_hour", "close_hour", "source",
               "source_url", "date", "image_url", "phone_public"]


CSV_PATH = DATA_DIR / "restaurants.csv"


CSV_DTYPES = {"id": str, "phone_public": str, "price_usd": str, "page_url": str}


In [13]:
if not CSV_PATH.exists():
    raise RuntimeError("data/restaurants.csv not found: run data_preprocessing.ipynb first (it collects and cleans the data).")
raw_df = pd.read_csv(CSV_PATH, dtype=CSV_DTYPES)
print(f"Loaded {len(raw_df)} places from data/restaurants.csv")

Loaded 6544 places from data/restaurants.csv


### Places across Cambodia: khans, cities and provinces

**What:** the 14 Phnom Penh khans and the 25 provinces, with their Khmer names and the spellings people use
("BKK1", "Sihanoukville", "សៀមរាប"), and the dish table that links Khmer and English food words.

**Why:** the data writes the same place in different ways ("Boeung Keng Kang" / "Boeng Keng Kang"), so every place
gets one standard area and its city. A question can then say "pizza in Siem Reap" or "កាហ្វេ នៅសៀមរាប", and a
question that names no place looks in Phnom Penh first.

In [ ]:
# 14 Phnom Penh khans, Khmer names, and popular aliases / communes
KHAN_DETAILS = {
    "Boeng Keng Kang": {"kh": "បឹងកេងកង", "aliases": ["bkk", "bkk1", "bkk2", "bkk3", "បកក", "បឹងកេងកង១", "បឹងកេងកង២", "បឹងកេងកង៣", "boeung keng kang", "boeng keng kang"]},
    "Chamkar Mon": {"kh": "ចំការមន", "aliases": ["chamkarmon", "chamkar morn", "ttp", "ttp1", "ttp2", "ទួលទំពូង", "toul tom poung", "tuol tom poung", "russian market", "ផ្សារទួលទំពូង", "បឹងត្របែក", "boeung trabek", "phsar doeum thkov", "ផ្សារដើមថ្កូវ"]},
    "Toul Kork": {"kh": "ទួលគោក", "aliases": ["tk", "tuol kouk", "toul kouk", "toul kork", "ទឹកល្អក់", "teuk laak", "បឹងកក់", "boeng kak"]},
    "Daun Penh": {"kh": "ដូនពេញ", "aliases": ["dp", "doun penh", "daun penh", "ផ្សារថ្មី", "phsar thmey", "central market", "riverside", "មាត់ទន្លេ", "វត្តភ្នំ", "wat phnom", "old market", "ផ្សារចាស់", "kandal market", "phsar kandal", "ផ្សារកណ្ដាល"]},
    "Sen Sok": {"kh": "សែនសុខ", "aliases": ["aeon 2", "aeon sen sok", "aeon2", "ភ្នំពេញថ្មី", "phnom penh thmey", "ទឹកថ្លា", "teuk thla", "sen sok"]},
    "Prampir Makara": {"kh": "ប្រាំពីរមករា", "aliases": ["7 makara", "7makara", "៧មករា", "៧ មករា", "7មករា", "អូរឫស្សី", "orussey", "mittapheap", "olympic", "ផ្សារអូឡាំពិក", "prampir makara"]},
    "Chroy Changvar": {"kh": "ជ្រោយចង្វារ", "aliases": ["chroy changva", "ជ្រោយចង្វា", "chroy changvar"]},
    "Chbar Ampov": {"kh": "ច្បារអំពៅ", "aliases": ["chbar ampov 1", "chbar ampov 2", "ច្បារអំពៅ១", "ច្បារអំពៅ២", "niroth", "prek pra", "ព្រែកប្រា", "chbar ampov"]},
    "Mean Chey": {"kh": "មានជ័យ", "aliases": ["meanchey", "stung meanchey", "ស្ទឹងមានជ័យ", "boeng tumpun", "បឹងទំពុន", "mean chey"]},
    "Russey Keo": {"kh": "ឫស្សីកែវ", "aliases": ["russei keo", "ឬស្សីកែវ", "toul sangke", "ទួលសង្កែ", "kilomet 6", "គីឡូម៉ែត្រ៦", "russey keo"]},
    "Pou Senchey": {"kh": "ពោធិ៍សែនជ័យ", "aliases": ["por senchey", "ពោធិសែនជ័យ", "choam chao", "ចោមចៅ", "kakab", "កាកាប", "pou senchey"]},
    "Dangkao": {"kh": "ដង្កោ", "aliases": ["dangkor", "dangkao"]},
    "Prek Pnov": {"kh": "ព្រែកព្នៅ", "aliases": ["prek pnou", "prek pnov"]},
    "Kamboul": {"kh": "កំបូល", "aliases": ["kombol", "kamboul"]},
}


AREA_MAP = {}


KHAN_KH = {}


for khan, info in KHAN_DETAILS.items():
    AREA_MAP[khan.lower()] = khan
    AREA_MAP[info["kh"]] = khan
    KHAN_KH[khan] = info["kh"]
    for a in info["aliases"]:
        AREA_MAP[a.lower()] = khan


# Phnom Penh and the 24 provinces: Khmer names and common spellings (towns people say instead of the province)
PROVINCE_DETAILS = {
    "Phnom Penh": {"kh": "ភ្នំពេញ", "aliases": ["phnompenh", "phnom phen", "pnh", "ក្រុងភ្នំពេញ"]},
    "Siem Reap": {"kh": "សៀមរាប", "aliases": ["siemreap", "siem reab"]},
    "Battambang": {"kh": "បាត់ដំបង", "aliases": ["battambong", "batdambang"]},
    "Preah Sihanouk": {"kh": "ព្រះសីហនុ", "aliases": ["sihanoukville", "sihanouk ville", "sihanouk", "kampong som", "kompong som",
                                                  "កំពង់សោម", "ក្រុងព្រះសីហនុ", "otres", "otres beach", "ochheuteal"]},
    "Kampot": {"kh": "កំពត", "aliases": ["kompot"]},
    "Kep": {"kh": "កែប", "aliases": ["kep city", "krong kep"]},
    "Kandal": {"kh": "កណ្ដាល", "aliases": ["កណ្តាល", "ta khmau", "takhmao", "takhmau", "តាខ្មៅ"]},
    "Kampong Cham": {"kh": "កំពង់ចាម", "aliases": ["kompong cham", "kampongcham"]},
    "Kampong Chhnang": {"kh": "កំពង់ឆ្នាំង", "aliases": ["kompong chhnang"]},
    "Kampong Speu": {"kh": "កំពង់ស្ពឺ", "aliases": ["kompong speu", "chbar mon", "ច្បារមន"]},
    "Kampong Thom": {"kh": "កំពង់ធំ", "aliases": ["kompong thom", "stung sen"]},
    "Svay Rieng": {"kh": "ស្វាយរៀង", "aliases": ["bavet", "បាវិត"]},
    "Prey Veng": {"kh": "ព្រៃវែង", "aliases": []},
    "Takeo": {"kh": "តាកែវ", "aliases": ["takev", "doun kaev"]},
    "Koh Kong": {"kh": "កោះកុង", "aliases": ["kohkong"]},
    "Pursat": {"kh": "ពោធិ៍សាត់", "aliases": ["ពោធិសាត់", "posat", "pouthisat"]},
    "Kratie": {"kh": "ក្រចេះ", "aliases": ["kracheh"]},
    "Stung Treng": {"kh": "ស្ទឹងត្រែង", "aliases": ["steung treng"]},
    "Ratanakiri": {"kh": "រតនគិរី", "aliases": ["ratanakkiri", "rattanakiri", "banlung", "បានលុង"]},
    "Mondulkiri": {"kh": "មណ្ឌលគិរី", "aliases": ["mondolkiri", "sen monorom", "senmonorom", "សែនមនោរម្យ"]},
    "Preah Vihear": {"kh": "ព្រះវិហារ", "aliases": ["tbeng meanchey"]},
    "Oddar Meanchey": {"kh": "ឧត្ដរមានជ័យ", "aliases": ["ឧត្តរមានជ័យ", "oddar mean chey", "otdar meanchey", "samraong"]},
    "Banteay Meanchey": {"kh": "បន្ទាយមានជ័យ", "aliases": ["banteay mean chey", "poipet", "ប៉ោយប៉ែត", "sisophon", "serei saophoan"]},
    "Pailin": {"kh": "ប៉ៃលិន", "aliases": []},
    "Tbong Khmum": {"kh": "ត្បូងឃ្មុំ", "aliases": ["tboung khmum"]},
}


HOME_CITY = "Phnom Penh"                     # where the search looks first when the question names no place


CITY_MAP = {}


CITY_KH = {}


for city, info in PROVINCE_DETAILS.items():
    CITY_MAP[city.lower()] = city
    CITY_MAP[info["kh"]] = city
    CITY_KH[city] = info["kh"]
    for a in info["aliases"]:
        CITY_MAP[a.lower()] = city


PLACE_NAMES_KH = {**KHAN_KH, **CITY_KH}      # area or city -> Khmer name, for replies in Khmer


# every place name the parser knows, longest first, so "Banteay Meanchey" wins over the khan "Mean Chey"
PLACE_ALIASES = sorted([(a, "area", v) for a, v in AREA_MAP.items()] + [(a, "city", v) for a, v in CITY_MAP.items()],
                       key=lambda x: len(x[0]), reverse=True)


def canonical_place(value) -> tuple:
    """A place written in the data -> (area, city): 'Boeung Keng Kang' -> ('Boeng Keng Kang', 'Phnom Penh'),
    'Sihanoukville' -> ('Preah Sihanouk', 'Preah Sihanouk'). Unknown names are kept as written, with no city."""
    key = normalize_text(value)
    if key in AREA_MAP:
        return AREA_MAP[key], HOME_CITY
    if key in CITY_MAP:
        return CITY_MAP[key], CITY_MAP[key]
    return str(value or ""), ""


# Bilingual dish categories and synonyms (Khmer <-> English)
FOOD_CONCEPTS = [
    {"en_canonical": "noodles", "kh_canonical": "គុយទាវ / មី",
     "en": ["noodle", "noodles", "kuy teav", "kuyteav", "kuy tiew", "koy teav", "ramen", "udon", "pho", "wonton"],
     "kh": ["គុយទាវ", "មី", "មីគាវ", "រ៉ាមេន", "អ៊ូដុង", "គុយទាវទូក"]},
    {"en_canonical": "nom banh chok (khmer noodles)", "kh_canonical": "នំបញ្ចុក",
     "en": ["nom banh chok", "numbanchok", "khmer noodle", "curry noodle"],
     "kh": ["នំបញ្ចុក"]},
    {"en_canonical": "fried rice", "kh_canonical": "បាយឆា",
     "en": ["fried rice", "friedrice"],
     "kh": ["បាយឆា"]},
    {"en_canonical": "chicken rice", "kh_canonical": "បាយមាន់",
     "en": ["chicken rice", "hainanese"],
     "kh": ["បាយមាន់"]},
    {"en_canonical": "pork rice", "kh_canonical": "បាយសាច់ជ្រូក",
     "en": ["pork rice", "broken rice", "bai sach chrouk", "com tam"],
     "kh": ["បាយសាច់ជ្រូក"]},
    {"en_canonical": "rice", "kh_canonical": "បាយ",
     "en": ["rice"],
     "kh": ["បាយ"]},
    {"en_canonical": "lok lak", "kh_canonical": "ឡុកឡាក់",
     "en": ["lok lak", "beef lok lak"],
     "kh": ["ឡុកឡាក់", "បាយឡុកឡាក់"]},
    {"en_canonical": "amok", "kh_canonical": "អាម៉ុក",
     "en": ["amok", "fish amok"],
     "kh": ["អាម៉ុក", "អាមុក"]},
    {"en_canonical": "curry", "kh_canonical": "ការី",
     "en": ["curry", "red curry"],
     "kh": ["ការី", "សម្លការី"]},
    {"en_canonical": "coffee", "kh_canonical": "កាហ្វេ",
     "en": ["coffee", "cafe", "espresso", "latte", "cappuccino"],
     "kh": ["កាហ្វេ"]},
    {"en_canonical": "tea / boba", "kh_canonical": "តែ / តែគុជ",
     "en": ["tea", "milk tea", "bubble tea", "boba", "green tea", "matcha"],
     "kh": ["តែ", "តែគុជ", "តែបៃតង"]},
    {"en_canonical": "pizza", "kh_canonical": "ភីហ្សា",
     "en": ["pizza", "pizzeria"],
     "kh": ["ភីហ្សា"]},
    {"en_canonical": "burger", "kh_canonical": "ប៊ឺហ្គឺ",
     "en": ["burger", "hamburger"],
     "kh": ["ប៊ឺហ្គឺ", "បឺហ្គឺ"]},
    {"en_canonical": "bbq / grilled meat", "kh_canonical": "សាច់អាំង",
     "en": ["bbq", "barbecue", "grill", "grilled", "roast"],
     "kh": ["សាច់អាំង", "សាច់គោអាំង", "សាច់ជ្រូកអាំង"]},
    {"en_canonical": "seafood", "kh_canonical": "គ្រឿងសមុទ្រ",
     "en": ["seafood", "crab", "shrimp", "prawn", "squid", "fish"],
     "kh": ["គ្រឿងសមុទ្រ", "ក្តាម", "បង្គា", "មឹក", "ត្រី"]},
    {"en_canonical": "sushi / japanese", "kh_canonical": "ស៊ូស៊ី / ម្ហូបជប៉ុន",
     "en": ["sushi", "sashimi", "japanese"],
     "kh": ["ស៊ូស៊ី", "សាស៊ីមី", "ម្ហូបជប៉ុន"]},
    {"en_canonical": "soup / hotpot", "kh_canonical": "ស៊ុប / ហតផត",
     "en": ["soup", "hotpot", "shabu", "steamboat"],
     "kh": ["ស៊ុប", "ហតផត", "ស៊ុបគោ", "ស៊ុបឆ្នាំងភ្លើង"]},
    {"en_canonical": "dim sum", "kh_canonical": "ឌីមសាំ",
     "en": ["dim sum", "dimsum", "dumpling", "bao"],
     "kh": ["ឌីមសាំ"]},
    {"en_canonical": "bakery / bread", "kh_canonical": "នំប៉័ង / បង្អែម",
     "en": ["bakery", "bread", "baguette", "pastry", "sandwich", "num pang"],
     "kh": ["នំប៉័ង", "ប័ុង"]},
    {"en_canonical": "dessert / ice cream", "kh_canonical": "បង្អែម / ការ៉េម",
     "en": ["dessert", "ice cream", "sweet", "cake"],
     "kh": ["បង្អែម", "ការ៉េម", "នំខេក"]},
    {"en_canonical": "vegetarian", "kh_canonical": "ម្ហូបបួស",
     "en": ["vegetarian", "vegan"],
     "kh": ["ម្ហូបបួស", "បួស"]},
    {"en_canonical": "khmer food", "kh_canonical": "ម្ហូបខ្មែរ",
     "en": ["khmer food", "cambodian food", "khmer", "cambodian"],
     "kh": ["ម្ហូបខ្មែរ"]},
    {"en_canonical": "thai food", "kh_canonical": "ម្ហូបថៃ",
     "en": ["thai food", "thai"],
     "kh": ["ម្ហូបថៃ"]},
    {"en_canonical": "chinese food", "kh_canonical": "ម្ហូបចិន",
     "en": ["chinese food", "chinese"],
     "kh": ["ម្ហូបចិន"]},
    {"en_canonical": "korean food", "kh_canonical": "ម្ហូបកូរ៉េ",
     "en": ["korean food", "korean"],
     "kh": ["ម្ហូបកូរ៉េ"]},
    {"en_canonical": "vietnamese food", "kh_canonical": "ម្ហូបវៀតណាម",
     "en": ["vietnamese food", "vietnamese", "pho"],
     "kh": ["ម្ហូបវៀតណាម"]},
    {"en_canonical": "indian food", "kh_canonical": "ម្ហូបឥណ្ឌា",
     "en": ["indian food", "indian"],
     "kh": ["ម្ហូបឥណ្ឌា"]},
    {"en_canonical": "french food", "kh_canonical": "ម្ហូបបារាំង",
     "en": ["french food", "french"],
     "kh": ["ម្ហូបបារាំង"]},
    {"en_canonical": "italian food", "kh_canonical": "ម្ហូបអ៊ីតាលី",
     "en": ["italian food", "italian", "pasta"],
     "kh": ["ម្ហូបអ៊ីតាលី"]},
    {"en_canonical": "mexican food / tacos", "kh_canonical": "ម្ហូបម៉ិកស៊ិក",
     "en": ["mexican food", "mexican", "tacos", "taco", "burrito"],
     "kh": ["ម្ហូបម៉ិកស៊ិក", "តាកូ"]},
    {"en_canonical": "fast food", "kh_canonical": "អាហាររហ័ស",
     "en": ["fast food"],
     "kh": ["អាហាររហ័ស"]},
    {"en_canonical": "breakfast", "kh_canonical": "អាហារពេលព្រឹក",
     "en": ["breakfast", "brunch"],
     "kh": ["អាហារពេលព្រឹក"]},
    {"en_canonical": "halal food", "kh_canonical": "អាហារហាឡាល់",
     "en": ["halal food", "halal", "shawarma", "kebab", "middle eastern"],
     "kh": ["អាហារហាឡាល់", "ហាឡាល់", "កេបាប់", "សាវ៉ាម៉ា"]},
    {"en_canonical": "buffet", "kh_canonical": "អាហារប៊ូហ្វេ",
     "en": ["buffet", "all you can eat"],
     "kh": ["អាហារប៊ូហ្វេ", "ប៊ូហ្វេ"]},
    {"en_canonical": "steak", "kh_canonical": "ស្តេក",
     "en": ["steak", "steakhouse", "beef steak", "ribeye"],
     "kh": ["ស្តេក", "សាច់គោស្តេក"]},
    {"en_canonical": "salad / poke", "kh_canonical": "សាឡាដ / អាហារសុខភាព",
     "en": ["salad", "poke", "healthy food", "clean food"],
     "kh": ["សាឡាដ", "អាហារសុខភាព"]},
]


KHMER_VOCAB = set()


for _c in FOOD_CONCEPTS:
    KHMER_VOCAB.update(_c["kh"])


for _k, _info in {**KHAN_DETAILS, **PROVINCE_DETAILS}.items():
    KHMER_VOCAB.add(_info["kh"])
    for _a in _info["aliases"]:
        if KHMER_RE.search(_a):
            KHMER_VOCAB.add(_a)


In [14]:
TEXT_COLS = ["name", "dish_en", "dish_kh", "description", "tags", "area",
             "source", "source_url", "date", "image_url", "phone_public"]


def clean_df(raw: pd.DataFrame) -> pd.DataFrame:
    raw = raw.copy()
    if "image_url" not in raw.columns and "image_path" in raw.columns:
        raw["image_url"] = raw["image_path"]
    elif "image_path" not in raw.columns and "image_url" in raw.columns:
        raw["image_path"] = raw["image_url"]
    missing = set(ALL_COLUMNS) - set(raw.columns)
    if missing:
        raise ValueError(f"restaurants.csv is missing columns: {sorted(missing)}")
    d = raw[ALL_COLUMNS].copy()
    d["image_path"] = d["image_url"].copy()
    d["id"] = d["id"].fillna("").astype(str).str.strip()
    d = d[~d["id"].isin(["", "id"])]                           # "id" = header row pasted again by a sheet import
    d = d.drop_duplicates("id", keep="first")                    # ids must be unique for ChromaDB
    # the seller's Facebook page / website (optional column): only real http(s) links are kept
    page = raw["page_url"] if "page_url" in raw.columns else pd.Series("", index=raw.index)
    d["page_url"] = page.reindex(d.index).fillna("").astype(str).str.strip()
    d.loc[~d["page_url"].str.match(r"^https?://\S+$"), "page_url"] = ""
    # restaurant menu items (optional column)
    menu_col = raw["menu"] if "menu" in raw.columns else pd.Series("", index=raw.index)
    d["menu"] = menu_col.reindex(d.index).fillna("").astype(str).str.strip()
    # text: NFC + single spaces + strip; empty string instead of NaN
    for c in TEXT_COLS:
        d[c] = d[c].fillna("").astype(str).map(
            lambda s: re.sub(r"\s+", " ", ZERO_WIDTH_RE.sub("", unicodedata.normalize("NFC", s))).strip())
    # lowercase the fields we search on (names and areas keep their case for display)
    for c in ["dish_en", "description", "tags"]:
        d[c] = d[c].str.lower()
    # one spelling per place ("Boeung Keng Kang" -> "Boeng Keng Kang") and the city/province it is in
    places = d["area"].map(canonical_place)
    d["area"], d["city"] = places.str[0], places.str[1]
    d["area_key"], d["city_key"] = d["area"].str.lower(), d["city"].str.lower()
    # numbers: prices may be written in riel; unknown values stay NaN (never guessed)
    d["price_usd"] = d["price_usd"].map(parse_price)
    for c in ["lat", "lng", "rating", "open_hour", "close_hour"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    # Bilingual enrichment: auto-fill missing dish categories and enrich doc_text with area and food concepts
    doc_texts = []
    for idx, r in d.iterrows():
        name_val = r["name"]
        dish_en_val = r["dish_en"]
        dish_kh_val = r["dish_kh"]
        desc_val = r["description"]
        tags_val = r["tags"].replace(";", " ")
        area_val = r["area"]

        combined = f"{name_val} {dish_en_val} {dish_kh_val} {desc_val} {tags_val}".lower()
        additions = []

        if area_val in KHAN_DETAILS:
            kd = KHAN_DETAILS[area_val]
            additions.append(f"{area_val} {kd['kh']} {' '.join(kd['aliases'])}")
        if r["city"] in PROVINCE_DETAILS:
            cd = PROVINCE_DETAILS[r["city"]]
            additions.append(f"{r['city']} {cd['kh']} {' '.join(cd['aliases'])}")

        for c in FOOD_CONCEPTS:
            matched = False
            for k in c["kh"]:
                if k in combined:
                    matched = True
                    break
            if not matched:
                for e in c["en"]:
                    if f" {e} " in f" {combined} " or combined.startswith(f"{e} ") or combined.endswith(f" {e}"):
                        matched = True
                        break
            if matched:
                additions.extend(c["kh"])
                additions.extend(c["en"])
                if not dish_kh_val:
                    d.at[idx, "dish_kh"] = c["kh_canonical"]
                    dish_kh_val = c["kh_canonical"]
                if not dish_en_val:
                    d.at[idx, "dish_en"] = c["en_canonical"]
                    dish_en_val = c["en_canonical"]

        extra = " ".join(dict.fromkeys(additions))
        full_doc = f"{name_val} {dish_en_val} {dish_kh_val} {desc_val} {tags_val} {area_val} {extra}"
        doc_texts.append(normalize_text(full_doc))

    d["doc_text"] = doc_texts
    return d


print("Missing / empty values before cleaning:")


print(raw_df.replace("", np.nan).isna().sum().loc[lambda s: s > 0].to_string())


df = clean_df(raw_df)


display(df[["id", "name", "dish_en", "dish_kh", "price_usd", "area", "rating", "open_hour", "close_hour"]].head())


Missing / empty values before cleaning:
dish_en         1271
dish_kh         3617
tags             119
price_usd       6544
area               5
lat                1
lng                1
rating          6544
open_hour       6248
close_hour      6248
source_url      4725
image_path      6544
phone_public    6251
page_url        1350


/var/folders/96/209bkmy91y5gl7897d5zdfdr0000gn/T/ipykernel_85072/3185663324.py:15: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  d[c] = d[c].fillna("").astype(str).map(


,id,name,dish_en,dish_kh,price_usd,area,rating,open_hour,close_hour
0,osm-n10010456626,Bay Area Tacos,mexican,,NaN,Chamkar Mon,NaN,NaN,NaN
1,osm-n10010456718,Kungfu Kitchen,chinese food,ម្ហូបចិន,NaN,Chamkar Mon,NaN,12.0,21.0
2,osm-n10010469517,Elia Greek Kitchen,greek,,NaN,Chamkar Mon,NaN,10.0,23.0
3,osm-n10010469617,Cakes & Craft,american,,NaN,Chamkar Mon,NaN,11.0,23.0
4,osm-n10012603022,Tube Coffee SEA TV,breakfast,,NaN,Sen Sok,NaN,7.0,20.0


## 2. Evaluation set

**What:** a list of test queries, each with the ids of the places that *should* be found. Mix English, Khmer,
typos (`carry noodle`, `kuytiew`), meaning-only queries (`something sweet`) and conditions (`cheap`, `open late`).

**Why:** without a test set we can only *feel* whether search got better. With it we can **measure** it:
* **Recall@5**: the share of the relevant places that appear in the top 5
* **MRR**: 1 / rank of the first relevant result (1.0 = the first result is relevant). We compute it on the top 5 (MRR@5).
* **nDCG@5**: rewards relevant results that are ranked *higher* (1.0 = perfect order)

**Your job:** no ready-made test set ships with this project, because the labels must be checked by a person who
knows the places. Create `data/eval/queries.csv` (`data_preprocessing.ipynb` downloads it from a Google Sheet tab if you set `SHEET_QUERIES_URL`) with the columns
`query`, `relevant_ids` (ids from `restaurants.csv`, separated by `;`), `lang`, `kind`. Aim for 30–50 queries.
Until it has rows, the evaluation cells are skipped and the rest of the notebook still runs.

In [15]:
QUERIES_PATH = EVAL_DIR / "queries.csv"
EVAL_COLUMNS = ["query", "relevant_ids", "lang", "kind"]
eval_df = None
if QUERIES_PATH.exists():
    eval_df = pd.read_csv(QUERIES_PATH, dtype={"relevant_ids": str})
    print(f"Loaded {len(eval_df)} queries from data/eval/queries.csv")
else:
    eval_df = pd.DataFrame(columns=EVAL_COLUMNS)
    eval_df.to_csv(QUERIES_PATH, index=False)
    print("Created an empty data/eval/queries.csv (header only): add your test queries there.")
for col in ("lang", "kind"):
    if col not in eval_df.columns:
        eval_df[col] = ""
eval_df["relevant"] = eval_df["relevant_ids"].fillna("").astype(str).str.split(";").map(
    lambda ids: {i.strip() for i in ids if i.strip()})

# Keep only queries that point at ids that exist in the data
known_ids = set(df["id"])
unknown = sorted(set().union(*eval_df["relevant"]) - known_ids)
if unknown:
    print(f"⚠️ {len(unknown)} relevant id(s) are not in the data: {unknown[:10]}. Check data/eval/queries.csv.")
usable = eval_df["relevant"].map(lambda ids: bool(ids & known_ids)).astype(bool)
eval_df = eval_df[usable].reset_index(drop=True)
HAS_EVAL = len(eval_df) > 0
if HAS_EVAL:
    print(f"{len(eval_df)} usable evaluation queries")
    display(eval_df[["query", "relevant_ids", "lang", "kind"]].head(8))
else:
    print("📋 No evaluation queries yet: the evaluation cells below are skipped (see the note above).")

Loaded 0 queries from data/eval/queries.csv
📋 No evaluation queries yet: the evaluation cells below are skipped (see the note above).


In [16]:
def recall_at_k(ranked, relevant, k=5) -> float:
    return len(set(ranked[:k]) & set(relevant)) / len(relevant) if relevant else 0.0


def mrr(ranked, relevant) -> float:
    """Reciprocal rank of the first relevant result (averaged over queries = MRR)."""
    for rank, doc in enumerate(ranked, 1):
        if doc in relevant:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(ranked, relevant, k=5) -> float:
    """Binary-relevance nDCG: DCG of our ranking / DCG of the perfect ranking."""
    dcg = sum(1 / math.log2(rank + 1) for rank, doc in enumerate(ranked[:k], 1) if doc in relevant)
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(len(relevant), k) + 1))
    return dcg / ideal if ideal else 0.0

In [17]:
# Toy test: 5 ranked docs, relevant = b, e and z (z was never retrieved)
ranked, relevant = ["a", "b", "c", "d", "e"], {"b", "e", "z"}
assert recall_at_k(ranked, relevant, 5) == 2 / 3
assert recall_at_k(ranked, relevant, 1) == 0.0
assert mrr(ranked, relevant) == 0.5
expected = (1 / math.log2(3) + 1 / math.log2(6)) / (1 + 1 / math.log2(3) + 1 / math.log2(4))
assert abs(ndcg_at_k(ranked, relevant, 5) - expected) < 1e-9
assert ndcg_at_k(["b", "e", "z"], relevant, 3) == 1.0

# Cross-check nDCG against scikit-learn (same case without the missing doc z)
from sklearn.metrics import ndcg_score
sk = ndcg_score([[0, 1, 0, 0, 1]], [[5, 4, 3, 2, 1]], k=5)
assert abs(ndcg_at_k(ranked, {"b", "e"}, 5) - sk) < 1e-9
print("✅ metric tests passed")

✅ metric tests passed


In [18]:
RESULTS, PER_QUERY = {}, {}     # method name -> mean metrics / per-query table


def evaluate(search_fn, queries=None, k=5):
    """search_fn(query, k) -> ranked list of ids. Returns (mean metrics, per-query table)."""
    queries = eval_df if queries is None else queries
    rows = []
    for q, rel in zip(queries["query"], queries["relevant"]):
        ranked = search_fn(q, k)
        rows.append({"query": q, f"Recall@{k}": recall_at_k(ranked, rel, k),
                     "MRR": mrr(ranked[:k], rel), f"nDCG@{k}": ndcg_at_k(ranked, rel, k)})
    per_query = pd.DataFrame(rows)
    return per_query.drop(columns="query").mean().round(3).to_dict(), per_query

## 3. Baseline: BM25 + fuzzy matching

**What:** **BM25** scores a document by how many query words it contains, giving more weight to rare
words and less to very long documents. It searches over name, dish (EN + KM), description and tags.

**Tokenizing Khmer:** Khmer is written **without spaces between words**, so splitting on spaces does
not work. As a simple default we cut Khmer text into **character 2-grams and 3-grams**
(`គុយទាវ` → `គុ, ុយ, យទ, …`), so a query matches documents that share these pieces. A real
Khmer word segmenter would be better (see *next steps*).

**Typos:** an English query word that is not in the vocabulary is replaced by the closest vocabulary word
with **rapidfuzz** (`carry` → `curry`). If BM25 still finds nothing, a **fuzzy fallback** compares the query
with the place and dish names character by character (`kuytiew` ≈ `kuyteav`).

In [19]:
def tokenize(text) -> list:
    """English words (with light plural stemming) + Khmer tokens (domain vocabulary words + runs + 2/3-grams)."""
    text = normalize_text(text)
    tokens = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
              for w in WORD_RE.findall(text)]
    for word in KHMER_VOCAB:
        if word in text:
            tokens.append(word)
    for run in KHMER_RE.findall(text):          # Khmer is written without spaces
        if len(run) <= 15:
            tokens.append(run)
        if len(run) < 2:
            tokens.append(run)
        for n in (2, 3):
            tokens += [run[i:i + n] for i in range(len(run) - n + 1)]
    return tokens


print(tokenize("Curry Noodles នំបញ្ចុក"))


['curry', 'noodle', 'នំ', 'ំប', 'បញ', 'ញ្', '្ច', 'ចុ', 'ុក', 'នំប', 'ំបញ', 'បញ្', 'ញ្ច', '្ចុ', 'ចុក']


In [20]:
def build_bm25(d: pd.DataFrame):
    """BM25 index over the combined text + the English vocabulary (for typo fixing)."""
    corpus = [tokenize(t) for t in d["doc_text"]]
    vocab = sorted({t for doc in corpus for t in doc if WORD_RE.fullmatch(t)})
    return BM25Okapi(corpus), vocab


def build_food_vocab(d: pd.DataFrame) -> list:
    """Words that name food or a kind of place (from the dish names, tags and FOOD_CONCEPTS), not every word of the
    descriptions: 'pizza', 'cafe', 'kuy', 'teav', 'គុយទាវ'. Used to tell a food question from small talk."""
    words = set()
    for c in FOOD_CONCEPTS:
        for term in c["en"] + c["kh"]:
            words.update(tokenize(term))
    for col in ("dish_en", "dish_kh", "tags"):
        for text in d[col].drop_duplicates():
            words.update(tokenize(str(text).replace(";", " ")))
    words.update(("food", "eat", "restaurant", "cafe", "coffee", "drink", "breakfast", "lunch", "dinner", "snack",
                  "dessert", "street", "menu", "ម្ហូប", "ញ៉ាំ", "ហាង", "អាហារ", "ភេសជ្ជៈ"))
    return sorted(w for w in words if len(w) >= 2)


def place_count() -> int:
    """How many places the search knows (0 before the data is loaded)."""
    return 0 if df is None else len(df)


def about_food(query) -> bool:
    """Does the message ask for food or places to eat? 'piza' -> yes (a typo of a food word), 'near CADT' -> yes,
    'what's the weather today?' -> no. A place name we do not know also says no: the chat model then decides."""
    text, flt = parse_query_filters(query)
    if flt:
        return True
    vocab = set(FOOD_VOCAB or [])
    tokens = tokenize(text)
    return any(t in vocab for t in correct_tokens(tokens, FOOD_VOCAB or [], cutoff=85)) if tokens else False


bm25, BM25_VOCAB = build_bm25(df)


FOOD_VOCAB = build_food_vocab(df)      # food words: tells a food question from small talk (section 7)


print(f"BM25 index: {len(df)} documents, {len(BM25_VOCAB)} English words in the vocabulary")


BM25 index: 6544 documents, 4449 English words in the vocabulary


In [21]:
def correct_tokens(tokens, vocab, cutoff=None):
    """Replace unknown English words by the closest vocabulary word ('carry' -> 'curry')."""
    vocab_set, fixed = set(vocab), []
    for t in tokens:
        if t in vocab_set or len(t) < 4 or not WORD_RE.fullmatch(t):
            fixed.append(t)
            continue
        match = process.extractOne(t, vocab, scorer=fuzz.ratio, score_cutoff=cutoff or TYPO_CUTOFF)
        fixed.append(match[0] if match else t)
    return fixed


def fuzzy_scores(query, d: pd.DataFrame) -> np.ndarray:
    """Typo fallback: fuzzy match the query against names and dish names (spaces removed)."""
    q = normalize_text(query).replace(" ", "")
    fields = (d["name"] + " " + d["dish_en"] + " " + d["dish_kh"]).map(normalize_text).str.replace(" ", "")
    scores = np.array([fuzz.partial_ratio(q, f) for f in fields]) / 100
    scores[scores < FUZZY_MIN] = 0.0
    return scores


print(correct_tokens(tokenize("carry noodle"), BM25_VOCAB))


['curry', 'noodle']


In [22]:
def bm25_scores(query):
    """Raw BM25 score per row of df (+ which method produced them)."""
    tokens = tokenize(query)
    if not tokens:
        return np.zeros(len(df)), "empty"
    fixed = correct_tokens(tokens, BM25_VOCAB)
    scores = np.asarray(bm25.get_scores(fixed), dtype=float)
    method = "bm25+typo-fix" if fixed != tokens else "bm25"
    if scores.max() <= 0:                    # no keyword hit at all -> fuzzy fallback
        scores, method = fuzzy_scores(query, df), "fuzzy"
    return scores, method


def bm25_search(query, k=TOP_K) -> pd.DataFrame:
    scores, method = bm25_scores(query)
    order = [i for i in np.argsort(-scores, kind="stable") if scores[i] > 0][:k]
    out = df.iloc[order][["id", "name", "dish_en", "dish_kh", "price_usd", "area"]].copy()
    out["score"] = scores[order].round(3)
    out["method"] = method
    return out.reset_index(drop=True)


In [23]:
DEMO_QUERIES = ["curry noodle", "carry noodle", "នំបញ្ចុក"]
for q in DEMO_QUERIES:
    print(f"🔎 {q}")
    display(bm25_search(q))

🔎 curry noodle


,id,name,dish_en,dish_kh,price_usd,area,score,method
0,osm-n5586787217,Khmer curry stall,,,NaN,Chamkar Mon,8.947,bm25
1,ovt-8bd3673f-e4c7-4d3e-a04a-dd2a0e4b08b8,Curry Zone Restaurant,restaurant,,NaN,Daun Penh,8.751,bm25
2,ovt-04f8a050-2b30-4f24-9ef4-0e94dbbee1c2,Curry Leaf India Restaurant,restaurant,,NaN,Prampir Makara,8.563,bm25
3,ovt-00db2c18-942b-4f7d-8581-6cf5b09c2925,Curry Pot,nepalese restaurant,,NaN,Toul Kork,7.178,bm25
4,osm-n4178059189,Marugame Udon,noodles,មី,NaN,Boeng Keng Kang,6.977,bm25


🔎 carry noodle


,id,name,dish_en,dish_kh,price_usd,area,score,method
0,osm-n5586787217,Khmer curry stall,,,NaN,Chamkar Mon,8.947,bm25+typo-fix
1,ovt-8bd3673f-e4c7-4d3e-a04a-dd2a0e4b08b8,Curry Zone Restaurant,restaurant,,NaN,Daun Penh,8.751,bm25+typo-fix
2,ovt-04f8a050-2b30-4f24-9ef4-0e94dbbee1c2,Curry Leaf India Restaurant,restaurant,,NaN,Prampir Makara,8.563,bm25+typo-fix
3,ovt-00db2c18-942b-4f7d-8581-6cf5b09c2925,Curry Pot,nepalese restaurant,,NaN,Toul Kork,7.178,bm25+typo-fix
4,osm-n4178059189,Marugame Udon,noodles,មី,NaN,Boeng Keng Kang,6.977,bm25+typo-fix


🔎 នំបញ្ចុក


,id,name,dish_en,dish_kh,price_usd,area,score,method
0,osm-n10292468209,នំបញ្ចុក482,asian,,NaN,Chamkar Mon,64.494,bm25
1,osm-n4861669721,បបរ នំបញ្ចុក,,,NaN,Toul Kork,64.494,bm25
2,ovt-b34f1a6d-8f20-49ea-887d-7b43e4bf05f3,នំបញ្ចុក កេរម៉ែ,restaurant,,NaN,Daun Penh,57.508,bm25
3,ovt-7bf721b4-fa12-42c6-9135-d6a8433b21d7,នំបញ្ចុកស្រុះ,restaurant,,NaN,Daun Penh,56.631,bm25
4,ovt-770f2a5d-7dff-48a1-abca-7ad4e8cb12cb,នំបញ្ចុកសាច់គោ,restaurant,,NaN,Daun Penh,54.956,bm25


In [24]:
if HAS_EVAL:
    RESULTS["BM25 + fuzzy"], PER_QUERY["BM25 + fuzzy"] = evaluate(lambda q, k: bm25_search(q, k)["id"].tolist())
    print(RESULTS["BM25 + fuzzy"])
else:
    print("⏭️ skipped: no evaluation queries yet (see section 2)")

⏭️ skipped: no evaluation queries yet (see section 2)


## 4. Semantic search: embeddings + ChromaDB

**What:** a **multilingual sentence-embedding model** turns each place's text into a vector, so texts with similar
*meaning* get nearby vectors, even across languages (`dessert` ≈ `បង្អែម` ≈ `something sweet`). The vectors
are stored in **ChromaDB** (a local vector database in `chroma_db/`) together with metadata (name, area, price, …).

**Model:** `paraphrase-multilingual-MiniLM-L12-v2` by default (small and fast, 50+ languages).
To try a stronger model, set `EMBED_MODEL=BAAI/bge-m3` in `.env`. Each model gets its **own collection**, so
vector sizes never mix.

**Idempotent:** building uses `upsert` (insert or update by id) and deletes ids that left the CSV, so
re-running the cell never creates duplicates.

In [25]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print(f"Loaded {EMBED_MODEL} on {DEVICE}, vector size {embedder.get_sentence_embedding_dimension()}")

W1103 09:22:00.580000 85072 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8605.79it/s]

Loaded paraphrase-multilingual-MiniLM-L12-v2 on mps, vector size 384


In [26]:
def embed_texts(texts) -> np.ndarray:
    """Unit-length sentence embeddings (so dot product = cosine similarity)."""
    return embedder.encode(list(texts), batch_size=32, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)


@lru_cache(maxsize=2048)
def embed_query(query: str) -> np.ndarray:
    return embed_texts([normalize_text(query)])[0]


def open_chroma():
    return chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(anonymized_telemetry=False))


def get_text_collection(client):
    # one collection per embedding model, so switching EMBED_MODEL never mixes vector sizes
    name = ("food_text_" + re.sub(r"[^a-z0-9]+", "_", EMBED_MODEL.lower()))[:60].strip("_")
    return client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})


def row_metadata(r) -> dict:
    meta = {"name": r["name"], "area": r["area"], "source": r["source"]}
    for c in ["price_usd", "rating", "lat", "lng", "open_hour", "close_hour"]:
        if pd.notna(r[c]):                   # Chroma metadata cannot hold None/NaN
            meta[c] = float(r[c])
    return meta


def build_text_index(d: pd.DataFrame, collection, batch: int = 2000) -> int:
    """Upsert new or updated rows and delete ids that are no longer in the CSV."""
    stored = collection.get(include=["documents"])
    existing = dict(zip(stored["ids"], stored["documents"]))
    existing_ids = set(existing)
    needed = d[d["id"].map(existing.get) != d["doc_text"]]       # new ids, and rows whose text changed
    if len(needed) > 0:
        for start in range(0, len(needed), batch):
            part = needed.iloc[start:start + batch]
            collection.upsert(ids=part["id"].tolist(),
                              embeddings=embed_texts(part["doc_text"]).tolist(),
                              documents=part["doc_text"].tolist(),
                              metadatas=[row_metadata(r) for _, r in part.iterrows()])
    stale = sorted(existing_ids - set(d["id"]))
    for start in range(0, len(stale), batch):
        collection.delete(ids=stale[start:start + batch])
    return collection.count()


chroma = open_chroma()


text_col = get_text_collection(chroma)


print(f"Collection {text_col.name!r}: {build_text_index(df, text_col)} vectors (re-run = same count)")


Collection 'food_text_paraphrase_multilingual_minilm_l12_v2': 6544 vectors (re-run = same count)


In [27]:
def embedding_scores(query) -> np.ndarray:
    """Cosine similarity per row of df (rows beyond EMBED_CANDIDATES, if any, get 0)."""
    n = min(EMBED_CANDIDATES, text_col.count())
    res = text_col.query(query_embeddings=[embed_query(query).tolist()], n_results=n,
                         include=["distances"])
    sims = {i: 1.0 - dist for i, dist in zip(res["ids"][0], res["distances"][0])}
    return np.array([sims.get(i, 0.0) for i in df["id"]])


def embedding_search(query, k=TOP_K) -> pd.DataFrame:
    scores = embedding_scores(query)
    order = np.argsort(-scores, kind="stable")[:k]
    out = df.iloc[order][["id", "name", "dish_en", "dish_kh", "price_usd", "area"]].copy()
    out["score"] = scores[order].round(3)
    return out.reset_index(drop=True)


In [28]:
for q in DEMO_QUERIES:
    print(f"🔎 {q}")
    display(embedding_search(q))

🔎 curry noodle


,id,name,dish_en,dish_kh,price_usd,area,score
0,ovt-7bda2203-0742-436a-98a5-a84c468c0878,VN noodles,,,NaN,Daun Penh,0.645
1,ovt-7b7e38dc-063b-4b91-8356-34cd88fd564e,Universal Noodles,,,NaN,Daun Penh,0.639
2,osm-n7214995387,Curry pot,indian food,ម្ហូបឥណ្ឌា,NaN,Chamkar Mon,0.625
3,ovt-58961dca-afae-4373-a33e-60b5f4230a67,មីគាវទាខ្វៃវត្តភ្នំ Wanton noodle soup with ro...,chinese restaurant,ម្ហូបចិន,NaN,Daun Penh,0.615
4,osm-n4253751389,Heng Heng - Fried 'silver needle' noodles,,,NaN,Daun Penh,0.612


🔎 carry noodle


,id,name,dish_en,dish_kh,price_usd,area,score
0,osm-n4178059189,Marugame Udon,noodles,មី,NaN,Boeng Keng Kang,0.552
1,ovt-7b7e38dc-063b-4b91-8356-34cd88fd564e,Universal Noodles,,,NaN,Daun Penh,0.504
2,osm-n4586509693,$1 noodles,local,,NaN,Chamkar Mon,0.487
3,ovt-7bda2203-0742-436a-98a5-a84c468c0878,VN noodles,,,NaN,Daun Penh,0.482
4,ovt-58961dca-afae-4373-a33e-60b5f4230a67,មីគាវទាខ្វៃវត្តភ្នំ Wanton noodle soup with ro...,chinese restaurant,ម្ហូបចិន,NaN,Daun Penh,0.481


🔎 នំបញ្ចុក


,id,name,dish_en,dish_kh,price_usd,area,score
0,osm-n13084021612,នំបញ្ចុកប្រហើរ បាយស្រូបសាច់ជ្រូក បបរសាច់ត្រីសា...,,,NaN,Dangkao,0.573
1,ovt-f7d7d7bf-ead7-4979-bbe9-492b7f2d3adb,គីមឈី ម៉ាក់អូនភ្លោះ,,,NaN,Mean Chey,0.544
2,ovt-da9fd8ee-85eb-4b28-b124-2c97c3866795,Little Duckling,,,NaN,Mean Chey,0.537
3,ovt-94d889fa-fb88-479f-87a8-16192df70e0f,អង្ករ Mr. RICE,,,NaN,Mean Chey,0.527
4,ovt-b0eebbd0-1af0-4a96-97cc-aa4af7021355,Mak NingNing,,,NaN,Sen Sok,0.521


In [29]:
if HAS_EVAL:
    RESULTS["Embeddings"], PER_QUERY["Embeddings"] = evaluate(lambda q, k: embedding_search(q, k)["id"].tolist())
    print(RESULTS["Embeddings"])
else:
    print("⏭️ skipped: no evaluation queries yet (see section 2)")

⏭️ skipped: no evaluation queries yet (see section 2)


## 5. Hybrid search + geo + filters

**What:** combine both signals. BM25 is good at exact words and names; embeddings are good at meaning and
cross-language matches. Their scores are on different scales, so each is **min-max normalized to 0..1** first:

`final = w · embed + (1 − w) · bm25 + rating_boost + distance_score`  (w = `HYBRID_WEIGHT`)

* `rating_boost = 0.05 · rating / 5` is a small tie-breaker for good places
* `distance_score = 0.15 / (1 + km)`, used only when the user shares a location (**Haversine** distance)
* **filters** parsed from the query text: `under $3` / `ក្រោម 5 ដុល្លារ` → max price, `cheap` / `ថោក` → ≤ $3,
  `open now` (Phnom Penh time), `open late` (closes ≥ 23:00 or after midnight), `top rated` → rating ≥ 4.5, khan names (`BKK1`, `ទួលគោក`) and cities or provinces (`Siem Reap`, `សៀមរាប`, `Sihanoukville`).
  A question that names no place gives Phnom Penh places a small bonus (`HOME_CITY_BONUS`)
* **near me / near CADT:** `pizza near me` uses the user's shared location, `pizza near CADT` looks the place up on
  OpenStreetMap (Nominatim): in Phnom Penh first, or anywhere in Cambodia when a well-known place matches better (`near Angkor Wat`). The closest of the clearly relevant places come first, with their distance in km
* rows that match neither by keyword nor by meaning (cosine < `MIN_EMBED_SIM`) are dropped, so an unrelated query like "car repair" returns *nothing*
  instead of random places. The threshold 0.45 was picked by looking at a few queries with the default MiniLM model
  (unrelated queries such as *car repair* score close to 0, real meaning matches are ≥ 0.45 on the few queries I checked). **Re-check it if you change `EMBED_MODEL`.**

In [30]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance in km (works on numbers or numpy/pandas arrays)."""
    lat1, lng1, lat2, lng2 = (np.radians(np.asarray(x, dtype=float)) for x in (lat1, lng1, lat2, lng2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def minmax(x) -> np.ndarray:
    """Scale to 0..1 so BM25 and cosine scores become comparable."""
    x = np.asarray(x, dtype=float)
    lo, hi = x.min(), x.max()
    return np.zeros_like(x) if hi - lo < 1e-12 else (x - lo) / (hi - lo)


# Sanity check: Royal Palace area -> BKK1 should be about 1.4 km
print(f"{float(haversine_km(11.5640, 104.9310, 11.5530, 104.9248)):.2f} km")


1.40 km


In [31]:
PRICE_RE = re.compile(r"(?:under|below|less than|cheaper than|max|<|ក្រោម|តិចជាង|តម្លៃក្រោម|តំលៃក្រោម)\s*\$?\s*(\d+(?:\.\d+)?)"
                      r"\s*(\$|usd|dollars?|ដុល្លារ|khr|riel|៛|រៀល)?")


FILTER_PHRASES = [  # (regex, filter name, value)
    (r"\bopen (?:right )?now\b|បើកឥឡូវ|កំពុងបើក|បើកពេលនេះ", "open_now", True),
    (r"\bopen late\b|\blate night\b|\blate\b|បើកយប់|យប់ជ្រៅ|ពេលយប់", "open_late", True),
    (r"\bcheap\b|\bbudget\b|ថោក|ថោកៗ|ធូរថ្លៃ|តម្លៃសមរម្យ|ធូរៗ", "max_price", CHEAP_MAX_USD),
    (r"\btop rated\b|\bbest rated\b|ល្បី|ពិន្ទុខ្ពស់", "min_rating", 4.5),
]


NEAR_ME_RE = re.compile(r"\b(?:near\s*me|nearby|near\s+here|around\s+me|around\s+here|close\s+to\s+me|closest|nearest)\b"
                        r"|ជិតខ្ញុំ|នៅជិតខ្ញុំ|ក្បែរខ្ញុំ|ជុំវិញខ្ញុំ|ជិតៗ")


SPATIAL_PREFIXES = r"(?:នៅម្ដុំ|នៅម្តុំ|ម្ដុំ|ម្តុំ|នៅក្បែរ|ក្បែរ|នៅជិត|ជិត|នៅ|ឯ|in|around|at|near)\s*"


CONVERSATIONAL_STOP_PATTERNS = [
    r"តើមាន\s*ហាង", r"តើមាន\s*កន្លែង", r"តើមាន",
    r"មាន\s*ហាង", r"មាន\s*កន្លែង", r"មាន",
    r"ហាង", r"កន្លែង",
    r"ឆ្ងាញ់ៗ?", r"ល្អ", r"ល្បី",
    r"ចង់ញ៉ាំ", r"ចង់រក", r"ចង់បាន",
    r"ជួយណែនាំ", r"សូមណែនាំ", r"ណែនាំ",
    r"ជួយរក", r"រក",
    r"អត់\s*\??", r"ទេ\s*\??", r"ដែរ\s*\??", r"ផង\s*\??",
    r"ណាខ្លះ\s*\??", r"ខ្លះ\s*\??", r"មួយណា\s*\??", r"ឯណា\s*\??", r"នៅឯណា\s*\??",
    r"ណា(?=[^ក-៿]|$)", r"មួយ(?=[^ក-៿]|$)",
    r"\b(?:is|are)\s+there\s+any\b", r"\bwhere\s+(?:can\s+i|to)\s+(?:find|get|eat)\b",
    r"\blooking\s+for\b", r"\bi\s+want\s+(?:to\s+eat)?\b",
    r"\b(?:can\s+you\s+)?recommend(?:\s+me)?\b",
    r"\b(?:delicious|good|tasty|best|popular)\b",
    r"\b(?:restaurant|restaurants|place|places|shop|shops)\b",
    r"\b(?:please|any)\b",
]


def expand_query(text: str) -> str:
    """Expand search text with cross-lingual food synonyms (Khmer <-> English)."""
    t = normalize_text(text)
    if not t:
        return ""
    additions = []
    for c in FOOD_CONCEPTS:
        matched = False
        for k in c["kh"]:
            if k in t:
                matched = True
                break
        if not matched:
            for e in c["en"]:
                if re.search(r"\b" + re.escape(e) + r"\b", t):
                    matched = True
                    break
        if matched:
            additions.extend(c["kh"])
            additions.extend(c["en"])
    extra = " ".join(dict.fromkeys(additions))
    return f"{text} {extra}".strip() if extra else text


def parse_query_filters(query):
    """Pull conditions and location out of the query text, cleaning conversational filler.
    Returns (clean search text, filters dict)."""
    orig = normalize_text(query)
    q = orig
    f = {}

    # 1. Near me
    if NEAR_ME_RE.search(q):
        f["near"] = "me"
        q = NEAR_ME_RE.sub(" ", q)

    # 2. Price
    m = PRICE_RE.search(q)
    if m:
        amount = float(m.group(1))
        unit = m.group(2) or "$"
        f["max_price"] = khr_to_usd(amount) if unit in ("khr", "riel", "៛", "រៀល") else amount
        q = q.replace(m.group(0), " ")

    # 3. Filter phrases (open now, open late, cheap, top rated)
    for pattern, key, value in FILTER_PHRASES:
        if re.search(pattern, q):
            f.setdefault(key, value)
            q = re.sub(pattern, " ", q)

    # 4. Place detection: a Phnom Penh khan ("BKK1", "ទួលគោក") and/or a city or province ("Siem Reap", "សៀមរាប")
    found = {}
    for alias, kind, value in PLACE_ALIASES:
        if kind in found:
            continue
        if KHMER_RE.search(alias):
            pattern = r"(?:" + SPATIAL_PREFIXES + r")?" + re.escape(alias)
        else:
            pattern = r"(?:" + SPATIAL_PREFIXES + r")?(?<![a-z0-9])" + re.escape(alias) + r"(?![a-z0-9])"
        match = re.search(pattern, q)
        if match:
            found[kind] = value
            q = q[:match.start()] + " " + q[match.end():]
    found_area = found.get("area")
    if found_area:
        f["area"] = found_area               # every khan is in Phnom Penh, so the khan is enough
    elif found.get("city"):
        f["city"] = found["city"]

    # Strip remaining aliases of the same place if present (e.g. BKK1 when Boeung Keng Kang was already matched)
    for alias, kind, value in PLACE_ALIASES:
        if found.get(kind) == value:
            if KHMER_RE.search(alias):
                q = re.sub(re.escape(alias), " ", q)
            else:
                q = re.sub(r"(?<![a-z0-9])" + re.escape(alias) + r"(?![a-z0-9])", " ", q)

    # 5. Near landmark (e.g. "near CADT", "ជិត CADT", "នៅម្តុំ CADT")
    if "near" not in f and not found_area:
        near_word = r"(?:near|around|ជិត|នៅជិត|ក្បែរ|នៅក្បែរ|នៅម្ដុំ|នៅម្តុំ)\s+"
        # a landmark of several words when it ends the query ("near Aeon Mall"), else the one word after "near"
        m_near = (re.search(near_word + r"([^,?.!]{2,60}?)\s*[?.!]*$", q)
                  or re.search(near_word + r"([^\s,?.!]{2,40})", q))
        if m_near:
            target = m_near.group(1).strip()
            if target.lower() not in AREA_MAP and target.lower() not in CITY_MAP:
                f["near"] = target
                q = q[:m_near.start()] + " " + q[m_near.end():]

    # 6. Conversational stop phrases & punctuation stripping
    for pat in CONVERSATIONAL_STOP_PATTERNS:
        q = re.sub(pat, " ", q, flags=re.IGNORECASE)

    q = re.sub(r"[?.,!():;\"\'/\\\[\]{}]+", " ", q)
    clean_q = re.sub(r"\s+", " ", q).strip()

    # If conversational stripping left nothing, fallback
    if not clean_q and not f:
        clean_q = re.sub(r"\s+", " ", orig).strip()

    return clean_q, f


def is_open_at(open_h, close_h, hour) -> bool:
    """Opening hours may cross midnight (e.g. 18 -> 2). Unknown hours = not open."""
    if pd.isna(open_h) or pd.isna(close_h):
        return False
    open_h, close_h = float(open_h), float(close_h) % 24
    if open_h == close_h:
        return True                          # open 24 hours
    if open_h < close_h:
        return open_h <= hour < close_h
    return hour >= open_h or hour < close_h


def closes_late(open_h, close_h) -> bool:
    if pd.isna(open_h) or pd.isna(close_h):
        return False
    return close_h >= LATE_HOUR or close_h < open_h


def apply_filters(d: pd.DataFrame, f: dict, now=None, relevant=None, unverified=None) -> np.ndarray:
    """Boolean mask. relevant = the rows that match the words of the question: a price, rating or opening-hours
    condition is applied only if at least one of those rows has the value to check. If none does, the condition
    cannot be checked in our data: it is not applied and its name ('price', 'rating', 'hours') is added to
    `unverified`, so the answer can say so instead of returning unrelated places that happen to have a price."""
    mask = np.ones(len(d), dtype=bool)
    relevant = np.ones(len(d), dtype=bool) if relevant is None else relevant
    unverified = [] if unverified is None else unverified
    if f.get("area"):
        mask &= d["area_key"].eq(normalize_text(f["area"])).to_numpy()
    if f.get("city"):
        mask &= d["city_key"].eq(normalize_text(f["city"])).to_numpy()
    relevant = relevant & mask                    # only places in the named khan or city can tell whether a condition is checkable

    def condition(name, match):
        nonlocal mask
        if (match & relevant).any():
            mask &= match
        elif name not in unverified:
            unverified.append(name)

    if f.get("max_price") is not None:
        condition("price", d["price_usd"].le(f["max_price"]).to_numpy())
    if f.get("min_rating") is not None:
        condition("rating", d["rating"].ge(f["min_rating"]).to_numpy())
    if f.get("open_now"):
        now = now or datetime.now(TZ)
        hour = now.hour + now.minute / 60
        condition("hours", np.array([is_open_at(o, c, hour) for o, c in zip(d["open_hour"], d["close_hour"])], dtype=bool))
    if f.get("open_late"):
        condition("hours", np.array([closes_late(o, c) for o, c in zip(d["open_hour"], d["close_hour"])], dtype=bool))
    return mask


for q in ["bbq under $5", "cheap kuy teav", "noodles open late", "korean in toul kork", "គុយទាវ ថោក", "coffee open now",
          "pizza near me", "coffee near CADT", "ជិតខ្ញុំ គុយទាវ"]:
    print(f"{q!r:22} -> {parse_query_filters(q)}")


'bbq under $5'         -> ('bbq', {'max_price': 5.0})
'cheap kuy teav'       -> ('kuy teav', {'max_price': 3.0})
'noodles open late'    -> ('noodles', {'open_late': True})
'korean in toul kork'  -> ('korean in', {'area': 'toul kork'})
'គុយទាវ ថោក'           -> ('គុយទាវ', {'max_price': 3.0})
'coffee open now'      -> ('coffee', {'open_now': True})
'pizza near me'        -> ('pizza', {'near': 'me'})
'coffee near CADT'     -> ('coffee', {'near': 'cadt'})
'ជិតខ្ញុំ គុយទាវ'      -> ('គុយទាវ', {'near': 'me'})


In [32]:
_geocache = {}


_last_geocode = [0.0]


LANDMARK_MARGIN = 0.15                       # how much better known (OSM "importance") a place outside Phnom Penh must be


def _nominatim(params):
    wait = 1.1 - (time.monotonic() - _last_geocode[0])
    if wait > 0:
        time.sleep(wait)
    _last_geocode[0] = time.monotonic()
    r = requests.get("https://nominatim.openstreetmap.org/search", timeout=6,
                     headers={"User-Agent": "food-search-student-project (Information Retrieval course)"},
                     params={"format": "jsonv2", "limit": 1, "countrycodes": "kh", **params})
    r.raise_for_status()
    hits = r.json()
    if not hits:
        return None
    return float(hits[0]["lat"]), float(hits[0]["lon"]), float(hits[0].get("importance") or 0)


def geocode_place(name: str, city: str = None):
    """Landmark name -> (lat, lng) via OpenStreetMap's Nominatim (max 1 request per second).
    Looks in the given city, else in Phnom Penh first and then anywhere in Cambodia ("near Angkor Wat")."""
    name = re.sub(r"\s+", " ", name or "").strip()[:60]
    if len(name) < 2:
        return None
    key = f"{name.lower()}|{(city or '').lower()}"
    if key in _geocache:
        return _geocache[key]
    try:
        if city and city != HOME_CITY:
            hit = _nominatim({"q": f"{name}, {city}"})
        else:
            hit = _nominatim({"q": f"{name}, {HOME_CITY}", "viewbox": "104.75,11.75,105.05,11.40", "bounded": 1})
            if not city:                     # a famous place elsewhere ("Angkor Wat") beats a small namesake in Phnom Penh
                wide = _nominatim({"q": name})
                if wide and (not hit or wide[2] > hit[2] + LANDMARK_MARGIN):
                    hit = wide
    except Exception as e:
        log.warning("geocoding failed (%s)", type(e).__name__)
        return None                          # not cached, so a later try can work
    spot = hit[:2] if hit else None
    _geocache[key] = spot
    return spot


# "near <place>": the place name is looked up on OpenStreetMap (Nominatim), then places are sorted by distance from it
print("CADT ->", geocode_place("CADT"))


CADT -> (11.6537486, 104.9116648)


In [33]:
def hybrid_search(query, k=TOP_K, user_loc=None, weight=None, filters=None,
                  use_query_filters=True, now=None) -> pd.DataFrame:
    """BM25 + embeddings + rating + distance, after filters. user_loc = (lat, lng)."""
    weight = HYBRID_WEIGHT if weight is None else weight
    text, parsed = parse_query_filters(query) if use_query_filters else (normalize_text(query), {})
    flt = {**parsed, **(filters or {})}
    out = df.copy()
    if text:
        search_text = expand_query(text)
        bm, method = bm25_scores(search_text)
        em = embedding_scores(search_text)
    else:                                    # query was only conditions, e.g. "open now"
        bm, em, method = np.zeros(len(df)), np.zeros(len(df)), "filters-only"
    out["bm25"], out["embed"] = minmax(bm), minmax(em)
    out["rating_boost"] = RATING_BOOST * out["rating"].fillna(0) / 5
    out["distance_km"], out["distance"] = np.nan, 0.0
    if user_loc:
        out["distance_km"] = haversine_km(user_loc[0], user_loc[1], out["lat"], out["lng"])
        out["distance"] = DISTANCE_WEIGHT / (1 + out["distance_km"].fillna(1e6))
    out["home"] = 0.0
    if not (flt.get("area") or flt.get("city") or user_loc):      # no place named: Phnom Penh first
        out["home"] = HOME_CITY_BONUS * out["city"].eq(HOME_CITY)
    out["final"] = (weight * out["embed"] + (1 - weight) * out["bm25"] + out["rating_boost"] + out["distance"]
                    + out["home"])
    match_mask = (bm > 0) | (em >= MIN_EMBED_SIM) if text else np.ones(len(out), dtype=bool)
    unverified = []
    keep = apply_filters(out, flt, now, match_mask, unverified)
    if text and (keep & match_mask).any():     # drop rows that match neither by keyword nor by meaning
        keep &= match_mask
    out = out[keep].sort_values("final", ascending=False, kind="stable")
    if flt.get("near") and user_loc and len(out):      # "near me" / "near X": closest of the clearly relevant places first
        pool = out[out["final"] >= 0.7 * out["final"].iloc[0]].head(NEAR_POOL)
        out = pool.sort_values("distance_km", kind="stable")
    out = out.head(k).reset_index(drop=True)
    out.attrs.update(filters=flt, method=method, query_text=text, unverified=unverified)
    return out


def get_rows(ids) -> pd.DataFrame:
    """Rows for the given ids, in that order (unknown ids are skipped)."""
    known = set(df["id"])
    ids = [i for i in dict.fromkeys(ids) if i in known]
    return df.set_index("id").loc[ids].reset_index()


In [34]:
SHOW = ["id", "name", "dish_en", "price_usd", "rating", "bm25", "embed", "distance", "final"]
for q in DEMO_QUERIES:
    res = hybrid_search(q)
    print(f"🔎 {q}   (keyword method: {res.attrs['method']})")
    display(res[SHOW].round(3))

🔎 curry noodle   (keyword method: bm25)


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final
0,ovt-8bd3673f-e4c7-4d3e-a04a-dd2a0e4b08b8,Curry Zone Restaurant,restaurant,NaN,NaN,0.978,0.854,0.0,0.916
1,ovt-04f8a050-2b30-4f24-9ef4-0e94dbbee1c2,Curry Leaf India Restaurant,restaurant,NaN,NaN,0.957,0.872,0.0,0.915
2,osm-n5586787217,Khmer curry stall,,NaN,NaN,1.000,0.729,0.0,0.864
3,ovt-7bda2203-0742-436a-98a5-a84c468c0878,VN noodles,,NaN,NaN,0.686,1.000,0.0,0.843
4,ovt-7b7e38dc-063b-4b91-8356-34cd88fd564e,Universal Noodles,,NaN,NaN,0.686,0.991,0.0,0.839


🔎 carry noodle   (keyword method: bm25+typo-fix)


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final
0,osm-n4178059189,Marugame Udon,noodles,NaN,NaN,0.780,1.000,0.0,0.890
1,osm-n4586489994,Malop Chat Toul Tom Poung,noodles; local,NaN,NaN,0.780,0.845,0.0,0.812
2,ovt-7b7e38dc-063b-4b91-8356-34cd88fd564e,Universal Noodles,,NaN,NaN,0.686,0.913,0.0,0.800
3,ovt-7bda2203-0742-436a-98a5-a84c468c0878,VN noodles,,NaN,NaN,0.686,0.873,0.0,0.780
4,osm-n4711644591,Spicy Noodle,,NaN,NaN,0.686,0.838,0.0,0.762


🔎 នំបញ្ចុក   (keyword method: bm25)


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final
0,osm-n4861669721,បបរ នំបញ្ចុក,,NaN,NaN,1.000,0.739,0.0,0.870
1,ovt-7bf721b4-fa12-42c6-9135-d6a8433b21d7,នំបញ្ចុកស្រុះ,restaurant,NaN,NaN,0.878,0.808,0.0,0.843
2,osm-n10292468209,នំបញ្ចុក482,asian,NaN,NaN,1.000,0.645,0.0,0.822
3,ovt-b34f1a6d-8f20-49ea-887d-7b43e4bf05f3,នំបញ្ចុក កេរម៉ែ,restaurant,NaN,NaN,0.892,0.742,0.0,0.817
4,ovt-770f2a5d-7dff-48a1-abca-7ad4e8cb12cb,នំបញ្ចុកសាច់គោ,restaurant,NaN,NaN,0.852,0.748,0.0,0.800


In [35]:
# Filters and location
now = datetime.now(TZ)
print("Phnom Penh time now:", now.strftime("%H:%M"))
for q in ["korean bbq", "coffee open now"]:
    res = hybrid_search(q)
    print(f"🔎 {q}   filters={res.attrs['filters']}")
    display(res[SHOW].round(3))

INDEPENDENCE_MONUMENT = (11.5564, 104.9282)
res = hybrid_search("iced coffee", user_loc=INDEPENDENCE_MONUMENT)
print("🔎 iced coffee, near Independence Monument")
display(res[SHOW + ["distance_km"]].round(3))

print("🔎 pizza near CADT (the place is looked up on OpenStreetMap; closest relevant places first)")
display(hybrid_search("pizza near CADT", user_loc=geocode_place("CADT"))[SHOW + ["distance_km"]].round(3))
print(f"🔎 car repair -> {len(hybrid_search('car repair'))} results (0 = the bot answers 'nothing matches')")

Phnom Penh time now: 09:22
🔎 korean bbq   filters={}


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final
0,osm-n11304262061,Misigga BBQ - Korean BBQ & Hot Pot,korean food,NaN,NaN,1.000,0.991,0.0,0.996
1,ovt-4d28a5b7-7ced-46dd-bf36-224d9ecf4577,WangGi Korean BBQ,korean restaurant,NaN,NaN,0.878,0.977,0.0,0.928
2,ovt-774201db-bc25-473e-8d0a-7b56416a6d7a,Ember Korean BBQ,korean restaurant,NaN,NaN,0.886,0.963,0.0,0.925
3,ovt-675ec9fe-4c81-456d-8d1f-cb7e67c231f2,Samwon Garden Korean BBQ,korean restaurant,NaN,NaN,0.878,0.964,0.0,0.921
4,ovt-eeeab863-07f9-4c24-8f78-e3726ce82357,Bulgogi Korean BBQ,korean restaurant,NaN,NaN,0.886,0.940,0.0,0.913


🔎 coffee open now   filters={'open_now': True}


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final
0,osm-w188369597,Well Coffee,"coffee, cake sandwich",NaN,NaN,1.000,0.900,0.0,0.950
1,osm-n11245003537,Tube Coffee Riverside,coffee,NaN,NaN,0.944,0.894,0.0,0.919
2,osm-n13607375002,1999 Coffee,coffee,NaN,NaN,0.944,0.837,0.0,0.891
3,osm-n4209266194,Cozy Coffee,coffee,NaN,NaN,0.944,0.817,0.0,0.880
4,osm-n8292085717,VR Coffee,,NaN,NaN,0.772,0.973,0.0,0.873


🔎 iced coffee, near Independence Monument


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final,distance_km
0,osm-n11005544321,Nuk Café,coffee; ice cream,NaN,NaN,0.868,0.909,0.135,1.023,0.113
1,ovt-e276faf0-2658-4121-a032-2b3551f7cf83,Yaas Gelato & Coffee,ice cream shop,NaN,NaN,1.000,0.687,0.107,0.951,0.402
2,ovt-a7dd30b4-38b6-4302-907a-4c2bc15f6b14,Mr. Coffee and Ice Cream Rolls,cafe,NaN,NaN,0.690,1.000,0.055,0.900,1.730
3,osm-n4281500393,Bingo Gourmand Home Ice Cream & Coffee,,NaN,NaN,0.864,0.825,0.039,0.883,2.894
4,osm-n5354304221,Aun Café,coffee; tea; ice cream,NaN,NaN,0.723,0.820,0.087,0.858,0.728


🔎 pizza near CADT (the place is looked up on OpenStreetMap; closest relevant places first)


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final,distance_km
0,ovt-b0fbe9d3-d4ee-4807-a4e9-6dd96d402b27,The Pizza Company - Big Tree,pizza restaurant,NaN,NaN,0.914,0.950,0.033,0.964,3.573
1,ovt-5eeb5056-981e-45be-9d37-8a7cd5ac3825,Pizza Hut Domrey Park Mall,pizza restaurant,NaN,NaN,0.914,0.955,0.031,0.965,3.898
2,ovt-7e7568f7-62b0-41fe-a414-c743532980f5,The Pizza Company-Cambodia,pizza restaurant,NaN,NaN,0.920,0.916,0.020,0.938,6.599
3,ovt-99a96bf3-5c4b-46c1-9f64-71d823c86bfd,Baraka Pizza House,pizza restaurant,NaN,NaN,0.927,0.924,0.018,0.943,7.418
4,osm-n5743727939,Domino's Pizza TK,pizza,NaN,NaN,0.894,0.949,0.017,0.938,7.858


🔎 car repair -> 5 results (0 = the bot answers 'nothing matches')


### BM25 vs Embeddings vs Hybrid

*Hybrid* means the full system, including the query filters. *Hybrid (no filters)* shows how much comes
from the score combination alone. Keep in mind that a small test set is noisy: a single query can move a metric a lot.
Use the table to spot patterns, not to declare a winner.

In [36]:
if HAS_EVAL:
    hybrid_ids = lambda q, k: hybrid_search(q, k)["id"].tolist()
    RESULTS["Hybrid (no filters)"], PER_QUERY["Hybrid (no filters)"] = evaluate(
        lambda q, k: hybrid_search(q, k, use_query_filters=False)["id"].tolist())
    RESULTS["Hybrid"], PER_QUERY["Hybrid"] = evaluate(hybrid_ids)

    eval_table = pd.DataFrame(RESULTS).T
    eval_table.to_csv(RESULTS_DIR / "eval_results.csv")
    print("saved results/eval_results.csv")
    display(eval_table)
else:
    print("⏭️ skipped: no evaluation queries yet (see section 2)")

⏭️ skipped: no evaluation queries yet (see section 2)


In [37]:
if HAS_EVAL:
    fig, ax = plt.subplots(figsize=(8, 3.6))
    methods, metrics = list(eval_table.index), list(eval_table.columns)
    width = 0.8 / len(methods)
    x = np.arange(len(metrics))
    for i, m in enumerate(methods):
        bars = ax.bar(x + i * width, eval_table.loc[m], width - 0.02, label=m, color=SERIES[i % len(SERIES)])
        ax.bar_label(bars, fmt="%.2f", fontsize=7, color="#52514e", padding=2)
    ax.set_xticks(x + width * (len(methods) - 1) / 2, metrics)
    ax.set_ylim(0, 1.1)
    ax.set_title(f"Retrieval quality on {len(eval_df)} test queries (higher is better)", loc="left")
    ax.legend(frameon=False, fontsize=8, ncol=len(methods), loc="upper left", bbox_to_anchor=(0, -0.1))
    plt.show()
else:
    print("⏭️ skipped: no evaluation queries yet (see section 2)")

⏭️ skipped: no evaluation queries yet (see section 2)


In [38]:
if HAS_EVAL:
    # Per-query view: where does each method fail? (nDCG@5)
    display(pd.DataFrame({m: PER_QUERY[m]["nDCG@5"].values for m in PER_QUERY},
                         index=eval_df["query"]).round(2))
else:
    print("⏭️ skipped: no evaluation queries yet (see section 2)")

⏭️ skipped: no evaluation queries yet (see section 2)


In [39]:
if HAS_EVAL:
    # Sweep the hybrid weight: 0 = only BM25, 1 = only embeddings
    weights = np.round(np.linspace(0, 1, 11), 2)
    sweep = [evaluate(lambda q, k, w=w: hybrid_search(q, k, weight=w)["id"].tolist())[0]["nDCG@5"] for w in weights]
    best_w = float(weights[int(np.argmax(sweep))])

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(weights, sweep, color=SERIES[0], linewidth=2, marker="o", markersize=6)
    ax.axvline(HYBRID_WEIGHT, color="#b5b4ae", linestyle="--", linewidth=1)
    ax.annotate(f"current w={HYBRID_WEIGHT}", (HYBRID_WEIGHT, min(sweep)), xytext=(4, 0),
                textcoords="offset points", fontsize=8, color="#52514e")
    ax.set_xlabel("HYBRID_WEIGHT (embedding share)")
    ax.set_ylabel("nDCG@5")
    ax.set_title("nDCG@5 vs hybrid weight", loc="left")
    plt.show()
    print(pd.Series(sweep, index=weights, name="nDCG@5").round(3).to_dict())
    print(f"best weight on this tiny eval set: {best_w} (we keep {HYBRID_WEIGHT}; tuning on a small set overfits)")
else:
    print("⏭️ skipped: no evaluation queries yet (see section 2)")

⏭️ skipped: no evaluation queries yet (see section 2)


## 6. Image search with CLIP

**What:** **CLIP** (ViT-B-32) maps images *and* text into the same vector space. So we can search
photos with text (*"grilled meat on skewers"*) and find similar photos from a user's photo (image → image).
Photo vectors go into a second Chroma collection.

**How to enable:** put food photos into `data/photos/`. Name them like the place id (`<id>.jpg`, e.g. `osm-n4011096528.jpg`) or use
the file name from the `image_path` column, then re-run this section. CLIP's text side is English-only, so
text→image works best with English queries.

If the folder is empty, this section prints a message and is skipped, and the rest of the notebook still works.

In [40]:
CLIP_ARCH, CLIP_PRETRAINED = "ViT-B-32", "laion2b_s34b_b79k"


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}


_clip = {}          # model, preprocess, tokenizer (loaded on first use)


def list_photos(folder=None) -> list:
    folder = Path(folder or PHOTOS_DIR)
    return sorted(p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS) if folder.exists() else []


def load_clip():
    if not _clip:
        import open_clip
        model, _, preprocess = open_clip.create_model_and_transforms(
            CLIP_ARCH, pretrained=CLIP_PRETRAINED, device=DEVICE)
        model.eval()
        _clip.update(model=model, preprocess=preprocess, tokenizer=open_clip.get_tokenizer(CLIP_ARCH))
    return _clip["model"], _clip["preprocess"], _clip["tokenizer"]


@torch.no_grad()
def clip_embed_images(paths) -> np.ndarray:
    from PIL import Image
    model, preprocess, _ = load_clip()
    batch = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths]).to(DEVICE)
    feats = model.encode_image(batch)
    return (feats / feats.norm(dim=-1, keepdim=True)).float().cpu().numpy()


@torch.no_grad()
def clip_embed_text(texts) -> np.ndarray:
    model, _, tokenizer = load_clip()
    feats = model.encode_text(tokenizer(list(texts)).to(DEVICE))
    return (feats / feats.norm(dim=-1, keepdim=True)).float().cpu().numpy()


def get_image_collection(client):
    return client.get_or_create_collection("food_images_" + CLIP_ARCH.lower().replace("-", "_"),
                                           metadata={"hnsw:space": "cosine"})


def build_image_index(paths, collection, d: pd.DataFrame) -> int:
    """Photo -> restaurant id: by image_url file name, else by file stem (<id>.jpg -> <id>)."""
    img_col = d["image_url"] if "image_url" in d.columns else d["image_path"]
    by_name = {Path(p).name: i for i, p in zip(d["id"], img_col) if p}
    ids = [p.relative_to(PHOTOS_DIR).as_posix() for p in paths]
    embs = np.vstack([clip_embed_images(paths[i:i + 16]) for i in range(0, len(paths), 16)])
    collection.upsert(ids=ids, embeddings=embs.tolist(),
                      metadatas=[{"path": p.relative_to(ROOT).as_posix(),
                                  "restaurant_id": by_name.get(p.name, p.stem)} for p in paths])
    stale = set(collection.get(include=[])["ids"]) - set(ids)
    if stale:
        collection.delete(ids=sorted(stale))
    return collection.count()


def _image_query(emb, k):
    res = image_col.query(query_embeddings=[emb.tolist()], n_results=min(k, image_col.count()),
                          include=["metadatas", "distances"])
    return [{"path": ROOT / m["path"], "restaurant_id": m["restaurant_id"], "similarity": round(1 - dist, 3)}
            for m, dist in zip(res["metadatas"][0], res["distances"][0])]


def search_images_by_text(query, k=TOP_K):
    return _image_query(clip_embed_text([query])[0], k)


def search_images_by_image(path, k=TOP_K):
    return _image_query(clip_embed_images([path])[0], k)


HAS_PHOTOS, image_col = False, None


def image_search_available() -> bool:
    return bool(HAS_PHOTOS) and image_col is not None


def init():
    """Load data/restaurants.csv and build (or refresh) every index. Call once at startup."""
    global df, bm25, BM25_VOCAB, FOOD_VOCAB, embedder, text_col, image_col, HAS_PHOTOS
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    from sentence_transformers import SentenceTransformer
    df = clean_df(pd.read_csv(DATA_DIR / "restaurants.csv", dtype={"id": str, "phone_public": str, "price_usd": str}))
    bm25, BM25_VOCAB = build_bm25(df)
    FOOD_VOCAB = build_food_vocab(df)
    embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
    client = open_chroma()
    text_col = get_text_collection(client)
    build_text_index(df, text_col)
    photos = list_photos()
    HAS_PHOTOS = bool(photos)
    if HAS_PHOTOS:
        image_col = get_image_collection(client)
        build_image_index(photos, image_col, df)
    log.info("search ready: %d places, %d photos, device=%s", len(df), len(photos), DEVICE)


In [41]:
photo_files = list_photos()
HAS_PHOTOS = len(photo_files) > 0
if HAS_PHOTOS:
    image_col = get_image_collection(chroma)
    print(f"Indexed {build_image_index(photo_files, image_col, df)} photos into {image_col.name!r} (CLIP on {DEVICE})")
else:
    print("📷 data/photos is empty: skipping CLIP image search. "
          "Add photos (e.g. data/photos/<id>.jpg) and re-run this section to enable it.")

📷 data/photos is empty: skipping CLIP image search. Add photos (e.g. data/photos/<id>.jpg) and re-run this section to enable it.


In [42]:
def show_image_results(matches, title=""):
    fig, axes = plt.subplots(1, len(matches), figsize=(3 * len(matches), 3), squeeze=False)
    from PIL import Image
    for ax, m in zip(axes[0], matches):
        ax.imshow(Image.open(m["path"]).convert("RGB"))
        ax.set_title(f"{m['restaurant_id']}  sim={m['similarity']:.2f}", fontsize=8)
        ax.axis("off")
    fig.suptitle(title, x=0.01, ha="left")
    plt.show()


if HAS_PHOTOS:
    show_image_results(search_images_by_text("grilled meat skewers", k=4), "text → image: 'grilled meat skewers'")
    show_image_results(search_images_by_image(photo_files[0], k=4), f"image → image: {photo_files[0].name}")
else:
    print("⏭️ skipped (no photos)")

⏭️ skipped (no photos)


## 7. LLM answers (RAG) with automatic fallback

**What:** **R**etrieval-**A**ugmented **G**eneration. We retrieve the best places first, then ask an LLM to write a short
friendly answer **using only those places**. The LLM never sees the whole database, so it cannot recommend
places that do not exist. The prompt also forbids inventing prices or hours.

**Models and fallback** (`LLM_MODELS` in `.env`, e.g. `gemini:gemini-3.5-flash-lite,gemini:gemini-3.6-flash`): the user
chooses a model; if it fails (no key, no internet, busy, timeout) the next ones are tried (`MAX_FALLBACKS`), and if none
answers the plain result cards are shown. `none` means plain cards only.

**Conversation** (the subsections below): after a search, the user can keep asking about the places shown. Each message
is planned (follow-up, new search or small talk), and each follow-up is answered in the shape that fits it.

**Ollama cloud:** to use ollama.com instead of a local Ollama, create a key at <https://ollama.com/settings/keys> and set
`OLLAMA_API_KEY`, `OLLAMA_HOST=https://ollama.com`, `OLLAMA_MODEL=<a model from check_ollama()>` and `LLM_PROVIDER=ollama`
in `.env`. The key is sent only over HTTPS and is never printed or logged.

Every call has a timeout and is wrapped in `try/except`, so the demo keeps working even without internet.
The log shows which provider answered (never any keys). Model names come from `GEMINI_MODEL` / `OLLAMA_MODEL`.

In [43]:
NO_MATCH = {
    "en": "Sorry, nothing in our list matches that. Try another dish or remove a filter.",
    "kh": "សូមអភ័យទោស មិនមានហាងណាត្រូវនឹងការស្វែងរករបស់អ្នកទេ។ សូមសាកល្បងម្ហូបផ្សេង។",
}


NO_MATCH_IN = {
    "en": "Sorry, our list has nothing like that in {place} yet. Try another dish, or ask without the place to search all of Cambodia.",
    "kh": "សូមអភ័យទោស បញ្ជីរបស់យើងមិនទាន់មានហាងបែបនេះនៅ{place}ទេ។ សូមសាកល្បងម្ហូបផ្សេង ឬសួរដោយមិនដាក់ទីកន្លែង។",
}


def no_match(lang, results=None) -> str:
    """The 'nothing found' reply; names the khan or city when the search was limited to one."""
    flt = results.attrs.get("filters", {}) if results is not None else {}
    place = flt.get("area") or flt.get("city")
    if not place:
        return NO_MATCH[lang]
    return NO_MATCH_IN[lang].format(place=PLACE_NAMES_KH.get(place, place) if lang == "kh" else place)


UNVERIFIED_WHAT = {"en": {"price": "prices", "hours": "opening hours", "rating": "ratings"},
                   "kh": {"price": "តម្លៃ", "hours": "ម៉ោងបើក", "rating": "ការវាយតម្លៃ"}}


UNVERIFIED_TEXT = {"en": "Our data has no {what} for these places, so I could not check that condition.",
                   "kh": "ទិន្នន័យរបស់យើងមិនមាន{what}សម្រាប់ហាងទាំងនេះ ដូច្នេះមិនអាចពិនិត្យលក្ខខណ្ឌនោះបានទេ។"}


def unverified_note(results, lang) -> str:
    """One sentence when a condition of the question (under $5, open now, rated 4+) could not be checked because the
    data has no such value for the places that match the question; '' when every condition was applied."""
    names = [UNVERIFIED_WHAT[lang][u] for u in (results.attrs.get("unverified") or []) if u in UNVERIFIED_WHAT[lang]]
    return UNVERIFIED_TEXT[lang].format(what=(" / " if lang == "en" else " និង ").join(names)) if names else ""


LEAD_TEXT = {
    "en": {"found": "Here's what I found", "for": " for “{what}”", "in": " in {place}", "open_now": "open now",
           "open_late": "open late", "under": "under ${v}", "rated": "rated {v}+", "near_me": "closest first",
           "near": "closest to {name} first", "end": "."},
    "kh": {"found": "នេះជាអ្វីដែលខ្ញុំរកឃើញ", "for": " សម្រាប់ “{what}”", "in": " នៅ{place}", "open_now": "កំពុងបើក",
           "open_late": "បើកយប់", "under": "ក្រោម ${v}", "rated": "ពិន្ទុ {v}+", "near_me": "ជិតខ្ញុំមុន",
           "near": "ជិត {name} មុន", "end": "។"},
}


def search_lead(results, lang, near=None) -> str:
    """The one line above the result cards of a search: what was searched, where, and which conditions were applied
    ("Here's what I found for “pizza” in Siem Reap, open now, closest first."). Built from the search itself, with no
    AI model: the cards below already list the places, so a model's text would only repeat their names.
    near = 'me', a place name the results are sorted by, or None."""
    W = LEAD_TEXT[lang]
    flt, unverified = results.attrs.get("filters", {}), results.attrs.get("unverified") or []
    what = (results.attrs.get("query_text") or "").strip()
    place = flt.get("area") or flt.get("city")
    if place and lang == "kh":
        place = PLACE_NAMES_KH.get(place, place)
    notes = []                                       # only conditions that were really applied (see unverified_note)
    if flt.get("open_now") and "hours" not in unverified:
        notes.append(W["open_now"])
    if flt.get("open_late") and "hours" not in unverified:
        notes.append(W["open_late"])
    if flt.get("max_price") is not None and "price" not in unverified:
        notes.append(W["under"].format(v=f"{flt['max_price']:g}"))
    if flt.get("min_rating") is not None and "rating" not in unverified:
        notes.append(W["rated"].format(v=f"{flt['min_rating']:g}"))
    if near:
        notes.append(W["near_me"] if near == "me" else W["near"].format(name=near))
    text = W["found"] + (W["for"].format(what=what) if what else "") + (W["in"].format(place=place) if place else "")
    return text + "".join(", " + n for n in notes) + W["end"]


def fmt_price(p) -> str:
    if pd.isna(p) or str(p).strip() in ("", "nan", "price unknown"):
        return "price unknown"
    try:
        return f"${float(p):.2f}"
    except (ValueError, TypeError):
        return str(p)


def fmt_hours(open_h, close_h) -> str:
    if pd.isna(open_h) or pd.isna(close_h):
        return "hours unknown"
    open_h, close_h = float(open_h), float(close_h)
    if open_h % 24 == close_h % 24:
        return "open 24 hours"
    hhmm = lambda h: f"{int(h) % 24:02d}:{round(h % 1 * 60):02d}"
    return f"{hhmm(open_h)}–{hhmm(close_h)}"


def format_cards(results, lang="en", max_items=TOP_K) -> str:
    """Plain-text result cards (the no-LLM answer)."""
    if results is None or len(results) == 0:
        return no_match(lang, results)
    lines = []
    for i, r in enumerate(results.head(max_items).itertuples(), 1):
        dish = r.dish_kh if lang == "kh" and r.dish_kh else r.dish_en
        bits = [fmt_price(r.price_usd), f"⭐ {r.rating:.1f}" if pd.notna(r.rating) else "⭐ n/a",
                r.area, fmt_hours(r.open_hour, r.close_hour)]
        if pd.notna(getattr(r, "distance_km", np.nan)):
            bits.append(f"{r.distance_km:.1f} km away")
        lines.append(f"{i}. {r.name} — {dish}\n   " + " · ".join(bits))
    return "\n".join(lines)


print(format_cards(hybrid_search("coffee"), "en", max_items=3))


1. Starbucks Coffee — coffee
   price unknown · ⭐ n/a · Toul Kork · hours unknown
2. The Coffee Bean & Tea Leaf — coffee
   price unknown · ⭐ n/a · Prampir Makara · hours unknown
3. Coffee Today — coffee
   price unknown · ⭐ n/a · Boeng Keng Kang · hours unknown


In [44]:
def build_prompt(query, results, lang="en") -> str:
    """The LLM only sees the retrieved rows, so it cannot recommend anything else."""
    rows = []
    for i, r in enumerate(results.head(TOP_K).itertuples(), 1):
        d = getattr(r, "distance_km", np.nan)
        # same format as the result list (470 m / 1.4 km), so the summary and the list agree
        dist = "unknown" if pd.isna(d) else (f"{round(d * 100) * 10:.0f} m" if d < 1 else f"{d:.1f} km")
        dish_val = getattr(r, "dish_kh", "")
        phone_val = getattr(r, "phone_public", "") or "none"
        page_val = getattr(r, "page_url", "") or "none"
        tags_val = getattr(r, "tags", "") or ""
        menu_val = getattr(r, "menu", "") or ""
        rows.append(f"{i}. name: {r.name} | dish: {r.dish_en} / {dish_val} | price: {fmt_price(r.price_usd)}"
                    f" | rating: {r.rating if pd.notna(r.rating) else 'unknown'} | area: {r.area}"
                    f" | hours: {fmt_hours(r.open_hour, r.close_hour)} | distance: {dist}"
                    f" | phone: {phone_val} | link: {page_val} | tags: {tags_val}"
                    f" | menu: {menu_val}"
                    f" | notes: {r.description}")
    language = "Khmer" if lang == "kh" else "English"
    return (
        "You are a friendly food guide for Cambodia.\n"
        "Answer the QUESTION using ONLY the places listed in RESULTS.\n"
        "Rules:\n"
        f"- Answer in {language} (the language of the question).\n"
        "- The full list of places with their details is shown right under your answer, so do not repeat it.\n"
        "- Answer exactly what the QUESTION asks, nothing more:\n"
        "  * a request for food or places (\"kuy teav\", \"pizza in BKK\"): name the best 1 to 3 places with their list\n"
        "    number (\"1. Costa\") and, in a few words, why they fit; no prices or hours unless the question asks;\n"
        "  * a question about one detail (\"cheapest\", \"closest\", \"open now\", \"under $3\"): name the place(s) that fit\n"
        "    and give that detail only;\n"
        "  * a yes/no question (\"is there vegan food in Toul Kork?\"): start with yes or no.\n"
        "- Never invent restaurants, prices, opening hours or other facts. Mention a value only if it is in RESULTS.\n"
        "- Do not mention missing information (unknown prices, hours or ratings) unless the question asks about it.\n"
        "- If the question says near me / near a place, prefer the closest places and give their distance.\n"
        "- Use only facts that are written in RESULTS: no opinions about taste, quality or popularity.\n"
        "- If none of the RESULTS fit the question, say clearly that nothing matches.\n"
        + (f"- The data cannot confirm the question's condition on {', '.join(results.attrs['unverified'])}: do not say or "
           "imply that a place meets it.\n" if results.attrs.get("unverified") else "") +
        "- 1 or 2 short sentences, under 50 words, plain text: no greeting, no list, no closing question.\n\n"
        f"QUESTION: {query}\n\nRESULTS:\n" + "\n".join(rows)
    )


# The exact prompt for one query: only the retrieved rows are included
print(build_prompt("korean restaurant", hybrid_search("korean restaurant"), "en"))


You are a friendly food guide for Phnom Penh, Cambodia.
Answer the QUESTION using ONLY the places listed in RESULTS.
Rules:
- Answer in English (the language of the question).
- Recommend at most 3 places, and only places from RESULTS.
- Never invent restaurants, prices, opening hours or other facts. Mention a value only if it is in RESULTS.
- Do not mention missing information (unknown prices, hours or ratings) unless the question asks about it.
- If the question says near me / near a place, prefer the closest places and give their distance.
- Use only facts that are written in RESULTS: no opinions about taste, quality or popularity.
- If none of the RESULTS fit the question, say clearly that nothing matches.
- Keep the answer under 120 words.

QUESTION: korean restaurant

RESULTS:
1. name: Jaru Korean Restaurant | dish: korean restaurant / ម្ហូបកូរ៉េ | price: price unknown | rating: unknown | area: Boeng Keng Kang | hours: hours unknown | distance: unknown | notes: korean restaurant 

In [45]:
def safe_error(e) -> str:
    """Short error text with any secret value masked (safe to log)."""
    msg = f"{type(e).__name__}: {e}"[:200]
    for secret in (GEMINI_API_KEY, TELEGRAM_TOKEN, TG_API_HASH, OLLAMA_API_KEY):
        if secret:
            msg = msg.replace(secret, "***")
    return msg


def call_gemini(prompt, model=None, json_mode=False, timeout_s=None) -> str:
    model = model or GEMINI_MODEL
    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY is not set")
    if not model:
        raise RuntimeError("GEMINI_MODEL is not set")
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_API_KEY,
                          http_options=types.HttpOptions(timeout=(timeout_s or LLM_TIMEOUT_S) * 1000))  # milliseconds
    resp = client.models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=1024,
                                           response_mime_type="application/json" if json_mode else None))
    text = (resp.text or "").strip()
    if not text:
        raise RuntimeError("Gemini returned an empty answer")
    return text


def ollama_headers() -> dict:
    """Ollama cloud needs the API key; it is sent only over HTTPS, never to a plain-http host."""
    if OLLAMA_API_KEY and OLLAMA_HOST.startswith("https://"):
        return {"Authorization": f"Bearer {OLLAMA_API_KEY}"}
    return {}


def call_ollama(prompt, model=None, json_mode=False, timeout_s=None) -> str:
    model = model or OLLAMA_MODEL
    if not model:
        raise RuntimeError("OLLAMA_MODEL is not set")
    r = requests.post(f"{OLLAMA_HOST}/api/generate", headers=ollama_headers(),
                      json={"model": model, "prompt": prompt, "stream": False, "options": {"temperature": 0.2},
                            **({"format": "json"} if json_mode else {})},
                      timeout=(5, timeout_s or OLLAMA_TIMEOUT_S))        # (connect, read) seconds
    r.raise_for_status()
    text = r.json().get("response", "").strip()
    if not text:
        raise RuntimeError("Ollama returned an empty answer")
    return text


# ---------- which model answers (users choose from LLM_MODELS; nothing else is accepted) ----------
NO_LLM = "none"            # "results only": the retrieved rows as plain cards, no model call


MAX_FALLBACKS = 2          # if the chosen model fails (busy, quota, ...), try at most this many others from the list


def model_label(provider, model) -> str:
    """'gemini-3.5-flash-lite' -> 'Gemini 3.5 Flash Lite', 'llama3.2' (ollama) -> 'Llama3.2 (Ollama)'."""
    words = [w for w in re.split(r"[-_:\s]+", model) if w]
    nice = " ".join(w.upper() if w.lower() in ("gpt", "oss", "glm") else w if any(c.isdigit() for c in w) else w.capitalize()
                    for w in words)
    return nice + (" (Ollama)" if provider == "ollama" else "")


def parse_models(spec) -> list:
    """'gemini:gemini-3.5-flash-lite, ollama:gpt-oss:20b=GPT-OSS' -> [{id, provider, model, label}, ...]"""
    models = []
    for item in (spec or "").split(","):
        item, _, label = item.strip().partition("=")
        provider, _, model = item.partition(":")
        provider, model = provider.strip().lower(), model.strip()
        if not item:
            continue
        if provider not in ("gemini", "ollama") or not model:
            log.warning("LLM_MODELS: skipping %r (use provider:model, provider = gemini or ollama)", item)
            continue
        models.append({"id": f"{provider}:{model}", "provider": provider, "model": model,
                       "label": label.strip() or model_label(provider, model)})
    return list({m["id"]: m for m in models}.values())          # no duplicates, order kept


def _default_spec() -> str:
    items = ([f"gemini:{GEMINI_MODEL}"] if GEMINI_MODEL else []) + ([f"ollama:{OLLAMA_MODEL}"] if OLLAMA_MODEL else [])
    return ",".join(reversed(items) if LLM_PROVIDER == "ollama" else items)


MODELS = parse_models(LLM_MODELS or _default_spec())


MODELS_BY_ID = {m["id"]: m for m in MODELS}


DEFAULT_MODEL_ID = NO_LLM if LLM_PROVIDER == "none" or not MODELS else MODELS[0]["id"]


def valid_model_id(model_id):
    """A model id from the list, 'none', or None (= the default). Anything else is ignored."""
    return model_id if model_id == NO_LLM or model_id in MODELS_BY_ID else None


def model_chain(model_id=None) -> list:
    """The chosen model first, then up to MAX_FALLBACKS others from the list (in list order)."""
    model_id = valid_model_id(model_id) or DEFAULT_MODEL_ID
    if model_id == NO_LLM:
        return []
    first = MODELS_BY_ID[model_id]
    return [first] + [m for m in MODELS if m is not first][:MAX_FALLBACKS]


class NoModelAnswered(Exception):
    """No model was chosen ('none'), or every model tried failed (busy, quota, offline, ...)."""


def complete(prompt, model_id=None, json_mode=False, max_models=None, timeout_s=None):
    """Runs the prompt on the chosen model, then on up to MAX_FALLBACKS others from the list.
    max_models / timeout_s limit the attempts (for quick steps that have a non-AI fallback).
    Returns (text, model) or raises NoModelAnswered."""
    callers = {"gemini": call_gemini, "ollama": call_ollama}
    for m in model_chain(model_id)[:max_models]:
        try:
            return callers[m["provider"]](prompt, m["model"], json_mode=json_mode, timeout_s=timeout_s), m
        except Exception as e:                       # no key, offline, busy, quota, timeout, ...
            log.warning("%s failed (%s) -> next model", m["id"], safe_error(e))
    raise NoModelAnswered()


def as_sentences(text) -> str:
    """Joins bullet or numbered lines a model wrote into plain sentences (answers that must not be lists)."""
    # only bullet marks go: "1. Costa" at the start of a line is a place's list number and stays
    lines = [re.sub(r"^\s*[-*•]\s+", "", l).strip() for l in str(text).strip().splitlines()]
    return " ".join(l for l in lines if l)


def generate_answer(query, results, model_id=None) -> dict:
    """RAG answer from the retrieved rows. Returns {text, provider, model, label, chosen}:
    provider = gemini | ollama | cards (no model answered) | no-results (nothing retrieved, no model call)."""
    lang = detect_lang(query)
    chosen = valid_model_id(model_id) or DEFAULT_MODEL_ID
    if results is None or len(results) == 0:
        return {"text": no_match(lang, results), "provider": "no-results", "model": None, "label": None, "chosen": chosen}
    try:
        text, m = complete(build_prompt(query, results, lang), chosen)
        text = as_sentences(text)
        log.info("answer model: %s", m["id"])
        return {"text": text, "provider": m["provider"], "model": m["id"], "label": m["label"], "chosen": chosen}
    except NoModelAnswered:
        log.info("answer provider: cards")
        return {"text": format_cards(results, lang), "provider": "cards", "model": None, "label": None, "chosen": chosen}


In [46]:
PROVIDER_CHAINS = {"gemini": ["gemini", "ollama", "cards"],
                   "ollama": ["ollama", "cards"],
                   "none": ["cards"]}


def answer_with_llm(query, results, provider=None, model=None):
    """Returns (answer_text, provider_used), as before. provider='none' = plain cards; 'gemini' / 'ollama' = the
    first model of that provider in the list. model = a model id from LLM_MODELS (wins over provider)."""
    if model is None and provider:
        p = provider.lower()
        model = NO_LLM if p == NO_LLM else next((m["id"] for m in MODELS if m["provider"] == p), None)
    out = generate_answer(query, results, model)
    return out["text"], out["provider"]


### Conversation: what does a message need?

**What:** every chat message is one of three things. A **follow-up** asks about the places already shown ("is the first
one open?", "menu of Lucky Burger"), a **search** asks for other places ("cheaper ones?", "what about Siem Reap?"),
and **chat** is small talk ("hi", "who are you?").

**How:** clear cases are decided by word rules (no AI call, instant): a greeting, "number 2", part of a place's name
("lucky burger"), or "it" right after an answer about one place. Unclear ones go to the AI model, which returns JSON.
A new search keeps what the user did not change ("cheaper ones" after "pizza near me" -> "pizza cheap near me").

In [ ]:
# ---------- conversation: follow-up questions about the places already shown ----------
# A chat turn is one of three things, decided by plan_turn():
#   followup  a question about the places shown last ("what's on the menu?", "is the first one open?"): answered from
#             those rows only, without a new search
#   search    a request for other places ("cheaper ones?", "pizza instead"): rewritten into a standalone search query
#             that keeps the earlier wishes ("cheap coffee near me"), then the normal search + answer
#   chat      greetings, thanks, small talk
HISTORY_TURNS = 6          # earlier messages (user + assistant) the model sees


CONTEXT_PLACES = 10        # places shown most recently that a follow-up can refer to


PLAN_TIMEOUT_S = 10        # the "decide" step makes one quick attempt (Gemini's minimum); if it is slow or busy, word rules decide


CHAT_RE = re.compile(r"^\s*(hi+|hello|hey|yo|thanks?|thank you|thx|ok(ay)?|cool|great|nice|bye|good (morning|afternoon|"
                     r"evening|night)|សួស្តី|ជំរាបសួរ|អរគុណ|បាទ|ចាស|អូខេ|លា)\b[\s!.?😀-🙏]*$", re.I)


FOLLOWUP_RE = re.compile(r"\b(it|its|it's|that|this|those|these|they|them|their|there|first|second|third|last|"
                         r"closest|nearest|number\s*\d|no\.?\s*\d|menu|menus|dishes|serve|serves|open|opens|close|"
                         r"closes|closed|hours?|phone|call|contact|far|distance|address|where|located|facebook|page|"
                         r"website|link|price|prices|cost|how much|rating|rated|stars?|deliver|delivery)\b|#\d|"
                         r"ម៉ឺនុយ|បើក|បិទ|ម៉ោង|ទូរស័ព្ទ|ឆ្ងាយ|ទីតាំង|តម្លៃ|ហាងនេះ|ហាងនោះ|ទីមួយ|ទីពីរ|ទីបី|មួយណា|ណាមួយ|ជាងគេ", re.I)


NEW_SEARCH_RE = re.compile(r"\b(other|another|else|instead|more|different|cheaper|cheap|any\s+\w+\s+(near|in)|near\s+\w+|"
                           r"in\s+[a-z]+\s+[a-z]+)\b|ផ្សេង|ទៀត", re.I)


ORDINALS = {"first": 1, "1st": 1, "second": 2, "2nd": 2, "third": 3, "3rd": 3, "fourth": 4, "4th": 4, "fifth": 5,
            "5th": 5, "ទីមួយ": 1, "ទីពីរ": 2, "ទីបី": 3, "ទីបួន": 4, "ទីប្រាំ": 5}


FILLER_RE = re.compile(r"^\s*(show me|find me|find|search for|what about|how about|i want|i'd like|give me|any)\s+|"
                       r"\s+(instead|please|then)\s*[?.!]*$", re.I)


ABOUT_RE = re.compile(r"\b(who are you|what are you|what can you do|what do you do|how (do|does) (this|it|you) work|"
                      r"your name)\b|^\s*help\s*[?!.]*$|អ្នកជានរណា|អ្នកអាចធ្វើអ្វី", re.I)


HUNGRY_RE = re.compile(r"^\s*(i'?m|i am|im)\s+(so |very |really )?(hungry|starving)\b|^\s*(so |very )?(hungry|starving)\s*[!.?]*$|\bwhat (should|can|do) i eat\b|"
                       r"\b(don'?t|do not) know what to eat\b|\bany (food )?(ideas|suggestions)\b|^\s*ឃ្លាន|ញ៉ាំអីល្អ|ញ៉ាំអ្វីល្អ", re.I)


OFF_TOPIC_RE = re.compile(r"\b(weather|poem|story|joke|song|lyrics|homework|translate|news|politics|essay|math|"
                          r"programming|write (me )?(a|an))\b", re.I)


CHAT_FALLBACK = {"en": "Happy to help! Tell me a dish, an area or a time, for example “kuy teav near me”.",
                 "kh": "រីករាយជួយ! ប្រាប់ខ្ញុំពីម្ហូប តំបន់ ឬម៉ោង ឧ. “គុយទាវ ជិតខ្ញុំ”។"}


NOT_IN_DATA = "not in our data"


def history_text(history) -> str:
    lines = [f"{h['role']}: {' '.join(str(h['text']).split())[:300]}" for h in (history or [])[-HISTORY_TURNS:]]
    return "\n".join(lines) or "(this is the first message)"


NUMBER_LIST_RE = re.compile(r"(?<![\d$.:])\b\d{1,2}(?:\s*(?:,|and|&|or|vs\.?|versus|និង|ឬ)\s*\d{1,2}\b(?![.:$%]|\s*(?:km|m|usd|\$|"
                            r"dollars?|riel|h|hours?|min)))+")


NAME_WORDS_RE = re.compile(r"[a-z0-9\u1780-\u17ff']+")


NAME_STOP = {"the", "and", "restaurant", "restaurants", "cafe", "café", "coffee", "shop", "house", "kitchen", "food",
             "bar", "grill", "bbq", "pizza", "pizzeria", "burger", "burgers", "noodle", "noodles", "bakery", "tea",
             "store", "phnom", "penh", "siem", "reap", "branch", "mall", "aeon", "street", "delivery", "express",
             "new", "best", "place", "menu", "one", "for", "with", "show", "what", "about"}


def named_by_part(message, places) -> list:
    """Shown places named by a distinctive part of their name: 'lucky burger' -> '3. Lucky Burger Big Tree'.
    Words most of the shown names share ('pizza' after a pizza search) do not count; the best match wins."""
    said = set(NAME_WORDS_RE.findall(message.lower()))
    names = [[w for w in NAME_WORDS_RE.findall(str(n).lower()) if len(w) >= 3] for n in places["name"]]
    common = {w for w in set().union(*map(set, names)) if len(names) >= 3 and sum(w in n for n in names) * 2 >= len(names)}
    scores = {}
    for i, words in enumerate(names, 1):
        key = [w for w in words if w not in NAME_STOP and w not in common]
        if any(w in said for w in key):                      # at least one distinctive word of the name
            scores[i] = sum(w in said for w in set(words))
    best = max(scores.values(), default=0)
    return [i for i, sc in scores.items() if sc == best]


def mentioned_places(message, places) -> list:
    """1-based numbers of the shown places the message names: by name (or a distinctive part of it), 'number 2',
    '#2', 'the first', 'the last'."""
    low, picked = message.lower(), []
    for i, name in enumerate(places["name"], 1):
        if name and len(name) > 2 and name.lower() in low:
            picked.append(i)
    if not picked:
        picked.extend(named_by_part(message, places))
    for m in re.finditer(r"(?:number|no\.?|#|លេខ|ទី)\s*(\d+)", low):
        picked.append(int(m.group(1)))
    for m in NUMBER_LIST_RE.finditer(low):               # "compare 1 and 2", "1, 3 or 4", "1 vs 2"
        picked.extend(int(x) for x in re.findall(r"\d+", m.group(0)))
    m = (re.search(r"\b(?:of|about|for|is|at)\s+(\d{1,2})\s*[?.!]*$", low)    # "what's the phone of 2?"
         or re.search(r"\b(?:is|does|do|are)\s+(\d{1,2})\s+(?!km|m\b|usd|dollars?|riel|hours?|min)\w", low))  # "is 3 open?"
    if m:
        picked.append(int(m.group(1)))
    for word, n in ORDINALS.items():
        khmer = not word.isascii()                      # Khmer has no spaces: "ហាងទីពីរ" = "the second place"
        if (word in low) if khmer else re.search(rf"(?<!\w){re.escape(word)}(?!\w)", low):
            picked.append(n)
    if re.search(r"\blast (one|place)\b", low):
        picked.append(len(places))
    return [n for n in dict.fromkeys(picked) if 1 <= n <= len(places)]


GENERIC_WORDS = {"one", "ones", "place", "places", "spot", "spots", "something", "option", "options", "more", "other",
                 "others", "another", "else", "any", "some", "better", "closer", "nearer", "different", "the", "a",
                 "what", "how", "about", "instead", "there", "too", "also", "then", "and", "or", "in", "at",
                 "ចុះ", "វិញ", "ដែរ", "នៅ", "អីចឹង",
                 "ផ្សេង", "ទៀត", "ផ្សេងទៀត", "ហាង", "កន្លែង", "ហាងផ្សេង", "ហាងផ្សេងទៀត"}


def carry_context(query, last_query) -> str:
    """Keeps what the user is looking for when a new search does not say it again:
    'cheaper ones' after 'pizza near me' -> 'pizza cheap near me'; after 'pizza in Toul Kork' -> '... in Toul Kork'."""
    if not last_query:
        return query
    query = re.sub(r"\bcheaper\b", "cheap", query, flags=re.I)        # "cheap" is a filter the search understands
    text, _ = parse_query_filters(query)
    if not [w for w in text.split() if w.strip("?.!,") not in GENERIC_WORDS]:     # no dish of its own
        words = [w for w in query.split() if w.lower().strip("?.!,") not in GENERIC_WORDS]
        query = " ".join([parse_query_filters(last_query)[0], *words]).strip()
    _, new = parse_query_filters(query)
    _, old = parse_query_filters(last_query)
    if new.get("near") or new.get("area") or new.get("city"):
        return query
    if old.get("near") == "me":
        return f"{query} near me"
    if old.get("near"):
        i = last_query.lower().rfind(old["near"].lower())
        return f"{query} near {last_query[i:i + len(old['near'])] if i >= 0 else old['near']}"
    if old.get("area") or old.get("city"):
        return f"{query} in {old.get('area') or old['city']}"
    return query


def heuristic_plan(message, places) -> dict:
    """The plan without an AI model (none chosen, or all busy): word rules."""
    if CHAT_RE.match(message) or ABOUT_RE.search(message) or HUNGRY_RE.search(message) or OFF_TOPIC_RE.search(message):
        return {"action": "chat", "query": "", "places": []}       # small talk, "I'm hungry", "write me a poem"
    if places is not None and len(places):
        picked = mentioned_places(message, places)
        about_all = SUPERLATIVE_RE.search(message) or COMPARE_RE.search(message)       # "which one is cheapest?"
        if picked or ((FOLLOWUP_RE.search(message) or about_all) and not NEW_SEARCH_RE.search(message)):
            return {"action": "followup", "query": "", "places": picked}
    return {"action": "search", "query": FILLER_RE.sub(" ", message).strip() or message, "places": []}


def focus_from_history(history, places) -> list:
    """The place(s) the last answer was about, so 'is it open?' / 'and its phone?' stay on them: the numbered names
    ('2. Papa's burger') in the last assistant message. [] after a list of results (more than 2 places named)."""
    last = next((h["text"] for h in reversed(history or []) if h.get("role") == "assistant"), "")
    found = [i for i, name in enumerate(places["name"], 1) if name and f"{i}. {str(name)[:24]}" in last]
    return found if 1 <= len(found) <= 2 else []


def build_plan_prompt(message, history, places) -> str:
    shown = ("\n".join(f"{i}. {r.name} ({', '.join(x for x in (r.dish_en, r.area) if x)})"
                       for i, r in enumerate(places.itertuples(), 1)) if places is not None and len(places)
             else "(none yet)")
    return (
        "You route messages in a food-guide chat for Cambodia.\n\n"
        f"CONVERSATION SO FAR (oldest first):\n{history_text(history)}\n\n"
        f"PLACES SHOWN TO THE USER (numbered as the user saw them):\n{shown}\n\n"
        f"NEW MESSAGE: {message}\n\n"
        "Decide what the new message needs:\n"
        '- "followup": it asks about the places already shown: their food, price, menu, opening hours, whether they are '
        "open, distance, phone, order link, Facebook page or website, location, or which of them is best, closest or open; or it "
        'refers to them ("it", "that one", "the first", "number 2", or a name from the list).\n'
        '- "search": it asks for other or new places: another dish, another area or province, a new condition ("cheaper", '
        '"open late", "near CADT", "in Siem Reap"), "something else" or "more options".\n'
        '- "chat": a greeting, thanks, small talk, or a question about you.\n'
        'For "search", write "query": a short standalone search query in the language of the new message that keeps '
        'the earlier wishes the user did not change (dish, area, "near me", open now). Example: after "coffee near me", '
        '"any open now?" becomes "coffee open now near me". For the other actions "query" is "".\n'
        'For "followup", "places": the numbers of the places the message is about, or [] if it is about all of them '
        "or it is unclear.\n"
        'Reply with JSON only: {"action": "followup" | "search" | "chat", "query": "...", "places": [1]}'
    )


def plan_turn(message, history=None, places=None, model_id=None, last_query=None) -> dict:
    """What a chat message needs: {action: followup | search | chat, query, places (1-based numbers)}.
    places = the rows shown last (a DataFrame, in the order the user saw them), or None;
    last_query = the previous search, whose place or area a new search keeps (carry_context)."""
    plan = _plan(message, history, places, model_id)
    if plan["action"] == "search":                   # in a conversation, keep where the user is looking
        plan["query"] = re.sub(r"\s*\b(in )?cambodia\b|\s*(នៅ)?កម្ពុជា", "", plan["query"], flags=re.I).strip() or message
        plan["query"] = carry_context(plan["query"], last_query)
    return plan


def _plan(message, history, places, model_id) -> dict:
    if places is None or len(places) == 0:          # nothing to refer back to: a search (or small talk)
        quick = heuristic_plan(message, None)
        if quick["action"] == "chat" or about_food(quick["query"]):
            return quick                             # "pizza", "hi": no AI call
    else:
        quick = heuristic_plan(message, places)
        if quick["action"] == "chat" or (quick["action"] == "followup" and quick["places"]):
            return quick                             # obvious (a greeting, or "number 2" / a place's name): no AI call
        if quick["action"] == "followup" and (WHICH_RE.search(message) or SUPERLATIVE_RE.search(message)
                                              or COMPARE_RE.search(message) or LIST_RE.search(message)):
            return quick                             # about all the places shown
        if quick["action"] == "followup":            # "is it open?" right after an answer about one place
            focus = focus_from_history(history, places)
            if focus and not (WHICH_RE.search(message) or COMPARE_RE.search(message) or LIST_RE.search(message)):
                return {**quick, "places": focus}
    try:
        text, _ = complete(build_plan_prompt(message, history, places), model_id, json_mode=True,
                           max_models=1, timeout_s=PLAN_TIMEOUT_S)
        plan = json.loads(text[text.find("{"):text.rfind("}") + 1])
        action = plan.get("action")
        if action not in ("followup", "search", "chat"):
            raise ValueError(f"unknown action {action!r}")
        if action == "followup" and (places is None or len(places) == 0):
            action = "search"                        # nothing shown yet to follow up on
        query = " ".join(str(plan.get("query") or "").split())[:200]
        n_shown = 0 if places is None else len(places)
        nums = [int(n) for n in plan.get("places") or [] if str(n).isdigit() and 1 <= int(n) <= n_shown]
        if action == "followup":                     # trust explicit mentions over the model's guess
            nums = mentioned_places(message, places) or nums
        return {"action": action, "query": query or message if action == "search" else "", "places": nums}
    except (NoModelAnswered, ValueError, TypeError, json.JSONDecodeError) as e:
        if not isinstance(e, NoModelAnswered):
            log.warning("plan_turn: unreadable plan (%s) -> word rules", safe_error(e))
        return heuristic_plan(message, places)


def place_facts(n, r, hour_now) -> str:
    """Everything our data knows about one place, and what it does not know (so the model never fills a gap)."""
    name_val = r["name"] if isinstance(r, (pd.Series, dict)) else getattr(r, "name", "")
    dish_en = r["dish_en"] if isinstance(r, (pd.Series, dict)) else getattr(r, "dish_en", "")
    dish_kh = r["dish_kh"] if (isinstance(r, (pd.Series, dict)) and "dish_kh" in r) else getattr(r, "dish_kh", "")
    food = ", ".join(x for x in (dish_en, dish_kh) if x) or NOT_IN_DATA
    tags = r["tags"] if isinstance(r, (pd.Series, dict)) else getattr(r, "tags", "")
    kinds = ", ".join(t.strip() for t in str(tags or "").split(";") if t.strip()) or NOT_IN_DATA
    d = r.get("distance_km", np.nan) if isinstance(r, (pd.Series, dict)) else getattr(r, "distance_km", np.nan)
    open_h = r["open_hour"] if isinstance(r, (pd.Series, dict)) else getattr(r, "open_hour", np.nan)
    close_h = r["close_hour"] if isinstance(r, (pd.Series, dict)) else getattr(r, "close_hour", np.nan)
    price_val = r["price_usd"] if isinstance(r, (pd.Series, dict)) else getattr(r, "price_usd", np.nan)
    has_hours = pd.notna(open_h) and pd.notna(close_h)
    hours = (f"{fmt_hours(open_h, close_h)} (open now: {'yes' if is_open_at(open_h, close_h, hour_now) else 'no'})"
             if has_hours else NOT_IN_DATA)
    page = str(r["page_url"] if isinstance(r, (pd.Series, dict)) else getattr(r, "page_url", "") or "")
    phone = str(r["phone_public"] if isinstance(r, (pd.Series, dict)) else getattr(r, "phone_public", "") or "").strip()
    desc = str(r["description"] if isinstance(r, (pd.Series, dict)) else getattr(r, "description", "") or "").strip()
    area_val = r["area"] if isinstance(r, (pd.Series, dict)) else getattr(r, "area", "")
    menu_raw = r.get("menu", "") if isinstance(r, (pd.Series, dict)) else getattr(r, "menu", "")
    menu_str = ""
    if menu_raw and isinstance(menu_raw, str):
        try:
            import json as _json
            items = _json.loads(menu_raw)
            if isinstance(items, list):
                menu_str = ", ".join(f"{it.get('name_en')} (${float(it.get('price_usd', 0)):.2f})" for it in items)
        except Exception:
            menu_str = ""
    return (f"{n}. name: {name_val} | food: {food} | type: {kinds} | area: {area_val or NOT_IN_DATA}"
            f" | distance: {'unknown' if pd.isna(d) else (f'{round(d * 100) * 10:.0f} m' if d < 1 else f'{d:.1f} km')}"
            f" | opening hours: {hours} | price: {fmt_price(price_val) if pd.notna(price_val) else NOT_IN_DATA}"
            f" | menu: {menu_str or NOT_IN_DATA}"
            f" | phone: {phone if phone and phone.lower() != 'nan' else NOT_IN_DATA}"
            f" | link: {page if page else NOT_IN_DATA}"
            f" | notes: {desc or NOT_IN_DATA}")


def _hour_now():
    now = datetime.now(TZ)
    return now.hour + now.minute / 60


### Answer shapes: a sentence, a list or a table

**What:** each follow-up gets the form that fits it, with only what it asks.

| Question | Shape |
|---|---|
| one detail of one place ("is it open now?", "phone of 2?") | one sentence, answer first |
| one detail of several places, or a menu | a list |
| a comparison, or several details of several places | a table (side by side for 2-3 places) |
| "which one is cheapest?" | one sentence from the AI model |

**Why:** lists, tables and one-detail sentences are built here from the rows, never written by a model, so their
values cannot be made up, and they work without AI. Missing values are said plainly ("Our data has no phone number
for 1. Costa").

In [ ]:
# ---------- answer shapes: every question gets the form that fits it, with only what it asks ----------
#   direct  one fact about one place, a yes/no, or "which one ...?"   -> one sentence, the answer first (AI)
#   list    one fact about several places, or a menu                   -> one line per place / per dish (from the data)
#   table   a comparison, or several facts about several places        -> only the asked columns (from the data)
# Lists and tables are built here from the rows, never written by a model, so their values cannot be made up.
FIELD_RES = {      # what a question can ask about a place (checked in this order = the column order)
    "food": r"\b(food|serve|serves|sell|sells|kind of|type of|cuisine)\b|ម្ហូបអ្វី|លក់អ្វី|ប្រភេទ",
    "menu": r"\b(menu|menus|dish|dishes)\b|ម៉ឺនុយ|មុខម្ហូប",
    "price": r"\b(price|prices|cost|costs|how much|cheap|cheaper|cheapest|expensive|budget)\b|តម្លៃ|ថ្លៃ|ថោក",
    "hours": r"\b(open|opens|opening|close|closes|closing|closed|hours?|what time)\b|បើក|បិទ|ម៉ោង",
    "distance": r"\b(far|distance|closest|nearest|how long)\b|ឆ្ងាយ|ចម្ងាយ|ជិតជាងគេ",
    "area": r"\b(where|address|located|location|which area|which khan)\b|នៅឯណា|ទីតាំង|អាសយដ្ឋាន",
    "phone": r"\b(phone|call|telephone|contact)\b|ទូរស័ព្ទ|លេខទូរស័ព្ទ",
    "link": r"\b(facebook|fb|page|website|site|link|order online)\b|ហ្វេសប៊ុក|គេហទំព័រ",
    "rating": r"\b(rating|ratings|rated|stars?|reviews?)\b|ផ្កាយ|ការវាយតម្លៃ",
}


FIELD_RES = {k: re.compile(v, re.I) for k, v in FIELD_RES.items()}


DEFAULT_FIELDS = ["food", "area", "price", "hours", "distance"]      # "compare 1 and 2" without saying what


COMPARE_RE = re.compile(r"\b(compare|comparison|vs\.?|versus|difference|differences|differ|side by side|table)\b|"
                        r"ប្រៀបធៀប|ខុសគ្នា|តារាង", re.I)


LIST_RE = re.compile(r"\b(list|all of them|each of them|each one|every one|them all|all|both)\b|ទាំងអស់|ទាំងពីរ|រាយ", re.I)


SUPERLATIVE_RE = re.compile(r"\b(cheapest|closest|nearest|best rated|highest rated|top rated|most expensive|priciest)\b|"
                            r"ថោកជាងគេ|ជិតជាងគេ|ថ្លៃជាងគេ", re.I)


WHICH_RE = re.compile(r"\b(which|who has|is there one|are any|any of them)\b|មួយណា|ណាមួយ|ហាងណា", re.I)


LABELS = {
    "en": {"place": "Place", "food": "Food", "menu": "Menu", "price": "Price", "hours": "Hours", "distance": "Distance",
           "area": "Area", "phone": "Phone", "link": "Page", "rating": "Rating", "unknown": "not in our data",
           "open": "open now", "closed": "closed now", "dash": "— = not in our data",
           "none": "Sorry, our data has no {what} for {places}.", "menu_of": "menu",
           "missing": "Our data has no {what} for these places.",
           "these": "these {n} places", "gaps": "Not in our data for number {nums}.",
           "which": "Which place's menu? Send its number or name, e.g. “menu of 2”."},
    "kh": {"place": "ហាង", "food": "ម្ហូប", "menu": "ម៉ឺនុយ", "price": "តម្លៃ", "hours": "ម៉ោងបើក", "distance": "ចម្ងាយ",
           "area": "ទីតាំង", "phone": "ទូរស័ព្ទ", "link": "ទំព័រ", "rating": "ការវាយតម្លៃ", "unknown": "មិនមានក្នុងទិន្នន័យ",
           "open": "កំពុងបើក", "closed": "បិទហើយ", "dash": "— = មិនមានក្នុងទិន្នន័យ",
           "none": "សូមអភ័យទោស ទិន្នន័យរបស់យើងមិនមាន{what}សម្រាប់ {places} ទេ។", "menu_of": "ម៉ឺនុយ",
           "missing": "ទិន្នន័យរបស់យើងមិនមាន{what}សម្រាប់ហាងទាំងនេះទេ។",
           "these": "ហាងទាំង {n} នេះ", "gaps": "មិនមានក្នុងទិន្នន័យសម្រាប់លេខ {nums}។",
           "which": "ចង់មើលម៉ឺនុយហាងណា? សូមផ្ញើលេខ ឬឈ្មោះហាង ឧ. “ម៉ឺនុយលេខ 2”។"},
}


def asked_fields(message) -> list:
    """The place details a question asks for, e.g. 'phone and hours?' -> ['hours', 'phone']."""
    return [f for f, rx in FIELD_RES.items() if rx.search(message)]


def answer_shape(message, n_places, fields) -> str:
    """direct | list | table (see above). n_places = how many places the question is about."""
    if n_places >= 2 and COMPARE_RE.search(message):
        return "table"
    if WHICH_RE.search(message) or n_places == 0:
        return "direct"                       # "which one is open now?": the model picks and says why
    if "menu" in fields:
        return "list"
    if n_places >= 2 and len(fields) >= 2:
        return "table"
    if n_places >= 2 and (fields or LIST_RE.search(message)):
        return "list"
    return "direct"


def _val(r, col):
    v = r.get(col, "") if isinstance(r, (pd.Series, dict)) else getattr(r, col, "")
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip().lower() == "nan" else v


def menu_items(r) -> list:
    raw = _val(r, "menu")
    try:
        items = json.loads(raw) if isinstance(raw, str) and raw.strip().startswith("[") else []
    except ValueError:
        return []
    return [it for it in items if isinstance(it, dict) and (it.get("name_en") or it.get("name_kh"))]


def field_value(r, field, lang, hour_now) -> str:
    """One detail of one place as short text, or '' when our data does not have it."""
    L = LABELS[lang]
    if field == "food":
        food = (_val(r, "dish_kh") if lang == "kh" else "") or _val(r, "dish_en")
        food = str(food).replace(";", ",").replace("_", " ").strip()
        return food[:1].upper() + food[1:]
    if field == "price":
        p = _val(r, "price_usd")
        return fmt_price(p) if p != "" else ""
    if field == "hours":
        o, c = _val(r, "open_hour"), _val(r, "close_hour")
        if o == "" or c == "":
            return ""
        return f"{fmt_hours(o, c)} ({L['open'] if is_open_at(o, c, hour_now) else L['closed']})"
    if field == "distance":
        d = _val(r, "distance_km")
        return "" if d == "" else (f"{round(d * 100) * 10:.0f} m" if d < 1 else f"{d:.1f} km")
    if field == "area":
        area, city = _val(r, "area"), _val(r, "city")
        if lang == "kh":
            area, city = PLACE_NAMES_KH.get(area, area), PLACE_NAMES_KH.get(city, city)
        return ", ".join(dict.fromkeys(x for x in (area, city) if x))
    if field == "phone":
        return str(_val(r, "phone_public")).strip()
    if field == "link":
        url = str(_val(r, "page_url")).strip()
        return f"[{'Facebook' if 'facebook.com' in url else 'Website'}]({url})" if url.startswith("http") else ""
    if field == "rating":
        v = _val(r, "rating")
        return f"⭐ {float(v):.1f}" if v != "" else ""
    if field == "menu":
        return ", ".join(menu_line(it, lang) for it in menu_items(r))
    return ""


def menu_line(item, lang) -> str:
    name = (item.get("name_kh") if lang == "kh" else "") or item.get("name_en") or item.get("name_kh")
    price = item.get("price_usd")
    return f"{name} — {fmt_price(price)}" if price not in (None, "") else str(name)


def _cell(text) -> str:
    return str(text).replace("|", "/").replace("\n", " ") or "—"


def facts_answer(shape, fields, rows, numbers, lang, hour_now):
    """The list or table for `rows` (numbered as shown), with only `fields`. None when the data has none of it."""
    L = LABELS[lang]
    fields = fields or DEFAULT_FIELDS
    values = {n: {f: field_value(r, f, lang, hour_now) for f in fields} for n, r in zip(numbers, rows)}
    asked_known = [f for f in fields if any(values[n][f] for n in numbers)]
    if not asked_known:                       # nothing of what was asked is known: say so in one sentence
        what = ", ".join(L[f].lower() if lang == "en" else L[f] for f in fields)
        places = (", ".join(f"{n}. {_val(r, 'name')}" for n, r in zip(numbers, rows)) if len(numbers) <= 3
                  else L["these"].format(n=len(numbers)))
        return L["none"].format(what=what, places=places)
    if shape == "list" and "menu" in fields:  # one bullet per dish, under each place
        blocks = []
        for n, r in zip(numbers, rows):
            items = menu_items(r)
            head = f"**{n}. {_val(r, 'name')}**"
            other = [f"{L[f]}: {values[n][f]}" for f in fields if f != "menu" and values[n][f]]
            if other:
                head += " — " + " · ".join(other)
            blocks.append(head + ("\n" + "\n".join(f"- {menu_line(it, lang)}" for it in items[:12])
                                  if items else f" — {L['menu_of']}: {L['unknown']}"))
        return "\n\n".join(blocks)
    if shape == "list":                       # the places that have it; the others in one line, not one each
        lines, gaps = [], []
        for n, r in zip(numbers, rows):
            known = [values[n][f] for f in fields if values[n][f]]
            if known:
                lines.append(f"- **{n}. {_val(r, 'name')}** — " + " · ".join(known))
            else:
                gaps.append(n)
        if len(gaps) == 1 and len(numbers) <= 3:
            n = gaps[0]
            lines.append(f"- **{n}. {_val(rows[numbers.index(n)], 'name')}** — {L['unknown']}")
        elif gaps:
            lines.append("\n" + L["gaps"].format(nums=", ".join(map(str, gaps))))
        return "\n".join(lines)
    cols = [f for f in asked_known if f != "menu"] or asked_known
    if len(numbers) <= 3:                     # side by side: one column per place, one row per detail (fits a phone)
        head = ["", *[f"{n}. {_val(r, 'name')}" for n, r in zip(numbers, rows)]]
        lines = ["| " + " | ".join(_cell(h) if h else "" for h in head) + " |", "|" + "---|" * len(head)]
        for f in cols:
            lines.append("| " + " | ".join([L[f], *[_cell(values[n][f]) for n in numbers]]) + " |")
    else:                                     # one row per place
        head = ["#", L["place"], *[L[f] for f in cols]]
        lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
        for n, r in zip(numbers, rows):
            lines.append("| " + " | ".join([str(n), _cell(_val(r, "name")), *[_cell(values[n][f]) for f in cols]]) + " |")
    notes = [L["dash"]] if any(not values[n][f] for n in numbers for f in cols) else []
    missing = [f for f in fields if f not in asked_known and fields is not DEFAULT_FIELDS]
    if missing:
        notes.append(L["missing"].format(what=", ".join(L[f].lower() if lang == "en" else L[f] for f in missing)))
    return "\n".join(lines) + ("\n\n" + "\n".join(notes) if notes else "")


TELL_RE = re.compile(r"\b(tell me (more )?about|more (info|details) (about|on)|details (of|for|about)|info (on|about))\b|"
                     r"ប្រាប់ខ្ញុំអំពី|ព័ត៌មានអំពី", re.I)


OPEN_Q_RE = re.compile(r"\b(is|are)\b.*\bopen\b|\bopen (now|today|tonight|right now)\b|បើកឥឡូវ|បើក(ទេ|អត់)|"
                       r"បើកនៅ", re.I)


FACT_TEXT = {
    "en": {"open_yes": "Yes, {p} is open now ({v}).", "open_no": "No, {p} is closed now. Its hours are {v}.",
           "hours": "{p} is open {v}.", "price": "{p}: about {v}.", "phone": "{p}: {v}", "distance": "{p} is {v} away.",
           "area": "{p} is in {v}.", "link": "{p}: {v}", "rating": "{p} is rated {v}.", "food": "{p} serves {v}.",
           "unknown": "Our data has no {what} for {p}.",
           "no_location": "I don't know where you are yet. Share your location, or ask again with “near me”, and I'll "
                          "give the distance.",
           "no_xy": "Our data has no map location for {p}, so I can't tell the distance.",
           "try_page": " Their {link} may have it.", "try_phone": " You could call them: {phone}."},
    "kh": {"open_yes": "បាទ {p} កំពុងបើក ({v})។", "open_no": "ទេ {p} បិទហើយ។ ម៉ោងបើក៖ {v}។",
           "hours": "{p} បើក {v}។", "price": "{p}៖ ប្រហែល {v}។", "phone": "{p}៖ {v}", "distance": "{p} ឆ្ងាយ {v}។",
           "area": "{p} នៅ {v}។", "link": "{p}៖ {v}", "rating": "{p} ទទួលបាន {v}។", "food": "{p} លក់ {v}។",
           "unknown": "ទិន្នន័យរបស់យើងមិនមាន{what}សម្រាប់ {p} ទេ។",
           "no_location": "ខ្ញុំមិនទាន់ដឹងថាអ្នកនៅឯណាទេ។ សូមចែករំលែកទីតាំង ឬសួរម្តងទៀតដោយថា “ជិតខ្ញុំ”។",
           "no_xy": "ទិន្នន័យរបស់យើងមិនមានទីតាំងលើផែនទីសម្រាប់ {p} ទេ ដូច្នេះមិនអាចប្រាប់ចម្ងាយបាន។",
           "try_page": " {link} របស់ពួកគេប្រហែលជាមាន។", "try_phone": " អាចទូរស័ព្ទទៅ៖ {phone}។"},
}


def fact_sentence(message, fields, n, r, lang, hour_now, places):
    """One-sentence answer from the data when a question asks one detail of one place ("is it open now?",
    "phone of 2?", "how far is number 3?", "tell me about the first one"). None = let the model answer."""
    F, L = FACT_TEXT[lang], LABELS[lang]
    p = f"{n}. {_val(r, 'name')}"
    if not fields:                                   # "tell me about number 2": its known facts in one line
        return facts_answer("list", [], [r], [n], lang, hour_now) if TELL_RE.search(message) else None
    if len(fields) != 1 or fields[0] == "menu":
        return None
    f = fields[0]
    v = field_value(r, f, lang, hour_now)
    if f == "distance" and not v:
        located = "distance_km" in places.columns and places["distance_km"].notna().any()
        return F["no_xy"].format(p=p) if located else F["no_location"]
    if f == "hours" and v:
        o, c = _val(r, "open_hour"), _val(r, "close_hour")
        if OPEN_Q_RE.search(message):
            return F["open_yes" if is_open_at(o, c, hour_now) else "open_no"].format(p=p, v=fmt_hours(o, c))
        return F["hours"].format(p=p, v=v)
    if v:
        return F[f].format(p=p, v=v)
    what = L[f].lower() if lang == "en" else L[f]
    text = F["unknown"].format(what=what, p=p)
    page, phone = field_value(r, "link", lang, hour_now), field_value(r, "phone", lang, hour_now)
    if f != "link" and page:
        text += F["try_page"].format(link=page)
    elif f not in ("phone", "link") and phone:
        text += F["try_phone"].format(phone=phone)
    return text


EXPENSIVE_RE = re.compile(r"\b(most expensive|priciest|highest price)\b|ថ្លៃជាងគេ", re.I)


WHICH_TEXT = {
    "en": {"price_min": "{w} {be} the cheapest, about {v}.", "price_max": "{w} {be} the most expensive, about {v}.",
           "price_all": "They all cost about {v}.", "distance": "{w} {be} the closest, {v} away.",
           "rating": "{w} {be} the best rated, {v}.", "rating_all": "They are all rated {v}.",
           "many": "{k} places tie at {v}: number {nums}.", "open": "Open now: {w}.",
           "open_many": "{k} of them are open now: number {nums}.",
           "open_none": "None of them is open now (opening hours known for {k} of {n}).",
           "has": "{w} {have} a {what} in our data.", "has_all": "All of them have a {what} in our data.",
           "has_many": "{k} of them have a {what} in our data: number {nums}.", "is": "is", "are": "are",
           "have1": "has", "have2": "have"},
    "kh": {"price_min": "{w} ថោកជាងគេ ប្រហែល {v}។", "price_max": "{w} ថ្លៃជាងគេ ប្រហែល {v}។",
           "price_all": "ហាងទាំងអស់មានតម្លៃប្រហែល {v}។", "distance": "{w} ជិតជាងគេ ចម្ងាយ {v}។",
           "rating": "{w} ទទួលបានការវាយតម្លៃខ្ពស់ជាងគេ {v}។", "rating_all": "ហាងទាំងអស់ទទួលបាន {v}។",
           "many": "ហាង {k} កន្លែងស្មើគ្នា {v}៖ លេខ {nums}។", "open": "កំពុងបើក៖ {w}។",
           "open_many": "ហាង {k} កន្លែងកំពុងបើក៖ លេខ {nums}។",
           "open_none": "គ្មានហាងណាកំពុងបើកទេ (ដឹងម៉ោងបើក {k} ក្នុងចំណោម {n})។",
           "has": "{w} មាន{what}ក្នុងទិន្នន័យរបស់យើង។", "has_all": "ហាងទាំងអស់មាន{what}ក្នុងទិន្នន័យរបស់យើង។",
           "has_many": "ហាង {k} កន្លែងមាន{what}៖ លេខ {nums}។", "is": "", "are": "", "have1": "", "have2": ""},
}


def which_answer(message, fields, rows, numbers, lang, hour_now, places):
    """'Which one is cheapest / closest / best rated / open now / has a phone?' answered from the data in one
    sentence. Ties are summed up ('They all cost about $2.50') instead of listing ten places. None = not such a question."""
    if len(fields) != 1 or fields[0] not in ("price", "distance", "rating", "hours", "phone", "link"):
        return None
    f, W, L = fields[0], WHICH_TEXT[lang], LABELS[lang]
    named = lambda ns: (" " if lang == "kh" else ", ").join(f"{n}. {_val(rows[numbers.index(n)], 'name')}" for n in ns)
    be = lambda ns: W["is"] if len(ns) == 1 else W["are"]
    if f == "hours":
        known = [n for n, r in zip(numbers, rows) if _val(r, "open_hour") != "" and _val(r, "close_hour") != ""]
        if not known:
            return LABELS[lang]["none"].format(what=L["hours"].lower() if lang == "en" else L["hours"],
                                               places=L["these"].format(n=len(numbers)))
        open_now = [n for n in known if is_open_at(_val(rows[numbers.index(n)], "open_hour"),
                                                   _val(rows[numbers.index(n)], "close_hour"), hour_now)]
        if not open_now:
            return W["open_none"].format(k=len(known), n=len(numbers))
        return (W["open"].format(w=named(open_now)) if len(open_now) <= 3
                else W["open_many"].format(k=len(open_now), nums=", ".join(map(str, open_now))))
    if f in ("phone", "link"):
        has = [n for n, r in zip(numbers, rows) if field_value(r, f, lang, hour_now)]
        what = {"phone": "phone number" if lang == "en" else "លេខទូរស័ព្ទ",
                "link": "Facebook page or website" if lang == "en" else "ហ្វេសប៊ុក ឬគេហទំព័រ"}[f]
        if not has:
            return LABELS[lang]["none"].format(what=what, places=L["these"].format(n=len(numbers)))
        if len(has) == len(numbers) and len(has) > 1:
            return W["has_all"].format(what=what)
        if len(has) > 3:
            return W["has_many"].format(k=len(has), what=what, nums=", ".join(map(str, has)))
        return W["has"].format(w=named(has), have=W["have1"] if len(has) == 1 else W["have2"], what=what)
    if f == "distance":
        known = {n: _val(r, "distance_km") for n, r in zip(numbers, rows) if _val(r, "distance_km") != ""}
        if not known:
            return FACT_TEXT[lang]["no_location"]
        best = min(known.values())
        win = [n for n, d in known.items() if d == best]
        return W["distance"].format(w=named(win), be=be(win), v=field_value(rows[numbers.index(win[0])], f, lang, hour_now))
    col = {"price": "price_usd", "rating": "rating"}[f]
    known = {n: float(_val(r, col)) for n, r in zip(numbers, rows) if _val(r, col) != ""}
    if not known:
        return LABELS[lang]["none"].format(what=L[f].lower() if lang == "en" else L[f], places=L["these"].format(n=len(numbers)))
    high = f == "rating" or bool(EXPENSIVE_RE.search(message))
    best = max(known.values()) if high else min(known.values())
    win = [n for n, v in known.items() if v == best]
    v = field_value(rows[numbers.index(win[0])], f, lang, hour_now)
    if len(win) == len(numbers) and len(win) > 1:
        return W["rating_all" if f == "rating" else "price_all"].format(v=v)
    if len(win) > 3:
        return W["many"].format(k=len(win), v=v, nums=", ".join(map(str, win)))
    key = "rating" if f == "rating" else ("price_max" if high else "price_min")
    return W[key].format(w=named(win), be=be(win), v=v)


def answer_followup(message, history, places, numbers=None, model_id=None) -> dict:
    """Answers a question about places already shown, from their rows only (no new search), in the shape that fits
    the question (see 'answer shapes'). places = the rows shown last; numbers = which of them (1-based) it is about,
    [] = all. Returns {text, provider, model, label, numbers, shape}; provider 'facts' = built from the data, no model."""
    lang = detect_lang(message)
    named = [n for n in (numbers or []) if 1 <= n <= len(places)]
    numbers = named or list(range(1, len(places) + 1))
    hour = _hour_now()
    rows = list(places.itertuples())                 # (a pandas Series' .name is its index, not the place's name)
    picked = [rows[n - 1] for n in numbers]
    fields = asked_fields(message)
    shape = answer_shape(message, len(numbers), fields)
    if "menu" in fields and not named and len(numbers) > 1 and not LIST_RE.search(message):
        text = LABELS[lang]["which"] + "\n" + "\n".join(f"- {n}. {_val(r, 'name')}" for n, r in zip(numbers, picked))
        return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": [], "shape": "list"}
    if shape == "direct" and len(numbers) > 1 and (WHICH_RE.search(message) or SUPERLATIVE_RE.search(message)):
        text = which_answer(message, fields, picked, numbers, lang, hour, places)
        if text:
            return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": numbers,
                    "shape": "direct"}
    if shape == "direct" and named and len(numbers) == 1:
        text = fact_sentence(message, fields, numbers[0], picked[0], lang, hour, places)
        if text:
            return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": numbers,
                    "shape": "direct"}
    if shape in ("list", "table"):
        text = facts_answer(shape, fields, picked, numbers, lang, hour)
        return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": numbers, "shape": shape}
    facts = "\n".join(place_facts(n, r, hour) for n, r in zip(numbers, picked))
    prompt = (
        "You are a friendly food guide for Cambodia in a chat. The user asks a follow-up question about "
        "places you already showed them.\n"
        "Rules:\n"
        f"- Answer in {'Khmer' if lang == 'kh' else 'English'} (the language of the question).\n"
        "- Answer exactly what is asked, nothing more: start with the answer itself (yes / no / the value / the "
        "place), in one short sentence (two only if the question has two parts). No greeting, no list, no extra "
        "details the user did not ask for.\n"
        "- Use ONLY the facts in PLACES. When you name a place, write its number and name as in PLACES, e.g. "
        "\"1. Costa\".\n"
        "- If the question asks which place fits (cheapest, closest, open now…), name the place(s) that fit and the "
        "one value that shows it.\n"
        "- If PLACES does not have the fact (\"not in our data\"), say that in one sentence and suggest one next "
        "step (call them, check their page, or search again).\n\n"
        f"CONVERSATION SO FAR:\n{history_text(history)}\n\nPLACES:\n{facts}\n\nQUESTION: {message}"
    )
    try:
        text, m = complete(prompt, model_id)
        return {"text": as_sentences(text), "provider": m["provider"], "model": m["id"], "label": m["label"],
                "numbers": numbers, "shape": "direct"}
    except NoModelAnswered:                          # no AI: the asked facts as a list
        text = facts_answer("list", fields, picked, numbers, lang, hour)
        return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": numbers, "shape": "list"}


### Small talk and questions about the bot

Greetings, thanks and "who are you?" get fixed replies (instant, never off-topic). Other small talk goes to the AI
model, which is told to bring the user back to food: "what's the weather?" -> "I only help find food in Cambodia…".

In [ ]:
CHAT_REPLIES = {    # greetings, thanks and "who are you?" need no AI: instant, always the same, never off-topic
    "en": {"hello": "Hi! 👋 What would you like to eat? Tell me a dish, an area or a time, e.g. “kuy teav near me” "
                    "or “pizza in Siem Reap”.",
           "thanks": "You're welcome! 😊 Anything else you'd like to find?",
           "hungry": "Let's fix that! 🍜 Do you feel like noodles (kuy teav), rice (bai sach chrouk), Khmer BBQ, pizza "
                     "or coffee? Tell me one, and add “near me” to see what's close.",
           "bye": "Bye! Enjoy your meal 🍜",
           "ok": "Great! Ask me anything about these places, or tell me what else you'd like to eat.",
           "about": "I'm Klean Nham, a food guide for Cambodia 🍜. I search our list of {n:,} places: ask for a dish, "
                    "an area or a time (“coffee near me”, “pizza in Siem Reap”, “bbq under $5”), then keep asking "
                    "about the places I show: opening hours, menu, phone, distance, or “compare 1 and 2”."},
    "kh": {"hello": "សួស្តី! 👋 តើអ្នកចង់ញ៉ាំអ្វី? ប្រាប់ខ្ញុំពីម្ហូប តំបន់ ឬម៉ោង ឧ. “គុយទាវ ជិតខ្ញុំ”។",
           "thanks": "មិនអីទេ! 😊 ចង់រកអ្វីទៀតទេ?",
           "hungry": "តោះរកម្ហូប! 🍜 ចង់ញ៉ាំគុយទាវ បាយសាច់ជ្រូក សាច់អាំង ភីហ្សា ឬកាហ្វេ? ប្រាប់ខ្ញុំមួយ ហើយថែម “ជិតខ្ញុំ” "
                     "ដើម្បីមើលហាងនៅជិត។",
           "bye": "លាហើយ! ញ៉ាំឲ្យឆ្ងាញ់ 🍜",
           "ok": "បាទ! សួរខ្ញុំអំពីហាងទាំងនេះ ឬប្រាប់ថាចង់ញ៉ាំអ្វីទៀត។",
           "about": "ខ្ញុំជា ឃ្លានញ៉ាំ ជំនួយការស្វែងរកម្ហូបនៅកម្ពុជា 🍜។ ខ្ញុំស្វែងរកក្នុងបញ្ជីហាង {n:,} កន្លែង៖ "
                    "សួររកម្ហូប តំបន់ ឬម៉ោង (“កាហ្វេ ជិតខ្ញុំ”) រួចសួរបន្តអំពីហាង៖ ម៉ោងបើក ម៉ឺនុយ "
                    "លេខទូរស័ព្ទ ចម្ងាយ ឬ “ប្រៀបធៀប 1 និង 2”។"},
}


CHAT_KINDS = [("about", ABOUT_RE), ("hungry", HUNGRY_RE),
              ("thanks", re.compile(r"^\s*(thanks?|thank you|thx|ty)\b|អរគុណ", re.I)),
              ("bye", re.compile(r"^\s*(bye|goodbye|see you|good night)\b|^\s*លា", re.I)),
              ("hello", re.compile(r"^\s*(hi+|hello|hey|yo|good (morning|afternoon|evening))\b|សួស្តី|ជំរាបសួរ", re.I)),
              ("ok", re.compile(r"^\s*(ok(ay)?|cool|great|nice|got it)\b|អូខេ|បាទ|ចាស", re.I))]


def chat_reply(message, history=None, model_id=None) -> dict:
    """A short reply to greetings, thanks, small talk and off-topic messages. Returns {text, provider, model, label}."""
    lang = detect_lang(message)
    if CHAT_RE.match(message) or ABOUT_RE.search(message) or HUNGRY_RE.search(message):
        kind = next((k for k, rx in CHAT_KINDS if rx.search(message)), "hello")
        return {"text": CHAT_REPLIES[lang][kind].format(n=place_count()), "provider": "facts", "model": None, "label": None}
    prompt = (
        "You are ឃ្លានញ៉ាំ (Klean Nham), a friendly food guide chatbot for Cambodia. You find places to eat "
        "from a list of real places across Cambodia, in English or Khmer.\n"
        f"Reply to the new message in {'Khmer' if lang == 'kh' else 'English'} in one or two short sentences. Do not "
        "recommend or name any place.\n"
        "If the message is not about food or places to eat (weather, news, homework, poems…), say kindly that you only "
        "help find food in Cambodia, and suggest one example question. If it is about being hungry or not knowing "
        "what to eat, suggest two or three kinds of food to try and ask which one.\n\n"
        f"CONVERSATION SO FAR:\n{history_text(history)}\n\nNEW MESSAGE: {message}"
    )
    try:
        text, m = complete(prompt, model_id)
        return {"text": as_sentences(text), "provider": m["provider"], "model": m["id"], "label": m["label"]}
    except NoModelAnswered:
        return {"text": CHAT_FALLBACK[lang], "provider": "cards", "model": None, "label": None}


In [47]:
def check_ollama() -> bool:
    """Is Ollama reachable, and is OLLAMA_MODEL available? Prints a hint if not."""
    cloud = OLLAMA_HOST.startswith("https://")
    where = f"Ollama cloud ({OLLAMA_HOST})" if cloud else f"Ollama at {OLLAMA_HOST}"
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", headers=ollama_headers(), timeout=5)
        if r.status_code == 401:
            print(f"❌ {where} rejected the key. Hint: check OLLAMA_API_KEY in .env (create a new one at ollama.com/settings/keys).")
            return False
        r.raise_for_status()
    except Exception:
        hint = "check your internet connection" if cloud else "open the Ollama app (or run: ollama serve)"
        print(f"❌ {where} is not reachable. Hint: {hint}.")
        return False
    names = [m.get("name", "") for m in r.json().get("models", [])]
    print(f"✅ {where} is reachable. Models: " + (", ".join(names[:12]) + (" …" if len(names) > 12 else "") or "none"))
    if not OLLAMA_MODEL:
        print("ℹ️ OLLAMA_MODEL is not set in .env. Choose a model from the list above" + ("." if cloud else ", then run: ollama pull <model>"))
        return False
    if OLLAMA_MODEL not in names and f"{OLLAMA_MODEL}:latest" not in names:
        print(f"❌ Model {OLLAMA_MODEL!r} is not available. Hint: " +
              ("set OLLAMA_MODEL to one of the models listed above." if cloud else f"run: ollama pull {OLLAMA_MODEL}"))
        return False
    print(f"✅ Model {OLLAMA_MODEL!r} is available.")
    return True


OLLAMA_READY = check_ollama()


❌ Ollama at http://localhost:11434 is not reachable. Hint: open the Ollama app (or run: ollama serve).


In [48]:
# LLM answer vs plain cards, side by side
def compare_row(query):
    res = hybrid_search(query)
    answer, provider = answer_with_llm(query, res)
    cards = format_cards(res, detect_lang(query))
    cell = "<td style='vertical-align:top;text-align:left'><pre style='white-space:pre-wrap'>{}</pre></td>"
    return (f"<tr><td style='vertical-align:top'><b>{html.escape(query)}</b><br><i>answered by: {provider}</i></td>"
            + cell.format(html.escape(answer)) + cell.format(html.escape(cards)) + "</tr>")


rows = "".join(compare_row(q) for q in ["korean food", "ម្ហូបខ្មែរ", "car repair"])
display(HTML("<table><tr><th>Query</th><th>LLM answer</th><th>Plain cards</th></tr>" + rows + "</table>"))
if LLM_PROVIDER != "none" and not GEMINI_API_KEY and not OLLAMA_READY:
    print("ℹ️ No Gemini key and Ollama not ready: the 'LLM answer' column fell back to plain cards.")

INFO food_search: answer provider: gemini


INFO food_search: answer provider: gemini


INFO food_search: answer provider: gemini


Query,LLM answer,Plain cards
korean foodanswered by: gemini,"If you are looking for Korean food in Phnom Penh, here are three options: 1. **Misigga BBQ - Korean BBQ & Hot Pot**: Located in the Sen Sok area. 2. **Tokkiya Modern Korean Comfort Food**: Located in the Boeng Keng Kang area. 3. **Geumdwaeji Korean Food & Pork Grill**: Located in the Toul Kork area.",1. Misigga BBQ - Korean BBQ & Hot Pot — korean food price unknown · ⭐ n/a · Sen Sok · hours unknown 2. Tokkiya Modern Korean Comfort Food — korean restaurant price unknown · ⭐ n/a · Boeng Keng Kang · hours unknown 3. Geumdwaeji Korean Food & Pork Grill — korean restaurant price unknown · ⭐ n/a · Toul Kork · hours unknown 4. Home Korea Food — korean restaurant price unknown · ⭐ n/a · Toul Kork · hours unknown 5. ស្មោះ-SMOS_Korean Street Food — korean restaurant price unknown · ⭐ n/a · Chbar Ampov · hours unknown
ម្ហូបខ្មែរanswered by: gemini,សម្រាប់ជម្រើសអាហារខ្មែរនៅភ្នំពេញ អ្នកអាចពិចារណាភោជនីយដ្ឋានទាំងនេះបាន៖ ១. Winner (ស្ថិតនៅតំបន់មានជ័យ) ២. Mahob Mommy (ស្ថិតនៅតំបន់ពោធិ៍សែនជ័យ បើកលក់ ២៤ ម៉ោង) ៣. Malis (ស្ថិតនៅតំបន់ចំការមន),1. Winner — ម្ហូបខ្មែរ price unknown · ⭐ n/a · Mean Chey · hours unknown 2. Mahob Mommy — ម្ហូបខ្មែរ price unknown · ⭐ n/a · Pou Senchey · open 24 hours 3. Malis — ម្ហូបខ្មែរ price unknown · ⭐ n/a · Chamkar Mon · hours unknown 4. Bopha titanic — ម្ហូបខ្មែរ price unknown · ⭐ n/a · Daun Penh · hours unknown 5. Frizz — ម្ហូបខ្មែរ price unknown · ⭐ n/a · Daun Penh · hours unknown
car repairanswered by: gemini,Nothing matches your request for car repair. The places listed in the results are restaurants or cafes.,1. S.M-Auto car 7777 — cafe price unknown · ⭐ n/a · Toul Kork · hours unknown 2. Ly Heng Car Care Center and Restaurant — restaurant price unknown · ⭐ n/a · Dangkao · hours unknown 3. PSP Car Decor — thai restaurant price unknown · ⭐ n/a · Toul Kork · hours unknown 4. PSP Car Decor — thai restaurant price unknown · ⭐ n/a · Toul Kork · hours unknown 5. Led Car - លេងគុជឡាន — italian restaurant price unknown · ⭐ n/a · Daun Penh · hours unknown


## 8. Telegram bot

**What:** a bot (python-telegram-bot v20+, `async`) that answers:
* text queries → hybrid search → LLM answer (if available) + up to 5 results with price, rating, distance and a Google Maps link
* follow-up questions about those results (section 7): `/new` starts a new conversation, `/model` chooses the AI model
* tables in answers become one line per place, because Telegram messages have no tables
* a shared **location** → kept **in memory only** (`context.user_data`) to add distances
* a **photo** → saved to a temporary file, image search, file deleted right away
* `/start` (English + Khmer help), `/privacy`, `/deleteme`, admin-only `/stats` (ids from `ADMIN_IDS`)
* inline buttons: **Open in Maps** (URL) and **More results**
* a per-user **rate limit** of 10 requests per minute, plus a *typing…* indicator

**Privacy in the logs:** message text is never logged together with user ids, only the language and the length.

Get a token from **@BotFather** on Telegram and put it in `.env` as `TELEGRAM_TOKEN=...`.

In [49]:
from telegram import (InlineKeyboardButton, InlineKeyboardMarkup, KeyboardButton, LinkPreviewOptions,
                      ReplyKeyboardMarkup, ReplyKeyboardRemove)
from telegram.constants import ChatAction, ParseMode
from telegram.ext import Application, CallbackQueryHandler, CommandHandler, MessageHandler
from telegram.ext import filters as tg_filters

In [50]:
def maps_url(lat, lng, name="", city="") -> str:
    """Google Maps link that opens the place itself (its listing with rating, photos and hours), not just a pin.
    The name is what is searched; the coordinates only centre the map, so a chain opens its closest branch."""
    name = re.sub(r"\s+", " ", str(name or "")).strip()
    has_xy = not (pd.isna(lat) or pd.isna(lng))
    if not name:
        return f"https://www.google.com/maps/search/?api=1&query={lat:.6f},{lng:.6f}" if has_xy else "https://www.google.com/maps"
    where = f"{name} {city or HOME_CITY}"
    if not has_xy:
        return "https://www.google.com/maps/search/?api=1&query=" + quote_plus(where)
    return f"https://www.google.com/maps/search/{quote(where, safe='')}/@{lat:.6f},{lng:.6f},17z"


TABLE_ROW_RE = re.compile(r"^\s*\|.*\|\s*$")


def _md_inline(line) -> str:
    """One line of light Markdown -> Telegram HTML: escaped < > &, **bold**, [text](https://link), "- " bullets."""
    line = html.escape(line, quote=False)
    line = re.sub(r"\[([^\]]+)\]\((https?://[^)\s]+)\)", lambda m: f'<a href="{m[2].replace(chr(34), "%22")}">{m[1]}</a>', line)
    line = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", line)
    line = re.sub(r"^\s*[*-]\s+", "• ", line)
    return re.sub(r"(?<![<\w])\*|\*(?![\w>])", "", line)


def _table_lines(rows) -> list:
    """Telegram has no tables: '| 1 | Costa | $2 | 07:00–22:00 |' -> '• <b>1. Costa</b> — Price: $2 · Hours: 07:00–22:00'."""
    cells = [[c.strip() for c in r.strip().strip("|").split("|")] for r in rows]
    cells = [c for c in cells if not all(re.fullmatch(r":?-{2,}:?", x) for x in c)]     # the |---| line
    if len(cells) < 2:
        return [_md_inline(r) for r in rows]
    head, out = cells[0], []
    for c in cells[1:]:
        name = f"{c[0]}. {c[1]}" if len(c) > 1 and c[0].isdigit() else c[0]
        rest = [f"{h}: {v}" for h, v in zip(head[2:], c[2:]) if v and v != "—"] if c[0].isdigit() else \
               [f"{h}: {v}" for h, v in zip(head[1:], c[1:]) if v and v != "—"]
        out.append(f"• **{name}**" + (" — " + " · ".join(rest) if rest else ""))
    return [_md_inline(x) for x in out]


def llm_to_telegram_html(text) -> str:
    """Answers use light Markdown (bold, bullets, links, tables); Telegram HTML mode has <b> and <a> but no tables."""
    lines, out, i = text.strip().split("\n"), [], 0
    while i < len(lines):
        if TABLE_ROW_RE.match(lines[i]):
            block = []
            while i < len(lines) and TABLE_ROW_RE.match(lines[i]):
                block.append(lines[i])
                i += 1
            out.extend(_table_lines(block))
            continue
        if lines[i].startswith("— = ") and any(TABLE_ROW_RE.match(x) for x in lines):
            lines[i] = ""                                     # the legend: empty cells are left out here
        out.append(_md_inline(lines[i]))
        i += 1
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()


SOURCE_LABELS = {"openstreetmap": "OpenStreetMap", "telegram": "Telegram channel", "telegram-public": "Telegram channel",
                 "facebook": "Facebook page", "survey": "Field survey", "overture": "Overture Maps (Meta)"}


def page_label(url) -> str:
    """'facebook' or 'website' for a seller's page link; empty when it is not a real http(s) link."""
    url = str(url or "").strip()
    if not url.startswith(("http://", "https://")):
        return ""
    host = url.split("/")[2].lower()
    return "facebook" if host.endswith(("facebook.com", "fb.com")) else "website"


# Attribution required by the data licences, shown once per message (not on every place)
DATA_CREDITS = {"openstreetmap": "© OpenStreetMap contributors", "overture": "Overture Maps (Meta)"}


FOOTER_TEXT = {"en": {"from": "📏 From {where}", "me": "your location", "data": "Data: {credits}"},
               "kh": {"from": "📏 ចម្ងាយគិតពី{where}", "me": "ទីតាំងរបស់អ្នក", "data": "ទិន្នន័យ៖ {credits}"}}


NO_LLM_LABEL = "No AI (results only)"


def fmt_km(km) -> str:
    return f"{round(km * 100) * 10:.0f} m" if km < 1 else f"{km:.1f} km"


def place_card(i, r, lang="en", hour_now=None) -> str:
    """One place, three short lines at most:
    <b>1. Name</b> · 0.6 km
    Coffee · Chroy Changvar · Facebook
    🕒 07:00–17:00 · open now · 💲 $2.00 · ⭐ 4.5      (only the facts we know)"""
    esc = lambda x: html.escape(str(x), quote=False)
    title = f"<b>{i}. {esc(r.name)}</b>"
    if pd.notna(getattr(r, "distance_km", np.nan)):
        title += f" · {fmt_km(r.distance_km)}"
    dish = ((r.dish_kh if lang == "kh" and r.dish_kh else r.dish_en) or "").replace(";", ",").replace("_", " ").strip()
    sub = []
    if dish and dish.lower() not in str(r.name).lower():          # "Starbucks Coffee": no extra "coffee"
        sub.append(esc(dish[:1].upper() + dish[1:]))
    if r.area:
        area_name = PLACE_NAMES_KH.get(r.area, r.area) if lang == "kh" else r.area
        sub.append(esc(area_name))
    url = str(getattr(r, "page_url", "") or "").strip()
    kind = page_label(url)
    if kind:
        sub.append(f'<a href="{html.escape(url, quote=True)}">{"Facebook" if kind == "facebook" else "Website"}</a>')
    facts = []
    if pd.notna(r.open_hour) and pd.notna(r.close_hour):
        facts.append(f"🕒 {fmt_hours(r.open_hour, r.close_hour)}")
        if hour_now is not None and is_open_at(r.open_hour, r.close_hour, hour_now):
            facts.append("open now" if lang != "kh" else "កំពុងបើក")
    if pd.notna(r.price_usd):
        facts.append(f"💲 {fmt_price(r.price_usd)}")
    if pd.notna(r.rating):
        facts.append(f"⭐ {r.rating:.1f}")
    return "\n".join([title] + [" · ".join(x) for x in (sub, facts) if x])


def footer_line(shown, lang="en", origin="") -> str:
    """One quiet line: where distances are measured from, and the data credits."""
    words = FOOTER_TEXT.get(lang, FOOTER_TEXT["en"])
    sources = [str(x).strip().lower() for x in shown["source"]] if "source" in shown else []
    credits = list(dict.fromkeys(DATA_CREDITS.get(x, SOURCE_LABELS.get(x, x.capitalize())) for x in sources if x and x != "nan"))
    parts = []
    if origin:
        parts.append(words["from"].format(where=words["me"] if origin == "me" else html.escape(origin, quote=False)))
    if credits:
        parts.append(words["data"].format(credits=", ".join(credits)))
    return f"<i>{' · '.join(parts)}</i>" if parts else ""


def source_line(r) -> str:
    """'🔗 Source: OpenStreetMap' (a link when the row has a source_url); empty when the source is unknown."""
    raw = str(getattr(r, "source", "") or "").strip()
    if not raw or raw.lower() == "nan":
        return ""
    label = html.escape(SOURCE_LABELS.get(raw.lower(), raw.capitalize()), quote=False)
    url = str(getattr(r, "source_url", "") or "").strip()
    if url.startswith(("http://", "https://")):
        return f'🔗 <a href="{html.escape(url, quote=True)}">Source: {label}</a>'
    return f"🔗 Source: {label}"


def format_telegram(results, lang="en", start=0, max_items=5, origin="") -> str:
    """Compact Telegram HTML list: numbered places + one footer line. Map links are the buttons below the message."""
    if results is None or len(results) == 0:
        return html.escape(no_match(lang, results), quote=False)
    now = datetime.now(TZ)
    shown = results.iloc[start:start + max_items]
    cards = [place_card(i, r, lang, now.hour + now.minute / 60) for i, r in enumerate(shown.itertuples(), start + 1)]
    footer = footer_line(shown, lang, origin)
    return "\n\n".join(cards + ([footer] if footer else []))


print(format_telegram(hybrid_search("coffee", user_loc=INDEPENDENCE_MONUMENT), "en", max_items=2))


<b>1. The Coffee Bean &amp; Tea Leaf</b> — coffee
📍 Prampir Makara · 📏 0.9 km · <a href="https://www.google.com/maps/search/?api=1&amp;query=11.558362,104.919860">🗺️ Google Maps</a>
🔗 <a href="https://www.openstreetmap.org/node/4652551754">Source: OpenStreetMap</a>

<b>2. The Coffee Bean &amp; Tea Leaf</b> — coffee
📍 Boeng Keng Kang · 📏 0.6 km · <a href="https://www.google.com/maps/search/?api=1&amp;query=11.552374,104.925264">🗺️ Google Maps</a>
🔗 <a href="https://www.openstreetmap.org/node/4652527008">Source: OpenStreetMap</a>


In [51]:
HELP_TEXT = (
    "🍜 Klean Nham – Cambodian food search bot\n"
    "Send a dish name, e.g. 'curry noodle', 'kuy teav open now', 'bbq under $5'.\n"
    "Add 'near me' (I ask for your 📍 location) or 'near <place>', e.g. 'pizza near me', 'coffee near CADT'.\n"
    "Name a khan, city or province for anywhere in Cambodia, e.g. 'pizza in BKK1', 'coffee in Siem Reap' (no place = Phnom Penh first).\n"
    "Every result shows its distance, price, rating, map link, Facebook page or website, and data source when known. You can also send a 📷 food photo.\n"
    "💬 Then just keep chatting: 'is the first one open now?', 'how far is number 2?', 'any cheaper ones?', 'pizza instead'.\n"
    "/new – start a new conversation · /setlocation <place> – tell me where you are · /model – choose the AI model · "
    "/privacy – how your data is used · /deleteme – forget my location\n"
    "Place data: © OpenStreetMap contributors (ODbL); data from Meta, via the Overture Maps Foundation (CDLA Permissive 2.0)\n\n"
    "🍜 ឃ្លានញ៉ាំ (Klean Nham) – បូតស្វែងរកម្ហូបខ្មែរ\n"
    "សូមផ្ញើឈ្មោះម្ហូប ឧ. 'នំបញ្ចុក', 'គុយទាវ ថោក'។\n"
    "សរសេរ 'ជិតខ្ញុំ' ឬ 'near <ទីកន្លែង>' ដើម្បីដឹងចម្ងាយ ឬផ្ញើ 📷 រូបម្ហូប។\n"
    "ដាក់ឈ្មោះខណ្ឌ ក្រុង ឬខេត្ត ឧ. 'កាហ្វេ នៅសៀមរាប'។\n"
    "💬 បន្ទាប់មកសួរបន្តបាន ឧ. 'ហាងទីមួយបើកឥឡូវទេ?'"
)


PRIVACY_TEXT = (
    "🔒 No account needed. Your location and photos are used only to answer your request "
    "and are not stored: your location stays in memory until /deleteme or a bot restart, "
    "photos are deleted right after the search. A place name after 'near' (e.g. CADT) is sent to OpenStreetMap's "
    "map search to find its coordinates; your own coordinates are never sent there. Distances (not coordinates) can "
    "be part of the question sent to the AI model that writes the short summary.\n\n"
    "🔒 មិនចាំបាច់មានគណនីទេ។ ទីតាំង និងរូបថតរបស់អ្នក ប្រើតែដើម្បីឆ្លើយតបប៉ុណ្ណោះ ហើយមិនត្រូវបានរក្សាទុកទេ។"
)


RATE_LIMIT_TEXT = "⏳ Too many requests – please wait a minute. / សូមរង់ចាំមួយនាទី។"


ASK_LOCATION_TEXT = (
    "📍 Where are you? Tap the closest landmark below. On a phone you can also share your exact location with the next message. "
    "On a computer, type /setlocation and a place, e.g. /setlocation CADT. It is kept in memory only, for this chat.\n"
    "📍 តើអ្នកនៅឯណា? សូមជ្រើសរើសទីតាំងដែលនៅជិតបំផុតខាងក្រោម។")


STATS = Counter()           # in-memory counters for /stats (no message text, no ids)


_seen_users = set()


_user_hits = defaultdict(deque)


def allow_request(user_id, now=None) -> bool:
    """Sliding-window rate limit: RATE_LIMIT_PER_MIN requests per user per 60 s."""
    now = time.monotonic() if now is None else now
    hits = _user_hits[user_id]
    while hits and now - hits[0] > 60:
        hits.popleft()
    if len(hits) >= RATE_LIMIT_PER_MIN:
        return False
    hits.append(now)
    return True


async def _guard(update, kind) -> bool:
    """Count the request and enforce the rate limit."""
    uid = update.effective_user.id
    _seen_users.add(uid)
    STATS[kind] += 1
    if not allow_request(uid):
        STATS["rate_limited"] += 1
        await update.effective_message.reply_text(RATE_LIMIT_TEXT)
        return False
    return True


def map_buttons(numbered):
    """[(number, row), ...] -> Google Maps buttons, numbered like the list, two per row."""
    short = lambda name: name if len(name) <= 20 else name[:19].rstrip() + "…"
    buttons = [InlineKeyboardButton(f"📍 {n}. {short(str(r.name))}", url=maps_url(r.lat, r.lng, r.name, getattr(r, "city", ""))) for n, r in numbered]
    return [buttons[k:k + 2] for k in range(0, len(buttons), 2)]


def results_keyboard(results, start=0, max_items=5):
    """A Google Maps button per shown place (numbered like the list, two per row) + 'More results'."""
    rows = map_buttons(list(enumerate(results.iloc[start:start + max_items].itertuples(), start + 1)))
    if len(results) > start + max_items:
        rows.append([InlineKeyboardButton("➕ More results", callback_data="more")])
    return InlineKeyboardMarkup(rows) if rows else None


NEAR_SPOTS = [   # (button label, lat, lng): well-known landmarks, so "near me" also works where sharing a location does not
    ("Independence Monument", 11.5564, 104.9282), ("Royal Palace", 11.5638, 104.9301),
    ("Wat Phnom", 11.5761, 104.9232), ("Central Market", 11.5696, 104.9211),
    ("Russian Market", 11.5405, 104.9147), ("Olympic Stadium", 11.5584, 104.9122),
    ("AEON Mall 1", 11.5481, 104.9326), ("AEON Mall Sen Sok", 11.6002, 104.8854),
    ("ITC", 11.5706, 104.8980), ("CADT", 11.6537, 104.9117),
]


def spots_keyboard():
    """Ten landmark buttons, two per row. Tapping one sets 'where I am' (see spot_callback)."""
    buttons = [InlineKeyboardButton(f"📍 {name}", callback_data=f"spot:{i}") for i, (name, _, _) in enumerate(NEAR_SPOTS)]
    return InlineKeyboardMarkup([buttons[i:i + 2] for i in range(0, len(buttons), 2)])


def location_keyboard():
    """A one-tap 'share my location' button (works in the phone apps)."""
    return ReplyKeyboardMarkup([[KeyboardButton("📍 Share my location", request_location=True)]],
                               resize_keyboard=True, one_time_keyboard=True)


def webapp_ready() -> bool:
    return WEBAPP_URL.startswith("https://")     # Telegram opens Mini Apps over HTTPS only


def webapp_keyboard():
    if not webapp_ready():
        return None
    return InlineKeyboardMarkup([[InlineKeyboardButton("🍜 Open the app / បើកកម្មវិធី", web_app=WebAppInfo(url=WEBAPP_URL))]])


In [52]:
async def start_cmd(update, context):
    await update.message.reply_text(HELP_TEXT, reply_markup=webapp_keyboard())


async def app_cmd(update, context):
    if not webapp_ready():
        await update.message.reply_text("The app is not online yet. Send a dish name here instead.")
        return
    await update.message.reply_text("Search with the app: / ស្វែងរកតាមកម្មវិធី៖", reply_markup=webapp_keyboard())


BOT_PROFILE = {   # (keys are Telegram language codes: "km" is the ISO code for Khmer) what Telegram shows on the bot's profile and in an empty chat (set at every start; cheap and idempotent)
    None: {"description": "Klean Nham: Find food across Cambodia. Send a dish, a place or a time, in English or Khmer, e.g. “kuy teav”, "
                          "“coffee near me”, “pizza in Toul Kork”, “coffee in Siem Reap”. Answers come only from our list of places, so nothing "
                          "is made up.",
           "short": "Klean Nham: Find food across Cambodia, in English or Khmer. Answers only from real places.",
           "commands": [("start", "How to use the bot"), ("new", "Start a new conversation"), ("model", "Choose the AI model"),
                        ("setlocation", "Tell me where you are"),
                        ("privacy", "How your data is used"), ("deleteme", "Forget my location")]},
    "km": {"description": "ឃ្លានញ៉ាំ (Klean Nham)៖ ស្វែងរកម្ហូបទូទាំងប្រទេសកម្ពុជា។ ផ្ញើឈ្មោះម្ហូប ទីកន្លែង ឬម៉ោង ជាភាសាខ្មែរ ឬអង់គ្លេស ឧ. “គុយទាវ” "
                          "“កាហ្វេ ជិតខ្ញុំ”។ ចម្លើយមកពីបញ្ជីហាងពិតប្រាកដរបស់យើងតែប៉ុណ្ណោះ។",
           "short": "ឃ្លានញ៉ាំ (Klean Nham)៖ ស្វែងរកម្ហូបទូទាំងប្រទេសកម្ពុជា ជាភាសាខ្មែរ ឬអង់គ្លេស។",
           "commands": [("start", "របៀបប្រើបូត"), ("new", "ចាប់ផ្តើមការសន្ទនាថ្មី"), ("model", "ជ្រើសរើសម៉ូដែល AI"),
                        ("setlocation", "ប្រាប់ទីតាំងរបស់អ្នក"),
                        ("privacy", "របៀបប្រើទិន្នន័យរបស់អ្នក"), ("deleteme", "លុបទីតាំងរបស់ខ្ញុំ")]},
}


async def setup_bot(app):
    """Bot profile text, the / command menu, and the 'Open app' button (when WEBAPP_URL is an https:// address)."""
    bot = app.bot
    for lang, p in BOT_PROFILE.items():
        commands = list(p["commands"]) + ([("app", "Open the app" if lang is None else "បើកកម្មវិធី")] if webapp_ready() else [])
        try:
            await bot.set_my_commands([BotCommand(c, d) for c, d in commands], language_code=lang)
            await bot.set_my_description(p["description"], language_code=lang)
            await bot.set_my_short_description(p["short"], language_code=lang)
        except Exception as e:
            log.warning("could not set the bot profile (%s)", safe_error(e))
    if webapp_ready():
        try:
            await bot.set_chat_menu_button(menu_button=MenuButtonWebApp("Open app", WebAppInfo(url=WEBAPP_URL)))
        except Exception as e:
            log.warning("could not set the menu button (%s)", safe_error(e))


def models_keyboard(current):
    """One button per model from LLM_MODELS + 'No AI'; the current choice has a ✓."""
    choices = [(m["id"], m["label"]) for m in MODELS] + [(NO_LLM, NO_LLM_LABEL)]
    return InlineKeyboardMarkup([[InlineKeyboardButton(("✓ " if mid == current else "") + label, callback_data=f"model:{i}")]
                                 for i, (mid, label) in enumerate(choices)])


def current_model(context):
    return context.user_data.get("model") or DEFAULT_MODEL_ID


async def model_cmd(update, context):
    await update.message.reply_text(
        "🤖 Which AI model should write the short answer? Every answer still comes only from the places I find. "
        "If a model is busy, another one from the list answers.\n🤖 ជ្រើសរើសម៉ូដែល AI៖",
        reply_markup=models_keyboard(current_model(context)))


async def model_callback(update, context):
    q = update.callback_query
    choices = [m["id"] for m in MODELS] + [NO_LLM]
    try:
        chosen = choices[int(q.data.split(":", 1)[1])]
    except (ValueError, IndexError):
        await q.answer()
        return
    context.user_data["model"] = chosen                      # memory only, per user
    label = MODELS_BY_ID[chosen]["label"] if chosen in MODELS_BY_ID else NO_LLM_LABEL
    await q.answer(f"✓ {label}")
    try:
        await q.edit_message_reply_markup(reply_markup=models_keyboard(chosen))
    except Exception:
        pass                                                  # "message is not modified" when tapping the same model


async def privacy_cmd(update, context):
    await update.message.reply_text(PRIVACY_TEXT)


async def deleteme_cmd(update, context):
    context.user_data.clear()                # location + last results
    await update.message.reply_text("🗑️ Your saved location was deleted. / ទីតាំងរបស់អ្នកត្រូវបានលុប។")


async def setlocation_cmd(update, context):
    place = " ".join(context.args or []).strip()[:60]
    if not place:
        await update.message.reply_text("Type /setlocation and where you are, e.g. /setlocation CADT or /setlocation Aeon Mall Sen Sok, "
                                        "or tap a landmark.\nឧទាហរណ៍៖ /setlocation CADT", reply_markup=spots_keyboard())
        return
    if not await _guard(update, "locations"):
        return
    spot = await asyncio.to_thread(geocode_place, place)
    if not spot:
        await update.message.reply_text(f"I could not find “{place}” on the map. Try the full name, or share your 📍 location.")
        return
    context.user_data["location"] = spot                       # memory only
    pending = context.user_data.pop("pending_query", None)
    await update.message.reply_text(f"📍 Your location is now {place} (kept in memory only, /deleteme forgets it)."
                                    + (" Searching…" if pending else " Now try: pizza near me"),
                                    reply_markup=ReplyKeyboardRemove())
    if pending:
        await run_query(update.message, context, pending, spot)


async def stats_cmd(update, context):
    if update.effective_user.id not in ADMIN_IDS:
        await update.message.reply_text("⛔ Admins only.")
        return
    lines = [f"{k}: {v}" for k, v in sorted(STATS.items())] + [f"unique users: {len(_seen_users)}"]
    await update.message.reply_text("📊 Bot stats (since start)\n" + "\n".join(lines))


In [53]:
async def send_results(message, context, results, lang, intro="", warning="", origin=""):
    """intro = the LLM summary; warning = a problem worth reading first; origin = where distances are measured from."""
    context.user_data.update(last_results=results, last_lang=lang, offset=0, last_origin=origin)
    head = "\n\n".join(x for x in (intro, warning) if x)
    text = (head + "\n\n" if head else "") + format_telegram(results, lang, origin=origin)
    await message.reply_text(text, parse_mode=ParseMode.HTML, reply_markup=results_keyboard(results),
                             link_preview_options=LinkPreviewOptions(is_disabled=True))


async def run_query(message, context, query, user_loc=None, rewritten=False):
    """Search, add the LLM summary (if configured) and reply. Used for text queries and for 'near me' after a location share.
    rewritten = the query was built from the conversation ('any cheaper ones?' -> 'cheap pizza near me'): say so."""
    lang = detect_lang(query)
    _, flt = parse_query_filters(query)
    near, origin, warning = flt.get("near"), "", ""
    if near and near != "me":                # "pizza near CADT": look the place up on OpenStreetMap
        spot = await asyncio.to_thread(geocode_place, near, flt.get("city"))
        i = query.lower().rfind(near.lower())
        typed = query[i:i + len(near)] if i >= 0 else near.title()     # the user's spelling: "CADT", not "Cadt"
        if spot:
            user_loc, origin = spot, typed
        else:
            user_loc = None
            warning = (f"⚠️ I could not find “{html.escape(typed, quote=False)}” on the map, so there are no distances. "
                       "Try the full name, or say 'near me'.")
    elif user_loc:
        origin = "me"
    results = await asyncio.to_thread(hybrid_search, query, 15, user_loc)
    unchecked = unverified_note(results, lang)               # "under $5" when the data has no prices for these places
    if unchecked:
        warning = "\n".join(x for x in (warning, f"ℹ️ {html.escape(unchecked, quote=False)}") if x)
    lead = search_lead(results, lang, origin or None) if len(results) else ""   # no AI: the cards list the places
    intro = html.escape(lead, quote=False)
    if rewritten:
        intro = f"🔎 <i>{html.escape(query, quote=False)}</i>" + (f"\n\n{intro}" if intro else "")
    await send_results(message, context, results, lang, intro, warning, origin)
    context.user_data["last_query"] = query
    remember(context, "assistant", lead + " Places shown: " + places_summary(results.head(5)))


# ---------- conversation memory (per chat, in memory only; /new or /deleteme clears it) ----------
def remember(context, role, text):
    """Adds a message to this chat's short history (the last HISTORY_TURNS exchanges)."""
    history = context.user_data.setdefault("history", [])
    history.append({"role": role, "text": " ".join(str(text).split())[:500]})
    del history[:-HISTORY_TURNS * 2]


def places_summary(rows, start=1) -> str:
    return "; ".join(f"{n}. {r.name} ({', '.join(x for x in (r.dish_en, r.area) if x)})"
                     for n, r in enumerate(rows.itertuples(), start))


def shown_places(context):
    """The places the user has seen, numbered as shown (the first 5, plus any 'More results'), or None."""
    results = context.user_data.get("last_results")
    if results is None or len(results) == 0:
        return None
    return results.iloc[:min(context.user_data.get("offset", 0) + 5, CONTEXT_PLACES)]


async def new_cmd(update, context):
    for key in ("history", "last_results", "last_query", "offset", "last_origin", "pending_query"):
        context.user_data.pop(key, None)                     # keeps the location and the chosen model
    await update.message.reply_text("🆕 New conversation. What would you like to eat? / ការសន្ទនាថ្មី។ តើអ្នកចង់ញ៉ាំអ្វី?")


async def text_handler(update, context):
    if not await _guard(update, "text_queries"):
        return
    query = update.message.text.strip()[:200]
    log.info("text query (lang=%s, %d chars)", detect_lang(query), len(query))   # never log text together with ids
    await context.bot.send_chat_action(update.effective_chat.id, ChatAction.TYPING)
    history = list(context.user_data.get("history", []))
    shown, model = shown_places(context), context.user_data.get("model")
    plan = await asyncio.to_thread(plan_turn, query, history, shown, model, context.user_data.get("last_query"))
    remember(context, "user", query)

    if plan["action"] == "chat":
        out = await asyncio.to_thread(chat_reply, query, history, model)
        await update.message.reply_text(llm_to_telegram_html(out["text"]), parse_mode=ParseMode.HTML)
        remember(context, "assistant", out["text"])
        return

    if plan["action"] == "followup" and shown is not None:     # about the places already shown: no new search
        out = await asyncio.to_thread(answer_followup, query, history, shown, plan["places"], model)
        text = llm_to_telegram_html(out["text"])               # AI sentence, or a list / table built from the data
        rows = list(shown.itertuples())                         # (a pandas Series' .name is its index, not the place)
        numbered = [(n, rows[n - 1]) for n in out["numbers"][:6]]
        await update.message.reply_text(text, parse_mode=ParseMode.HTML,
                                        reply_markup=InlineKeyboardMarkup(map_buttons(numbered)),
                                        link_preview_options=LinkPreviewOptions(is_disabled=True))
        remember(context, "assistant", out["text"])
        return

    search_query = plan["query"] or query
    _, flt = parse_query_filters(search_query)
    if flt.get("near") == "me" and not context.user_data.get("location"):
        context.user_data["pending_query"] = search_query          # answered as soon as the location arrives
        await update.message.reply_text(ASK_LOCATION_TEXT, reply_markup=spots_keyboard())
        await update.message.reply_text("📍 Or share your exact location (phone):", reply_markup=location_keyboard())
        return
    await run_query(update.message, context, search_query, context.user_data.get("location"),
                    rewritten=search_query.strip().lower() != query.strip().lower())


async def spot_callback(update, context):
    q = update.callback_query
    await q.answer()
    try:
        name, lat, lng = NEAR_SPOTS[int(q.data.split(":", 1)[1])]
    except (ValueError, IndexError):
        return
    if not await _guard(update, "locations"):
        return
    context.user_data["location"] = (lat, lng)                    # memory only
    pending = context.user_data.pop("pending_query", None)
    await q.message.reply_text(f"📍 Your location is now near {name} (kept in memory only, /deleteme forgets it)."
                               + (" Searching…" if pending else " Now try: pizza near me"), reply_markup=ReplyKeyboardRemove())
    if pending:
        await run_query(q.message, context, pending, (lat, lng))


async def more_callback(update, context):
    q = update.callback_query
    await q.answer()
    results, lang = context.user_data.get("last_results"), context.user_data.get("last_lang", "en")
    offset = context.user_data.get("offset", 0) + 5
    if results is None or offset >= len(results):
        await q.message.reply_text("No more results. / មិនមានលទ្ធផលទៀតទេ។")
        return
    context.user_data["offset"] = offset
    await q.message.reply_text(format_telegram(results, lang, start=offset, origin=context.user_data.get("last_origin", "")),
                               parse_mode=ParseMode.HTML,
                               reply_markup=results_keyboard(results, start=offset),
                               link_preview_options=LinkPreviewOptions(is_disabled=True))
    remember(context, "assistant", "More places shown: " + places_summary(results.iloc[offset:offset + 5], offset + 1))


async def location_handler(update, context):
    if update.edited_message is not None:        # a live location that moves: Telegram sends each new point as an edit
        if update.edited_message.location:
            context.user_data["location"] = (update.edited_message.location.latitude,
                                             update.edited_message.location.longitude)
        return                                   # no reply, no rate limit: just keep the newest point
    if not await _guard(update, "locations"):
        return
    loc = update.message.location
    context.user_data["location"] = (loc.latitude, loc.longitude)   # memory only
    pending = context.user_data.pop("pending_query", None)
    await update.message.reply_text(
        "📍 Location received (kept in memory only, /deleteme forgets it)." + (" Searching…" if pending else " Now send a dish name.")
        + "\n📍 បានទទួលទីតាំង។", reply_markup=ReplyKeyboardRemove())
    if pending:
        await context.bot.send_chat_action(update.effective_chat.id, ChatAction.TYPING)
        await run_query(update.message, context, pending, context.user_data["location"])


In [54]:
async def photo_handler(update, context):
    if not await _guard(update, "photo_queries"):
        return
    if not image_search_available():
        await update.message.reply_text("📷 Photo search is not set up yet. Please type the dish name instead.")
        return
    await context.bot.send_chat_action(update.effective_chat.id, ChatAction.TYPING)
    tg_file = await update.message.photo[-1].get_file()
    fd, tmp_path = tempfile.mkstemp(suffix=".jpg")
    os.close(fd)
    try:
        await tg_file.download_to_drive(tmp_path)
        matches = await asyncio.to_thread(search_images_by_image, tmp_path, 10)
    finally:
        Path(tmp_path).unlink(missing_ok=True)   # the user's photo is never kept
    results = get_rows([m["restaurant_id"] for m in matches])
    await send_results(update.message, context, results, "en", "📷 Places with similar-looking food:")


async def error_handler(update, context):
    log.error("bot error: %s", safe_error(context.error))


def build_bot_app(token):
    # generous timeouts: the default 5 seconds is too tight on a slow or unstable connection
    app = (Application.builder().token(token).connect_timeout(15).read_timeout(25).write_timeout(25)
           .pool_timeout(15).get_updates_connect_timeout(15).get_updates_read_timeout(35)
           .post_init(setup_bot).build())
    app.add_handler(CommandHandler(["start", "help"], start_cmd))
    app.add_handler(CommandHandler("app", app_cmd))
    app.add_handler(CommandHandler("model", model_cmd))
    app.add_handler(CommandHandler("new", new_cmd))
    app.add_handler(CallbackQueryHandler(model_callback, pattern="^model:"))
    app.add_handler(CommandHandler("privacy", privacy_cmd))
    app.add_handler(CommandHandler("deleteme", deleteme_cmd))
    app.add_handler(CommandHandler("stats", stats_cmd))
    app.add_handler(CommandHandler("setlocation", setlocation_cmd))
    app.add_handler(CallbackQueryHandler(more_callback, pattern="^more$"))
    app.add_handler(CallbackQueryHandler(spot_callback, pattern="^spot:"))
    app.add_handler(MessageHandler(tg_filters.LOCATION, location_handler))
    app.add_handler(MessageHandler(tg_filters.PHOTO, photo_handler))
    app.add_handler(MessageHandler(tg_filters.TEXT & ~tg_filters.COMMAND, text_handler))
    app.add_error_handler(error_handler)
    return app


### Start / stop the bot inside the notebook

The notebook already runs an event loop, so we **do not** call `app.run_polling()` here (that is for `backend/bot.py`).
Instead the start cell runs `initialize → start → updater.start_polling`, and the stop cell reverses it.

> ⚠️ **Always run the stop cell before restarting the kernel or re-running the notebook.** Otherwise the old
> bot may keep polling and Telegram reports *Conflict: two bots are using the same token*.
> The bot only answers **while this kernel is running and the Mac is awake**. For 24/7, deploy `backend/bot.py` (section 9).

In [55]:
# ▶️ START the bot (safe to run twice: it checks BOT_APP first)
START_BOT = False      # set True to run the bot inside this notebook (never at the same time as backend/bot.py or backend/web.py)
BOT_APP = globals().get("BOT_APP")
if not TELEGRAM_TOKEN:
    print("⏭️ TELEGRAM_TOKEN is missing in .env: skipping the bot. Add it and re-run this cell.")
elif not START_BOT:
    print("⏭️ START_BOT is False, so Run All does not start the bot. Set START_BOT = True and run this cell to start it.")
elif BOT_APP is not None:
    print("ℹ️ The bot is already running. Run the stop cell first if you want to restart it.")
else:
    app = build_bot_app(TELEGRAM_TOKEN)
    try:
        await app.initialize()
        await app.start()
        await app.updater.start_polling(drop_pending_updates=True)
        BOT_APP = app
        me = await app.bot.get_me()
        print(f"✅ Bot @{me.username} is running. Send it a message on Telegram.")
    except Exception as e:
        print("❌ Could not start the bot:", safe_error(e))
        if app.running:
            await app.stop()
        await app.shutdown()

⏭️ START_BOT is False, so Run All does not start the bot. Set START_BOT = True and run this cell to start it.


In [56]:
# ⏹️ STOP the bot (run this before restarting the kernel)
if globals().get("BOT_APP") is None:
    print("ℹ️ The bot is not running.")
else:
    await BOT_APP.updater.stop()
    await BOT_APP.stop()
    await BOT_APP.shutdown()
    BOT_APP = None
    print("🛑 Bot stopped.")

ℹ️ The bot is not running.


## 9. Export for deployment

**What:** this section is the **only place to change the backend**. Each cell below holds one file of the deployed app.
Running the section writes them into the project, so the website, the Telegram bot and Docker run **without Jupyter**:

```
backend/food_search/config.py        settings + secrets from environment variables (.env)
backend/food_search/search.py        retrieval: cleaning, BM25 + typo fix, embeddings + ChromaDB, hybrid ranking, CLIP photos
backend/food_search/dishes.py        photo -> dish (EfficientNet from food_classification.ipynb, or CLIP zero-shot) + recipes
backend/food_search/llm.py           generation: prompt, AI models (Gemini / Ollama) with fallback, result cards
backend/food_search/bot_handlers.py  the Telegram bot
backend/food_search/api.py           FastAPI: /api/chat, /api/demo/..., and serves the React frontend
backend/bot.py                       runs the Telegram bot only
backend/web.py                       runs the website + API (and the bot)
requirements.txt, .dockerignore, Dockerfile
```

The React frontend (`frontend/`) is not generated here: it is a separate app that calls the API in `api.py`.

**How to change the backend:** edit the cell of that file, run this whole section, then restart `python backend/web.py`
(or `python backend/bot.py`). Every generated file starts with a line saying it comes from this notebook. A change made
directly in a generated file is replaced by the next export; the replaced version is first saved in
`export_backups/<date_time>/`, so nothing is lost.

The earlier sections explain and evaluate each step interactively. The cells below are the deployed version of that
code, including later additions (the AI model choice, the compact Telegram replies, the web API for the React app).


In [57]:
EXPORT_DIR = ROOT / "export"                 # staging folder (in .gitignore)
shutil.rmtree(EXPORT_DIR, ignore_errors=True)           # start empty, so no stale file from an earlier run can be copied
(EXPORT_DIR / "backend" / "food_search").mkdir(parents=True, exist_ok=True)
(EXPORT_DIR / "backend" / "food_search" / "__init__.py").write_text('"""Cambodian food search package (exported from food_search.ipynb)."""\n', encoding="utf-8")
print("export/ ready")

export/ ready


In [ ]:
%%writefile export/backend/food_search/config.py
"""Settings and secrets. Values come from environment variables (or .env when run locally)."""
import os
import re
from pathlib import Path
from zoneinfo import ZoneInfo

from dotenv import load_dotenv

ROOT = Path(__file__).resolve().parents[2]    # project root: backend/food_search/config.py -> food-search/
load_dotenv(ROOT / ".env", override=False)    # does nothing if there is no .env (e.g. in Docker)


def get_secret(name, default=""):
    return (os.getenv(name) or default).strip()


DATA_DIR = ROOT / "data"
EVAL_DIR = DATA_DIR / "eval"
PHOTOS_DIR = DATA_DIR / "photos"
RESULTS_DIR = ROOT / "results"
CHROMA_DIR = ROOT / "chroma_db"
BACKEND_DIR = ROOT / "backend"


# --- secrets & provider settings (values are never printed) ---
TELEGRAM_TOKEN = get_secret("TELEGRAM_TOKEN")
GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
GEMINI_MODEL = get_secret("GEMINI_MODEL")          # no hardcoded model: set it in .env
OLLAMA_MODEL = get_secret("OLLAMA_MODEL")          # e.g. any model you pulled with `ollama pull`
OLLAMA_HOST = get_secret("OLLAMA_HOST", "http://localhost:11434").rstrip("/")
OLLAMA_API_KEY = get_secret("OLLAMA_API_KEY")      # only for Ollama cloud: OLLAMA_HOST=https://ollama.com
LLM_PROVIDER = get_secret("LLM_PROVIDER", "gemini").lower()   # gemini | ollama | none
# Models users can choose from (web app, demo, Telegram /model). Comma-separated provider:model, first = default,
# optional "=Label". Empty = just GEMINI_MODEL / OLLAMA_MODEL. e.g. gemini:gemini-3.5-flash-lite,gemini:gemini-3.5-flash
LLM_MODELS = get_secret("LLM_MODELS")
EMBED_MODEL = get_secret("EMBED_MODEL", "paraphrase-multilingual-MiniLM-L12-v2")
SHEET_CSV_URL = get_secret("SHEET_CSV_URL")          # optional: Google Sheet with the places
SHEET_QUERIES_URL = get_secret("SHEET_QUERIES_URL")  # optional: Google Sheet tab with eval queries
TG_API_ID = get_secret("TG_API_ID")                  # optional: Telegram API (my.telegram.org) for seller channels
TG_API_HASH = get_secret("TG_API_HASH")
ADMIN_IDS = {int(x) for x in re.split(r"[,\s]+", get_secret("ADMIN_IDS")) if x.isdigit()}

# --- website / Telegram Mini App / demo (python web.py) ---
FRONTEND_DIR = ROOT / "frontend"                    # the React app (website + Telegram Mini App + demo)
FRONTEND_DIST = FRONTEND_DIR / "dist"               # its production build (npm run build), served by web.py
WEBAPP_URL = get_secret("WEBAPP_URL").rstrip("/")    # public https:// address of web.py: enables the bot's "Open app" button
HOST = get_secret("HOST", "127.0.0.1")               # 0.0.0.0 inside Docker / on a server
PORT = int(get_secret("PORT", "8080"))
RUN_BOT = get_secret("RUN_BOT", "1") != "0"          # web.py also runs the bot (polling) when TELEGRAM_TOKEN is set
MAX_PHOTO_MB = 8
ENABLE_DEMO = get_secret("ENABLE_DEMO", "1") != "0"     # the "How it works" page (/demo) and its API (/api/demo/...)

# --- search settings ---
TOP_K = 5                 # results shown per query
HYBRID_WEIGHT = 0.5       # 0 = only BM25, 1 = only embeddings
RATING_BOOST = 0.05       # small bonus for well-rated places
HOME_CITY_BONUS = 0.05    # bonus for Phnom Penh places when the question names no city or khan
DISTANCE_WEIGHT = 0.15    # bonus for nearby places (only when a location is given)
NEAR_POOL = 40            # "near me": the closest of the (up to) 40 most relevant places are shown first
EMBED_CANDIDATES = 5000   # places scored by embeddings per query: all of them for datasets up to this size
MIN_EMBED_SIM = 0.45      # no keyword hit and cosine below this = "not a match" (chosen for MiniLM)
TYPO_CUTOFF = 75          # rapidfuzz score needed to auto-correct a query word
FUZZY_MIN = 0.70          # minimum fuzzy score for the typo fallback
CHEAP_MAX_USD = 3.0       # what "cheap" means
LATE_HOUR = 23            # "open late" = closes at 23:00 or later (or after midnight)
KHR_PER_USD = 4100        # riel per dollar (approximate, change if needed)
TZ = ZoneInfo("Asia/Phnom_Penh")

# --- LLM & bot settings ---
LLM_TIMEOUT_S = 20
OLLAMA_TIMEOUT_S = 60     # local models can be slow on the first call
RATE_LIMIT_PER_MIN = 10
TG_POST_LIMIT = 100       # recent posts read per seller channel


In [ ]:
%%writefile export/backend/food_search/search.py
"""Search engine: cleaning, BM25 + fuzzy, embeddings + ChromaDB, hybrid ranking, filters."""
import logging
import math
import re
import time
import unicodedata
from datetime import datetime
from functools import lru_cache
from pathlib import Path

import chromadb
import numpy as np
import pandas as pd
import requests
import torch
from chromadb.config import Settings
from rank_bm25 import BM25Okapi
from rapidfuzz import fuzz, process

from .config import *  # noqa: F401,F403  (paths + settings)

log = logging.getLogger("food_search")

# Filled by init(): the cleaned table, indexes and models
df = bm25 = BM25_VOCAB = FOOD_VOCAB = embedder = text_col = image_col = None
HAS_PHOTOS = False


def pick_device() -> str:
    """Prefer an NVIDIA GPU, then Apple Silicon (MPS), then CPU."""
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


DEVICE = pick_device()


KHMER_RE = re.compile(r"[ក-៿᧠-᧿]+")   # Khmer Unicode blocks
WORD_RE = re.compile(r"[a-z0-9]+")
ZERO_WIDTH_RE = re.compile("[\u200b\u2060\ufeff]")      # invisible characters common in Khmer text


def normalize_text(s) -> str:
    """NFC-normalize (important for Khmer), collapse spaces, strip, lowercase."""
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = ZERO_WIDTH_RE.sub("", unicodedata.normalize("NFC", str(s)))
    return re.sub(r"\s+", " ", s).strip().lower()


def detect_lang(text) -> str:
    """'kh' if the text contains Khmer script, otherwise 'en'."""
    return "kh" if KHMER_RE.search(str(text)) else "en"


def khr_to_usd(amount_khr, rate=None) -> float:
    """Convert Cambodian riel to US dollars."""
    return round(float(amount_khr) / (rate or KHR_PER_USD), 2)


def parse_price(value) -> float:
    """'$2', '2.5', '8000 KHR', '8000៛' -> price in USD. Unknown -> NaN."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return float("nan")
    s = normalize_text(value).replace(",", "")
    num = re.search(r"\d+(?:\.\d+)?", s)
    if not num:
        return float("nan")
    amount = float(num.group())
    if any(unit in s for unit in ("khr", "riel", "៛", "រៀល")):
        return khr_to_usd(amount)
    return amount


ALL_COLUMNS = ["id", "name", "dish_en", "dish_kh", "description", "tags", "price_usd",
               "area", "lat", "lng", "rating", "open_hour", "close_hour", "source",
               "source_url", "date", "image_url", "phone_public"]
TEXT_COLS = ["name", "dish_en", "dish_kh", "description", "tags", "area",
             "source", "source_url", "date", "image_url", "phone_public"]

# 14 Phnom Penh khans, Khmer names, and popular aliases / communes
KHAN_DETAILS = {
    "Boeng Keng Kang": {"kh": "បឹងកេងកង", "aliases": ["bkk", "bkk1", "bkk2", "bkk3", "បកក", "បឹងកេងកង១", "បឹងកេងកង២", "បឹងកេងកង៣", "boeung keng kang", "boeng keng kang"]},
    "Chamkar Mon": {"kh": "ចំការមន", "aliases": ["chamkarmon", "chamkar morn", "ttp", "ttp1", "ttp2", "ទួលទំពូង", "toul tom poung", "tuol tom poung", "russian market", "ផ្សារទួលទំពូង", "បឹងត្របែក", "boeung trabek", "phsar doeum thkov", "ផ្សារដើមថ្កូវ"]},
    "Toul Kork": {"kh": "ទួលគោក", "aliases": ["tk", "tuol kouk", "toul kouk", "toul kork", "ទឹកល្អក់", "teuk laak", "បឹងកក់", "boeng kak"]},
    "Daun Penh": {"kh": "ដូនពេញ", "aliases": ["dp", "doun penh", "daun penh", "ផ្សារថ្មី", "phsar thmey", "central market", "riverside", "មាត់ទន្លេ", "វត្តភ្នំ", "wat phnom", "old market", "ផ្សារចាស់", "kandal market", "phsar kandal", "ផ្សារកណ្ដាល"]},
    "Sen Sok": {"kh": "សែនសុខ", "aliases": ["aeon 2", "aeon sen sok", "aeon2", "ភ្នំពេញថ្មី", "phnom penh thmey", "ទឹកថ្លា", "teuk thla", "sen sok"]},
    "Prampir Makara": {"kh": "ប្រាំពីរមករា", "aliases": ["7 makara", "7makara", "៧មករា", "៧ មករា", "7មករា", "អូរឫស្សី", "orussey", "mittapheap", "olympic", "ផ្សារអូឡាំពិក", "prampir makara"]},
    "Chroy Changvar": {"kh": "ជ្រោយចង្វារ", "aliases": ["chroy changva", "ជ្រោយចង្វា", "chroy changvar"]},
    "Chbar Ampov": {"kh": "ច្បារអំពៅ", "aliases": ["chbar ampov 1", "chbar ampov 2", "ច្បារអំពៅ១", "ច្បារអំពៅ២", "niroth", "prek pra", "ព្រែកប្រា", "chbar ampov"]},
    "Mean Chey": {"kh": "មានជ័យ", "aliases": ["meanchey", "stung meanchey", "ស្ទឹងមានជ័យ", "boeng tumpun", "បឹងទំពុន", "mean chey"]},
    "Russey Keo": {"kh": "ឫស្សីកែវ", "aliases": ["russei keo", "ឬស្សីកែវ", "toul sangke", "ទួលសង្កែ", "kilomet 6", "គីឡូម៉ែត្រ៦", "russey keo"]},
    "Pou Senchey": {"kh": "ពោធិ៍សែនជ័យ", "aliases": ["por senchey", "ពោធិសែនជ័យ", "choam chao", "ចោមចៅ", "kakab", "កាកាប", "pou senchey"]},
    "Dangkao": {"kh": "ដង្កោ", "aliases": ["dangkor", "dangkao"]},
    "Prek Pnov": {"kh": "ព្រែកព្នៅ", "aliases": ["prek pnou", "prek pnov"]},
    "Kamboul": {"kh": "កំបូល", "aliases": ["kombol", "kamboul"]},
}

AREA_MAP = {}
KHAN_KH = {}
for khan, info in KHAN_DETAILS.items():
    AREA_MAP[khan.lower()] = khan
    AREA_MAP[info["kh"]] = khan
    KHAN_KH[khan] = info["kh"]
    for a in info["aliases"]:
        AREA_MAP[a.lower()] = khan

# Phnom Penh and the 24 provinces: Khmer names and common spellings (towns people say instead of the province)
PROVINCE_DETAILS = {
    "Phnom Penh": {"kh": "ភ្នំពេញ", "aliases": ["phnompenh", "phnom phen", "pnh", "ក្រុងភ្នំពេញ"]},
    "Siem Reap": {"kh": "សៀមរាប", "aliases": ["siemreap", "siem reab"]},
    "Battambang": {"kh": "បាត់ដំបង", "aliases": ["battambong", "batdambang"]},
    "Preah Sihanouk": {"kh": "ព្រះសីហនុ", "aliases": ["sihanoukville", "sihanouk ville", "sihanouk", "kampong som", "kompong som",
                                                  "កំពង់សោម", "ក្រុងព្រះសីហនុ", "otres", "otres beach", "ochheuteal"]},
    "Kampot": {"kh": "កំពត", "aliases": ["kompot"]},
    "Kep": {"kh": "កែប", "aliases": ["kep city", "krong kep"]},
    "Kandal": {"kh": "កណ្ដាល", "aliases": ["កណ្តាល", "ta khmau", "takhmao", "takhmau", "តាខ្មៅ"]},
    "Kampong Cham": {"kh": "កំពង់ចាម", "aliases": ["kompong cham", "kampongcham"]},
    "Kampong Chhnang": {"kh": "កំពង់ឆ្នាំង", "aliases": ["kompong chhnang"]},
    "Kampong Speu": {"kh": "កំពង់ស្ពឺ", "aliases": ["kompong speu", "chbar mon", "ច្បារមន"]},
    "Kampong Thom": {"kh": "កំពង់ធំ", "aliases": ["kompong thom", "stung sen"]},
    "Svay Rieng": {"kh": "ស្វាយរៀង", "aliases": ["bavet", "បាវិត"]},
    "Prey Veng": {"kh": "ព្រៃវែង", "aliases": []},
    "Takeo": {"kh": "តាកែវ", "aliases": ["takev", "doun kaev"]},
    "Koh Kong": {"kh": "កោះកុង", "aliases": ["kohkong"]},
    "Pursat": {"kh": "ពោធិ៍សាត់", "aliases": ["ពោធិសាត់", "posat", "pouthisat"]},
    "Kratie": {"kh": "ក្រចេះ", "aliases": ["kracheh"]},
    "Stung Treng": {"kh": "ស្ទឹងត្រែង", "aliases": ["steung treng"]},
    "Ratanakiri": {"kh": "រតនគិរី", "aliases": ["ratanakkiri", "rattanakiri", "banlung", "បានលុង"]},
    "Mondulkiri": {"kh": "មណ្ឌលគិរី", "aliases": ["mondolkiri", "sen monorom", "senmonorom", "សែនមនោរម្យ"]},
    "Preah Vihear": {"kh": "ព្រះវិហារ", "aliases": ["tbeng meanchey"]},
    "Oddar Meanchey": {"kh": "ឧត្ដរមានជ័យ", "aliases": ["ឧត្តរមានជ័យ", "oddar mean chey", "otdar meanchey", "samraong"]},
    "Banteay Meanchey": {"kh": "បន្ទាយមានជ័យ", "aliases": ["banteay mean chey", "poipet", "ប៉ោយប៉ែត", "sisophon", "serei saophoan"]},
    "Pailin": {"kh": "ប៉ៃលិន", "aliases": []},
    "Tbong Khmum": {"kh": "ត្បូងឃ្មុំ", "aliases": ["tboung khmum"]},
}
HOME_CITY = "Phnom Penh"                     # where the search looks first when the question names no place

CITY_MAP = {}
CITY_KH = {}
for city, info in PROVINCE_DETAILS.items():
    CITY_MAP[city.lower()] = city
    CITY_MAP[info["kh"]] = city
    CITY_KH[city] = info["kh"]
    for a in info["aliases"]:
        CITY_MAP[a.lower()] = city
PLACE_NAMES_KH = {**KHAN_KH, **CITY_KH}      # area or city -> Khmer name, for replies in Khmer

# every place name the parser knows, longest first, so "Banteay Meanchey" wins over the khan "Mean Chey"
PLACE_ALIASES = sorted([(a, "area", v) for a, v in AREA_MAP.items()] + [(a, "city", v) for a, v in CITY_MAP.items()],
                       key=lambda x: len(x[0]), reverse=True)


def canonical_place(value) -> tuple:
    """A place written in the data -> (area, city): 'Boeung Keng Kang' -> ('Boeng Keng Kang', 'Phnom Penh'),
    'Sihanoukville' -> ('Preah Sihanouk', 'Preah Sihanouk'). Unknown names are kept as written, with no city."""
    key = normalize_text(value)
    if key in AREA_MAP:
        return AREA_MAP[key], HOME_CITY
    if key in CITY_MAP:
        return CITY_MAP[key], CITY_MAP[key]
    return str(value or ""), ""

# Bilingual dish categories and synonyms (Khmer <-> English)
FOOD_CONCEPTS = [
    {"en_canonical": "noodles", "kh_canonical": "គុយទាវ / មី",
     "en": ["noodle", "noodles", "kuy teav", "kuyteav", "kuy tiew", "koy teav", "ramen", "udon", "pho", "wonton"],
     "kh": ["គុយទាវ", "មី", "មីគាវ", "រ៉ាមេន", "អ៊ូដុង", "គុយទាវទូក"]},
    {"en_canonical": "nom banh chok (khmer noodles)", "kh_canonical": "នំបញ្ចុក",
     "en": ["nom banh chok", "numbanchok", "khmer noodle", "curry noodle"],
     "kh": ["នំបញ្ចុក"]},
    {"en_canonical": "fried rice", "kh_canonical": "បាយឆា",
     "en": ["fried rice", "friedrice"],
     "kh": ["បាយឆា"]},
    {"en_canonical": "chicken rice", "kh_canonical": "បាយមាន់",
     "en": ["chicken rice", "hainanese"],
     "kh": ["បាយមាន់"]},
    {"en_canonical": "pork rice", "kh_canonical": "បាយសាច់ជ្រូក",
     "en": ["pork rice", "broken rice", "bai sach chrouk", "com tam"],
     "kh": ["បាយសាច់ជ្រូក"]},
    {"en_canonical": "rice", "kh_canonical": "បាយ",
     "en": ["rice"],
     "kh": ["បាយ"]},
    {"en_canonical": "lok lak", "kh_canonical": "ឡុកឡាក់",
     "en": ["lok lak", "beef lok lak"],
     "kh": ["ឡុកឡាក់", "បាយឡុកឡាក់"]},
    {"en_canonical": "amok", "kh_canonical": "អាម៉ុក",
     "en": ["amok", "fish amok"],
     "kh": ["អាម៉ុក", "អាមុក"]},
    {"en_canonical": "curry", "kh_canonical": "ការី",
     "en": ["curry", "red curry"],
     "kh": ["ការី", "សម្លការី"]},
    {"en_canonical": "coffee", "kh_canonical": "កាហ្វេ",
     "en": ["coffee", "cafe", "espresso", "latte", "cappuccino"],
     "kh": ["កាហ្វេ"]},
    {"en_canonical": "tea / boba", "kh_canonical": "តែ / តែគុជ",
     "en": ["tea", "milk tea", "bubble tea", "boba", "green tea", "matcha"],
     "kh": ["តែ", "តែគុជ", "តែបៃតង"]},
    {"en_canonical": "pizza", "kh_canonical": "ភីហ្សា",
     "en": ["pizza", "pizzeria"],
     "kh": ["ភីហ្សា"]},
    {"en_canonical": "burger", "kh_canonical": "ប៊ឺហ្គឺ",
     "en": ["burger", "hamburger"],
     "kh": ["ប៊ឺហ្គឺ", "បឺហ្គឺ"]},
    {"en_canonical": "bbq / grilled meat", "kh_canonical": "សាច់អាំង",
     "en": ["bbq", "barbecue", "grill", "grilled", "roast"],
     "kh": ["សាច់អាំង", "សាច់គោអាំង", "សាច់ជ្រូកអាំង"]},
    {"en_canonical": "seafood", "kh_canonical": "គ្រឿងសមុទ្រ",
     "en": ["seafood", "crab", "shrimp", "prawn", "squid", "fish"],
     "kh": ["គ្រឿងសមុទ្រ", "ក្តាម", "បង្គា", "មឹក", "ត្រី"]},
    {"en_canonical": "sushi / japanese", "kh_canonical": "ស៊ូស៊ី / ម្ហូបជប៉ុន",
     "en": ["sushi", "sashimi", "japanese"],
     "kh": ["ស៊ូស៊ី", "សាស៊ីមី", "ម្ហូបជប៉ុន"]},
    {"en_canonical": "soup / hotpot", "kh_canonical": "ស៊ុប / ហតផត",
     "en": ["soup", "hotpot", "shabu", "steamboat"],
     "kh": ["ស៊ុប", "ហតផត", "ស៊ុបគោ", "ស៊ុបឆ្នាំងភ្លើង"]},
    {"en_canonical": "dim sum", "kh_canonical": "ឌីមសាំ",
     "en": ["dim sum", "dimsum", "dumpling", "bao"],
     "kh": ["ឌីមសាំ"]},
    {"en_canonical": "bakery / bread", "kh_canonical": "នំប៉័ង / បង្អែម",
     "en": ["bakery", "bread", "baguette", "pastry", "sandwich", "num pang"],
     "kh": ["នំប៉័ង", "ប័ុង"]},
    {"en_canonical": "dessert / ice cream", "kh_canonical": "បង្អែម / ការ៉េម",
     "en": ["dessert", "ice cream", "sweet", "cake"],
     "kh": ["បង្អែម", "ការ៉េម", "នំខេក"]},
    {"en_canonical": "vegetarian", "kh_canonical": "ម្ហូបបួស",
     "en": ["vegetarian", "vegan"],
     "kh": ["ម្ហូបបួស", "បួស"]},
    {"en_canonical": "khmer food", "kh_canonical": "ម្ហូបខ្មែរ",
     "en": ["khmer food", "cambodian food", "khmer", "cambodian"],
     "kh": ["ម្ហូបខ្មែរ"]},
    {"en_canonical": "thai food", "kh_canonical": "ម្ហូបថៃ",
     "en": ["thai food", "thai"],
     "kh": ["ម្ហូបថៃ"]},
    {"en_canonical": "chinese food", "kh_canonical": "ម្ហូបចិន",
     "en": ["chinese food", "chinese"],
     "kh": ["ម្ហូបចិន"]},
    {"en_canonical": "korean food", "kh_canonical": "ម្ហូបកូរ៉េ",
     "en": ["korean food", "korean"],
     "kh": ["ម្ហូបកូរ៉េ"]},
    {"en_canonical": "vietnamese food", "kh_canonical": "ម្ហូបវៀតណាម",
     "en": ["vietnamese food", "vietnamese", "pho"],
     "kh": ["ម្ហូបវៀតណាម"]},
    {"en_canonical": "indian food", "kh_canonical": "ម្ហូបឥណ្ឌា",
     "en": ["indian food", "indian"],
     "kh": ["ម្ហូបឥណ្ឌា"]},
    {"en_canonical": "french food", "kh_canonical": "ម្ហូបបារាំង",
     "en": ["french food", "french"],
     "kh": ["ម្ហូបបារាំង"]},
    {"en_canonical": "italian food", "kh_canonical": "ម្ហូបអ៊ីតាលី",
     "en": ["italian food", "italian", "pasta"],
     "kh": ["ម្ហូបអ៊ីតាលី"]},
    {"en_canonical": "mexican food / tacos", "kh_canonical": "ម្ហូបម៉ិកស៊ិក",
     "en": ["mexican food", "mexican", "tacos", "taco", "burrito"],
     "kh": ["ម្ហូបម៉ិកស៊ិក", "តាកូ"]},
    {"en_canonical": "fast food", "kh_canonical": "អាហាររហ័ស",
     "en": ["fast food"],
     "kh": ["អាហាររហ័ស"]},
    {"en_canonical": "breakfast", "kh_canonical": "អាហារពេលព្រឹក",
     "en": ["breakfast", "brunch"],
     "kh": ["អាហារពេលព្រឹក"]},
    {"en_canonical": "halal food", "kh_canonical": "អាហារហាឡាល់",
     "en": ["halal food", "halal", "shawarma", "kebab", "middle eastern"],
     "kh": ["អាហារហាឡាល់", "ហាឡាល់", "កេបាប់", "សាវ៉ាម៉ា"]},
    {"en_canonical": "buffet", "kh_canonical": "អាហារប៊ូហ្វេ",
     "en": ["buffet", "all you can eat"],
     "kh": ["អាហារប៊ូហ្វេ", "ប៊ូហ្វេ"]},
    {"en_canonical": "steak", "kh_canonical": "ស្តេក",
     "en": ["steak", "steakhouse", "beef steak", "ribeye"],
     "kh": ["ស្តេក", "សាច់គោស្តេក"]},
    {"en_canonical": "salad / poke", "kh_canonical": "សាឡាដ / អាហារសុខភាព",
     "en": ["salad", "poke", "healthy food", "clean food"],
     "kh": ["សាឡាដ", "អាហារសុខភាព"]},
]

KHMER_VOCAB = set()
for _c in FOOD_CONCEPTS:
    KHMER_VOCAB.update(_c["kh"])
for _k, _info in {**KHAN_DETAILS, **PROVINCE_DETAILS}.items():
    KHMER_VOCAB.add(_info["kh"])
    for _a in _info["aliases"]:
        if KHMER_RE.search(_a):
            KHMER_VOCAB.add(_a)


def clean_df(raw: pd.DataFrame) -> pd.DataFrame:
    raw = raw.copy()
    if "image_url" not in raw.columns and "image_path" in raw.columns:
        raw["image_url"] = raw["image_path"]
    elif "image_path" not in raw.columns and "image_url" in raw.columns:
        raw["image_path"] = raw["image_url"]
    missing = set(ALL_COLUMNS) - set(raw.columns)
    if missing:
        raise ValueError(f"restaurants.csv is missing columns: {sorted(missing)}")
    d = raw[ALL_COLUMNS].copy()
    d["image_path"] = d["image_url"].copy()
    d["id"] = d["id"].fillna("").astype(str).str.strip()
    d = d[~d["id"].isin(["", "id"])]                           # "id" = header row pasted again by a sheet import
    d = d.drop_duplicates("id", keep="first")                    # ids must be unique for ChromaDB
    # the seller's Facebook page / website (optional column): only real http(s) links are kept
    page = raw["page_url"] if "page_url" in raw.columns else pd.Series("", index=raw.index)
    d["page_url"] = page.reindex(d.index).fillna("").astype(str).str.strip()
    d.loc[~d["page_url"].str.match(r"^https?://\S+$"), "page_url"] = ""
    # restaurant menu items (optional column)
    menu_col = raw["menu"] if "menu" in raw.columns else pd.Series("", index=raw.index)
    d["menu"] = menu_col.reindex(d.index).fillna("").astype(str).str.strip()
    # text: NFC + single spaces + strip; empty string instead of NaN
    for c in TEXT_COLS:
        d[c] = d[c].fillna("").astype(str).map(
            lambda s: re.sub(r"\s+", " ", ZERO_WIDTH_RE.sub("", unicodedata.normalize("NFC", s))).strip())
    # lowercase the fields we search on (names and areas keep their case for display)
    for c in ["dish_en", "description", "tags"]:
        d[c] = d[c].str.lower()
    # one spelling per place ("Boeung Keng Kang" -> "Boeng Keng Kang") and the city/province it is in
    places = d["area"].map(canonical_place)
    d["area"], d["city"] = places.str[0], places.str[1]
    d["area_key"], d["city_key"] = d["area"].str.lower(), d["city"].str.lower()
    # numbers: prices may be written in riel; unknown values stay NaN (never guessed)
    d["price_usd"] = d["price_usd"].map(parse_price)
    for c in ["lat", "lng", "rating", "open_hour", "close_hour"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    # Bilingual enrichment: auto-fill missing dish categories and enrich doc_text with area and food concepts
    doc_texts = []
    for idx, r in d.iterrows():
        name_val = r["name"]
        dish_en_val = r["dish_en"]
        dish_kh_val = r["dish_kh"]
        desc_val = r["description"]
        tags_val = r["tags"].replace(";", " ")
        area_val = r["area"]

        combined = f"{name_val} {dish_en_val} {dish_kh_val} {desc_val} {tags_val}".lower()
        additions = []

        if area_val in KHAN_DETAILS:
            kd = KHAN_DETAILS[area_val]
            additions.append(f"{area_val} {kd['kh']} {' '.join(kd['aliases'])}")
        if r["city"] in PROVINCE_DETAILS:
            cd = PROVINCE_DETAILS[r["city"]]
            additions.append(f"{r['city']} {cd['kh']} {' '.join(cd['aliases'])}")

        for c in FOOD_CONCEPTS:
            matched = False
            for k in c["kh"]:
                if k in combined:
                    matched = True
                    break
            if not matched:
                for e in c["en"]:
                    if f" {e} " in f" {combined} " or combined.startswith(f"{e} ") or combined.endswith(f" {e}"):
                        matched = True
                        break
            if matched:
                additions.extend(c["kh"])
                additions.extend(c["en"])
                if not dish_kh_val:
                    d.at[idx, "dish_kh"] = c["kh_canonical"]
                    dish_kh_val = c["kh_canonical"]
                if not dish_en_val:
                    d.at[idx, "dish_en"] = c["en_canonical"]
                    dish_en_val = c["en_canonical"]

        extra = " ".join(dict.fromkeys(additions))
        full_doc = f"{name_val} {dish_en_val} {dish_kh_val} {desc_val} {tags_val} {area_val} {extra}"
        doc_texts.append(normalize_text(full_doc))

    d["doc_text"] = doc_texts
    return d


def tokenize(text) -> list:
    """English words (with light plural stemming) + Khmer tokens (domain vocabulary words + runs + 2/3-grams)."""
    text = normalize_text(text)
    tokens = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w
              for w in WORD_RE.findall(text)]
    for word in KHMER_VOCAB:
        if word in text:
            tokens.append(word)
    for run in KHMER_RE.findall(text):          # Khmer is written without spaces
        if len(run) <= 15:
            tokens.append(run)
        if len(run) < 2:
            tokens.append(run)
        for n in (2, 3):
            tokens += [run[i:i + n] for i in range(len(run) - n + 1)]
    return tokens


def build_bm25(d: pd.DataFrame):
    """BM25 index over the combined text + the English vocabulary (for typo fixing)."""
    corpus = [tokenize(t) for t in d["doc_text"]]
    vocab = sorted({t for doc in corpus for t in doc if WORD_RE.fullmatch(t)})
    return BM25Okapi(corpus), vocab


def build_food_vocab(d: pd.DataFrame) -> list:
    """Words that name food or a kind of place (from the dish names, tags and FOOD_CONCEPTS), not every word of the
    descriptions: 'pizza', 'cafe', 'kuy', 'teav', 'គុយទាវ'. Used to tell a food question from small talk."""
    words = set()
    for c in FOOD_CONCEPTS:
        for term in c["en"] + c["kh"]:
            words.update(tokenize(term))
    for col in ("dish_en", "dish_kh", "tags"):
        for text in d[col].drop_duplicates():
            words.update(tokenize(str(text).replace(";", " ")))
    words.update(("food", "eat", "restaurant", "cafe", "coffee", "drink", "breakfast", "lunch", "dinner", "snack",
                  "dessert", "street", "menu", "ម្ហូប", "ញ៉ាំ", "ហាង", "អាហារ", "ភេសជ្ជៈ"))
    return sorted(w for w in words if len(w) >= 2)


def place_count() -> int:
    """How many places the search knows (0 before the data is loaded)."""
    return 0 if df is None else len(df)


def about_food(query) -> bool:
    """Does the message ask for food or places to eat? 'piza' -> yes (a typo of a food word), 'near CADT' -> yes,
    'what's the weather today?' -> no. A place name we do not know also says no: the chat model then decides."""
    text, flt = parse_query_filters(query)
    if flt:
        return True
    vocab = set(FOOD_VOCAB or [])
    tokens = tokenize(text)
    return any(t in vocab for t in correct_tokens(tokens, FOOD_VOCAB or [], cutoff=85)) if tokens else False


def correct_tokens(tokens, vocab, cutoff=None):
    """Replace unknown English words by the closest vocabulary word ('carry' -> 'curry')."""
    vocab_set, fixed = set(vocab), []
    for t in tokens:
        if t in vocab_set or len(t) < 4 or not WORD_RE.fullmatch(t):
            fixed.append(t)
            continue
        match = process.extractOne(t, vocab, scorer=fuzz.ratio, score_cutoff=cutoff or TYPO_CUTOFF)
        fixed.append(match[0] if match else t)
    return fixed


def fuzzy_scores(query, d: pd.DataFrame) -> np.ndarray:
    """Typo fallback: fuzzy match the query against names and dish names (spaces removed)."""
    q = normalize_text(query).replace(" ", "")
    fields = (d["name"] + " " + d["dish_en"] + " " + d["dish_kh"]).map(normalize_text).str.replace(" ", "")
    scores = np.array([fuzz.partial_ratio(q, f) for f in fields]) / 100
    scores[scores < FUZZY_MIN] = 0.0
    return scores


def bm25_scores(query):
    """Raw BM25 score per row of df (+ which method produced them)."""
    tokens = tokenize(query)
    if not tokens:
        return np.zeros(len(df)), "empty"
    fixed = correct_tokens(tokens, BM25_VOCAB)
    scores = np.asarray(bm25.get_scores(fixed), dtype=float)
    method = "bm25+typo-fix" if fixed != tokens else "bm25"
    if scores.max() <= 0:                    # no keyword hit at all -> fuzzy fallback
        scores, method = fuzzy_scores(query, df), "fuzzy"
    return scores, method


def bm25_search(query, k=TOP_K) -> pd.DataFrame:
    scores, method = bm25_scores(query)
    order = [i for i in np.argsort(-scores, kind="stable") if scores[i] > 0][:k]
    out = df.iloc[order][["id", "name", "dish_en", "dish_kh", "price_usd", "area"]].copy()
    out["score"] = scores[order].round(3)
    out["method"] = method
    return out.reset_index(drop=True)


def embed_texts(texts) -> np.ndarray:
    """Unit-length sentence embeddings (so dot product = cosine similarity)."""
    return embedder.encode(list(texts), batch_size=32, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=False)


@lru_cache(maxsize=2048)
def embed_query(query: str) -> np.ndarray:
    return embed_texts([normalize_text(query)])[0]


def open_chroma():
    return chromadb.PersistentClient(path=str(CHROMA_DIR), settings=Settings(anonymized_telemetry=False))


def get_text_collection(client):
    # one collection per embedding model, so switching EMBED_MODEL never mixes vector sizes
    name = ("food_text_" + re.sub(r"[^a-z0-9]+", "_", EMBED_MODEL.lower()))[:60].strip("_")
    return client.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})


def row_metadata(r) -> dict:
    meta = {"name": r["name"], "area": r["area"], "source": r["source"]}
    for c in ["price_usd", "rating", "lat", "lng", "open_hour", "close_hour"]:
        if pd.notna(r[c]):                   # Chroma metadata cannot hold None/NaN
            meta[c] = float(r[c])
    return meta


def build_text_index(d: pd.DataFrame, collection, batch: int = 2000) -> int:
    """Upsert new or updated rows and delete ids that are no longer in the CSV."""
    stored = collection.get(include=["documents"])
    existing = dict(zip(stored["ids"], stored["documents"]))
    existing_ids = set(existing)
    needed = d[d["id"].map(existing.get) != d["doc_text"]]       # new ids, and rows whose text changed
    if len(needed) > 0:
        for start in range(0, len(needed), batch):
            part = needed.iloc[start:start + batch]
            collection.upsert(ids=part["id"].tolist(),
                              embeddings=embed_texts(part["doc_text"]).tolist(),
                              documents=part["doc_text"].tolist(),
                              metadatas=[row_metadata(r) for _, r in part.iterrows()])
    stale = sorted(existing_ids - set(d["id"]))
    for start in range(0, len(stale), batch):
        collection.delete(ids=stale[start:start + batch])
    return collection.count()


def embedding_scores(query) -> np.ndarray:
    """Cosine similarity per row of df (rows beyond EMBED_CANDIDATES, if any, get 0)."""
    n = min(EMBED_CANDIDATES, text_col.count())
    res = text_col.query(query_embeddings=[embed_query(query).tolist()], n_results=n,
                         include=["distances"])
    sims = {i: 1.0 - dist for i, dist in zip(res["ids"][0], res["distances"][0])}
    return np.array([sims.get(i, 0.0) for i in df["id"]])


def embedding_search(query, k=TOP_K) -> pd.DataFrame:
    scores = embedding_scores(query)
    order = np.argsort(-scores, kind="stable")[:k]
    out = df.iloc[order][["id", "name", "dish_en", "dish_kh", "price_usd", "area"]].copy()
    out["score"] = scores[order].round(3)
    return out.reset_index(drop=True)


def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance in km (works on numbers or numpy/pandas arrays)."""
    lat1, lng1, lat2, lng2 = (np.radians(np.asarray(x, dtype=float)) for x in (lat1, lng1, lat2, lng2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def minmax(x) -> np.ndarray:
    """Scale to 0..1 so BM25 and cosine scores become comparable."""
    x = np.asarray(x, dtype=float)
    lo, hi = x.min(), x.max()
    return np.zeros_like(x) if hi - lo < 1e-12 else (x - lo) / (hi - lo)


PRICE_RE = re.compile(r"(?:under|below|less than|cheaper than|max|<|ក្រោម|តិចជាង|តម្លៃក្រោម|តំលៃក្រោម)\s*\$?\s*(\d+(?:\.\d+)?)"
                      r"\s*(\$|usd|dollars?|ដុល្លារ|khr|riel|៛|រៀល)?")
FILTER_PHRASES = [  # (regex, filter name, value)
    (r"\bopen (?:right )?now\b|បើកឥឡូវ|កំពុងបើក|បើកពេលនេះ", "open_now", True),
    (r"\bopen late\b|\blate night\b|\blate\b|បើកយប់|យប់ជ្រៅ|ពេលយប់", "open_late", True),
    (r"\bcheap\b|\bbudget\b|ថោក|ថោកៗ|ធូរថ្លៃ|តម្លៃសមរម្យ|ធូរៗ", "max_price", CHEAP_MAX_USD),
    (r"\btop rated\b|\bbest rated\b|ល្បី|ពិន្ទុខ្ពស់", "min_rating", 4.5),
]


NEAR_ME_RE = re.compile(r"\b(?:near\s*me|nearby|near\s+here|around\s+me|around\s+here|close\s+to\s+me|closest|nearest)\b"
                        r"|ជិតខ្ញុំ|នៅជិតខ្ញុំ|ក្បែរខ្ញុំ|ជុំវិញខ្ញុំ|ជិតៗ")

SPATIAL_PREFIXES = r"(?:នៅម្ដុំ|នៅម្តុំ|ម្ដុំ|ម្តុំ|នៅក្បែរ|ក្បែរ|នៅជិត|ជិត|នៅ|ឯ|in|around|at|near)\s*"

CONVERSATIONAL_STOP_PATTERNS = [
    r"តើមាន\s*ហាង", r"តើមាន\s*កន្លែង", r"តើមាន",
    r"មាន\s*ហាង", r"មាន\s*កន្លែង", r"មាន",
    r"ហាង", r"កន្លែង",
    r"ឆ្ងាញ់ៗ?", r"ល្អ", r"ល្បី",
    r"ចង់ញ៉ាំ", r"ចង់រក", r"ចង់បាន",
    r"ជួយណែនាំ", r"សូមណែនាំ", r"ណែនាំ",
    r"ជួយរក", r"រក",
    r"អត់\s*\??", r"ទេ\s*\??", r"ដែរ\s*\??", r"ផង\s*\??",
    r"ណាខ្លះ\s*\??", r"ខ្លះ\s*\??", r"មួយណា\s*\??", r"ឯណា\s*\??", r"នៅឯណា\s*\??",
    r"ណា(?=[^ក-៿]|$)", r"មួយ(?=[^ក-៿]|$)",
    r"\b(?:is|are)\s+there\s+any\b", r"\bwhere\s+(?:can\s+i|to)\s+(?:find|get|eat)\b",
    r"\blooking\s+for\b", r"\bi\s+want\s+(?:to\s+eat)?\b",
    r"\b(?:can\s+you\s+)?recommend(?:\s+me)?\b",
    r"\b(?:delicious|good|tasty|best|popular)\b",
    r"\b(?:restaurant|restaurants|place|places|shop|shops)\b",
    r"\b(?:please|any)\b",
]


def expand_query(text: str) -> str:
    """Expand search text with cross-lingual food synonyms (Khmer <-> English)."""
    t = normalize_text(text)
    if not t:
        return ""
    additions = []
    for c in FOOD_CONCEPTS:
        matched = False
        for k in c["kh"]:
            if k in t:
                matched = True
                break
        if not matched:
            for e in c["en"]:
                if re.search(r"\b" + re.escape(e) + r"\b", t):
                    matched = True
                    break
        if matched:
            additions.extend(c["kh"])
            additions.extend(c["en"])
    extra = " ".join(dict.fromkeys(additions))
    return f"{text} {extra}".strip() if extra else text


def parse_query_filters(query):
    """Pull conditions and location out of the query text, cleaning conversational filler.
    Returns (clean search text, filters dict)."""
    orig = normalize_text(query)
    q = orig
    f = {}

    # 1. Near me
    if NEAR_ME_RE.search(q):
        f["near"] = "me"
        q = NEAR_ME_RE.sub(" ", q)

    # 2. Price
    m = PRICE_RE.search(q)
    if m:
        amount = float(m.group(1))
        unit = m.group(2) or "$"
        f["max_price"] = khr_to_usd(amount) if unit in ("khr", "riel", "៛", "រៀល") else amount
        q = q.replace(m.group(0), " ")

    # 3. Filter phrases (open now, open late, cheap, top rated)
    for pattern, key, value in FILTER_PHRASES:
        if re.search(pattern, q):
            f.setdefault(key, value)
            q = re.sub(pattern, " ", q)

    # 4. Place detection: a Phnom Penh khan ("BKK1", "ទួលគោក") and/or a city or province ("Siem Reap", "សៀមរាប")
    found = {}
    for alias, kind, value in PLACE_ALIASES:
        if kind in found:
            continue
        if KHMER_RE.search(alias):
            pattern = r"(?:" + SPATIAL_PREFIXES + r")?" + re.escape(alias)
        else:
            pattern = r"(?:" + SPATIAL_PREFIXES + r")?(?<![a-z0-9])" + re.escape(alias) + r"(?![a-z0-9])"
        match = re.search(pattern, q)
        if match:
            found[kind] = value
            q = q[:match.start()] + " " + q[match.end():]
    found_area = found.get("area")
    if found_area:
        f["area"] = found_area               # every khan is in Phnom Penh, so the khan is enough
    elif found.get("city"):
        f["city"] = found["city"]

    # Strip remaining aliases of the same place if present (e.g. BKK1 when Boeung Keng Kang was already matched)
    for alias, kind, value in PLACE_ALIASES:
        if found.get(kind) == value:
            if KHMER_RE.search(alias):
                q = re.sub(re.escape(alias), " ", q)
            else:
                q = re.sub(r"(?<![a-z0-9])" + re.escape(alias) + r"(?![a-z0-9])", " ", q)

    # 5. Near landmark (e.g. "near CADT", "ជិត CADT", "នៅម្តុំ CADT")
    if "near" not in f and not found_area:
        near_word = r"(?:near|around|ជិត|នៅជិត|ក្បែរ|នៅក្បែរ|នៅម្ដុំ|នៅម្តុំ)\s+"
        # a landmark of several words when it ends the query ("near Aeon Mall"), else the one word after "near"
        m_near = (re.search(near_word + r"([^,?.!]{2,60}?)\s*[?.!]*$", q)
                  or re.search(near_word + r"([^\s,?.!]{2,40})", q))
        if m_near:
            target = m_near.group(1).strip()
            if target.lower() not in AREA_MAP and target.lower() not in CITY_MAP:
                f["near"] = target
                q = q[:m_near.start()] + " " + q[m_near.end():]

    # 6. Conversational stop phrases & punctuation stripping
    for pat in CONVERSATIONAL_STOP_PATTERNS:
        q = re.sub(pat, " ", q, flags=re.IGNORECASE)

    q = re.sub(r"[?.,!():;\"\'/\\\[\]{}]+", " ", q)
    clean_q = re.sub(r"\s+", " ", q).strip()

    # If conversational stripping left nothing, fallback
    if not clean_q and not f:
        clean_q = re.sub(r"\s+", " ", orig).strip()

    return clean_q, f


def is_open_at(open_h, close_h, hour) -> bool:
    """Opening hours may cross midnight (e.g. 18 -> 2). Unknown hours = not open."""
    if pd.isna(open_h) or pd.isna(close_h):
        return False
    open_h, close_h = float(open_h), float(close_h) % 24
    if open_h == close_h:
        return True                          # open 24 hours
    if open_h < close_h:
        return open_h <= hour < close_h
    return hour >= open_h or hour < close_h


def closes_late(open_h, close_h) -> bool:
    if pd.isna(open_h) or pd.isna(close_h):
        return False
    return close_h >= LATE_HOUR or close_h < open_h


def apply_filters(d: pd.DataFrame, f: dict, now=None, relevant=None, unverified=None) -> np.ndarray:
    """Boolean mask. relevant = the rows that match the words of the question: a price, rating or opening-hours
    condition is applied only if at least one of those rows has the value to check. If none does, the condition
    cannot be checked in our data: it is not applied and its name ('price', 'rating', 'hours') is added to
    `unverified`, so the answer can say so instead of returning unrelated places that happen to have a price."""
    mask = np.ones(len(d), dtype=bool)
    relevant = np.ones(len(d), dtype=bool) if relevant is None else relevant
    unverified = [] if unverified is None else unverified
    if f.get("area"):
        mask &= d["area_key"].eq(normalize_text(f["area"])).to_numpy()
    if f.get("city"):
        mask &= d["city_key"].eq(normalize_text(f["city"])).to_numpy()
    relevant = relevant & mask                    # only places in the named khan or city can tell whether a condition is checkable

    def condition(name, match):
        nonlocal mask
        if (match & relevant).any():
            mask &= match
        elif name not in unverified:
            unverified.append(name)

    if f.get("max_price") is not None:
        condition("price", d["price_usd"].le(f["max_price"]).to_numpy())
    if f.get("min_rating") is not None:
        condition("rating", d["rating"].ge(f["min_rating"]).to_numpy())
    if f.get("open_now"):
        now = now or datetime.now(TZ)
        hour = now.hour + now.minute / 60
        condition("hours", np.array([is_open_at(o, c, hour) for o, c in zip(d["open_hour"], d["close_hour"])], dtype=bool))
    if f.get("open_late"):
        condition("hours", np.array([closes_late(o, c) for o, c in zip(d["open_hour"], d["close_hour"])], dtype=bool))
    return mask


_geocache = {}
_last_geocode = [0.0]
LANDMARK_MARGIN = 0.15                       # how much better known (OSM "importance") a place outside Phnom Penh must be


def _nominatim(params):
    wait = 1.1 - (time.monotonic() - _last_geocode[0])
    if wait > 0:
        time.sleep(wait)
    _last_geocode[0] = time.monotonic()
    r = requests.get("https://nominatim.openstreetmap.org/search", timeout=6,
                     headers={"User-Agent": "food-search-student-project (Information Retrieval course)"},
                     params={"format": "jsonv2", "limit": 1, "countrycodes": "kh", **params})
    r.raise_for_status()
    hits = r.json()
    if not hits:
        return None
    return float(hits[0]["lat"]), float(hits[0]["lon"]), float(hits[0].get("importance") or 0)


def geocode_place(name: str, city: str = None):
    """Landmark name -> (lat, lng) via OpenStreetMap's Nominatim (max 1 request per second).
    Looks in the given city, else in Phnom Penh first and then anywhere in Cambodia ("near Angkor Wat")."""
    name = re.sub(r"\s+", " ", name or "").strip()[:60]
    if len(name) < 2:
        return None
    key = f"{name.lower()}|{(city or '').lower()}"
    if key in _geocache:
        return _geocache[key]
    try:
        if city and city != HOME_CITY:
            hit = _nominatim({"q": f"{name}, {city}"})
        else:
            hit = _nominatim({"q": f"{name}, {HOME_CITY}", "viewbox": "104.75,11.75,105.05,11.40", "bounded": 1})
            if not city:                     # a famous place elsewhere ("Angkor Wat") beats a small namesake in Phnom Penh
                wide = _nominatim({"q": name})
                if wide and (not hit or wide[2] > hit[2] + LANDMARK_MARGIN):
                    hit = wide
    except Exception as e:
        log.warning("geocoding failed (%s)", type(e).__name__)
        return None                          # not cached, so a later try can work
    spot = hit[:2] if hit else None
    _geocache[key] = spot
    return spot


def hybrid_search(query, k=TOP_K, user_loc=None, weight=None, filters=None,
                  use_query_filters=True, now=None) -> pd.DataFrame:
    """BM25 + embeddings + rating + distance, after filters. user_loc = (lat, lng)."""
    weight = HYBRID_WEIGHT if weight is None else weight
    text, parsed = parse_query_filters(query) if use_query_filters else (normalize_text(query), {})
    flt = {**parsed, **(filters or {})}
    out = df.copy()
    if text:
        search_text = expand_query(text)
        bm, method = bm25_scores(search_text)
        em = embedding_scores(search_text)
    else:                                    # query was only conditions, e.g. "open now"
        bm, em, method = np.zeros(len(df)), np.zeros(len(df)), "filters-only"
    out["bm25"], out["embed"] = minmax(bm), minmax(em)
    out["rating_boost"] = RATING_BOOST * out["rating"].fillna(0) / 5
    out["distance_km"], out["distance"] = np.nan, 0.0
    if user_loc:
        out["distance_km"] = haversine_km(user_loc[0], user_loc[1], out["lat"], out["lng"])
        out["distance"] = DISTANCE_WEIGHT / (1 + out["distance_km"].fillna(1e6))
    out["home"] = 0.0
    if not (flt.get("area") or flt.get("city") or user_loc):      # no place named: Phnom Penh first
        out["home"] = HOME_CITY_BONUS * out["city"].eq(HOME_CITY)
    out["final"] = (weight * out["embed"] + (1 - weight) * out["bm25"] + out["rating_boost"] + out["distance"]
                    + out["home"])
    match_mask = (bm > 0) | (em >= MIN_EMBED_SIM) if text else np.ones(len(out), dtype=bool)
    unverified = []
    keep = apply_filters(out, flt, now, match_mask, unverified)
    if text and (keep & match_mask).any():     # drop rows that match neither by keyword nor by meaning
        keep &= match_mask
    out = out[keep].sort_values("final", ascending=False, kind="stable")
    if flt.get("near") and user_loc and len(out):      # "near me" / "near X": closest of the clearly relevant places first
        pool = out[out["final"] >= 0.7 * out["final"].iloc[0]].head(NEAR_POOL)
        out = pool.sort_values("distance_km", kind="stable")
    out = out.head(k).reset_index(drop=True)
    out.attrs.update(filters=flt, method=method, query_text=text, unverified=unverified)
    return out


def get_rows(ids) -> pd.DataFrame:
    """Rows for the given ids, in that order (unknown ids are skipped)."""
    known = set(df["id"])
    ids = [i for i in dict.fromkeys(ids) if i in known]
    return df.set_index("id").loc[ids].reset_index()


CLIP_ARCH, CLIP_PRETRAINED = "ViT-B-32", "laion2b_s34b_b79k"
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
_clip = {}          # model, preprocess, tokenizer (loaded on first use)


def list_photos(folder=None) -> list:
    folder = Path(folder or PHOTOS_DIR)
    return sorted(p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS) if folder.exists() else []


def load_clip():
    if not _clip:
        import open_clip
        model, _, preprocess = open_clip.create_model_and_transforms(
            CLIP_ARCH, pretrained=CLIP_PRETRAINED, device=DEVICE)
        model.eval()
        _clip.update(model=model, preprocess=preprocess, tokenizer=open_clip.get_tokenizer(CLIP_ARCH))
    return _clip["model"], _clip["preprocess"], _clip["tokenizer"]


@torch.no_grad()
def clip_embed_images(paths) -> np.ndarray:
    from PIL import Image
    model, preprocess, _ = load_clip()
    batch = torch.stack([preprocess(Image.open(p).convert("RGB")) for p in paths]).to(DEVICE)
    feats = model.encode_image(batch)
    return (feats / feats.norm(dim=-1, keepdim=True)).float().cpu().numpy()


@torch.no_grad()
def clip_embed_text(texts) -> np.ndarray:
    model, _, tokenizer = load_clip()
    feats = model.encode_text(tokenizer(list(texts)).to(DEVICE))
    return (feats / feats.norm(dim=-1, keepdim=True)).float().cpu().numpy()


def get_image_collection(client):
    return client.get_or_create_collection("food_images_" + CLIP_ARCH.lower().replace("-", "_"),
                                           metadata={"hnsw:space": "cosine"})


def build_image_index(paths, collection, d: pd.DataFrame) -> int:
    """Photo -> restaurant id: by image_url file name, else by file stem (<id>.jpg -> <id>)."""
    img_col = d["image_url"] if "image_url" in d.columns else d["image_path"]
    by_name = {Path(p).name: i for i, p in zip(d["id"], img_col) if p}
    ids = [p.relative_to(PHOTOS_DIR).as_posix() for p in paths]
    embs = np.vstack([clip_embed_images(paths[i:i + 16]) for i in range(0, len(paths), 16)])
    collection.upsert(ids=ids, embeddings=embs.tolist(),
                      metadatas=[{"path": p.relative_to(ROOT).as_posix(),
                                  "restaurant_id": by_name.get(p.name, p.stem)} for p in paths])
    stale = set(collection.get(include=[])["ids"]) - set(ids)
    if stale:
        collection.delete(ids=sorted(stale))
    return collection.count()


def _image_query(emb, k):
    res = image_col.query(query_embeddings=[emb.tolist()], n_results=min(k, image_col.count()),
                          include=["metadatas", "distances"])
    return [{"path": ROOT / m["path"], "restaurant_id": m["restaurant_id"], "similarity": round(1 - dist, 3)}
            for m, dist in zip(res["metadatas"][0], res["distances"][0])]


def search_images_by_text(query, k=TOP_K):
    return _image_query(clip_embed_text([query])[0], k)


def search_images_by_image(path, k=TOP_K):
    return _image_query(clip_embed_images([path])[0], k)


def image_search_available() -> bool:
    return bool(HAS_PHOTOS) and image_col is not None


def init():
    """Load data/restaurants.csv and build (or refresh) every index. Call once at startup."""
    global df, bm25, BM25_VOCAB, FOOD_VOCAB, embedder, text_col, image_col, HAS_PHOTOS
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    from sentence_transformers import SentenceTransformer
    df = clean_df(pd.read_csv(DATA_DIR / "restaurants.csv", dtype={"id": str, "phone_public": str, "price_usd": str}))
    bm25, BM25_VOCAB = build_bm25(df)
    FOOD_VOCAB = build_food_vocab(df)
    embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
    client = open_chroma()
    text_col = get_text_collection(client)
    build_text_index(df, text_col)
    photos = list_photos()
    HAS_PHOTOS = bool(photos)
    if HAS_PHOTOS:
        image_col = get_image_collection(client)
        build_image_index(photos, image_col, df)
    log.info("search ready: %d places, %d photos, device=%s", len(df), len(photos), DEVICE)
    from . import dishes                 # photo -> dish + recipes (imported here: dishes needs this module loaded)
    dishes.init()


In [ ]:
%%writefile export/backend/food_search/dishes.py
"""Dish recognition (photo -> which dish) and recipes (how to make it, how long it takes).

Photo -> one of the 101 Food-101 dishes, with
  * the EfficientNet-B0 trained in food_classification.ipynb (models/food101_efficientnet_b0.pt), or, while that
    file is missing, CLIP zero-shot (the photo is compared with sentences like "a photo of pizza, a type of food.").
Dish -> its recipe from data/recipes.csv (Food.com: ingredients, steps, prep / cook minutes; built by
scrapers/recipes_foodcom.py).
Message -> is it a recipe question ("how do I make it?", "how many minutes?"), and about which dish?

  init()                                   called by search.init(): dish list, recipes, model
  available() / model_label()              can photos be recognised, and by which model
  classify(path, k=3)                      [{"id", "name", "confidence"}, ...], best first
  dish_json(id, ...) / recipe_json(id)     what the API returns
  find_dish(text)                          a dish named in a message, or None
  recipe_intent(text, has_dish)            None, or what the question asks: time | ingredients | servings |
                                           calories | steps | other
  recipe_answer(question, id, lang, ...)   {text, provider, model, label, focus}
"""
import json
import logging
import re

import pandas as pd
import torch

from .config import DATA_DIR, ROOT
from .llm import NoModelAnswered, as_sentences, complete

log = logging.getLogger("food_search.dishes")

MODELS_DIR = ROOT / "models"
MODEL_PATH = MODELS_DIR / "food101_efficientnet_b0.pt"
CLASSES_PATH = MODELS_DIR / "food101_classes.json"
RECIPES_PATH = DATA_DIR / "recipes.csv"
SURE_AT = 0.50            # below this the app says "it might be ..." and offers the other guesses
CLIP_TEMPLATES = ["a photo of {}, a type of food.", "a close-up photo of {}.", "a photo of a plate of {}.",
                  "{}, food photography."]      # the same sentences as food_classification.ipynb, section 5

DISHES = {}               # id -> display name ("pad_thai" -> "Pad thai")
RECIPES = {}              # id -> recipe dict
_model = {}               # kind, label, model, transform, classes, text (CLIP)
_name_re = None           # every dish name and alias, longest first
_name_ids = {}            # lower-case name -> dish id

# Other names people use for a dish (the display names are matched too)
ALIASES = {"hamburger": ["burger", "burgers", "cheeseburger"], "donuts": ["doughnut", "doughnuts", "donut"],
           "cup_cakes": ["cupcake", "cupcakes"], "french_fries": ["fries"], "hot_dog": ["hot dogs", "hotdog"],
           "macaroni_and_cheese": ["mac and cheese", "mac n cheese", "mac & cheese"], "tacos": ["taco"],
           "dumplings": ["dumpling", "dim sum", "siu mai", "shumai"], "gyoza": ["potstickers", "pot stickers"],
           "spring_rolls": ["spring roll"], "chicken_wings": ["wings", "buffalo wings"], "waffles": ["waffle"],
           "pancakes": ["pancake"], "churros": ["churro"], "macarons": ["macaron"], "samosa": ["samosas"],
           "beignets": ["beignet"], "escargots": ["escargot", "snails"], "omelette": ["omelet"],
           "spaghetti_bolognese": ["bolognese"], "spaghetti_carbonara": ["carbonara"], "pho": ["phở"],
           "creme_brulee": ["creme brulee", "crème brûlée"], "ice_cream": ["icecream"], "lasagna": ["lasagne"],
           "steak": ["beef steak"], "pork_chop": ["pork chops"], "oysters": ["oyster"], "scallops": ["scallop"],
           "mussels": ["mussel"], "deviled_eggs": ["devilled eggs"], "crab_cakes": ["crab cake"],
           "onion_rings": ["onion ring"], "lobster_roll_sandwich": ["lobster roll"], "cheese_plate": ["cheese board"],
           "pulled_pork_sandwich": ["pulled pork"], "grilled_cheese_sandwich": ["grilled cheese"],
           "chicken_quesadilla": ["quesadilla", "quesadillas"], "breakfast_burrito": ["burrito"]}


# ======================= loading =======================

def init():
    """Loads the dish list, the recipes and the photo model. Safe to call again; never raises."""
    global _name_re, _name_ids
    try:
        DISHES.clear()
        DISHES.update({d["id"]: d["name"] for d in json.loads(CLASSES_PATH.read_text(encoding="utf-8"))})
    except (OSError, ValueError) as e:
        log.warning("no dish list (%s): photo recognition and recipes are off", e)
        return
    RECIPES.clear()
    if RECIPES_PATH.exists():
        for r in pd.read_csv(RECIPES_PATH, dtype={"recipe_id": str}).to_dict("records"):
            r["ingredients"], r["steps"] = json.loads(r["ingredients"]), json.loads(r["steps"])
            r = {k: (None if isinstance(v, float) and pd.isna(v) else v) for k, v in r.items()}
            for k in ("prep_minutes", "cook_minutes", "total_minutes", "servings", "calories_per_serving"):
                r[k] = int(r[k]) if r[k] is not None else None       # read as float when a column has gaps
            RECIPES[r["dish_id"]] = r
    names = {}
    for dish_id, name in DISHES.items():
        for n in [name, dish_id.replace("_", " ")] + ALIASES.get(dish_id, []):
            names[n.lower()] = dish_id
    # longest first, so "chicken wings" wins over "wings" and "french onion soup" over "french fries"' "fries"
    _name_re = re.compile(r"\b(" + "|".join(re.escape(n) for n in sorted(names, key=len, reverse=True)) + r")\b", re.I)
    _name_ids = names
    _load_model()
    log.info("dishes ready: %d dishes, %d recipes, photo model: %s", len(DISHES), len(RECIPES), model_label() or "none")


def _load_model():
    _model.clear()
    try:
        if MODEL_PATH.exists():
            import torchvision
            from torchvision import transforms as T
            from .search import DEVICE
            ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
            net = torchvision.models.efficientnet_b0(weights=None)
            net.classifier[1] = torch.nn.Linear(net.classifier[1].in_features, len(ckpt["classes"]))
            net.load_state_dict(ckpt["state_dict"])
            _model.update(kind="efficientnet", model=net.eval().to(DEVICE), classes=list(ckpt["classes"]),
                          transform=T.Compose([T.Resize(ckpt["resize"]), T.CenterCrop(ckpt["img_size"]), T.ToTensor(),
                                               T.Normalize(ckpt["mean"], ckpt["std"])]),
                          label="EfficientNet-B0 (Food-101)" + (" · quick run" if ckpt.get("quick_run") else ""),
                          top1=ckpt.get("test_top1"))
        else:
            from .search import DEVICE, load_clip
            model, preprocess, tokenizer = load_clip()
            classes = list(DISHES)
            with torch.no_grad():
                vectors = []
                for dish_id in classes:
                    tokens = tokenizer([t.format(DISHES[dish_id].lower()) for t in CLIP_TEMPLATES]).to(DEVICE)
                    v = model.encode_text(tokens).float()
                    v = (v / v.norm(dim=-1, keepdim=True)).mean(dim=0)
                    vectors.append(v / v.norm())
            _model.update(kind="clip", model=model, transform=preprocess, classes=classes, text=torch.stack(vectors),
                          label="CLIP zero-shot")
    except Exception as e:                       # no weights, no CLIP download offline, ...
        log.warning("photo model not loaded (%s)", e)
        _model.clear()


def available() -> bool:
    return bool(_model) and bool(DISHES)


def model_label():
    return _model.get("label")


# ======================= photo -> dish =======================

@torch.no_grad()
def classify(path, k=3) -> list:
    """The k most likely dishes for a photo, best first. Raises if the file is not an image."""
    from PIL import Image
    from .search import DEVICE
    x = _model["transform"](Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE)
    if _model["kind"] == "clip":
        img = _model["model"].encode_image(x).float()
        logits = 100 * (img / img.norm(dim=-1, keepdim=True)) @ _model["text"].T
    else:
        logits = _model["model"](x).float()
    probs = logits.softmax(dim=1)[0].cpu()
    p, idx = probs.topk(min(k, len(probs)))
    classes = _model["classes"]
    return [{"id": classes[i], "name": DISHES.get(classes[i], classes[i].replace("_", " ").capitalize()),
             "confidence": round(float(q), 3)} for q, i in zip(p.tolist(), idx.tolist())]


# ======================= texts =======================

TEXT = {
    "en": {"sure": "This looks like **{dish}** ({p}% sure).",
           "unsure": "I'm not sure. It might be **{dish}** ({p}%).",
           "picked": "**{dish}**",
           "no_recipe": "I don't have a recipe for {dish} yet.",
           "time": "**{title}** takes about **{total}** in total: {prep} to prepare and {cook} to cook.",
           "time_no_cook": "**{title}** takes about **{total}** to prepare (no cooking).",
           "wait": " The recipe also has waiting time ({what}) that Food.com may not count.",
           "serves": " It serves {n}.",
           "ingredients": "**{title}** needs {n} ingredients:",
           "steps": "How to make **{title}**: {n} steps, about {total} in total.",
           "servings": "**{title}** serves {n}.",
           "servings_unknown": "The recipe for {title} does not say how many it serves.",
           "calories": "**{title}** has about **{cal} calories** per serving.",
           "calories_unknown": "The recipe for {title} has no calorie information.",
           "other": "Here is the recipe for **{title}**.",
           "min": "{n} min", "h": "{h} h", "hmin": "{h} h {m} min",
           "chill": "chilling", "marinate": "marinating", "rise": "letting the dough rise", "freeze": "freezing",
           "rest": "resting", "soak": "soaking"},
    "kh": {"sure": "នេះមើលទៅដូចជា **{dish}** (ប្រាកដ {p}%)។",
           "unsure": "ខ្ញុំមិនប្រាកដទេ។ ប្រហែលជា **{dish}** ({p}%)។",
           "picked": "**{dish}**",
           "no_recipe": "ខ្ញុំមិនទាន់មានរូបមន្តសម្រាប់ {dish} នៅឡើយទេ។",
           "time": "**{title}** ប្រើពេលប្រហែល **{total}** សរុប៖ រៀបចំ {prep} និងចម្អិន {cook}។",
           "time_no_cook": "**{title}** ប្រើពេលរៀបចំប្រហែល **{total}** (មិនចាំបាច់ចម្អិន)។",
           "wait": " រូបមន្តនេះក៏មានពេលរង់ចាំ ({what}) ដែល Food.com ប្រហែលមិនបានរាប់បញ្ចូល។",
           "serves": " សម្រាប់ {n} នាក់។",
           "ingredients": "**{title}** ត្រូវការគ្រឿងផ្សំ {n} មុខ៖",
           "steps": "របៀបធ្វើ **{title}**៖ {n} ជំហាន ប្រហែល {total} សរុប។",
           "servings": "**{title}** សម្រាប់ {n} នាក់។",
           "servings_unknown": "រូបមន្ត {title} មិនបានប្រាប់ថាសម្រាប់ប៉ុន្មាននាក់ទេ។",
           "calories": "**{title}** មានប្រហែល **{cal} កាឡូរី** ក្នុងមួយចំណែក។",
           "calories_unknown": "រូបមន្ត {title} មិនមានព័ត៌មានកាឡូរីទេ។",
           "other": "នេះជារូបមន្តសម្រាប់ **{title}**។",
           "min": "{n} នាទី", "h": "{h} ម៉ោង", "hmin": "{h} ម៉ោង {m} នាទី",
           "chill": "ដាក់ទូទឹកកក", "marinate": "ប្រឡាក់", "rise": "ទុកម្សៅឱ្យដុះ", "freeze": "ដាក់ក្លាសេ",
           "rest": "ទុកឱ្យសម្រាក", "soak": "ត្រាំ"},
}
WAIT_WORDS = {"chill": r"\b(chill|refrigerat|fridge|overnight)", "marinate": r"\bmarinat", "rise": r"\b(rise|double in size|proof)",
              "freeze": r"\bfreez", "soak": r"\bsoak", "rest": r"\b(let (it )?rest|rest for)"}


def fmt_minutes(n, lang="en") -> str:
    t = TEXT[lang]
    n = int(n or 0)
    if n < 60:
        return t["min"].format(n=n)
    return t["h"].format(h=n // 60) if n % 60 == 0 else t["hmin"].format(h=n // 60, m=n % 60)


def waiting(recipe) -> list:
    text = " ".join(recipe["steps"]).lower()
    return [k for k, rx in WAIT_WORDS.items() if re.search(rx, text)]


def photo_lead(dish, lang="en") -> str:
    """The sentence above the dish: '... looks like Pizza (93% sure).' (no confidence = the user picked the dish)."""
    t = TEXT.get(lang, TEXT["en"])
    if dish.get("confidence") is None:
        return t["picked"].format(dish=dish["name"])
    key = "sure" if dish["confidence"] >= SURE_AT else "unsure"
    return t[key].format(dish=dish["name"], p=min(99, round(dish["confidence"] * 100)))   # a model is never 100% right


def dish_json(dish_id, confidence=None, alternatives=None) -> dict:
    return {"id": dish_id, "name": DISHES.get(dish_id, dish_id.replace("_", " ").capitalize()),
            "confidence": confidence, "sure": confidence is None or confidence >= SURE_AT,
            "alternatives": alternatives or [], "model": model_label() if confidence is not None else None,
            "has_recipe": dish_id in RECIPES}


def recipe_json(dish_id):
    r = RECIPES.get(dish_id)
    if not r:
        return None
    return {"dish_id": dish_id, "dish_name": r["dish_name"], "title": r["recipe_title"],
            "description": r.get("description") or None, "prep_minutes": r["prep_minutes"],
            "cook_minutes": r["cook_minutes"], "total_minutes": r["total_minutes"], "servings": r.get("servings"),
            "calories": r.get("calories_per_serving"), "difficulty": r.get("difficulty"),
            "ingredients": r["ingredients"], "steps": r["steps"], "waiting": waiting(r),
            "source": r.get("source") or "Food.com", "source_url": r.get("source_url") or None}


# ======================= message -> recipe question =======================

STEPS_RE = re.compile(r"\b(recipes?|how (do|can|should|would|to) (i |you |we |one )?(make|cook|prepare|bake|do)|"
                      r"how (is|are) (it|this|that|they) (made|cooked|prepared)|steps?|method|instructions?|"
                      r"teach me|show me how)\b|រូបមន្ត|របៀបធ្វើ|ធ្វើ(យ៉ាង|ដូច)ម្តេច|ធ្វើម៉េច|ចម្អិន(យ៉ាង)?ម៉េច", re.I)
TIME_RE = re.compile(r"\b(how long|how many (minutes|mins|hours)|how much time|(cook|cooking|prep|preparation|baking|"
                     r"total) time|takes? to (make|cook|prepare|bake)|(minutes|hours) to (make|cook|prepare|bake))\b|"
                     r"ប៉ុន្មាននាទី|ប៉ុន្មានម៉ោង|យូរប៉ុណ្ណា|យូរទេ|ប្រើពេល", re.I)
INGREDIENTS_RE = re.compile(r"\b(ingredients?|what (do|will) (i|you|we) need|what('s| is) in (it|this|that|them)|"
                            r"what goes in|shopping list|made (of|from|with))\b|គ្រឿងផ្សំ|ត្រូវការអ្វីខ្លះ", re.I)
SERVINGS_RE = re.compile(r"\b(servings?|serves|how many (people|persons|portions|plates))\b|ប៉ុន្មាននាក់", re.I)
CALORIES_RE = re.compile(r"\b(calories?|kcal|nutrition|healthy)\b|កាឡូរី", re.I)
# a free question about the dish just shown ("can I use chicken instead?"): answered by the AI from the recipe
OTHER_RE = re.compile(r"\b(instead|substitut\w*|replace|without|swap|vegetarian|vegan|spicy|oven|air ?fryer|grill|"
                      r"fry|bake|store|keep|freeze|reheat|leftovers?|ahead|kids|easy|hard|difficult|serve with|"
                      r"pair|side dish|origin)\b", re.I)
REFERS_RE = re.compile(r"\b(it|this|that|they|them|this dish|that dish|the dish|the recipe)\b|វា|នេះ|នោះ", re.I)
PLACE_RE = re.compile(r"\b(near|nearby|where|open|deliver|delivery|restaurant|restaurants|shop|place|places|buy|eat out|"
                      r"order|price|cheap|cost|wait|queue|get there|drive|walk|reservation|book)\b|នៅណា|ជិត|ហាង|ទិញ|បើក", re.I)


def find_dish(text):
    """The dish a message names ('how do I make pad thai?' -> 'pad_thai'), or None."""
    if not _name_re:
        return None
    m = _name_re.search(text or "")
    return _name_ids.get(m.group(1).lower()) if m else None


def recipe_intent(text, has_dish=False):
    """What a recipe question asks, or None when it is not one (then the normal place search answers).
    has_dish: a dish is in focus (just recognised from a photo, or named in the message). Without one it is never a
    recipe question ("how do I make a reservation?")."""
    if not has_dish or not text or PLACE_RE.search(text):   # "where can I buy pizza", "pizza open now": a place search
        return None
    for kind, rx in (("time", TIME_RE), ("ingredients", INGREDIENTS_RE), ("servings", SERVINGS_RE),
                     ("calories", CALORIES_RE), ("steps", STEPS_RE)):
        if rx.search(text):
            return kind
    if OTHER_RE.search(text) and REFERS_RE.search(text):
        return "other"
    return None


def resolve_reference(text, dish_id) -> str:
    """'where can I eat it near me?' after a photo of pizza -> 'where can I eat pizza near me?'."""
    if not dish_id or find_dish(text):
        return text
    name = DISHES.get(dish_id, "").lower()
    return re.sub(r"\b(it|this|that|this dish|that dish|the dish)\b", name, text, count=1, flags=re.I) if name else text


def recipe_prompt(question, r, lang) -> str:
    return (
        "You answer a cooking question using ONLY the recipe below (from Food.com). "
        f"Reply in {'Khmer' if lang == 'kh' else 'English'} in at most three short sentences, without lists. "
        "If the recipe does not answer the question, say so in one sentence and give no guess. "
        "Never invent amounts, times or ingredients.\n\n"
        f"RECIPE: {r['recipe_title']} ({r['dish_name']})\n"
        f"Prep: {r['prep_minutes']} min. Cook: {r['cook_minutes']} min. Total: {r['total_minutes']} min. "
        f"Serves: {r.get('servings') or 'not given'}. Calories per serving: {r.get('calories_per_serving') or 'not given'}.\n"
        "Ingredients:\n" + "\n".join(f"- {i}" for i in r["ingredients"]) + "\n"
        "Steps:\n" + "\n".join(f"{n}. {s}" for n, s in enumerate(r["steps"], 1)) + "\n\n"
        f"QUESTION: {question}"
    )


def recipe_answer(question, dish_id, lang="en", model_id=None, full=False) -> dict:
    """The answer to a recipe question. Fixed questions (time, ingredients, servings, calories, steps) are answered
    from the data, without AI; other questions by the AI model from this recipe only.
    full=True (Telegram): the ingredients and steps are written out; the web app shows them in a recipe card.
    Returns {text, provider, model, label, focus}; focus = the part of the recipe card to open."""
    t = TEXT.get(lang, TEXT["en"])
    r = RECIPES.get(dish_id)
    facts = {"provider": "facts", "model": None, "label": None}
    if not r:
        return {"text": t["no_recipe"].format(dish=DISHES.get(dish_id, dish_id)), "focus": None, **facts}
    kind = recipe_intent(question, has_dish=True) or "steps"      # a button or a place word: show the recipe
    title, n_steps = r["recipe_title"], len(r["steps"])
    total = fmt_minutes(r["total_minutes"], lang)
    lists = ""
    if kind == "time":
        cook = r["cook_minutes"] or 0
        text = (t["time"] if cook else t["time_no_cook"]).format(
            title=title, total=total, prep=fmt_minutes(r["prep_minutes"], lang), cook=fmt_minutes(cook, lang))
        if r.get("servings"):
            text += t["serves"].format(n=r["servings"])
        wait = waiting(r)
        if wait:
            text += t["wait"].format(what=", ".join(t[w] for w in wait))
    elif kind == "ingredients":
        text = t["ingredients"].format(title=title, n=len(r["ingredients"]))
        lists = "\n" + "\n".join(f"- {i}" for i in r["ingredients"])
    elif kind == "servings":
        text = t["servings"].format(title=title, n=r["servings"]) if r.get("servings") else t["servings_unknown"].format(title=title)
    elif kind == "calories":
        cal = r.get("calories_per_serving")
        text = t["calories"].format(title=title, cal=cal) if cal else t["calories_unknown"].format(title=title)
    elif kind == "steps":
        text = t["steps"].format(title=title, n=n_steps, total=total)
        lists = "\n" + "\n".join(f"{n}. {s}" for n, s in enumerate(r["steps"], 1))
    else:                                         # a free question: the AI answers from this recipe only
        try:
            answer, m = complete(recipe_prompt(question, r, lang), model_id)
            return {"text": as_sentences(answer), "provider": m["provider"], "model": m["id"], "label": m["label"],
                    "focus": None}
        except NoModelAnswered:
            text = t["other"].format(title=title)
    return {"text": text + (lists if full else ""), "focus": kind if kind in ("ingredients", "steps") else None, **facts}


In [ ]:
%%writefile export/backend/food_search/llm.py
"""Result cards, RAG prompt and LLM providers with fallback (Gemini -> Ollama -> plain cards)."""
import json
import logging
import re
from datetime import datetime

import numpy as np
import pandas as pd
import requests

from .config import *  # noqa: F401,F403
from .search import PLACE_NAMES_KH, about_food, detect_lang, is_open_at, parse_query_filters, place_count

log = logging.getLogger("food_search.llm")


NO_MATCH = {
    "en": "Sorry, nothing in our list matches that. Try another dish or remove a filter.",
    "kh": "សូមអភ័យទោស មិនមានហាងណាត្រូវនឹងការស្វែងរករបស់អ្នកទេ។ សូមសាកល្បងម្ហូបផ្សេង។",
}
NO_MATCH_IN = {
    "en": "Sorry, our list has nothing like that in {place} yet. Try another dish, or ask without the place to search all of Cambodia.",
    "kh": "សូមអភ័យទោស បញ្ជីរបស់យើងមិនទាន់មានហាងបែបនេះនៅ{place}ទេ។ សូមសាកល្បងម្ហូបផ្សេង ឬសួរដោយមិនដាក់ទីកន្លែង។",
}


def no_match(lang, results=None) -> str:
    """The 'nothing found' reply; names the khan or city when the search was limited to one."""
    flt = results.attrs.get("filters", {}) if results is not None else {}
    place = flt.get("area") or flt.get("city")
    if not place:
        return NO_MATCH[lang]
    return NO_MATCH_IN[lang].format(place=PLACE_NAMES_KH.get(place, place) if lang == "kh" else place)


UNVERIFIED_WHAT = {"en": {"price": "prices", "hours": "opening hours", "rating": "ratings"},
                   "kh": {"price": "តម្លៃ", "hours": "ម៉ោងបើក", "rating": "ការវាយតម្លៃ"}}
UNVERIFIED_TEXT = {"en": "Our data has no {what} for these places, so I could not check that condition.",
                   "kh": "ទិន្នន័យរបស់យើងមិនមាន{what}សម្រាប់ហាងទាំងនេះ ដូច្នេះមិនអាចពិនិត្យលក្ខខណ្ឌនោះបានទេ។"}


def unverified_note(results, lang) -> str:
    """One sentence when a condition of the question (under $5, open now, rated 4+) could not be checked because the
    data has no such value for the places that match the question; '' when every condition was applied."""
    names = [UNVERIFIED_WHAT[lang][u] for u in (results.attrs.get("unverified") or []) if u in UNVERIFIED_WHAT[lang]]
    return UNVERIFIED_TEXT[lang].format(what=(" / " if lang == "en" else " និង ").join(names)) if names else ""


LEAD_TEXT = {
    "en": {"found": "Here's what I found", "for": " for “{what}”", "in": " in {place}", "open_now": "open now",
           "open_late": "open late", "under": "under ${v}", "rated": "rated {v}+", "near_me": "closest first",
           "near": "closest to {name} first", "end": "."},
    "kh": {"found": "នេះជាអ្វីដែលខ្ញុំរកឃើញ", "for": " សម្រាប់ “{what}”", "in": " នៅ{place}", "open_now": "កំពុងបើក",
           "open_late": "បើកយប់", "under": "ក្រោម ${v}", "rated": "ពិន្ទុ {v}+", "near_me": "ជិតខ្ញុំមុន",
           "near": "ជិត {name} មុន", "end": "។"},
}


def search_lead(results, lang, near=None) -> str:
    """The one line above the result cards of a search: what was searched, where, and which conditions were applied
    ("Here's what I found for “pizza” in Siem Reap, open now, closest first."). Built from the search itself, with no
    AI model: the cards below already list the places, so a model's text would only repeat their names.
    near = 'me', a place name the results are sorted by, or None."""
    W = LEAD_TEXT[lang]
    flt, unverified = results.attrs.get("filters", {}), results.attrs.get("unverified") or []
    what = (results.attrs.get("query_text") or "").strip()
    place = flt.get("area") or flt.get("city")
    if place and lang == "kh":
        place = PLACE_NAMES_KH.get(place, place)
    notes = []                                       # only conditions that were really applied (see unverified_note)
    if flt.get("open_now") and "hours" not in unverified:
        notes.append(W["open_now"])
    if flt.get("open_late") and "hours" not in unverified:
        notes.append(W["open_late"])
    if flt.get("max_price") is not None and "price" not in unverified:
        notes.append(W["under"].format(v=f"{flt['max_price']:g}"))
    if flt.get("min_rating") is not None and "rating" not in unverified:
        notes.append(W["rated"].format(v=f"{flt['min_rating']:g}"))
    if near:
        notes.append(W["near_me"] if near == "me" else W["near"].format(name=near))
    text = W["found"] + (W["for"].format(what=what) if what else "") + (W["in"].format(place=place) if place else "")
    return text + "".join(", " + n for n in notes) + W["end"]


def fmt_price(p) -> str:
    if pd.isna(p) or str(p).strip() in ("", "nan", "price unknown"):
        return "price unknown"
    try:
        return f"${float(p):.2f}"
    except (ValueError, TypeError):
        return str(p)


def fmt_hours(open_h, close_h) -> str:
    if pd.isna(open_h) or pd.isna(close_h):
        return "hours unknown"
    open_h, close_h = float(open_h), float(close_h)
    if open_h % 24 == close_h % 24:
        return "open 24 hours"
    hhmm = lambda h: f"{int(h) % 24:02d}:{round(h % 1 * 60):02d}"
    return f"{hhmm(open_h)}–{hhmm(close_h)}"


def format_cards(results, lang="en", max_items=TOP_K) -> str:
    """Plain-text result cards (the no-LLM answer)."""
    if results is None or len(results) == 0:
        return no_match(lang, results)
    lines = []
    for i, r in enumerate(results.head(max_items).itertuples(), 1):
        dish = r.dish_kh if lang == "kh" and r.dish_kh else r.dish_en
        bits = [fmt_price(r.price_usd), f"⭐ {r.rating:.1f}" if pd.notna(r.rating) else "⭐ n/a",
                r.area, fmt_hours(r.open_hour, r.close_hour)]
        if pd.notna(getattr(r, "distance_km", np.nan)):
            bits.append(f"{r.distance_km:.1f} km away")
        lines.append(f"{i}. {r.name} — {dish}\n   " + " · ".join(bits))
    return "\n".join(lines)


def build_prompt(query, results, lang="en") -> str:
    """The LLM only sees the retrieved rows, so it cannot recommend anything else."""
    rows = []
    for i, r in enumerate(results.head(TOP_K).itertuples(), 1):
        d = getattr(r, "distance_km", np.nan)
        # same format as the result list (470 m / 1.4 km), so the summary and the list agree
        dist = "unknown" if pd.isna(d) else (f"{round(d * 100) * 10:.0f} m" if d < 1 else f"{d:.1f} km")
        dish_val = getattr(r, "dish_kh", "")
        phone_val = getattr(r, "phone_public", "") or "none"
        page_val = getattr(r, "page_url", "") or "none"
        tags_val = getattr(r, "tags", "") or ""
        menu_val = getattr(r, "menu", "") or ""
        rows.append(f"{i}. name: {r.name} | dish: {r.dish_en} / {dish_val} | price: {fmt_price(r.price_usd)}"
                    f" | rating: {r.rating if pd.notna(r.rating) else 'unknown'} | area: {r.area}"
                    f" | hours: {fmt_hours(r.open_hour, r.close_hour)} | distance: {dist}"
                    f" | phone: {phone_val} | link: {page_val} | tags: {tags_val}"
                    f" | menu: {menu_val}"
                    f" | notes: {r.description}")
    language = "Khmer" if lang == "kh" else "English"
    return (
        "You are a friendly food guide for Cambodia.\n"
        "Answer the QUESTION using ONLY the places listed in RESULTS.\n"
        "Rules:\n"
        f"- Answer in {language} (the language of the question).\n"
        "- The full list of places with their details is shown right under your answer, so do not repeat it.\n"
        "- Answer exactly what the QUESTION asks, nothing more:\n"
        "  * a request for food or places (\"kuy teav\", \"pizza in BKK\"): name the best 1 to 3 places with their list\n"
        "    number (\"1. Costa\") and, in a few words, why they fit; no prices or hours unless the question asks;\n"
        "  * a question about one detail (\"cheapest\", \"closest\", \"open now\", \"under $3\"): name the place(s) that fit\n"
        "    and give that detail only;\n"
        "  * a yes/no question (\"is there vegan food in Toul Kork?\"): start with yes or no.\n"
        "- Never invent restaurants, prices, opening hours or other facts. Mention a value only if it is in RESULTS.\n"
        "- Do not mention missing information (unknown prices, hours or ratings) unless the question asks about it.\n"
        "- If the question says near me / near a place, prefer the closest places and give their distance.\n"
        "- Use only facts that are written in RESULTS: no opinions about taste, quality or popularity.\n"
        "- If none of the RESULTS fit the question, say clearly that nothing matches.\n"
        + (f"- The data cannot confirm the question's condition on {', '.join(results.attrs['unverified'])}: do not say or "
           "imply that a place meets it.\n" if results.attrs.get("unverified") else "") +
        "- 1 or 2 short sentences, under 50 words, plain text: no greeting, no list, no closing question.\n\n"
        f"QUESTION: {query}\n\nRESULTS:\n" + "\n".join(rows)
    )


def safe_error(e) -> str:
    """Short error text with any secret value masked (safe to log)."""
    msg = f"{type(e).__name__}: {e}"[:200]
    for secret in (GEMINI_API_KEY, TELEGRAM_TOKEN, TG_API_HASH, OLLAMA_API_KEY):
        if secret:
            msg = msg.replace(secret, "***")
    return msg


def call_gemini(prompt, model=None, json_mode=False, timeout_s=None) -> str:
    model = model or GEMINI_MODEL
    if not GEMINI_API_KEY:
        raise RuntimeError("GEMINI_API_KEY is not set")
    if not model:
        raise RuntimeError("GEMINI_MODEL is not set")
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=GEMINI_API_KEY,
                          http_options=types.HttpOptions(timeout=(timeout_s or LLM_TIMEOUT_S) * 1000))  # milliseconds
    resp = client.models.generate_content(
        model=model, contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=1024,
                                           response_mime_type="application/json" if json_mode else None))
    text = (resp.text or "").strip()
    if not text:
        raise RuntimeError("Gemini returned an empty answer")
    return text


def ollama_headers() -> dict:
    """Ollama cloud needs the API key; it is sent only over HTTPS, never to a plain-http host."""
    if OLLAMA_API_KEY and OLLAMA_HOST.startswith("https://"):
        return {"Authorization": f"Bearer {OLLAMA_API_KEY}"}
    return {}


def call_ollama(prompt, model=None, json_mode=False, timeout_s=None) -> str:
    model = model or OLLAMA_MODEL
    if not model:
        raise RuntimeError("OLLAMA_MODEL is not set")
    r = requests.post(f"{OLLAMA_HOST}/api/generate", headers=ollama_headers(),
                      json={"model": model, "prompt": prompt, "stream": False, "options": {"temperature": 0.2},
                            **({"format": "json"} if json_mode else {})},
                      timeout=(5, timeout_s or OLLAMA_TIMEOUT_S))        # (connect, read) seconds
    r.raise_for_status()
    text = r.json().get("response", "").strip()
    if not text:
        raise RuntimeError("Ollama returned an empty answer")
    return text


# ---------- which model answers (users choose from LLM_MODELS; nothing else is accepted) ----------

NO_LLM = "none"            # "results only": the retrieved rows as plain cards, no model call
MAX_FALLBACKS = 2          # if the chosen model fails (busy, quota, ...), try at most this many others from the list


def model_label(provider, model) -> str:
    """'gemini-3.5-flash-lite' -> 'Gemini 3.5 Flash Lite', 'llama3.2' (ollama) -> 'Llama3.2 (Ollama)'."""
    words = [w for w in re.split(r"[-_:\s]+", model) if w]
    nice = " ".join(w.upper() if w.lower() in ("gpt", "oss", "glm") else w if any(c.isdigit() for c in w) else w.capitalize()
                    for w in words)
    return nice + (" (Ollama)" if provider == "ollama" else "")


def parse_models(spec) -> list:
    """'gemini:gemini-3.5-flash-lite, ollama:gpt-oss:20b=GPT-OSS' -> [{id, provider, model, label}, ...]"""
    models = []
    for item in (spec or "").split(","):
        item, _, label = item.strip().partition("=")
        provider, _, model = item.partition(":")
        provider, model = provider.strip().lower(), model.strip()
        if not item:
            continue
        if provider not in ("gemini", "ollama") or not model:
            log.warning("LLM_MODELS: skipping %r (use provider:model, provider = gemini or ollama)", item)
            continue
        models.append({"id": f"{provider}:{model}", "provider": provider, "model": model,
                       "label": label.strip() or model_label(provider, model)})
    return list({m["id"]: m for m in models}.values())          # no duplicates, order kept


def _default_spec() -> str:
    items = ([f"gemini:{GEMINI_MODEL}"] if GEMINI_MODEL else []) + ([f"ollama:{OLLAMA_MODEL}"] if OLLAMA_MODEL else [])
    return ",".join(reversed(items) if LLM_PROVIDER == "ollama" else items)


MODELS = parse_models(LLM_MODELS or _default_spec())
MODELS_BY_ID = {m["id"]: m for m in MODELS}
DEFAULT_MODEL_ID = NO_LLM if LLM_PROVIDER == "none" or not MODELS else MODELS[0]["id"]


def valid_model_id(model_id):
    """A model id from the list, 'none', or None (= the default). Anything else is ignored."""
    return model_id if model_id == NO_LLM or model_id in MODELS_BY_ID else None


def model_chain(model_id=None) -> list:
    """The chosen model first, then up to MAX_FALLBACKS others from the list (in list order)."""
    model_id = valid_model_id(model_id) or DEFAULT_MODEL_ID
    if model_id == NO_LLM:
        return []
    first = MODELS_BY_ID[model_id]
    return [first] + [m for m in MODELS if m is not first][:MAX_FALLBACKS]


class NoModelAnswered(Exception):
    """No model was chosen ('none'), or every model tried failed (busy, quota, offline, ...)."""


def complete(prompt, model_id=None, json_mode=False, max_models=None, timeout_s=None):
    """Runs the prompt on the chosen model, then on up to MAX_FALLBACKS others from the list.
    max_models / timeout_s limit the attempts (for quick steps that have a non-AI fallback).
    Returns (text, model) or raises NoModelAnswered."""
    callers = {"gemini": call_gemini, "ollama": call_ollama}
    for m in model_chain(model_id)[:max_models]:
        try:
            return callers[m["provider"]](prompt, m["model"], json_mode=json_mode, timeout_s=timeout_s), m
        except Exception as e:                       # no key, offline, busy, quota, timeout, ...
            log.warning("%s failed (%s) -> next model", m["id"], safe_error(e))
    raise NoModelAnswered()


def as_sentences(text) -> str:
    """Joins bullet or numbered lines a model wrote into plain sentences (answers that must not be lists)."""
    # only bullet marks go: "1. Costa" at the start of a line is a place's list number and stays
    lines = [re.sub(r"^\s*[-*•]\s+", "", l).strip() for l in str(text).strip().splitlines()]
    return " ".join(l for l in lines if l)


def generate_answer(query, results, model_id=None) -> dict:
    """RAG answer from the retrieved rows. Returns {text, provider, model, label, chosen}:
    provider = gemini | ollama | cards (no model answered) | no-results (nothing retrieved, no model call)."""
    lang = detect_lang(query)
    chosen = valid_model_id(model_id) or DEFAULT_MODEL_ID
    if results is None or len(results) == 0:
        return {"text": no_match(lang, results), "provider": "no-results", "model": None, "label": None, "chosen": chosen}
    try:
        text, m = complete(build_prompt(query, results, lang), chosen)
        text = as_sentences(text)
        log.info("answer model: %s", m["id"])
        return {"text": text, "provider": m["provider"], "model": m["id"], "label": m["label"], "chosen": chosen}
    except NoModelAnswered:
        log.info("answer provider: cards")
        return {"text": format_cards(results, lang), "provider": "cards", "model": None, "label": None, "chosen": chosen}


def answer_with_llm(query, results, provider=None, model=None):
    """Returns (answer_text, provider_used), as before. provider='none' = plain cards; 'gemini' / 'ollama' = the
    first model of that provider in the list. model = a model id from LLM_MODELS (wins over provider)."""
    if model is None and provider:
        p = provider.lower()
        model = NO_LLM if p == NO_LLM else next((m["id"] for m in MODELS if m["provider"] == p), None)
    out = generate_answer(query, results, model)
    return out["text"], out["provider"]


# ---------- conversation: follow-up questions about the places already shown ----------
# A chat turn is one of three things, decided by plan_turn():
#   followup  a question about the places shown last ("what's on the menu?", "is the first one open?"): answered from
#             those rows only, without a new search
#   search    a request for other places ("cheaper ones?", "pizza instead"): rewritten into a standalone search query
#             that keeps the earlier wishes ("cheap coffee near me"), then the normal search + answer
#   chat      greetings, thanks, small talk

HISTORY_TURNS = 6          # earlier messages (user + assistant) the model sees
CONTEXT_PLACES = 10        # places shown most recently that a follow-up can refer to
PLAN_TIMEOUT_S = 10        # the "decide" step makes one quick attempt (Gemini's minimum); if it is slow or busy, word rules decide

CHAT_RE = re.compile(r"^\s*(hi+|hello|hey|yo|thanks?|thank you|thx|ok(ay)?|cool|great|nice|bye|good (morning|afternoon|"
                     r"evening|night)|សួស្តី|ជំរាបសួរ|អរគុណ|បាទ|ចាស|អូខេ|លា)\b[\s!.?😀-🙏]*$", re.I)
FOLLOWUP_RE = re.compile(r"\b(it|its|it's|that|this|those|these|they|them|their|there|first|second|third|last|"
                         r"closest|nearest|number\s*\d|no\.?\s*\d|menu|menus|dishes|serve|serves|open|opens|close|"
                         r"closes|closed|hours?|phone|call|contact|far|distance|address|where|located|facebook|page|"
                         r"website|link|price|prices|cost|how much|rating|rated|stars?|deliver|delivery)\b|#\d|"
                         r"ម៉ឺនុយ|បើក|បិទ|ម៉ោង|ទូរស័ព្ទ|ឆ្ងាយ|ទីតាំង|តម្លៃ|ហាងនេះ|ហាងនោះ|ទីមួយ|ទីពីរ|ទីបី|មួយណា|ណាមួយ|ជាងគេ", re.I)
NEW_SEARCH_RE = re.compile(r"\b(other|another|else|instead|more|different|cheaper|cheap|any\s+\w+\s+(near|in)|near\s+\w+|"
                           r"in\s+[a-z]+\s+[a-z]+)\b|ផ្សេង|ទៀត", re.I)
ORDINALS = {"first": 1, "1st": 1, "second": 2, "2nd": 2, "third": 3, "3rd": 3, "fourth": 4, "4th": 4, "fifth": 5,
            "5th": 5, "ទីមួយ": 1, "ទីពីរ": 2, "ទីបី": 3, "ទីបួន": 4, "ទីប្រាំ": 5}
FILLER_RE = re.compile(r"^\s*(show me|find me|find|search for|what about|how about|i want|i'd like|give me|any)\s+|"
                       r"\s+(instead|please|then)\s*[?.!]*$", re.I)
ABOUT_RE = re.compile(r"\b(who are you|what are you|what can you do|what do you do|how (do|does) (this|it|you) work|"
                      r"your name)\b|^\s*help\s*[?!.]*$|អ្នកជានរណា|អ្នកអាចធ្វើអ្វី", re.I)
HUNGRY_RE = re.compile(r"^\s*(i'?m|i am|im)\s+(so |very |really )?(hungry|starving)\b|^\s*(so |very )?(hungry|starving)\s*[!.?]*$|\bwhat (should|can|do) i eat\b|"
                       r"\b(don'?t|do not) know what to eat\b|\bany (food )?(ideas|suggestions)\b|^\s*ឃ្លាន|ញ៉ាំអីល្អ|ញ៉ាំអ្វីល្អ", re.I)
OFF_TOPIC_RE = re.compile(r"\b(weather|poem|story|joke|song|lyrics|homework|translate|news|politics|essay|math|"
                          r"programming|write (me )?(a|an))\b", re.I)
CHAT_FALLBACK = {"en": "Happy to help! Tell me a dish, an area or a time, for example “kuy teav near me”.",
                 "kh": "រីករាយជួយ! ប្រាប់ខ្ញុំពីម្ហូប តំបន់ ឬម៉ោង ឧ. “គុយទាវ ជិតខ្ញុំ”។"}
NOT_IN_DATA = "not in our data"


def history_text(history) -> str:
    lines = [f"{h['role']}: {' '.join(str(h['text']).split())[:300]}" for h in (history or [])[-HISTORY_TURNS:]]
    return "\n".join(lines) or "(this is the first message)"


NUMBER_LIST_RE = re.compile(r"(?<![\d$.:])\b\d{1,2}(?:\s*(?:,|and|&|or|vs\.?|versus|និង|ឬ)\s*\d{1,2}\b(?![.:$%]|\s*(?:km|m|usd|\$|"
                            r"dollars?|riel|h|hours?|min)))+")


NAME_WORDS_RE = re.compile(r"[a-z0-9\u1780-\u17ff']+")
NAME_STOP = {"the", "and", "restaurant", "restaurants", "cafe", "café", "coffee", "shop", "house", "kitchen", "food",
             "bar", "grill", "bbq", "pizza", "pizzeria", "burger", "burgers", "noodle", "noodles", "bakery", "tea",
             "store", "phnom", "penh", "siem", "reap", "branch", "mall", "aeon", "street", "delivery", "express",
             "new", "best", "place", "menu", "one", "for", "with", "show", "what", "about"}


def named_by_part(message, places) -> list:
    """Shown places named by a distinctive part of their name: 'lucky burger' -> '3. Lucky Burger Big Tree'.
    Words most of the shown names share ('pizza' after a pizza search) do not count; the best match wins."""
    said = set(NAME_WORDS_RE.findall(message.lower()))
    names = [[w for w in NAME_WORDS_RE.findall(str(n).lower()) if len(w) >= 3] for n in places["name"]]
    common = {w for w in set().union(*map(set, names)) if len(names) >= 3 and sum(w in n for n in names) * 2 >= len(names)}
    scores = {}
    for i, words in enumerate(names, 1):
        key = [w for w in words if w not in NAME_STOP and w not in common]
        if any(w in said for w in key):                      # at least one distinctive word of the name
            scores[i] = sum(w in said for w in set(words))
    best = max(scores.values(), default=0)
    return [i for i, sc in scores.items() if sc == best]


def mentioned_places(message, places) -> list:
    """1-based numbers of the shown places the message names: by name (or a distinctive part of it), 'number 2',
    '#2', 'the first', 'the last'."""
    low, picked = message.lower(), []
    for i, name in enumerate(places["name"], 1):
        if name and len(name) > 2 and name.lower() in low:
            picked.append(i)
    if not picked:
        picked.extend(named_by_part(message, places))
    for m in re.finditer(r"(?:number|no\.?|#|លេខ|ទី)\s*(\d+)", low):
        picked.append(int(m.group(1)))
    for m in NUMBER_LIST_RE.finditer(low):               # "compare 1 and 2", "1, 3 or 4", "1 vs 2"
        picked.extend(int(x) for x in re.findall(r"\d+", m.group(0)))
    m = (re.search(r"\b(?:of|about|for|is|at)\s+(\d{1,2})\s*[?.!]*$", low)    # "what's the phone of 2?"
         or re.search(r"\b(?:is|does|do|are)\s+(\d{1,2})\s+(?!km|m\b|usd|dollars?|riel|hours?|min)\w", low))  # "is 3 open?"
    if m:
        picked.append(int(m.group(1)))
    for word, n in ORDINALS.items():
        khmer = not word.isascii()                      # Khmer has no spaces: "ហាងទីពីរ" = "the second place"
        if (word in low) if khmer else re.search(rf"(?<!\w){re.escape(word)}(?!\w)", low):
            picked.append(n)
    if re.search(r"\blast (one|place)\b", low):
        picked.append(len(places))
    return [n for n in dict.fromkeys(picked) if 1 <= n <= len(places)]


GENERIC_WORDS = {"one", "ones", "place", "places", "spot", "spots", "something", "option", "options", "more", "other",
                 "others", "another", "else", "any", "some", "better", "closer", "nearer", "different", "the", "a",
                 "what", "how", "about", "instead", "there", "too", "also", "then", "and", "or", "in", "at",
                 "ចុះ", "វិញ", "ដែរ", "នៅ", "អីចឹង",
                 "ផ្សេង", "ទៀត", "ផ្សេងទៀត", "ហាង", "កន្លែង", "ហាងផ្សេង", "ហាងផ្សេងទៀត"}


def carry_context(query, last_query) -> str:
    """Keeps what the user is looking for when a new search does not say it again:
    'cheaper ones' after 'pizza near me' -> 'pizza cheap near me'; after 'pizza in Toul Kork' -> '... in Toul Kork'."""
    if not last_query:
        return query
    query = re.sub(r"\bcheaper\b", "cheap", query, flags=re.I)        # "cheap" is a filter the search understands
    text, _ = parse_query_filters(query)
    if not [w for w in text.split() if w.strip("?.!,") not in GENERIC_WORDS]:     # no dish of its own
        words = [w for w in query.split() if w.lower().strip("?.!,") not in GENERIC_WORDS]
        query = " ".join([parse_query_filters(last_query)[0], *words]).strip()
    _, new = parse_query_filters(query)
    _, old = parse_query_filters(last_query)
    if new.get("near") or new.get("area") or new.get("city"):
        return query
    if old.get("near") == "me":
        return f"{query} near me"
    if old.get("near"):
        i = last_query.lower().rfind(old["near"].lower())
        return f"{query} near {last_query[i:i + len(old['near'])] if i >= 0 else old['near']}"
    if old.get("area") or old.get("city"):
        return f"{query} in {old.get('area') or old['city']}"
    return query


def heuristic_plan(message, places) -> dict:
    """The plan without an AI model (none chosen, or all busy): word rules."""
    if CHAT_RE.match(message) or ABOUT_RE.search(message) or HUNGRY_RE.search(message) or OFF_TOPIC_RE.search(message):
        return {"action": "chat", "query": "", "places": []}       # small talk, "I'm hungry", "write me a poem"
    if places is not None and len(places):
        picked = mentioned_places(message, places)
        about_all = SUPERLATIVE_RE.search(message) or COMPARE_RE.search(message)       # "which one is cheapest?"
        if picked or ((FOLLOWUP_RE.search(message) or about_all) and not NEW_SEARCH_RE.search(message)):
            return {"action": "followup", "query": "", "places": picked}
    return {"action": "search", "query": FILLER_RE.sub(" ", message).strip() or message, "places": []}


def focus_from_history(history, places) -> list:
    """The place(s) the last answer was about, so 'is it open?' / 'and its phone?' stay on them: the numbered names
    ('2. Papa's burger') in the last assistant message. [] after a list of results (more than 2 places named)."""
    last = next((h["text"] for h in reversed(history or []) if h.get("role") == "assistant"), "")
    found = [i for i, name in enumerate(places["name"], 1) if name and f"{i}. {str(name)[:24]}" in last]
    return found if 1 <= len(found) <= 2 else []


def build_plan_prompt(message, history, places) -> str:
    shown = ("\n".join(f"{i}. {r.name} ({', '.join(x for x in (r.dish_en, r.area) if x)})"
                       for i, r in enumerate(places.itertuples(), 1)) if places is not None and len(places)
             else "(none yet)")
    return (
        "You route messages in a food-guide chat for Cambodia.\n\n"
        f"CONVERSATION SO FAR (oldest first):\n{history_text(history)}\n\n"
        f"PLACES SHOWN TO THE USER (numbered as the user saw them):\n{shown}\n\n"
        f"NEW MESSAGE: {message}\n\n"
        "Decide what the new message needs:\n"
        '- "followup": it asks about the places already shown: their food, price, menu, opening hours, whether they are '
        "open, distance, phone, order link, Facebook page or website, location, or which of them is best, closest or open; or it "
        'refers to them ("it", "that one", "the first", "number 2", or a name from the list).\n'
        '- "search": it asks for other or new places: another dish, another area or province, a new condition ("cheaper", '
        '"open late", "near CADT", "in Siem Reap"), "something else" or "more options".\n'
        '- "chat": a greeting, thanks, small talk, or a question about you.\n'
        'For "search", write "query": a short standalone search query in the language of the new message that keeps '
        'the earlier wishes the user did not change (dish, area, "near me", open now). Example: after "coffee near me", '
        '"any open now?" becomes "coffee open now near me". For the other actions "query" is "".\n'
        'For "followup", "places": the numbers of the places the message is about, or [] if it is about all of them '
        "or it is unclear.\n"
        'Reply with JSON only: {"action": "followup" | "search" | "chat", "query": "...", "places": [1]}'
    )


def plan_turn(message, history=None, places=None, model_id=None, last_query=None) -> dict:
    """What a chat message needs: {action: followup | search | chat, query, places (1-based numbers)}.
    places = the rows shown last (a DataFrame, in the order the user saw them), or None;
    last_query = the previous search, whose place or area a new search keeps (carry_context)."""
    plan = _plan(message, history, places, model_id)
    if plan["action"] == "search":                   # in a conversation, keep where the user is looking
        plan["query"] = re.sub(r"\s*\b(in )?cambodia\b|\s*(នៅ)?កម្ពុជា", "", plan["query"], flags=re.I).strip() or message
        plan["query"] = carry_context(plan["query"], last_query)
    return plan


def _plan(message, history, places, model_id) -> dict:
    if places is None or len(places) == 0:          # nothing to refer back to: a search (or small talk)
        quick = heuristic_plan(message, None)
        if quick["action"] == "chat" or about_food(quick["query"]):
            return quick                             # "pizza", "hi": no AI call
    else:
        quick = heuristic_plan(message, places)
        if quick["action"] == "chat" or (quick["action"] == "followup" and quick["places"]):
            return quick                             # obvious (a greeting, or "number 2" / a place's name): no AI call
        if quick["action"] == "followup" and (WHICH_RE.search(message) or SUPERLATIVE_RE.search(message)
                                              or COMPARE_RE.search(message) or LIST_RE.search(message)):
            return quick                             # about all the places shown
        if quick["action"] == "followup":            # "is it open?" right after an answer about one place
            focus = focus_from_history(history, places)
            if focus and not (WHICH_RE.search(message) or COMPARE_RE.search(message) or LIST_RE.search(message)):
                return {**quick, "places": focus}
    try:
        text, _ = complete(build_plan_prompt(message, history, places), model_id, json_mode=True,
                           max_models=1, timeout_s=PLAN_TIMEOUT_S)
        plan = json.loads(text[text.find("{"):text.rfind("}") + 1])
        action = plan.get("action")
        if action not in ("followup", "search", "chat"):
            raise ValueError(f"unknown action {action!r}")
        if action == "followup" and (places is None or len(places) == 0):
            action = "search"                        # nothing shown yet to follow up on
        query = " ".join(str(plan.get("query") or "").split())[:200]
        n_shown = 0 if places is None else len(places)
        nums = [int(n) for n in plan.get("places") or [] if str(n).isdigit() and 1 <= int(n) <= n_shown]
        if action == "followup":                     # trust explicit mentions over the model's guess
            nums = mentioned_places(message, places) or nums
        return {"action": action, "query": query or message if action == "search" else "", "places": nums}
    except (NoModelAnswered, ValueError, TypeError, json.JSONDecodeError) as e:
        if not isinstance(e, NoModelAnswered):
            log.warning("plan_turn: unreadable plan (%s) -> word rules", safe_error(e))
        return heuristic_plan(message, places)


def place_facts(n, r, hour_now) -> str:
    """Everything our data knows about one place, and what it does not know (so the model never fills a gap)."""
    name_val = r["name"] if isinstance(r, (pd.Series, dict)) else getattr(r, "name", "")
    dish_en = r["dish_en"] if isinstance(r, (pd.Series, dict)) else getattr(r, "dish_en", "")
    dish_kh = r["dish_kh"] if (isinstance(r, (pd.Series, dict)) and "dish_kh" in r) else getattr(r, "dish_kh", "")
    food = ", ".join(x for x in (dish_en, dish_kh) if x) or NOT_IN_DATA
    tags = r["tags"] if isinstance(r, (pd.Series, dict)) else getattr(r, "tags", "")
    kinds = ", ".join(t.strip() for t in str(tags or "").split(";") if t.strip()) or NOT_IN_DATA
    d = r.get("distance_km", np.nan) if isinstance(r, (pd.Series, dict)) else getattr(r, "distance_km", np.nan)
    open_h = r["open_hour"] if isinstance(r, (pd.Series, dict)) else getattr(r, "open_hour", np.nan)
    close_h = r["close_hour"] if isinstance(r, (pd.Series, dict)) else getattr(r, "close_hour", np.nan)
    price_val = r["price_usd"] if isinstance(r, (pd.Series, dict)) else getattr(r, "price_usd", np.nan)
    has_hours = pd.notna(open_h) and pd.notna(close_h)
    hours = (f"{fmt_hours(open_h, close_h)} (open now: {'yes' if is_open_at(open_h, close_h, hour_now) else 'no'})"
             if has_hours else NOT_IN_DATA)
    page = str(r["page_url"] if isinstance(r, (pd.Series, dict)) else getattr(r, "page_url", "") or "")
    phone = str(r["phone_public"] if isinstance(r, (pd.Series, dict)) else getattr(r, "phone_public", "") or "").strip()
    desc = str(r["description"] if isinstance(r, (pd.Series, dict)) else getattr(r, "description", "") or "").strip()
    area_val = r["area"] if isinstance(r, (pd.Series, dict)) else getattr(r, "area", "")
    menu_raw = r.get("menu", "") if isinstance(r, (pd.Series, dict)) else getattr(r, "menu", "")
    menu_str = ""
    if menu_raw and isinstance(menu_raw, str):
        try:
            import json as _json
            items = _json.loads(menu_raw)
            if isinstance(items, list):
                menu_str = ", ".join(f"{it.get('name_en')} (${float(it.get('price_usd', 0)):.2f})" for it in items)
        except Exception:
            menu_str = ""
    return (f"{n}. name: {name_val} | food: {food} | type: {kinds} | area: {area_val or NOT_IN_DATA}"
            f" | distance: {'unknown' if pd.isna(d) else (f'{round(d * 100) * 10:.0f} m' if d < 1 else f'{d:.1f} km')}"
            f" | opening hours: {hours} | price: {fmt_price(price_val) if pd.notna(price_val) else NOT_IN_DATA}"
            f" | menu: {menu_str or NOT_IN_DATA}"
            f" | phone: {phone if phone and phone.lower() != 'nan' else NOT_IN_DATA}"
            f" | link: {page if page else NOT_IN_DATA}"
            f" | notes: {desc or NOT_IN_DATA}")


def _hour_now():
    now = datetime.now(TZ)
    return now.hour + now.minute / 60


# ---------- answer shapes: every question gets the form that fits it, with only what it asks ----------
#   direct  one fact about one place, a yes/no, or "which one ...?"   -> one sentence, the answer first (AI)
#   list    one fact about several places, or a menu                   -> one line per place / per dish (from the data)
#   table   a comparison, or several facts about several places        -> only the asked columns (from the data)
# Lists and tables are built here from the rows, never written by a model, so their values cannot be made up.

FIELD_RES = {      # what a question can ask about a place (checked in this order = the column order)
    "food": r"\b(food|serve|serves|sell|sells|kind of|type of|cuisine)\b|ម្ហូបអ្វី|លក់អ្វី|ប្រភេទ",
    "menu": r"\b(menu|menus|dish|dishes)\b|ម៉ឺនុយ|មុខម្ហូប",
    "price": r"\b(price|prices|cost|costs|how much|cheap|cheaper|cheapest|expensive|budget)\b|តម្លៃ|ថ្លៃ|ថោក",
    "hours": r"\b(open|opens|opening|close|closes|closing|closed|hours?|what time)\b|បើក|បិទ|ម៉ោង",
    "distance": r"\b(far|distance|closest|nearest|how long)\b|ឆ្ងាយ|ចម្ងាយ|ជិតជាងគេ",
    "area": r"\b(where|address|located|location|which area|which khan)\b|នៅឯណា|ទីតាំង|អាសយដ្ឋាន",
    "phone": r"\b(phone|call|telephone|contact)\b|ទូរស័ព្ទ|លេខទូរស័ព្ទ",
    "link": r"\b(facebook|fb|page|website|site|link|order online)\b|ហ្វេសប៊ុក|គេហទំព័រ",
    "rating": r"\b(rating|ratings|rated|stars?|reviews?)\b|ផ្កាយ|ការវាយតម្លៃ",
}
FIELD_RES = {k: re.compile(v, re.I) for k, v in FIELD_RES.items()}
DEFAULT_FIELDS = ["food", "area", "price", "hours", "distance"]      # "compare 1 and 2" without saying what
COMPARE_RE = re.compile(r"\b(compare|comparison|vs\.?|versus|difference|differences|differ|side by side|table)\b|"
                        r"ប្រៀបធៀប|ខុសគ្នា|តារាង", re.I)
LIST_RE = re.compile(r"\b(list|all of them|each of them|each one|every one|them all|all|both)\b|ទាំងអស់|ទាំងពីរ|រាយ", re.I)
SUPERLATIVE_RE = re.compile(r"\b(cheapest|closest|nearest|best rated|highest rated|top rated|most expensive|priciest)\b|"
                            r"ថោកជាងគេ|ជិតជាងគេ|ថ្លៃជាងគេ", re.I)
WHICH_RE = re.compile(r"\b(which|who has|is there one|are any|any of them)\b|មួយណា|ណាមួយ|ហាងណា", re.I)
LABELS = {
    "en": {"place": "Place", "food": "Food", "menu": "Menu", "price": "Price", "hours": "Hours", "distance": "Distance",
           "area": "Area", "phone": "Phone", "link": "Page", "rating": "Rating", "unknown": "not in our data",
           "open": "open now", "closed": "closed now", "dash": "— = not in our data",
           "none": "Sorry, our data has no {what} for {places}.", "menu_of": "menu",
           "missing": "Our data has no {what} for these places.",
           "these": "these {n} places", "gaps": "Not in our data for number {nums}.",
           "which": "Which place's menu? Send its number or name, e.g. “menu of 2”."},
    "kh": {"place": "ហាង", "food": "ម្ហូប", "menu": "ម៉ឺនុយ", "price": "តម្លៃ", "hours": "ម៉ោងបើក", "distance": "ចម្ងាយ",
           "area": "ទីតាំង", "phone": "ទូរស័ព្ទ", "link": "ទំព័រ", "rating": "ការវាយតម្លៃ", "unknown": "មិនមានក្នុងទិន្នន័យ",
           "open": "កំពុងបើក", "closed": "បិទហើយ", "dash": "— = មិនមានក្នុងទិន្នន័យ",
           "none": "សូមអភ័យទោស ទិន្នន័យរបស់យើងមិនមាន{what}សម្រាប់ {places} ទេ។", "menu_of": "ម៉ឺនុយ",
           "missing": "ទិន្នន័យរបស់យើងមិនមាន{what}សម្រាប់ហាងទាំងនេះទេ។",
           "these": "ហាងទាំង {n} នេះ", "gaps": "មិនមានក្នុងទិន្នន័យសម្រាប់លេខ {nums}។",
           "which": "ចង់មើលម៉ឺនុយហាងណា? សូមផ្ញើលេខ ឬឈ្មោះហាង ឧ. “ម៉ឺនុយលេខ 2”។"},
}


def asked_fields(message) -> list:
    """The place details a question asks for, e.g. 'phone and hours?' -> ['hours', 'phone']."""
    return [f for f, rx in FIELD_RES.items() if rx.search(message)]


def answer_shape(message, n_places, fields) -> str:
    """direct | list | table (see above). n_places = how many places the question is about."""
    if n_places >= 2 and COMPARE_RE.search(message):
        return "table"
    if WHICH_RE.search(message) or n_places == 0:
        return "direct"                       # "which one is open now?": the model picks and says why
    if "menu" in fields:
        return "list"
    if n_places >= 2 and len(fields) >= 2:
        return "table"
    if n_places >= 2 and (fields or LIST_RE.search(message)):
        return "list"
    return "direct"


def _val(r, col):
    v = r.get(col, "") if isinstance(r, (pd.Series, dict)) else getattr(r, col, "")
    return "" if v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip().lower() == "nan" else v


def menu_items(r) -> list:
    raw = _val(r, "menu")
    try:
        items = json.loads(raw) if isinstance(raw, str) and raw.strip().startswith("[") else []
    except ValueError:
        return []
    return [it for it in items if isinstance(it, dict) and (it.get("name_en") or it.get("name_kh"))]


def field_value(r, field, lang, hour_now) -> str:
    """One detail of one place as short text, or '' when our data does not have it."""
    L = LABELS[lang]
    if field == "food":
        food = (_val(r, "dish_kh") if lang == "kh" else "") or _val(r, "dish_en")
        food = str(food).replace(";", ",").replace("_", " ").strip()
        return food[:1].upper() + food[1:]
    if field == "price":
        p = _val(r, "price_usd")
        return fmt_price(p) if p != "" else ""
    if field == "hours":
        o, c = _val(r, "open_hour"), _val(r, "close_hour")
        if o == "" or c == "":
            return ""
        return f"{fmt_hours(o, c)} ({L['open'] if is_open_at(o, c, hour_now) else L['closed']})"
    if field == "distance":
        d = _val(r, "distance_km")
        return "" if d == "" else (f"{round(d * 100) * 10:.0f} m" if d < 1 else f"{d:.1f} km")
    if field == "area":
        area, city = _val(r, "area"), _val(r, "city")
        if lang == "kh":
            area, city = PLACE_NAMES_KH.get(area, area), PLACE_NAMES_KH.get(city, city)
        return ", ".join(dict.fromkeys(x for x in (area, city) if x))
    if field == "phone":
        return str(_val(r, "phone_public")).strip()
    if field == "link":
        url = str(_val(r, "page_url")).strip()
        return f"[{'Facebook' if 'facebook.com' in url else 'Website'}]({url})" if url.startswith("http") else ""
    if field == "rating":
        v = _val(r, "rating")
        return f"⭐ {float(v):.1f}" if v != "" else ""
    if field == "menu":
        return ", ".join(menu_line(it, lang) for it in menu_items(r))
    return ""


def menu_line(item, lang) -> str:
    name = (item.get("name_kh") if lang == "kh" else "") or item.get("name_en") or item.get("name_kh")
    price = item.get("price_usd")
    return f"{name} — {fmt_price(price)}" if price not in (None, "") else str(name)


def _cell(text) -> str:
    return str(text).replace("|", "/").replace("\n", " ") or "—"


def facts_answer(shape, fields, rows, numbers, lang, hour_now):
    """The list or table for `rows` (numbered as shown), with only `fields`. None when the data has none of it."""
    L = LABELS[lang]
    fields = fields or DEFAULT_FIELDS
    values = {n: {f: field_value(r, f, lang, hour_now) for f in fields} for n, r in zip(numbers, rows)}
    asked_known = [f for f in fields if any(values[n][f] for n in numbers)]
    if not asked_known:                       # nothing of what was asked is known: say so in one sentence
        what = ", ".join(L[f].lower() if lang == "en" else L[f] for f in fields)
        places = (", ".join(f"{n}. {_val(r, 'name')}" for n, r in zip(numbers, rows)) if len(numbers) <= 3
                  else L["these"].format(n=len(numbers)))
        return L["none"].format(what=what, places=places)
    if shape == "list" and "menu" in fields:  # one bullet per dish, under each place
        blocks = []
        for n, r in zip(numbers, rows):
            items = menu_items(r)
            head = f"**{n}. {_val(r, 'name')}**"
            other = [f"{L[f]}: {values[n][f]}" for f in fields if f != "menu" and values[n][f]]
            if other:
                head += " — " + " · ".join(other)
            blocks.append(head + ("\n" + "\n".join(f"- {menu_line(it, lang)}" for it in items[:12])
                                  if items else f" — {L['menu_of']}: {L['unknown']}"))
        return "\n\n".join(blocks)
    if shape == "list":                       # the places that have it; the others in one line, not one each
        lines, gaps = [], []
        for n, r in zip(numbers, rows):
            known = [values[n][f] for f in fields if values[n][f]]
            if known:
                lines.append(f"- **{n}. {_val(r, 'name')}** — " + " · ".join(known))
            else:
                gaps.append(n)
        if len(gaps) == 1 and len(numbers) <= 3:
            n = gaps[0]
            lines.append(f"- **{n}. {_val(rows[numbers.index(n)], 'name')}** — {L['unknown']}")
        elif gaps:
            lines.append("\n" + L["gaps"].format(nums=", ".join(map(str, gaps))))
        return "\n".join(lines)
    cols = [f for f in asked_known if f != "menu"] or asked_known
    if len(numbers) <= 3:                     # side by side: one column per place, one row per detail (fits a phone)
        head = ["", *[f"{n}. {_val(r, 'name')}" for n, r in zip(numbers, rows)]]
        lines = ["| " + " | ".join(_cell(h) if h else "" for h in head) + " |", "|" + "---|" * len(head)]
        for f in cols:
            lines.append("| " + " | ".join([L[f], *[_cell(values[n][f]) for n in numbers]]) + " |")
    else:                                     # one row per place
        head = ["#", L["place"], *[L[f] for f in cols]]
        lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
        for n, r in zip(numbers, rows):
            lines.append("| " + " | ".join([str(n), _cell(_val(r, "name")), *[_cell(values[n][f]) for f in cols]]) + " |")
    notes = [L["dash"]] if any(not values[n][f] for n in numbers for f in cols) else []
    missing = [f for f in fields if f not in asked_known and fields is not DEFAULT_FIELDS]
    if missing:
        notes.append(L["missing"].format(what=", ".join(L[f].lower() if lang == "en" else L[f] for f in missing)))
    return "\n".join(lines) + ("\n\n" + "\n".join(notes) if notes else "")


TELL_RE = re.compile(r"\b(tell me (more )?about|more (info|details) (about|on)|details (of|for|about)|info (on|about))\b|"
                     r"ប្រាប់ខ្ញុំអំពី|ព័ត៌មានអំពី", re.I)
OPEN_Q_RE = re.compile(r"\b(is|are)\b.*\bopen\b|\bopen (now|today|tonight|right now)\b|បើកឥឡូវ|បើក(ទេ|អត់)|"
                       r"បើកនៅ", re.I)
FACT_TEXT = {
    "en": {"open_yes": "Yes, {p} is open now ({v}).", "open_no": "No, {p} is closed now. Its hours are {v}.",
           "hours": "{p} is open {v}.", "price": "{p}: about {v}.", "phone": "{p}: {v}", "distance": "{p} is {v} away.",
           "area": "{p} is in {v}.", "link": "{p}: {v}", "rating": "{p} is rated {v}.", "food": "{p} serves {v}.",
           "unknown": "Our data has no {what} for {p}.",
           "no_location": "I don't know where you are yet. Share your location, or ask again with “near me”, and I'll "
                          "give the distance.",
           "no_xy": "Our data has no map location for {p}, so I can't tell the distance.",
           "try_page": " Their {link} may have it.", "try_phone": " You could call them: {phone}."},
    "kh": {"open_yes": "បាទ {p} កំពុងបើក ({v})។", "open_no": "ទេ {p} បិទហើយ។ ម៉ោងបើក៖ {v}។",
           "hours": "{p} បើក {v}។", "price": "{p}៖ ប្រហែល {v}។", "phone": "{p}៖ {v}", "distance": "{p} ឆ្ងាយ {v}។",
           "area": "{p} នៅ {v}។", "link": "{p}៖ {v}", "rating": "{p} ទទួលបាន {v}។", "food": "{p} លក់ {v}។",
           "unknown": "ទិន្នន័យរបស់យើងមិនមាន{what}សម្រាប់ {p} ទេ។",
           "no_location": "ខ្ញុំមិនទាន់ដឹងថាអ្នកនៅឯណាទេ។ សូមចែករំលែកទីតាំង ឬសួរម្តងទៀតដោយថា “ជិតខ្ញុំ”។",
           "no_xy": "ទិន្នន័យរបស់យើងមិនមានទីតាំងលើផែនទីសម្រាប់ {p} ទេ ដូច្នេះមិនអាចប្រាប់ចម្ងាយបាន។",
           "try_page": " {link} របស់ពួកគេប្រហែលជាមាន។", "try_phone": " អាចទូរស័ព្ទទៅ៖ {phone}។"},
}


def fact_sentence(message, fields, n, r, lang, hour_now, places):
    """One-sentence answer from the data when a question asks one detail of one place ("is it open now?",
    "phone of 2?", "how far is number 3?", "tell me about the first one"). None = let the model answer."""
    F, L = FACT_TEXT[lang], LABELS[lang]
    p = f"{n}. {_val(r, 'name')}"
    if not fields:                                   # "tell me about number 2": its known facts in one line
        return facts_answer("list", [], [r], [n], lang, hour_now) if TELL_RE.search(message) else None
    if len(fields) != 1 or fields[0] == "menu":
        return None
    f = fields[0]
    v = field_value(r, f, lang, hour_now)
    if f == "distance" and not v:
        located = "distance_km" in places.columns and places["distance_km"].notna().any()
        return F["no_xy"].format(p=p) if located else F["no_location"]
    if f == "hours" and v:
        o, c = _val(r, "open_hour"), _val(r, "close_hour")
        if OPEN_Q_RE.search(message):
            return F["open_yes" if is_open_at(o, c, hour_now) else "open_no"].format(p=p, v=fmt_hours(o, c))
        return F["hours"].format(p=p, v=v)
    if v:
        return F[f].format(p=p, v=v)
    what = L[f].lower() if lang == "en" else L[f]
    text = F["unknown"].format(what=what, p=p)
    page, phone = field_value(r, "link", lang, hour_now), field_value(r, "phone", lang, hour_now)
    if f != "link" and page:
        text += F["try_page"].format(link=page)
    elif f not in ("phone", "link") and phone:
        text += F["try_phone"].format(phone=phone)
    return text


EXPENSIVE_RE = re.compile(r"\b(most expensive|priciest|highest price)\b|ថ្លៃជាងគេ", re.I)
WHICH_TEXT = {
    "en": {"price_min": "{w} {be} the cheapest, about {v}.", "price_max": "{w} {be} the most expensive, about {v}.",
           "price_all": "They all cost about {v}.", "distance": "{w} {be} the closest, {v} away.",
           "rating": "{w} {be} the best rated, {v}.", "rating_all": "They are all rated {v}.",
           "many": "{k} places tie at {v}: number {nums}.", "open": "Open now: {w}.",
           "open_many": "{k} of them are open now: number {nums}.",
           "open_none": "None of them is open now (opening hours known for {k} of {n}).",
           "has": "{w} {have} a {what} in our data.", "has_all": "All of them have a {what} in our data.",
           "has_many": "{k} of them have a {what} in our data: number {nums}.", "is": "is", "are": "are",
           "have1": "has", "have2": "have"},
    "kh": {"price_min": "{w} ថោកជាងគេ ប្រហែល {v}។", "price_max": "{w} ថ្លៃជាងគេ ប្រហែល {v}។",
           "price_all": "ហាងទាំងអស់មានតម្លៃប្រហែល {v}។", "distance": "{w} ជិតជាងគេ ចម្ងាយ {v}។",
           "rating": "{w} ទទួលបានការវាយតម្លៃខ្ពស់ជាងគេ {v}។", "rating_all": "ហាងទាំងអស់ទទួលបាន {v}។",
           "many": "ហាង {k} កន្លែងស្មើគ្នា {v}៖ លេខ {nums}។", "open": "កំពុងបើក៖ {w}។",
           "open_many": "ហាង {k} កន្លែងកំពុងបើក៖ លេខ {nums}។",
           "open_none": "គ្មានហាងណាកំពុងបើកទេ (ដឹងម៉ោងបើក {k} ក្នុងចំណោម {n})។",
           "has": "{w} មាន{what}ក្នុងទិន្នន័យរបស់យើង។", "has_all": "ហាងទាំងអស់មាន{what}ក្នុងទិន្នន័យរបស់យើង។",
           "has_many": "ហាង {k} កន្លែងមាន{what}៖ លេខ {nums}។", "is": "", "are": "", "have1": "", "have2": ""},
}


def which_answer(message, fields, rows, numbers, lang, hour_now, places):
    """'Which one is cheapest / closest / best rated / open now / has a phone?' answered from the data in one
    sentence. Ties are summed up ('They all cost about $2.50') instead of listing ten places. None = not such a question."""
    if len(fields) != 1 or fields[0] not in ("price", "distance", "rating", "hours", "phone", "link"):
        return None
    f, W, L = fields[0], WHICH_TEXT[lang], LABELS[lang]
    named = lambda ns: (" " if lang == "kh" else ", ").join(f"{n}. {_val(rows[numbers.index(n)], 'name')}" for n in ns)
    be = lambda ns: W["is"] if len(ns) == 1 else W["are"]
    if f == "hours":
        known = [n for n, r in zip(numbers, rows) if _val(r, "open_hour") != "" and _val(r, "close_hour") != ""]
        if not known:
            return LABELS[lang]["none"].format(what=L["hours"].lower() if lang == "en" else L["hours"],
                                               places=L["these"].format(n=len(numbers)))
        open_now = [n for n in known if is_open_at(_val(rows[numbers.index(n)], "open_hour"),
                                                   _val(rows[numbers.index(n)], "close_hour"), hour_now)]
        if not open_now:
            return W["open_none"].format(k=len(known), n=len(numbers))
        return (W["open"].format(w=named(open_now)) if len(open_now) <= 3
                else W["open_many"].format(k=len(open_now), nums=", ".join(map(str, open_now))))
    if f in ("phone", "link"):
        has = [n for n, r in zip(numbers, rows) if field_value(r, f, lang, hour_now)]
        what = {"phone": "phone number" if lang == "en" else "លេខទូរស័ព្ទ",
                "link": "Facebook page or website" if lang == "en" else "ហ្វេសប៊ុក ឬគេហទំព័រ"}[f]
        if not has:
            return LABELS[lang]["none"].format(what=what, places=L["these"].format(n=len(numbers)))
        if len(has) == len(numbers) and len(has) > 1:
            return W["has_all"].format(what=what)
        if len(has) > 3:
            return W["has_many"].format(k=len(has), what=what, nums=", ".join(map(str, has)))
        return W["has"].format(w=named(has), have=W["have1"] if len(has) == 1 else W["have2"], what=what)
    if f == "distance":
        known = {n: _val(r, "distance_km") for n, r in zip(numbers, rows) if _val(r, "distance_km") != ""}
        if not known:
            return FACT_TEXT[lang]["no_location"]
        best = min(known.values())
        win = [n for n, d in known.items() if d == best]
        return W["distance"].format(w=named(win), be=be(win), v=field_value(rows[numbers.index(win[0])], f, lang, hour_now))
    col = {"price": "price_usd", "rating": "rating"}[f]
    known = {n: float(_val(r, col)) for n, r in zip(numbers, rows) if _val(r, col) != ""}
    if not known:
        return LABELS[lang]["none"].format(what=L[f].lower() if lang == "en" else L[f], places=L["these"].format(n=len(numbers)))
    high = f == "rating" or bool(EXPENSIVE_RE.search(message))
    best = max(known.values()) if high else min(known.values())
    win = [n for n, v in known.items() if v == best]
    v = field_value(rows[numbers.index(win[0])], f, lang, hour_now)
    if len(win) == len(numbers) and len(win) > 1:
        return W["rating_all" if f == "rating" else "price_all"].format(v=v)
    if len(win) > 3:
        return W["many"].format(k=len(win), v=v, nums=", ".join(map(str, win)))
    key = "rating" if f == "rating" else ("price_max" if high else "price_min")
    return W[key].format(w=named(win), be=be(win), v=v)


def answer_followup(message, history, places, numbers=None, model_id=None) -> dict:
    """Answers a question about places already shown, from their rows only (no new search), in the shape that fits
    the question (see 'answer shapes'). places = the rows shown last; numbers = which of them (1-based) it is about,
    [] = all. Returns {text, provider, model, label, numbers, shape}; provider 'facts' = built from the data, no model."""
    lang = detect_lang(message)
    named = [n for n in (numbers or []) if 1 <= n <= len(places)]
    numbers = named or list(range(1, len(places) + 1))
    hour = _hour_now()
    rows = list(places.itertuples())                 # (a pandas Series' .name is its index, not the place's name)
    picked = [rows[n - 1] for n in numbers]
    fields = asked_fields(message)
    shape = answer_shape(message, len(numbers), fields)
    if "menu" in fields and not named and len(numbers) > 1 and not LIST_RE.search(message):
        text = LABELS[lang]["which"] + "\n" + "\n".join(f"- {n}. {_val(r, 'name')}" for n, r in zip(numbers, picked))
        return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": [], "shape": "list"}
    if shape == "direct" and len(numbers) > 1 and (WHICH_RE.search(message) or SUPERLATIVE_RE.search(message)):
        text = which_answer(message, fields, picked, numbers, lang, hour, places)
        if text:
            return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": numbers,
                    "shape": "direct"}
    if shape == "direct" and named and len(numbers) == 1:
        text = fact_sentence(message, fields, numbers[0], picked[0], lang, hour, places)
        if text:
            return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": numbers,
                    "shape": "direct"}
    if shape in ("list", "table"):
        text = facts_answer(shape, fields, picked, numbers, lang, hour)
        return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": numbers, "shape": shape}
    facts = "\n".join(place_facts(n, r, hour) for n, r in zip(numbers, picked))
    prompt = (
        "You are a friendly food guide for Cambodia in a chat. The user asks a follow-up question about "
        "places you already showed them.\n"
        "Rules:\n"
        f"- Answer in {'Khmer' if lang == 'kh' else 'English'} (the language of the question).\n"
        "- Answer exactly what is asked, nothing more: start with the answer itself (yes / no / the value / the "
        "place), in one short sentence (two only if the question has two parts). No greeting, no list, no extra "
        "details the user did not ask for.\n"
        "- Use ONLY the facts in PLACES. When you name a place, write its number and name as in PLACES, e.g. "
        "\"1. Costa\".\n"
        "- If the question asks which place fits (cheapest, closest, open now…), name the place(s) that fit and the "
        "one value that shows it.\n"
        "- If PLACES does not have the fact (\"not in our data\"), say that in one sentence and suggest one next "
        "step (call them, check their page, or search again).\n\n"
        f"CONVERSATION SO FAR:\n{history_text(history)}\n\nPLACES:\n{facts}\n\nQUESTION: {message}"
    )
    try:
        text, m = complete(prompt, model_id)
        return {"text": as_sentences(text), "provider": m["provider"], "model": m["id"], "label": m["label"],
                "numbers": numbers, "shape": "direct"}
    except NoModelAnswered:                          # no AI: the asked facts as a list
        text = facts_answer("list", fields, picked, numbers, lang, hour)
        return {"text": text, "provider": "facts", "model": None, "label": None, "numbers": numbers, "shape": "list"}


CHAT_REPLIES = {    # greetings, thanks and "who are you?" need no AI: instant, always the same, never off-topic
    "en": {"hello": "Hi! 👋 What would you like to eat? Tell me a dish, an area or a time, e.g. “kuy teav near me” "
                    "or “pizza in Siem Reap”.",
           "thanks": "You're welcome! 😊 Anything else you'd like to find?",
           "hungry": "Let's fix that! 🍜 Do you feel like noodles (kuy teav), rice (bai sach chrouk), Khmer BBQ, pizza "
                     "or coffee? Tell me one, and add “near me” to see what's close.",
           "bye": "Bye! Enjoy your meal 🍜",
           "ok": "Great! Ask me anything about these places, or tell me what else you'd like to eat.",
           "about": "I'm Klean Nham, a food guide for Cambodia 🍜. I search our list of {n:,} places: ask for a dish, "
                    "an area or a time (“coffee near me”, “pizza in Siem Reap”, “bbq under $5”), then keep asking "
                    "about the places I show: opening hours, menu, phone, distance, or “compare 1 and 2”."},
    "kh": {"hello": "សួស្តី! 👋 តើអ្នកចង់ញ៉ាំអ្វី? ប្រាប់ខ្ញុំពីម្ហូប តំបន់ ឬម៉ោង ឧ. “គុយទាវ ជិតខ្ញុំ”។",
           "thanks": "មិនអីទេ! 😊 ចង់រកអ្វីទៀតទេ?",
           "hungry": "តោះរកម្ហូប! 🍜 ចង់ញ៉ាំគុយទាវ បាយសាច់ជ្រូក សាច់អាំង ភីហ្សា ឬកាហ្វេ? ប្រាប់ខ្ញុំមួយ ហើយថែម “ជិតខ្ញុំ” "
                     "ដើម្បីមើលហាងនៅជិត។",
           "bye": "លាហើយ! ញ៉ាំឲ្យឆ្ងាញ់ 🍜",
           "ok": "បាទ! សួរខ្ញុំអំពីហាងទាំងនេះ ឬប្រាប់ថាចង់ញ៉ាំអ្វីទៀត។",
           "about": "ខ្ញុំជា ឃ្លានញ៉ាំ ជំនួយការស្វែងរកម្ហូបនៅកម្ពុជា 🍜។ ខ្ញុំស្វែងរកក្នុងបញ្ជីហាង {n:,} កន្លែង៖ "
                    "សួររកម្ហូប តំបន់ ឬម៉ោង (“កាហ្វេ ជិតខ្ញុំ”) រួចសួរបន្តអំពីហាង៖ ម៉ោងបើក ម៉ឺនុយ "
                    "លេខទូរស័ព្ទ ចម្ងាយ ឬ “ប្រៀបធៀប 1 និង 2”។"},
}
CHAT_KINDS = [("about", ABOUT_RE), ("hungry", HUNGRY_RE),
              ("thanks", re.compile(r"^\s*(thanks?|thank you|thx|ty)\b|អរគុណ", re.I)),
              ("bye", re.compile(r"^\s*(bye|goodbye|see you|good night)\b|^\s*លា", re.I)),
              ("hello", re.compile(r"^\s*(hi+|hello|hey|yo|good (morning|afternoon|evening))\b|សួស្តី|ជំរាបសួរ", re.I)),
              ("ok", re.compile(r"^\s*(ok(ay)?|cool|great|nice|got it)\b|អូខេ|បាទ|ចាស", re.I))]


def chat_reply(message, history=None, model_id=None) -> dict:
    """A short reply to greetings, thanks, small talk and off-topic messages. Returns {text, provider, model, label}."""
    lang = detect_lang(message)
    if CHAT_RE.match(message) or ABOUT_RE.search(message) or HUNGRY_RE.search(message):
        kind = next((k for k, rx in CHAT_KINDS if rx.search(message)), "hello")
        return {"text": CHAT_REPLIES[lang][kind].format(n=place_count()), "provider": "facts", "model": None, "label": None}
    prompt = (
        "You are ឃ្លានញ៉ាំ (Klean Nham), a friendly food guide chatbot for Cambodia. You find places to eat "
        "from a list of real places across Cambodia, in English or Khmer.\n"
        f"Reply to the new message in {'Khmer' if lang == 'kh' else 'English'} in one or two short sentences. Do not "
        "recommend or name any place.\n"
        "If the message is not about food or places to eat (weather, news, homework, poems…), say kindly that you only "
        "help find food in Cambodia, and suggest one example question. If it is about being hungry or not knowing "
        "what to eat, suggest two or three kinds of food to try and ask which one.\n\n"
        f"CONVERSATION SO FAR:\n{history_text(history)}\n\nNEW MESSAGE: {message}"
    )
    try:
        text, m = complete(prompt, model_id)
        return {"text": as_sentences(text), "provider": m["provider"], "model": m["id"], "label": m["label"]}
    except NoModelAnswered:
        return {"text": CHAT_FALLBACK[lang], "provider": "cards", "model": None, "label": None}


def check_ollama() -> bool:
    """Is Ollama reachable, and is OLLAMA_MODEL available? Prints a hint if not."""
    cloud = OLLAMA_HOST.startswith("https://")
    where = f"Ollama cloud ({OLLAMA_HOST})" if cloud else f"Ollama at {OLLAMA_HOST}"
    try:
        r = requests.get(f"{OLLAMA_HOST}/api/tags", headers=ollama_headers(), timeout=5)
        if r.status_code == 401:
            print(f"❌ {where} rejected the key. Hint: check OLLAMA_API_KEY in .env (create a new one at ollama.com/settings/keys).")
            return False
        r.raise_for_status()
    except Exception:
        hint = "check your internet connection" if cloud else "open the Ollama app (or run: ollama serve)"
        print(f"❌ {where} is not reachable. Hint: {hint}.")
        return False
    names = [m.get("name", "") for m in r.json().get("models", [])]
    print(f"✅ {where} is reachable. Models: " + (", ".join(names[:12]) + (" …" if len(names) > 12 else "") or "none"))
    if not OLLAMA_MODEL:
        print("ℹ️ OLLAMA_MODEL is not set in .env. Choose a model from the list above" + ("." if cloud else ", then run: ollama pull <model>"))
        return False
    if OLLAMA_MODEL not in names and f"{OLLAMA_MODEL}:latest" not in names:
        print(f"❌ Model {OLLAMA_MODEL!r} is not available. Hint: " +
              ("set OLLAMA_MODEL to one of the models listed above." if cloud else f"run: ollama pull {OLLAMA_MODEL}"))
        return False
    print(f"✅ Model {OLLAMA_MODEL!r} is available.")
    return True


In [ ]:
%%writefile export/backend/food_search/bot_handlers.py
"""Telegram bot (python-telegram-bot v20+): formatter, handlers and app builder."""
import asyncio
import html
import logging
import os
import re
import tempfile
import time
from collections import Counter, defaultdict, deque
from datetime import datetime
from pathlib import Path
from urllib.parse import quote, quote_plus

import numpy as np
import pandas as pd
from telegram import (BotCommand, InlineKeyboardButton, InlineKeyboardMarkup, KeyboardButton, LinkPreviewOptions,
                      MenuButtonWebApp, ReplyKeyboardMarkup, ReplyKeyboardRemove, WebAppInfo)
from telegram.constants import ChatAction, ParseMode
from telegram.ext import Application, CallbackQueryHandler, CommandHandler, MessageHandler
from telegram.ext import filters as tg_filters

from . import dishes
from .config import ADMIN_IDS, RATE_LIMIT_PER_MIN, TZ, WEBAPP_URL
from .llm import (CONTEXT_PLACES, DEFAULT_MODEL_ID, HISTORY_TURNS, MODELS, MODELS_BY_ID, NO_LLM, answer_followup, no_match,
                  chat_reply, fmt_hours, fmt_price, plan_turn, safe_error, search_lead, unverified_note)
from .search import (detect_lang, geocode_place, get_rows, hybrid_search, image_search_available, is_open_at,
                     parse_query_filters, search_images_by_image, HOME_CITY, PLACE_NAMES_KH)

log = logging.getLogger("food_search.bot")


def maps_url(lat, lng, name="", city="") -> str:
    """Google Maps link that opens the place itself (its listing with rating, photos and hours), not just a pin.
    The name is what is searched; the coordinates only centre the map, so a chain opens its closest branch."""
    name = re.sub(r"\s+", " ", str(name or "")).strip()
    has_xy = not (pd.isna(lat) or pd.isna(lng))
    if not name:
        return f"https://www.google.com/maps/search/?api=1&query={lat:.6f},{lng:.6f}" if has_xy else "https://www.google.com/maps"
    where = f"{name} {city or HOME_CITY}"
    if not has_xy:
        return "https://www.google.com/maps/search/?api=1&query=" + quote_plus(where)
    return f"https://www.google.com/maps/search/{quote(where, safe='')}/@{lat:.6f},{lng:.6f},17z"


TABLE_ROW_RE = re.compile(r"^\s*\|.*\|\s*$")


def _md_inline(line) -> str:
    """One line of light Markdown -> Telegram HTML: escaped < > &, **bold**, [text](https://link), "- " bullets."""
    line = html.escape(line, quote=False)
    line = re.sub(r"\[([^\]]+)\]\((https?://[^)\s]+)\)", lambda m: f'<a href="{m[2].replace(chr(34), "%22")}">{m[1]}</a>', line)
    line = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", line)
    line = re.sub(r"^\s*[*-]\s+", "• ", line)
    return re.sub(r"(?<![<\w])\*|\*(?![\w>])", "", line)


def _table_lines(rows) -> list:
    """Telegram has no tables: '| 1 | Costa | $2 | 07:00–22:00 |' -> '• <b>1. Costa</b> — Price: $2 · Hours: 07:00–22:00'."""
    cells = [[c.strip() for c in r.strip().strip("|").split("|")] for r in rows]
    cells = [c for c in cells if not all(re.fullmatch(r":?-{2,}:?", x) for x in c)]     # the |---| line
    if len(cells) < 2:
        return [_md_inline(r) for r in rows]
    head, out = cells[0], []
    for c in cells[1:]:
        name = f"{c[0]}. {c[1]}" if len(c) > 1 and c[0].isdigit() else c[0]
        rest = [f"{h}: {v}" for h, v in zip(head[2:], c[2:]) if v and v != "—"] if c[0].isdigit() else \
               [f"{h}: {v}" for h, v in zip(head[1:], c[1:]) if v and v != "—"]
        out.append(f"• **{name}**" + (" — " + " · ".join(rest) if rest else ""))
    return [_md_inline(x) for x in out]


def llm_to_telegram_html(text) -> str:
    """Answers use light Markdown (bold, bullets, links, tables); Telegram HTML mode has <b> and <a> but no tables."""
    lines, out, i = text.strip().split("\n"), [], 0
    while i < len(lines):
        if TABLE_ROW_RE.match(lines[i]):
            block = []
            while i < len(lines) and TABLE_ROW_RE.match(lines[i]):
                block.append(lines[i])
                i += 1
            out.extend(_table_lines(block))
            continue
        if lines[i].startswith("— = ") and any(TABLE_ROW_RE.match(x) for x in lines):
            lines[i] = ""                                     # the legend: empty cells are left out here
        out.append(_md_inline(lines[i]))
        i += 1
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()


SOURCE_LABELS = {"openstreetmap": "OpenStreetMap", "telegram": "Telegram channel", "telegram-public": "Telegram channel",
                 "facebook": "Facebook page", "survey": "Field survey", "overture": "Overture Maps (Meta)"}


def page_label(url) -> str:
    """'facebook' or 'website' for a seller's page link; empty when it is not a real http(s) link."""
    url = str(url or "").strip()
    if not url.startswith(("http://", "https://")):
        return ""
    host = url.split("/")[2].lower()
    return "facebook" if host.endswith(("facebook.com", "fb.com")) else "website"


# Attribution required by the data licences, shown once per message (not on every place)
DATA_CREDITS = {"openstreetmap": "© OpenStreetMap contributors", "overture": "Overture Maps (Meta)"}
FOOTER_TEXT = {"en": {"from": "📏 From {where}", "me": "your location", "data": "Data: {credits}"},
               "kh": {"from": "📏 ចម្ងាយគិតពី{where}", "me": "ទីតាំងរបស់អ្នក", "data": "ទិន្នន័យ៖ {credits}"}}
NO_LLM_LABEL = "No AI (results only)"


def fmt_km(km) -> str:
    return f"{round(km * 100) * 10:.0f} m" if km < 1 else f"{km:.1f} km"


def place_card(i, r, lang="en", hour_now=None) -> str:
    """One place, three short lines at most:
    <b>1. Name</b> · 0.6 km
    Coffee · Chroy Changvar · Facebook
    🕒 07:00–17:00 · open now · 💲 $2.00 · ⭐ 4.5      (only the facts we know)"""
    esc = lambda x: html.escape(str(x), quote=False)
    title = f"<b>{i}. {esc(r.name)}</b>"
    if pd.notna(getattr(r, "distance_km", np.nan)):
        title += f" · {fmt_km(r.distance_km)}"
    dish = ((r.dish_kh if lang == "kh" and r.dish_kh else r.dish_en) or "").replace(";", ",").replace("_", " ").strip()
    sub = []
    if dish and dish.lower() not in str(r.name).lower():          # "Starbucks Coffee": no extra "coffee"
        sub.append(esc(dish[:1].upper() + dish[1:]))
    if r.area:
        area_name = PLACE_NAMES_KH.get(r.area, r.area) if lang == "kh" else r.area
        sub.append(esc(area_name))
    url = str(getattr(r, "page_url", "") or "").strip()
    kind = page_label(url)
    if kind:
        sub.append(f'<a href="{html.escape(url, quote=True)}">{"Facebook" if kind == "facebook" else "Website"}</a>')
    facts = []
    if pd.notna(r.open_hour) and pd.notna(r.close_hour):
        facts.append(f"🕒 {fmt_hours(r.open_hour, r.close_hour)}")
        if hour_now is not None and is_open_at(r.open_hour, r.close_hour, hour_now):
            facts.append("open now" if lang != "kh" else "កំពុងបើក")
    if pd.notna(r.price_usd):
        facts.append(f"💲 {fmt_price(r.price_usd)}")
    if pd.notna(r.rating):
        facts.append(f"⭐ {r.rating:.1f}")
    return "\n".join([title] + [" · ".join(x) for x in (sub, facts) if x])


def footer_line(shown, lang="en", origin="") -> str:
    """One quiet line: where distances are measured from, and the data credits."""
    words = FOOTER_TEXT.get(lang, FOOTER_TEXT["en"])
    sources = [str(x).strip().lower() for x in shown["source"]] if "source" in shown else []
    credits = list(dict.fromkeys(DATA_CREDITS.get(x, SOURCE_LABELS.get(x, x.capitalize())) for x in sources if x and x != "nan"))
    parts = []
    if origin:
        parts.append(words["from"].format(where=words["me"] if origin == "me" else html.escape(origin, quote=False)))
    if credits:
        parts.append(words["data"].format(credits=", ".join(credits)))
    return f"<i>{' · '.join(parts)}</i>" if parts else ""


def format_telegram(results, lang="en", start=0, max_items=5, origin="") -> str:
    """Compact Telegram HTML list: numbered places + one footer line. Map links are the buttons below the message."""
    if results is None or len(results) == 0:
        return html.escape(no_match(lang, results), quote=False)
    now = datetime.now(TZ)
    shown = results.iloc[start:start + max_items]
    cards = [place_card(i, r, lang, now.hour + now.minute / 60) for i, r in enumerate(shown.itertuples(), start + 1)]
    footer = footer_line(shown, lang, origin)
    return "\n\n".join(cards + ([footer] if footer else []))


HELP_TEXT = (
    "🍜 Klean Nham – Cambodian food search bot\n"
    "Send a dish name, e.g. 'curry noodle', 'kuy teav open now', 'bbq under $5'.\n"
    "Add 'near me' (I ask for your 📍 location) or 'near <place>', e.g. 'pizza near me', 'coffee near CADT'.\n"
    "Name a khan, city or province for anywhere in Cambodia, e.g. 'pizza in BKK1', 'coffee in Siem Reap' (no place = Phnom Penh first).\n"
    "Every result shows its distance, price, rating, map link, Facebook page or website, and data source when known. You can also send a 📷 food photo.\n"
    "💬 Then just keep chatting: 'is the first one open now?', 'how far is number 2?', 'any cheaper ones?', 'pizza instead'.\n"
    "/new – start a new conversation · /setlocation <place> – tell me where you are · /model – choose the AI model · "
    "/privacy – how your data is used · /deleteme – forget my location\n"
    "Place data: © OpenStreetMap contributors (ODbL); data from Meta, via the Overture Maps Foundation (CDLA Permissive 2.0)\n\n"
    "🍜 ឃ្លានញ៉ាំ (Klean Nham) – បូតស្វែងរកម្ហូបខ្មែរ\n"
    "សូមផ្ញើឈ្មោះម្ហូប ឧ. 'នំបញ្ចុក', 'គុយទាវ ថោក'។\n"
    "សរសេរ 'ជិតខ្ញុំ' ឬ 'near <ទីកន្លែង>' ដើម្បីដឹងចម្ងាយ ឬផ្ញើ 📷 រូបម្ហូប។\n"
    "ដាក់ឈ្មោះខណ្ឌ ក្រុង ឬខេត្ត ឧ. 'កាហ្វេ នៅសៀមរាប'។\n"
    "💬 បន្ទាប់មកសួរបន្តបាន ឧ. 'ហាងទីមួយបើកឥឡូវទេ?'"
)
PRIVACY_TEXT = (
    "🔒 No account needed. Your location and photos are used only to answer your request "
    "and are not stored: your location stays in memory until /deleteme or a bot restart, "
    "photos are deleted right after the search. A place name after 'near' (e.g. CADT) is sent to OpenStreetMap's "
    "map search to find its coordinates; your own coordinates are never sent there. Distances (not coordinates) can "
    "be part of the question sent to the AI model that writes the short summary.\n\n"
    "🔒 មិនចាំបាច់មានគណនីទេ។ ទីតាំង និងរូបថតរបស់អ្នក ប្រើតែដើម្បីឆ្លើយតបប៉ុណ្ណោះ ហើយមិនត្រូវបានរក្សាទុកទេ។"
)
RATE_LIMIT_TEXT = "⏳ Too many requests – please wait a minute. / សូមរង់ចាំមួយនាទី។"
ASK_LOCATION_TEXT = (
    "📍 Where are you? Tap the closest landmark below. On a phone you can also share your exact location with the next message. "
    "On a computer, type /setlocation and a place, e.g. /setlocation CADT. It is kept in memory only, for this chat.\n"
    "📍 តើអ្នកនៅឯណា? សូមជ្រើសរើសទីតាំងដែលនៅជិតបំផុតខាងក្រោម។")

STATS = Counter()           # in-memory counters for /stats (no message text, no ids)
_seen_users = set()
_user_hits = defaultdict(deque)


def allow_request(user_id, now=None) -> bool:
    """Sliding-window rate limit: RATE_LIMIT_PER_MIN requests per user per 60 s."""
    now = time.monotonic() if now is None else now
    hits = _user_hits[user_id]
    while hits and now - hits[0] > 60:
        hits.popleft()
    if len(hits) >= RATE_LIMIT_PER_MIN:
        return False
    hits.append(now)
    return True


async def _guard(update, kind) -> bool:
    """Count the request and enforce the rate limit."""
    uid = update.effective_user.id
    _seen_users.add(uid)
    STATS[kind] += 1
    if not allow_request(uid):
        STATS["rate_limited"] += 1
        await update.effective_message.reply_text(RATE_LIMIT_TEXT)
        return False
    return True


def map_buttons(numbered):
    """[(number, row), ...] -> Google Maps buttons, numbered like the list, two per row."""
    short = lambda name: name if len(name) <= 20 else name[:19].rstrip() + "…"
    buttons = [InlineKeyboardButton(f"📍 {n}. {short(str(r.name))}", url=maps_url(r.lat, r.lng, r.name, getattr(r, "city", ""))) for n, r in numbered]
    return [buttons[k:k + 2] for k in range(0, len(buttons), 2)]


def results_keyboard(results, start=0, max_items=5):
    """A Google Maps button per shown place (numbered like the list, two per row) + 'More results'."""
    rows = map_buttons(list(enumerate(results.iloc[start:start + max_items].itertuples(), start + 1)))
    if len(results) > start + max_items:
        rows.append([InlineKeyboardButton("➕ More results", callback_data="more")])
    return InlineKeyboardMarkup(rows) if rows else None


NEAR_SPOTS = [   # (button label, lat, lng): well-known landmarks, so "near me" also works where sharing a location does not
    ("Independence Monument", 11.5564, 104.9282), ("Royal Palace", 11.5638, 104.9301),
    ("Wat Phnom", 11.5761, 104.9232), ("Central Market", 11.5696, 104.9211),
    ("Russian Market", 11.5405, 104.9147), ("Olympic Stadium", 11.5584, 104.9122),
    ("AEON Mall 1", 11.5481, 104.9326), ("AEON Mall Sen Sok", 11.6002, 104.8854),
    ("ITC", 11.5706, 104.8980), ("CADT", 11.6537, 104.9117),
]


def spots_keyboard():
    """Ten landmark buttons, two per row. Tapping one sets 'where I am' (see spot_callback)."""
    buttons = [InlineKeyboardButton(f"📍 {name}", callback_data=f"spot:{i}") for i, (name, _, _) in enumerate(NEAR_SPOTS)]
    return InlineKeyboardMarkup([buttons[i:i + 2] for i in range(0, len(buttons), 2)])


def location_keyboard():
    """A one-tap 'share my location' button (works in the phone apps)."""
    return ReplyKeyboardMarkup([[KeyboardButton("📍 Share my location", request_location=True)]],
                               resize_keyboard=True, one_time_keyboard=True)


def webapp_ready() -> bool:
    return WEBAPP_URL.startswith("https://")     # Telegram opens Mini Apps over HTTPS only


def webapp_keyboard():
    if not webapp_ready():
        return None
    return InlineKeyboardMarkup([[InlineKeyboardButton("🍜 Open the app / បើកកម្មវិធី", web_app=WebAppInfo(url=WEBAPP_URL))]])


async def start_cmd(update, context):
    await update.message.reply_text(HELP_TEXT, reply_markup=webapp_keyboard())


async def app_cmd(update, context):
    if not webapp_ready():
        await update.message.reply_text("The app is not online yet. Send a dish name here instead.")
        return
    await update.message.reply_text("Search with the app: / ស្វែងរកតាមកម្មវិធី៖", reply_markup=webapp_keyboard())


BOT_PROFILE = {   # (keys are Telegram language codes: "km" is the ISO code for Khmer) what Telegram shows on the bot's profile and in an empty chat (set at every start; cheap and idempotent)
    None: {"description": "Klean Nham: Find food across Cambodia. Send a dish, a place or a time, in English or Khmer, e.g. “kuy teav”, "
                          "“coffee near me”, “pizza in Toul Kork”, “coffee in Siem Reap”. Answers come only from our list of places, so nothing "
                          "is made up.",
           "short": "Klean Nham: Find food across Cambodia, in English or Khmer. Answers only from real places.",
           "commands": [("start", "How to use the bot"), ("new", "Start a new conversation"), ("model", "Choose the AI model"),
                        ("setlocation", "Tell me where you are"),
                        ("privacy", "How your data is used"), ("deleteme", "Forget my location")]},
    "km": {"description": "ឃ្លានញ៉ាំ (Klean Nham)៖ ស្វែងរកម្ហូបទូទាំងប្រទេសកម្ពុជា។ ផ្ញើឈ្មោះម្ហូប ទីកន្លែង ឬម៉ោង ជាភាសាខ្មែរ ឬអង់គ្លេស ឧ. “គុយទាវ” "
                          "“កាហ្វេ ជិតខ្ញុំ”។ ចម្លើយមកពីបញ្ជីហាងពិតប្រាកដរបស់យើងតែប៉ុណ្ណោះ។",
           "short": "ឃ្លានញ៉ាំ (Klean Nham)៖ ស្វែងរកម្ហូបទូទាំងប្រទេសកម្ពុជា ជាភាសាខ្មែរ ឬអង់គ្លេស។",
           "commands": [("start", "របៀបប្រើបូត"), ("new", "ចាប់ផ្តើមការសន្ទនាថ្មី"), ("model", "ជ្រើសរើសម៉ូដែល AI"),
                        ("setlocation", "ប្រាប់ទីតាំងរបស់អ្នក"),
                        ("privacy", "របៀបប្រើទិន្នន័យរបស់អ្នក"), ("deleteme", "លុបទីតាំងរបស់ខ្ញុំ")]},
}


async def setup_bot(app):
    """Bot profile text, the / command menu, and the 'Open app' button (when WEBAPP_URL is an https:// address)."""
    bot = app.bot
    for lang, p in BOT_PROFILE.items():
        commands = list(p["commands"]) + ([("app", "Open the app" if lang is None else "បើកកម្មវិធី")] if webapp_ready() else [])
        try:
            await bot.set_my_commands([BotCommand(c, d) for c, d in commands], language_code=lang)
            await bot.set_my_description(p["description"], language_code=lang)
            await bot.set_my_short_description(p["short"], language_code=lang)
        except Exception as e:
            log.warning("could not set the bot profile (%s)", safe_error(e))
    if webapp_ready():
        try:
            await bot.set_chat_menu_button(menu_button=MenuButtonWebApp("Open app", WebAppInfo(url=WEBAPP_URL)))
        except Exception as e:
            log.warning("could not set the menu button (%s)", safe_error(e))


def models_keyboard(current):
    """One button per model from LLM_MODELS + 'No AI'; the current choice has a ✓."""
    choices = [(m["id"], m["label"]) for m in MODELS] + [(NO_LLM, NO_LLM_LABEL)]
    return InlineKeyboardMarkup([[InlineKeyboardButton(("✓ " if mid == current else "") + label, callback_data=f"model:{i}")]
                                 for i, (mid, label) in enumerate(choices)])


def current_model(context):
    return context.user_data.get("model") or DEFAULT_MODEL_ID


async def model_cmd(update, context):
    await update.message.reply_text(
        "🤖 Which AI model should write the short answer? Every answer still comes only from the places I find. "
        "If a model is busy, another one from the list answers.\n🤖 ជ្រើសរើសម៉ូដែល AI៖",
        reply_markup=models_keyboard(current_model(context)))


async def model_callback(update, context):
    q = update.callback_query
    choices = [m["id"] for m in MODELS] + [NO_LLM]
    try:
        chosen = choices[int(q.data.split(":", 1)[1])]
    except (ValueError, IndexError):
        await q.answer()
        return
    context.user_data["model"] = chosen                      # memory only, per user
    label = MODELS_BY_ID[chosen]["label"] if chosen in MODELS_BY_ID else NO_LLM_LABEL
    await q.answer(f"✓ {label}")
    try:
        await q.edit_message_reply_markup(reply_markup=models_keyboard(chosen))
    except Exception:
        pass                                                  # "message is not modified" when tapping the same model


async def privacy_cmd(update, context):
    await update.message.reply_text(PRIVACY_TEXT)


async def deleteme_cmd(update, context):
    context.user_data.clear()                # location + last results
    await update.message.reply_text("🗑️ Your saved location was deleted. / ទីតាំងរបស់អ្នកត្រូវបានលុប។")


async def setlocation_cmd(update, context):
    place = " ".join(context.args or []).strip()[:60]
    if not place:
        await update.message.reply_text("Type /setlocation and where you are, e.g. /setlocation CADT or /setlocation Aeon Mall Sen Sok, "
                                        "or tap a landmark.\nឧទាហរណ៍៖ /setlocation CADT", reply_markup=spots_keyboard())
        return
    if not await _guard(update, "locations"):
        return
    spot = await asyncio.to_thread(geocode_place, place)
    if not spot:
        await update.message.reply_text(f"I could not find “{place}” on the map. Try the full name, or share your 📍 location.")
        return
    context.user_data["location"] = spot                       # memory only
    pending = context.user_data.pop("pending_query", None)
    await update.message.reply_text(f"📍 Your location is now {place} (kept in memory only, /deleteme forgets it)."
                                    + (" Searching…" if pending else " Now try: pizza near me"),
                                    reply_markup=ReplyKeyboardRemove())
    if pending:
        await run_query(update.message, context, pending, spot)


async def stats_cmd(update, context):
    if update.effective_user.id not in ADMIN_IDS:
        await update.message.reply_text("⛔ Admins only.")
        return
    lines = [f"{k}: {v}" for k, v in sorted(STATS.items())] + [f"unique users: {len(_seen_users)}"]
    await update.message.reply_text("📊 Bot stats (since start)\n" + "\n".join(lines))


async def send_results(message, context, results, lang, intro="", warning="", origin=""):
    """intro = the LLM summary; warning = a problem worth reading first; origin = where distances are measured from."""
    context.user_data.update(last_results=results, last_lang=lang, offset=0, last_origin=origin)
    head = "\n\n".join(x for x in (intro, warning) if x)
    text = (head + "\n\n" if head else "") + format_telegram(results, lang, origin=origin)
    await message.reply_text(text, parse_mode=ParseMode.HTML, reply_markup=results_keyboard(results),
                             link_preview_options=LinkPreviewOptions(is_disabled=True))


async def run_query(message, context, query, user_loc=None, rewritten=False):
    """Search, add the LLM summary (if configured) and reply. Used for text queries and for 'near me' after a location share.
    rewritten = the query was built from the conversation ('any cheaper ones?' -> 'cheap pizza near me'): say so."""
    lang = detect_lang(query)
    _, flt = parse_query_filters(query)
    near, origin, warning = flt.get("near"), "", ""
    if near and near != "me":                # "pizza near CADT": look the place up on OpenStreetMap
        spot = await asyncio.to_thread(geocode_place, near, flt.get("city"))
        i = query.lower().rfind(near.lower())
        typed = query[i:i + len(near)] if i >= 0 else near.title()     # the user's spelling: "CADT", not "Cadt"
        if spot:
            user_loc, origin = spot, typed
        else:
            user_loc = None
            warning = (f"⚠️ I could not find “{html.escape(typed, quote=False)}” on the map, so there are no distances. "
                       "Try the full name, or say 'near me'.")
    elif user_loc:
        origin = "me"
    results = await asyncio.to_thread(hybrid_search, query, 15, user_loc)
    unchecked = unverified_note(results, lang)               # "under $5" when the data has no prices for these places
    if unchecked:
        warning = "\n".join(x for x in (warning, f"ℹ️ {html.escape(unchecked, quote=False)}") if x)
    lead = search_lead(results, lang, origin or None) if len(results) else ""   # no AI: the cards list the places
    intro = html.escape(lead, quote=False)
    if rewritten:
        intro = f"🔎 <i>{html.escape(query, quote=False)}</i>" + (f"\n\n{intro}" if intro else "")
    await send_results(message, context, results, lang, intro, warning, origin)
    context.user_data["last_query"] = query
    remember(context, "assistant", lead + " Places shown: " + places_summary(results.head(5)))


# ---------- conversation memory (per chat, in memory only; /new or /deleteme clears it) ----------

def remember(context, role, text):
    """Adds a message to this chat's short history (the last HISTORY_TURNS exchanges)."""
    history = context.user_data.setdefault("history", [])
    history.append({"role": role, "text": " ".join(str(text).split())[:500]})
    del history[:-HISTORY_TURNS * 2]


def places_summary(rows, start=1) -> str:
    return "; ".join(f"{n}. {r.name} ({', '.join(x for x in (r.dish_en, r.area) if x)})"
                     for n, r in enumerate(rows.itertuples(), start))


def shown_places(context):
    """The places the user has seen, numbered as shown (the first 5, plus any 'More results'), or None."""
    results = context.user_data.get("last_results")
    if results is None or len(results) == 0:
        return None
    return results.iloc[:min(context.user_data.get("offset", 0) + 5, CONTEXT_PLACES)]


async def new_cmd(update, context):
    for key in ("history", "last_results", "last_query", "offset", "last_origin", "pending_query"):
        context.user_data.pop(key, None)                     # keeps the location and the chosen model
    await update.message.reply_text("🆕 New conversation. What would you like to eat? / ការសន្ទនាថ្មី។ តើអ្នកចង់ញ៉ាំអ្វី?")


async def text_handler(update, context):
    if not await _guard(update, "text_queries"):
        return
    query = update.message.text.strip()[:200]
    log.info("text query (lang=%s, %d chars)", detect_lang(query), len(query))   # never log text together with ids
    await context.bot.send_chat_action(update.effective_chat.id, ChatAction.TYPING)
    history = list(context.user_data.get("history", []))
    shown, model = shown_places(context), context.user_data.get("model")
    focus, named = context.user_data.get("dish"), dishes.find_dish(query)
    if dishes.recipe_intent(query, has_dish=bool(named or focus)):      # "how do I make it?", "how long?"
        await send_recipe_answer(update.message, context, query, named or focus)
        return
    query = dishes.resolve_reference(query, focus)    # "where can I eat it?" after a photo of pizza
    plan = await asyncio.to_thread(plan_turn, query, history, shown, model, context.user_data.get("last_query"))
    remember(context, "user", query)

    if plan["action"] == "chat":
        out = await asyncio.to_thread(chat_reply, query, history, model)
        await update.message.reply_text(llm_to_telegram_html(out["text"]), parse_mode=ParseMode.HTML)
        remember(context, "assistant", out["text"])
        return

    if plan["action"] == "followup" and shown is not None:     # about the places already shown: no new search
        out = await asyncio.to_thread(answer_followup, query, history, shown, plan["places"], model)
        text = llm_to_telegram_html(out["text"])               # AI sentence, or a list / table built from the data
        rows = list(shown.itertuples())                         # (a pandas Series' .name is its index, not the place)
        numbered = [(n, rows[n - 1]) for n in out["numbers"][:6]]
        await update.message.reply_text(text, parse_mode=ParseMode.HTML,
                                        reply_markup=InlineKeyboardMarkup(map_buttons(numbered)),
                                        link_preview_options=LinkPreviewOptions(is_disabled=True))
        remember(context, "assistant", out["text"])
        return

    search_query = plan["query"] or query
    _, flt = parse_query_filters(search_query)
    if flt.get("near") == "me" and not context.user_data.get("location"):
        context.user_data["pending_query"] = search_query          # answered as soon as the location arrives
        await update.message.reply_text(ASK_LOCATION_TEXT, reply_markup=spots_keyboard())
        await update.message.reply_text("📍 Or share your exact location (phone):", reply_markup=location_keyboard())
        return
    await run_query(update.message, context, search_query, context.user_data.get("location"),
                    rewritten=search_query.strip().lower() != query.strip().lower())


async def spot_callback(update, context):
    q = update.callback_query
    await q.answer()
    try:
        name, lat, lng = NEAR_SPOTS[int(q.data.split(":", 1)[1])]
    except (ValueError, IndexError):
        return
    if not await _guard(update, "locations"):
        return
    context.user_data["location"] = (lat, lng)                    # memory only
    pending = context.user_data.pop("pending_query", None)
    await q.message.reply_text(f"📍 Your location is now near {name} (kept in memory only, /deleteme forgets it)."
                               + (" Searching…" if pending else " Now try: pizza near me"), reply_markup=ReplyKeyboardRemove())
    if pending:
        await run_query(q.message, context, pending, (lat, lng))


async def more_callback(update, context):
    q = update.callback_query
    await q.answer()
    results, lang = context.user_data.get("last_results"), context.user_data.get("last_lang", "en")
    offset = context.user_data.get("offset", 0) + 5
    if results is None or offset >= len(results):
        await q.message.reply_text("No more results. / មិនមានលទ្ធផលទៀតទេ។")
        return
    context.user_data["offset"] = offset
    await q.message.reply_text(format_telegram(results, lang, start=offset, origin=context.user_data.get("last_origin", "")),
                               parse_mode=ParseMode.HTML,
                               reply_markup=results_keyboard(results, start=offset),
                               link_preview_options=LinkPreviewOptions(is_disabled=True))
    remember(context, "assistant", "More places shown: " + places_summary(results.iloc[offset:offset + 5], offset + 1))


async def location_handler(update, context):
    if update.edited_message is not None:        # a live location that moves: Telegram sends each new point as an edit
        if update.edited_message.location:
            context.user_data["location"] = (update.edited_message.location.latitude,
                                             update.edited_message.location.longitude)
        return                                   # no reply, no rate limit: just keep the newest point
    if not await _guard(update, "locations"):
        return
    loc = update.message.location
    context.user_data["location"] = (loc.latitude, loc.longitude)   # memory only
    pending = context.user_data.pop("pending_query", None)
    await update.message.reply_text(
        "📍 Location received (kept in memory only, /deleteme forgets it)." + (" Searching…" if pending else " Now send a dish name.")
        + "\n📍 បានទទួលទីតាំង។", reply_markup=ReplyKeyboardRemove())
    if pending:
        await context.bot.send_chat_action(update.effective_chat.id, ChatAction.TYPING)
        await run_query(update.message, context, pending, context.user_data["location"])


async def photo_handler(update, context):
    if not await _guard(update, "photo_queries"):
        return
    if not (dishes.available() or image_search_available()):
        await update.message.reply_text("📷 Photo search is not set up yet. Please type the dish name instead.")
        return
    await context.bot.send_chat_action(update.effective_chat.id, ChatAction.TYPING)
    tg_file = await update.message.photo[-1].get_file()
    fd, tmp_path = tempfile.mkstemp(suffix=".jpg")
    os.close(fd)
    try:
        await tg_file.download_to_drive(tmp_path)
        if dishes.available():
            guesses = await asyncio.to_thread(dishes.classify, tmp_path, 3)
        else:
            matches = await asyncio.to_thread(search_images_by_image, tmp_path, 10)
    finally:
        Path(tmp_path).unlink(missing_ok=True)   # the user's photo is never kept
    if dishes.available():
        best = guesses[0]
        await send_dish(update.message, context, dishes.dish_json(best["id"], best["confidence"], guesses[1:]))
        return
    results = get_rows([m["restaurant_id"] for m in matches])
    await send_results(update.message, context, results, "en", "📷 Places with similar-looking food:")


TG_MAX_CHARS = 4000          # Telegram allows 4096 characters per message


async def send_dish(message, context, dish):
    """The recognised (or picked) dish: what it is, its cooking time, buttons for the recipe and other guesses,
    then the places that serve it."""
    lang = "en"
    context.user_data["dish"] = dish["id"]
    lines = [llm_to_telegram_html(dishes.photo_lead(dish, lang))]
    r = dishes.recipe_json(dish["id"])
    buttons = []
    if r:
        lines.append(llm_to_telegram_html(dishes.recipe_answer("how long", dish["id"], lang)["text"]))
        lines.append("Ask me “how do I make it?”, “what are the ingredients?” or “where can I eat it?”.")
        buttons.append([InlineKeyboardButton("🍳 Recipe", callback_data=f"recipe:{dish['id']}"),
                        InlineKeyboardButton("🧺 Ingredients", callback_data=f"ingr:{dish['id']}")])
    else:
        lines.append(html.escape(dishes.recipe_answer("recipe", dish["id"], lang)["text"], quote=False))
    if not dish["sure"] and dish["alternatives"]:
        lines.append("Not it? Tap the right dish:")
        buttons.append([InlineKeyboardButton(f"{a['name']} ({round(a['confidence'] * 100)}%)", callback_data=f"dish:{a['id']}")
                        for a in dish["alternatives"]])
    await message.reply_text("\n\n".join(lines), parse_mode=ParseMode.HTML,
                             reply_markup=InlineKeyboardMarkup(buttons) if buttons else None)
    remember(context, "assistant", f"This photo looks like {dish['name']}.")
    results = await asyncio.to_thread(hybrid_search, dish["name"], 15, context.user_data.get("location"))
    if len(results):
        await send_results(message, context, results, lang, f"📍 Places that serve {html.escape(dish['name'], quote=False)}:")
        remember(context, "assistant", "Places shown: " + places_summary(results.head(5)))


async def send_recipe_answer(message, context, question, dish_id):
    out = await asyncio.to_thread(dishes.recipe_answer, question, dish_id, "en", context.user_data.get("model"), True)
    text = llm_to_telegram_html(out["text"])
    r = dishes.recipe_json(dish_id)
    if len(text) > TG_MAX_CHARS:
        text = text[:TG_MAX_CHARS].rsplit("\n", 1)[0] + "\n…"
    if r and r["source_url"] and out["focus"] in ("steps", "ingredients"):
        text += f'\n\n<a href="{html.escape(r["source_url"])}">Full recipe on {html.escape(r["source"])}</a>'
    await message.reply_text(text, parse_mode=ParseMode.HTML, link_preview_options=LinkPreviewOptions(is_disabled=True))
    context.user_data["dish"] = dish_id
    remember(context, "assistant", out["text"])


async def dish_callback(update, context):
    """Buttons under a recognised dish: recipe:<id>, ingr:<id> (answer from the recipe), dish:<id> (another guess)."""
    q = update.callback_query
    await q.answer()
    action, _, dish_id = q.data.partition(":")
    if dish_id not in dishes.DISHES or not await _guard(update, "dish_buttons"):
        return
    if action == "dish":
        await send_dish(q.message, context, dishes.dish_json(dish_id))
    else:
        await send_recipe_answer(q.message, context, "ingredients" if action == "ingr" else "how to make it", dish_id)


async def error_handler(update, context):
    log.error("bot error: %s", safe_error(context.error))


def build_bot_app(token):
    # generous timeouts: the default 5 seconds is too tight on a slow or unstable connection
    app = (Application.builder().token(token).connect_timeout(15).read_timeout(25).write_timeout(25)
           .pool_timeout(15).get_updates_connect_timeout(15).get_updates_read_timeout(35)
           .post_init(setup_bot).build())
    app.add_handler(CommandHandler(["start", "help"], start_cmd))
    app.add_handler(CommandHandler("app", app_cmd))
    app.add_handler(CommandHandler("model", model_cmd))
    app.add_handler(CommandHandler("new", new_cmd))
    app.add_handler(CallbackQueryHandler(model_callback, pattern="^model:"))
    app.add_handler(CommandHandler("privacy", privacy_cmd))
    app.add_handler(CommandHandler("deleteme", deleteme_cmd))
    app.add_handler(CommandHandler("stats", stats_cmd))
    app.add_handler(CommandHandler("setlocation", setlocation_cmd))
    app.add_handler(CallbackQueryHandler(more_callback, pattern="^more$"))
    app.add_handler(CallbackQueryHandler(spot_callback, pattern="^spot:"))
    app.add_handler(CallbackQueryHandler(dish_callback, pattern="^(recipe|ingr|dish):"))
    app.add_handler(MessageHandler(tg_filters.LOCATION, location_handler))
    app.add_handler(MessageHandler(tg_filters.PHOTO, photo_handler))
    app.add_handler(MessageHandler(tg_filters.TEXT & ~tg_filters.COMMAND, text_handler))
    app.add_error_handler(error_handler)
    return app


In [ ]:
%%writefile export/backend/food_search/api.py
"""The web app (FastAPI): the JSON API used by the React frontend, and the built frontend itself.

  GET  /api/config        what the UI can offer (photo search, number of places, AI models)
  POST /api/chat          a question -> places + a short AI answer written only from those places
  POST /api/photo         a food photo -> the dish (Food-101 classifier), its recipe and places that serve it
                          (without a dish model: places with similar-looking food photos)
  POST /api/dish          a dish chosen by the user (another guess for the photo) -> recipe + places
  POST /api/demo/ask      the demo: answer + the 3 steps (understand, retrieve, generate) + scored rows + prompt
  POST /api/demo/compare  the demo: BM25 vs embeddings vs hybrid ranking for one query
  GET  /api/demo/data     the demo: what the place list contains
  /, /demo                the React app from frontend/dist (cd frontend && npm install && npm run build)
  /docs                   API documentation (generated by FastAPI)

Inside Telegram the page sends its signed initData (X-Telegram-Init-Data); it is checked with the bot token and the
Telegram user id is used for the rate limit. Outside Telegram the client IP is used.
Call search.init() before create_app(). No X-Frame-Options header: Telegram Web shows Mini Apps in an iframe.
"""
import asyncio
import hashlib
import hmac
import json
import logging
import os
import tempfile
import time
from datetime import datetime
from pathlib import Path
from typing import Literal
from urllib.parse import parse_qsl

import numpy as np
import pandas as pd
from fastapi import APIRouter, FastAPI, File, Form, Request, UploadFile
from fastapi.exceptions import RequestValidationError
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from pydantic import BaseModel, Field

from . import dishes, search
from .bot_handlers import SOURCE_LABELS, STATS, allow_request, maps_url, page_label
from .config import ENABLE_DEMO, FRONTEND_DIST, HYBRID_WEIGHT, MAX_PHOTO_MB, TELEGRAM_TOKEN, TOP_K, TZ
from .llm import (CONTEXT_PLACES, DEFAULT_MODEL_ID, HISTORY_TURNS, MODELS, MODELS_BY_ID, NO_LLM, no_match,
                  answer_followup, asked_fields, build_prompt, chat_reply, generate_answer, plan_turn, safe_error, search_lead,
                  unverified_note,
                  valid_model_id)
from .search import (bm25_search, detect_lang, embedding_search, geocode_place, get_rows, haversine_km, hybrid_search,
                     image_search_available, is_open_at, parse_query_filters, search_images_by_image)

log = logging.getLogger("food_search.web")


# ======================= who is asking: Telegram sign-in check, rate-limit key, JSON errors =======================

INIT_DATA_MAX_AGE_S = 24 * 3600
RATE_LIMIT_TEXT = {"en": "Too many searches. Wait a minute, then try again.",
                   "kh": "ស្វែងរកច្រើនពេក។ សូមរង់ចាំមួយនាទី រួចសាកម្តងទៀត។"}


def telegram_user(init_data: str):
    """The Telegram user from Mini App initData, or None if it is missing, forged or too old."""
    if not init_data or not TELEGRAM_TOKEN:
        return None
    fields = dict(parse_qsl(init_data, keep_blank_values=True))
    received = fields.pop("hash", "")
    secret = hmac.new(b"WebAppData", TELEGRAM_TOKEN.encode(), hashlib.sha256).digest()

    def valid(items):
        check = "\n".join(f"{k}={v}" for k, v in sorted(items.items()))
        return hmac.compare_digest(hmac.new(secret, check.encode(), hashlib.sha256).hexdigest(), received)

    # newer clients add a "signature" field; depending on the client it is or is not part of the hashed string
    if not received or not (valid(fields) or valid({k: v for k, v in fields.items() if k != "signature"})):
        return None
    auth_date = fields.get("auth_date", "")
    if not auth_date.isdigit() or time.time() - int(auth_date) > INIT_DATA_MAX_AGE_S:
        return None
    try:
        user = json.loads(fields.get("user", "{}"))
    except ValueError:
        return None
    return user if isinstance(user, dict) and user.get("id") else None


def client_key(request: Request):
    """Rate-limit key: the verified Telegram user id (shared with the bot), else the client IP."""
    user = telegram_user(request.headers.get("X-Telegram-Init-Data", ""))
    return user["id"] if user else f"ip:{request.client.host if request.client else 'unknown'}"


def error(status, code, message):
    return JSONResponse({"error": code, "message": message}, status_code=status)


def too_many(lang="en"):
    STATS["rate_limited"] += 1
    return error(429, "rate_limited", RATE_LIMIT_TEXT.get(lang, RATE_LIMIT_TEXT["en"]))


# ======================= the RAG step: retrieve, then generate (shared by the chat and the demo) =======================

RESULTS_PER_QUERY = 15          # the chat shows 5 and reveals the rest with "Show more"


def as_typed(query, part):
    """The user's own spelling of a lowercased part of the query ('cadt' -> 'CADT')."""
    i = query.lower().rfind(part.lower())
    return query[i:i + len(part)] if i >= 0 else part.title()


def answer_query(query, user_loc=None, model=None, k=RESULTS_PER_QUERY, lead=False) -> dict:
    """Retrieve (hybrid search, with 'near <place>' geocoded) and generate (LLM over the top rows only).
    lead=True (the chat): no model, only a one-line lead built from the search, because the cards under it list the
    places. The demo keeps the model, to show the RAG step. Blocking: call it from a thread in async code."""
    t0 = time.perf_counter()
    _, flt = parse_query_filters(query)
    near, origin, llm_query = flt.get("near"), None, query
    if near and near != "me":                         # "pizza near CADT": look the place up on OpenStreetMap
        spot = geocode_place(near, flt.get("city"))
        if spot:
            user_loc, origin = spot, {"kind": "place", "name": as_typed(query, near)}
        else:
            user_loc, llm_query = None, parse_query_filters(query)[0] or query   # do not confuse the LLM
            origin = {"kind": "not_found", "name": as_typed(query, near)}
    elif user_loc:
        origin = {"kind": "me"}
    elif near == "me":                                # "near me" but the user chose not to share a location
        origin, llm_query = {"kind": "no_location"}, parse_query_filters(query)[0] or query
    results = hybrid_search(query, k, user_loc)
    t1 = time.perf_counter()
    if lead:
        near_name = "me" if origin and origin["kind"] == "me" else origin["name"] if origin and origin["kind"] == "place" else None
        gen = {"text": search_lead(results, detect_lang(query), near_name) if len(results) else no_match(detect_lang(query), results),
               "provider": "facts" if len(results) else "no-results", "model": None, "label": None,
               "chosen": valid_model_id(model) or DEFAULT_MODEL_ID}
    else:
        gen = generate_answer(llm_query, results.head(TOP_K), model)
    t2 = time.perf_counter()
    return {"results": results, "answer": gen["text"], "provider": gen["provider"], "model": gen["model"],
            "model_label": gen["label"], "chosen_model": gen["chosen"], "origin": origin, "llm_query": llm_query,
            "timing": {"retrieve_ms": round((t1 - t0) * 1000), "generate_ms": round((t2 - t1) * 1000)}}


def num(x, digits=2):
    return None if x is None or pd.isna(x) else round(float(x), digits)


def place_json(r, lang, hour_now) -> dict:
    """One result row for the UI. Unknown values are null (the UI leaves them out), never guessed."""
    name_val = r["name"] if isinstance(r, (pd.Series, dict)) else getattr(r, "name", "")
    dish_en_val = r["dish_en"] if isinstance(r, (pd.Series, dict)) else getattr(r, "dish_en", "")
    dish_kh_val = r["dish_kh"] if (isinstance(r, (pd.Series, dict)) and "dish_kh" in r) else getattr(r, "dish_kh", "")
    dish_display = (dish_kh_val if lang == "kh" and dish_kh_val else dish_en_val) or ""
    tags_val = r["tags"] if isinstance(r, (pd.Series, dict)) else getattr(r, "tags", "")
    area_val = r["area"] if isinstance(r, (pd.Series, dict)) else getattr(r, "area", "")
    city_val = (r.get("city", "") if isinstance(r, (pd.Series, dict)) else getattr(r, "city", "")) or ""
    open_h = r["open_hour"] if isinstance(r, (pd.Series, dict)) else getattr(r, "open_hour", np.nan)
    close_h = r["close_hour"] if isinstance(r, (pd.Series, dict)) else getattr(r, "close_hour", np.nan)
    price_val = r["price_usd"] if isinstance(r, (pd.Series, dict)) else getattr(r, "price_usd", np.nan)
    rating_val = r["rating"] if isinstance(r, (pd.Series, dict)) else getattr(r, "rating", np.nan)
    lat_val = r["lat"] if isinstance(r, (pd.Series, dict)) else getattr(r, "lat", np.nan)
    lng_val = r["lng"] if isinstance(r, (pd.Series, dict)) else getattr(r, "lng", np.nan)
    id_val = r["id"] if isinstance(r, (pd.Series, dict)) else getattr(r, "id", "")
    source_val = r["source"] if isinstance(r, (pd.Series, dict)) else getattr(r, "source", "")
    has_hours = pd.notna(open_h) and pd.notna(close_h)
    source = str(source_val or "").strip()
    source_url = str(r["source_url"] if isinstance(r, (pd.Series, dict)) else getattr(r, "source_url", "") or "").strip()
    phone = str(r["phone_public"] if isinstance(r, (pd.Series, dict)) else getattr(r, "phone_public", "") or "").strip()
    page_url = str(r["page_url"] if isinstance(r, (pd.Series, dict)) else getattr(r, "page_url", "") or "").strip()
    img_url = str(r.get("image_url", "") or r.get("image_path", "") if isinstance(r, (pd.Series, dict)) else getattr(r, "image_url", getattr(r, "image_path", "")) or "").strip()

    # Parse menu items for place card display
    menu_raw = r.get("menu", "") if isinstance(r, (pd.Series, dict)) else getattr(r, "menu", "")
    menu_list = []
    if menu_raw and isinstance(menu_raw, str):
        try:
            import json as _json
            items = _json.loads(menu_raw)
            if isinstance(items, list):
                is_kh = lang == "kh"
                for it in items:
                    name_dish = it.get("name_kh" if is_kh else "name_en") or it.get("name_en") or it.get("name_kh") or ""
                    menu_list.append({
                        "name": name_dish,
                        "name_en": it.get("name_en", ""),
                        "name_kh": it.get("name_kh", ""),
                        "price_usd": float(it.get("price_usd", 0)),
                        "price_khr": int(it.get("price_khr", 0)),
                    })
        except Exception:
            menu_list = []

    return {
        "id": id_val,
        "name": name_val,
        "dish": dish_display.replace(";", ",").replace("_", " "),
        "kind": str(tags_val or "").split(";")[0].strip().replace("_", " "),
        "area": area_val or "",
        "city": city_val,
        "price_usd": num(price_val),
        "rating": num(rating_val, 1),
        "open_hour": num(open_h, 0) if has_hours else None,
        "close_hour": num(close_h, 0) if has_hours else None,
        "open_now": bool(is_open_at(open_h, close_h, hour_now)) if has_hours else None,
        "distance_km": num(r.get("distance_km", np.nan) if isinstance(r, (pd.Series, dict)) else getattr(r, "distance_km", np.nan)),
        "maps_url": maps_url(lat_val, lng_val, name_val, city_val),
        "phone": phone if phone and phone.lower() != "nan" else None,
        "source": SOURCE_LABELS.get(source.lower(), source.capitalize()) if source and source.lower() != "nan" else None,
        "source_url": source_url if source_url.startswith(("http://", "https://")) else None,
        "page_url": page_url if page_label(page_url) else None,          # the seller's Facebook page / website
        "page_kind": page_label(page_url) or None,
        "image_url": img_url if img_url.startswith(("http://", "https://")) else None,
        "menu": menu_list,
    }


def places_json(results, lang, with_menu=False) -> list:
    """with_menu: the cards show each place's menu only when the question asked for it ("pizza menu in BKK1")."""
    now = datetime.now(TZ)
    hour = now.hour + now.minute / 60
    places = [place_json(r, lang, hour) for r in results.itertuples()]
    if not with_menu:
        for p in places:
            p["menu"] = []
    return places


# ======================= request bodies (checked by FastAPI, documented at /docs) =======================

class Location(BaseModel):
    lat: float = Field(ge=-90, le=90)
    lng: float = Field(ge=-180, le=180)


class Turn(BaseModel):
    role: Literal["user", "assistant"]
    text: str = Field(max_length=1000)


class ChatRequest(BaseModel):
    query: str = Field(max_length=2000, description="A dish, place or time, in English or Khmer. Only the first 200 characters are used.",
                       examples=["kuy teav open now", "coffee near CADT", "ម្ហូបខ្មែរ បើកឥឡូវ"])
    location: Location | None = Field(None, description="The user's position, only needed for 'near me'.")
    no_location: bool = Field(False, description="The user declined to share a location: answer 'near me' without distances.")
    model: str | None = Field(None, max_length=100, description="A model id from /api/config, or 'none' for results only. "
                                                              "Unknown ids are ignored (the default model answers).")
    # the conversation: the server keeps no state, the client sends what it showed
    history: list[Turn] = Field(default_factory=list, max_length=12, description="The last messages, oldest first.")
    context_ids: list[str] = Field(default_factory=list, max_length=15,
                                   description="Ids of the places shown last, in the order shown (for follow-up questions).")
    last_query: str | None = Field(None, max_length=200, description="The previous search (its place or area is kept).")
    direct: bool = Field(False, description="Search `query` as it is, without planning (e.g. after the location step).")
    dish_id: str | None = Field(None, max_length=60, description="The dish shown last (from a photo), so 'how long does "
                                                                 "it take?' is about that dish.")


class DishRequest(BaseModel):
    dish_id: str = Field(max_length=60, examples=["pad_thai"], description="A dish id (one of the 101 Food-101 dishes).")
    lang: Literal["en", "kh"] = "en"
    location: Location | None = None


class DemoAskRequest(BaseModel):
    query: str = Field(max_length=2000, examples=["kuy teav", "coffee near CADT"])
    model: str | None = Field(None, max_length=100, description="A model id from /api/config, or 'none'.")


class CompareRequest(BaseModel):
    query: str = Field(max_length=2000, examples=["curry noodle", "carry nodle"])
    weight: float = Field(0.5, ge=0, le=1, description="0 = BM25 only, 1 = embeddings only.")


# ======================= chat app routes (website and Telegram Mini App) =======================

chat_router = APIRouter(prefix="/api", tags=["chat"])


@chat_router.get("/config", summary="What the UI can offer")
async def config():
    return {"photo_search": dishes.available() or image_search_available(), "dish_model": dishes.model_label(),
            "places": int(len(search.df)),
            "models": [{"id": m["id"], "label": m["label"]} for m in MODELS], "default_model": DEFAULT_MODEL_ID,
            "no_llm": NO_LLM}


@chat_router.post("/chat", summary="Ask a question (search + AI answer)")
async def chat(body: ChatRequest, request: Request):
    query = body.query.strip()[:200]
    if not query:
        return error(400, "empty_query", "Type a dish or a place.")
    lang = detect_lang(query)
    STATS["web_text_queries"] += 1
    if not allow_request(client_key(request)):
        return too_many(lang)
    log.info("web query (lang=%s, %d chars)", lang, len(query))       # never log the text itself
    user_loc = (body.location.lat, body.location.lng) if body.location else None
    if user_loc == (0.0, 0.0):
        user_loc = None
    model = valid_model_id(body.model)
    history = [{"role": t.role, "text": t.text} for t in body.history][-HISTORY_TURNS * 2:]
    focus = body.dish_id if body.dish_id in dishes.DISHES else None
    named = dishes.find_dish(query)
    kind = dishes.recipe_intent(query, has_dish=bool(named or focus))
    if kind:                                     # "how do I make it?", "how long does pad thai take?": the recipe
        dish_id = named or focus
        out = await asyncio.to_thread(dishes.recipe_answer, query, dish_id, lang, model)
        return {"kind": "recipe", "query": query, "lang": lang, "answer": out["text"], "provider": out["provider"],
                "model": out["model"], "model_label": out["label"], "dish": dishes.dish_json(dish_id),
                "recipe": dishes.recipe_json(dish_id), "focus": out["focus"], "results": [], "filters": {}}
    query = dishes.resolve_reference(query, focus)   # "where can I eat it?" after a photo of pizza -> "... pizza?"
    search_query, rewritten = query, False
    if not body.direct:
        places = get_rows(body.context_ids[:CONTEXT_PLACES]) if body.context_ids else None
        if places is not None and len(places) and user_loc:
            places = places.assign(distance_km=haversine_km(user_loc[0], user_loc[1], places["lat"], places["lng"]))
        plan = await asyncio.to_thread(plan_turn, query, history, places, model, body.last_query)
        if plan["action"] == "chat":
            out = await asyncio.to_thread(chat_reply, query, history, model)
            return {"kind": "chat", "query": query, "lang": lang, "answer": out["text"], "provider": out["provider"],
                    "model": out["model"], "model_label": out["label"], "results": [], "filters": {}}
        if plan["action"] == "followup" and places is not None and len(places):    # about the shown places: no search
            out = await asyncio.to_thread(answer_followup, query, history, places, plan["places"], model)
            picked = places.iloc[[n - 1 for n in out["numbers"]]]
            return {"kind": "followup", "query": query, "lang": lang, "answer": out["text"], "provider": out["provider"],
                    "model": out["model"], "model_label": out["label"], "numbers": out["numbers"], "shape": out["shape"],
                    # an answer built from the data already says it: cards under it would repeat it
                    "results": places_json(picked, lang) if out["provider"] != "facts" else [], "filters": {}}
        search_query = plan["query"] or query
        rewritten = search_query.strip().lower() != query.strip().lower()
    if parse_query_filters(search_query)[1].get("near") == "me" and not user_loc and not body.no_location:
        # the UI asks for the location, then sends search_query again with direct=true
        return {"query": query, "search_query": search_query, "lang": lang, "need_location": True}
    out = await asyncio.to_thread(answer_query, search_query, user_loc, model, RESULTS_PER_QUERY, True)
    results = out["results"]
    return {
        "kind": "search",
        "query": query,
        "search_query": search_query if rewritten else None,                    # shown as "searched for ..."
        "lang": lang,
        "answer": out["answer"] if out["provider"] in ("gemini", "ollama", "facts") else None,   # no match: no_match says it
        "provider": out["provider"],
        "model": out["model"],
        "model_label": out["model_label"],                                      # which model actually answered
        "no_match": no_match(lang, results) if len(results) == 0 else None,
        "origin": out["origin"],
        "filters": {k: v for k, v in results.attrs.get("filters", {}).items() if k != "near"},
        "unverified_note": unverified_note(results, lang) or None,              # e.g. "under $5" with no prices in the data
        "results": places_json(results, lang, with_menu="menu" in asked_fields(search_query)),
    }


def dish_reply(dish, lang, user_loc=None) -> dict:
    """A dish (recognised or picked) -> the lead sentence, the dish, its recipe and the places that serve it."""
    results = hybrid_search(dish["name"], RESULTS_PER_QUERY, user_loc)
    return {"kind": "dish", "query": dish["name"], "lang": lang, "answer": dishes.photo_lead(dish, lang),
            "provider": "photo", "model": None, "model_label": dish.get("model"), "origin": {"kind": "me"} if user_loc else None,
            "dish": dish, "recipe": dishes.recipe_json(dish["id"]), "filters": {},
            "no_match": no_match(lang, results) if len(results) == 0 else None, "results": places_json(results, lang)}


@chat_router.post("/photo", summary="Search with a food photo")
async def photo(request: Request, photo: UploadFile = File(...), lang: Literal["en", "kh"] = Form("en"),
                lat: float | None = Form(None, ge=-90, le=90), lng: float | None = Form(None, ge=-180, le=180)):
    if not (dishes.available() or image_search_available()):
        return error(503, "photo_search_off", "Photo search is not set up yet. Type the dish name instead.")
    STATS["web_photo_queries"] += 1
    if not allow_request(client_key(request)):
        return too_many()
    data = await photo.read(MAX_PHOTO_MB * 1024 * 1024 + 1)
    if len(data) > MAX_PHOTO_MB * 1024 * 1024:
        return error(413, "too_large", f"Photos can be up to {MAX_PHOTO_MB} MB.")
    fd, tmp_path = tempfile.mkstemp(suffix=Path(photo.filename or "").suffix.lower() or ".jpg")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
        if dishes.available():
            guesses = await asyncio.to_thread(dishes.classify, tmp_path, 3)
        else:
            matches = await asyncio.to_thread(search_images_by_image, tmp_path, 10)
    except Exception as e:                        # not an image, unreadable file, ...
        log.warning("photo search failed (%s)", safe_error(e))
        return error(400, "bad_photo", "That file could not be read as a photo.")
    finally:
        Path(tmp_path).unlink(missing_ok=True)    # the user's photo is never kept
    if dishes.available():
        best = guesses[0]
        log.info("photo: %s (%.0f%%)", best["id"], best["confidence"] * 100)
        dish = dishes.dish_json(best["id"], best["confidence"], guesses[1:])
        user_loc = (lat, lng) if lat is not None and lng is not None else None
        return await asyncio.to_thread(dish_reply, dish, lang, user_loc)
    results = get_rows([m["restaurant_id"] for m in matches])
    return {"query": "", "lang": "en", "answer": None, "provider": "photo", "origin": None, "filters": {},
            "no_match": no_match("en", results) if len(results) == 0 else None, "results": places_json(results, "en")}


@chat_router.post("/dish", summary="A dish chosen by the user: recipe + places")
async def dish(body: DishRequest, request: Request):
    if body.dish_id not in dishes.DISHES:
        return error(400, "unknown_dish", "Unknown dish.")
    if not allow_request(client_key(request)):
        return too_many(body.lang)
    user_loc = (body.location.lat, body.location.lng) if body.location else None
    return await asyncio.to_thread(dish_reply, dishes.dish_json(body.dish_id), body.lang, user_loc)


# ======================= demo routes ("How it works" page, /demo) =======================

demo_router = APIRouter(prefix="/api/demo", tags=["demo"])


def filters_list(flt, query="") -> list:
    """{'open_now': True, 'area': 'toul kork'} -> ['open now', 'in Toul Kork'] (readable conditions)."""
    names = {"max_price": "under ${v}", "min_rating": "rated {v}+", "open_now": "open now", "open_late": "open late",
             "area": "in {v}"}
    out = []
    for k, v in flt.items():
        if k == "near":
            out.append("near me" if v == "me" else f"near {as_typed(query, v)}")
        elif k in names:
            out.append(names[k].format(v=v.title() if isinstance(v, str) else v))
    return out


def scored_rows(results) -> list:
    """The rows the model receives, with every score that ranked them (0-1 per query, plus small bonuses)."""
    return [{"name": r.name, "dish": (r.dish_en or "").replace(";", ","), "area": r.area,
             "bm25": num(r.bm25, 3), "embed": num(r.embed, 3), "rating_bonus": num(r.rating_boost, 3),
             "distance_bonus": num(r.distance, 3), "final": num(r.final, 3), "distance_km": num(r.distance_km)}
            for r in results.itertuples()]


def ranked(rows, score_key) -> list:
    if rows is None:
        return []
    return [{"id": r.id, "name": r.name, "dish": (getattr(r, "dish_en", "") or "").replace(";", ","), "area": r.area,
             "score": num(getattr(r, score_key), 3)} for r in rows.itertuples()]


@demo_router.post("/ask", summary="Ask, and see every step of the answer")
async def ask(body: DemoAskRequest, request: Request):
    query = body.query.strip()[:200]
    if not query:
        return error(400, "empty_query", "Type a dish or a place.")
    if not allow_request(client_key(request)):
        return too_many(detect_lang(query))
    # k = every place: the demo shows how many matched, not only the top rows
    out = await asyncio.to_thread(answer_query, query, None, valid_model_id(body.model), len(search.df))
    results = out["results"]
    top = results.head(TOP_K)
    prompt = build_prompt(out["llm_query"], top, detect_lang(query)) if len(top) else ""
    attrs = results.attrs
    chosen = out["chosen_model"]
    return {
        "answer": out["answer"],
        "provider": out["provider"],                      # gemini | ollama | cards | no-results
        "model_label": out["model_label"],
        "chosen_label": MODELS_BY_ID[chosen]["label"] if chosen in MODELS_BY_ID else None,
        "fallback": chosen in MODELS_BY_ID and out["model"] != chosen and out["model"] is not None,
        "origin": out["origin"],
        "steps": {
            "text": attrs.get("query_text", ""),
            "filters": filters_list(attrs.get("filters", {}), query),
            "lang": detect_lang(query),
            "matched": len(results),
            "total": len(search.df),
            "top_k": min(TOP_K, len(results)),
            "method": attrs.get("method", "-"),
            "geocoded": bool(out["origin"] and out["origin"]["kind"] in ("place", "not_found")),
            "retrieve_ms": out["timing"]["retrieve_ms"],
            "generate_ms": out["timing"]["generate_ms"],
            "prompt_chars": len(prompt),
        },
        "rows": scored_rows(top),
        "prompt": prompt,
        "weight": HYBRID_WEIGHT,
    }


@demo_router.post("/compare", summary="BM25 vs embeddings vs hybrid for one query")
async def compare(body: CompareRequest, request: Request):
    query = body.query.strip()[:200]
    if not query:
        return error(400, "empty_query", "Type a query.")
    if not allow_request(client_key(request)):
        return too_many(detect_lang(query))
    text, flt = parse_query_filters(query)

    def run():
        hy = hybrid_search(query, TOP_K, weight=body.weight)
        bm = bm25_search(text, TOP_K) if text else None
        em = embedding_search(text, TOP_K) if text else None
        return hy, bm, em
    hy, bm, em = await asyncio.to_thread(run)
    ids = lambda t: set() if t is None else set(t["id"])
    in_hybrid = ids(hy)
    return {
        "text": text, "filters": filters_list(flt, query), "method": hy.attrs.get("method", "-"), "weight": body.weight,
        "bm25": ranked(bm, "score"), "embeddings": ranked(em, "score"), "hybrid": ranked(hy, "final"),
        "in_hybrid": sorted(in_hybrid),
        "shared": len(ids(bm) & ids(em)),
        "hybrid_from_bm25": len(ids(bm) & in_hybrid),
        "hybrid_from_embeddings": len(ids(em) & in_hybrid),
        "top_k": TOP_K,
    }


@demo_router.get("/data", summary="What the place list contains")
async def data():
    d = search.df
    n = len(d)
    coverage = [
        ("Area (khan, city or province)", d["area"].ne("")),
        ("Map location", d["lat"].notna() & d["lng"].notna()),
        ("Khmer dish name", d["dish_kh"].ne("")),
        ("English dish name", d["dish_en"].ne("")),
        ("Opening hours", d["open_hour"].notna() & d["close_hour"].notna()),
        ("Phone number", d["phone_public"].ne("")),
        ("Facebook or website", d["page_url"].ne("") if "page_url" in d else pd.Series(False, index=d.index)),
        ("Photo", d["image_url"].ne("") if "image_url" in d else d["image_path"].ne("")),
        ("Price", d["price_usd"].notna()),
        ("Rating", d["rating"].notna()),
    ]
    coverage = sorted(({"field": f, "count": int(mask.sum())} for f, mask in coverage), key=lambda x: -x["count"])
    areas = d["area"].replace("", "Unknown").value_counts()
    kinds = d["tags"].str.split(";").str[0].str.strip().replace("", "other").value_counts()
    sources = d["source"].str.lower().value_counts()
    return {
        "total": n,
        "area_count": int(d.loc[d["area"].ne(""), "area"].nunique()),
        "with_hours": int((d["open_hour"].notna() & d["close_hour"].notna()).sum()),
        "with_dish": int((d["dish_en"].ne("") | d["dish_kh"].ne("")).sum()),
        "coverage": coverage,
        "areas": [{"name": k, "count": int(v)} for k, v in areas.items()],
        "kinds": [{"name": k, "count": int(v)} for k, v in kinds.items()],
        "sources": [{"name": k, "count": int(v)} for k, v in sources.items()],
    }


# ======================= the app: error handling, routes, and the built React frontend =======================


NOT_BUILT = """<!doctype html><meta charset="utf-8"><title>Frontend not built</title>
<body style="font-family:system-ui;max-width:40em;margin:4em auto;padding:0 1em;line-height:1.5">
<h1>The frontend is not built yet</h1>
<p>The API is running (see <a href="/docs">/docs</a>). To build the website, run:</p>
<pre style="background:#f2f4f7;padding:12px;border-radius:8px">cd frontend
npm install
npm run build</pre>
<p>Then reload this page. While developing, <code>npm run dev</code> serves the frontend with live reload.</p></body>"""


def create_app(lifespan=None) -> FastAPI:
    app = FastAPI(title="Food search API", version="2.0", lifespan=lifespan,
                  description="Hybrid search (BM25 + embeddings) over places across Cambodia, "
                              "with an AI answer written only from the retrieved rows.")

    @app.middleware("http")
    async def headers_and_errors(request: Request, call_next):
        try:
            resp = await call_next(request)
        except Exception as e:                        # never send a stack trace (or a secret) to the client
            log.error("web error on %s: %s", request.url.path, safe_error(e))
            resp = error(500, "server_error", "Something went wrong on the server. Try again.")
        resp.headers.setdefault("X-Content-Type-Options", "nosniff")
        resp.headers.setdefault("Referrer-Policy", "strict-origin-when-cross-origin")
        return resp

    @app.exception_handler(RequestValidationError)
    async def bad_request(request, exc):
        return error(400, "bad_request", "The request was not valid. Type a dish or a place.")

    app.include_router(chat_router)
    if ENABLE_DEMO:
        app.include_router(demo_router)

    # ---- the React app: real files from frontend/dist, every other page path -> index.html (client-side routing) ----
    dist = FRONTEND_DIST.resolve()

    @app.get("/{path:path}", include_in_schema=False)
    async def frontend(path: str):
        if path.startswith("api/") or path == "api":
            return error(404, "not_found", "No such API endpoint. See /docs.")
        index = dist / "index.html"
        if not index.exists():
            return HTMLResponse(NOT_BUILT, status_code=503)
        file = (dist / path).resolve()
        if path and file.is_file() and dist in file.parents:            # never serve files outside dist/
            # Vite puts a content hash in asset file names, so they can be cached for a long time
            cache = "public, max-age=31536000, immutable" if path.startswith("assets/") else "no-cache"
            return FileResponse(file, headers={"Cache-Control": cache})
        return FileResponse(index, headers={"Cache-Control": "no-cache"})

    return app


In [ ]:
%%writefile export/backend/bot.py
"""Cambodian food search Telegram bot.

Run (from the project folder):  python backend/bot.py      (TELEGRAM_TOKEN etc. come from the environment or a local .env)

Switch to a webhook (e.g. Cloud Run / a VPS with HTTPS) by replacing app.run_polling(...) with:
    app.run_webhook(listen="0.0.0.0", port=int(os.environ.get("PORT", 8080)),
                    url_path="telegram", webhook_url="https://YOUR-DOMAIN/telegram",
                    secret_token=os.environ["WEBHOOK_SECRET"])
(this needs: pip install "python-telegram-bot[webhooks]")
"""
import logging
import sys

from food_search import search
from food_search.bot_handlers import build_bot_app
from food_search.config import TELEGRAM_TOKEN


def main():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    logging.getLogger("httpx").setLevel(logging.WARNING)   # httpx logs URLs that contain the token
    if not TELEGRAM_TOKEN:
        sys.exit("TELEGRAM_TOKEN is not set (environment variable or .env).")
    search.init()
    app = build_bot_app(TELEGRAM_TOKEN)
    app.run_polling(drop_pending_updates=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export/backend/web.py
"""Website + Telegram Mini App + "How it works" demo for the food search (RAG) chatbot.

Run (from the project folder):
    cd frontend && npm install && npm run build && cd ..     # once, and after changing the frontend
    python backend/web.py
then open
    http://127.0.0.1:8080        the chat app (also the Telegram Mini App)
    http://127.0.0.1:8080/demo   how it works: the 3 steps, retrieved rows, prompt, BM25 vs embeddings vs hybrid
    http://127.0.0.1:8080/docs   API documentation

While working on the frontend, run `npm run dev` in frontend/ instead (live reload on http://localhost:5173,
API calls are forwarded to this server).

It also runs the Telegram bot in the same process when TELEGRAM_TOKEN is set (RUN_BOT=0 turns that off,
e.g. while backend/bot.py is already running somewhere else: two pollers on one token conflict).
For the Mini App, Telegram needs the page on a public https:// address: put it behind HTTPS
(a server with a certificate, Cloud Run, or a tunnel such as `ngrok http 8080`),
set WEBAPP_URL to that address and restart. The bot then shows an "Open app" button.
"""
import logging
import sys
from contextlib import asynccontextmanager
from pathlib import Path

# Ensure backend/ is in sys.path
sys.path.insert(0, str(Path(__file__).resolve().parent))

import uvicorn

from food_search import search
from food_search.api import create_app
from food_search.bot_handlers import build_bot_app, setup_bot
from food_search.config import HOST, PORT, RUN_BOT, TELEGRAM_TOKEN

log = logging.getLogger("food_search.web")


@asynccontextmanager
async def bot_lifespan(app):
    """Starts the bot's polling next to the web server and stops it on shutdown."""
    bot_app = build_bot_app(TELEGRAM_TOKEN)
    await bot_app.initialize()
    await setup_bot(bot_app)                  # post_init only runs with run_polling(), so call it here
    await bot_app.start()
    await bot_app.updater.start_polling(drop_pending_updates=True)
    log.info("telegram bot is polling")
    try:
        yield
    finally:
        await bot_app.updater.stop()
        await bot_app.stop()
        await bot_app.shutdown()


def main():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    logging.getLogger("httpx").setLevel(logging.WARNING)       # httpx logs URLs that contain the token
    search.init()
    with_bot = bool(TELEGRAM_TOKEN and RUN_BOT)
    if not with_bot:
        log.info("web only (no TELEGRAM_TOKEN, or RUN_BOT=0)")
    uvicorn.run(create_app(lifespan=bot_lifespan if with_bot else None), host=HOST, port=PORT, access_log=False)


if __name__ == "__main__":
    main()


In [ ]:
# requirements.txt pinned to the versions that work in this environment
from importlib.metadata import version

PACKAGES = ["rank-bm25", "rapidfuzz", "sentence-transformers", "torch", "torchvision", "chromadb", "open_clip_torch",
            "python-telegram-bot", "google-genai", "requests", "pandas", "numpy", "matplotlib",
            "scikit-learn", "python-dotenv", "pillow",
            "fastapi", "uvicorn", "python-multipart"]          # the web API (backend/food_search/api.py)
pins = [f"{p}=={version(p)}" for p in PACKAGES]
(EXPORT_DIR / "requirements.txt").write_text("\n".join(pins) + "\n", encoding="utf-8")
print("\n".join(pins))


In [ ]:
%%writefile export/.dockerignore
.env
*.session
venv/
__pycache__/
.ipynb_checkpoints/
*.ipynb
results/
docs/
export/
frontend/node_modules/
frontend/dist/
export_backups/


In [ ]:
%%writefile export/Dockerfile
# Website / Telegram Mini App + demo + Telegram bot in one image.   Build: docker build -t food-bot .
# Run (secrets come from the host, never from the image):  docker run --env-file .env -p 8080:8080 food-bot
# Bot only (no website):  docker run --env-file .env food-bot python backend/bot.py

# ---- 1. build the React frontend (only its output, frontend/dist, goes into the final image) ----
FROM node:22-slim AS frontend
WORKDIR /frontend
COPY frontend/package.json frontend/package-lock.json ./
RUN npm ci --no-audit --no-fund
COPY frontend/ ./
RUN npm run build

# ---- 2. the Python app ----
FROM python:3.13-slim

ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1 HF_HOME=/app/.cache/huggingface HOST=0.0.0.0 PORT=8080
WORKDIR /app

# CPU-only torch keeps the image small (the default Linux wheel bundles CUDA, several GB)
COPY requirements.txt .
RUN pip install --no-cache-dir torch==$(grep '^torch==' requirements.txt | cut -d= -f3) \
        --index-url https://download.pytorch.org/whl/cpu \
 && pip install --no-cache-dir -r requirements.txt

# Cache the embedding model in an image layer so containers start fast and work offline.
# Change EMBED_MODEL here AND at runtime if you switch models (e.g. BAAI/bge-m3).
ARG EMBED_MODEL=paraphrase-multilingual-MiniLM-L12-v2
RUN python -c "from sentence_transformers import SentenceTransformer; SentenceTransformer('${EMBED_MODEL}')"

COPY backend/ backend/
COPY --from=frontend /frontend/dist frontend/dist/
COPY data/ data/
# The dish photo model (food_classification.ipynb) and the 101 dish names. Without the .pt file the app
# uses CLIP zero-shot, which is downloaded on first use: run the app once with internet, or train the model.
COPY models/ models/
# Prebuilt vector index from the notebook (init() only refreshes it)
COPY chroma_db/ chroma_db/

EXPOSE 8080
CMD ["python", "backend/web.py"]


In [ ]:
# Copy the exported files into the project. The notebook is the source of the backend, so a file that differs is
# replaced; the version it replaces is first saved in export_backups/<date_time>/, so nothing is ever lost.
from datetime import datetime

GENERATED = "# Generated by food_search.ipynb (section 9, Export). Edit the notebook and re-run the export, not this file.\n"
BACKUP_DIR = ROOT / "export_backups" / datetime.now().strftime("%Y-%m-%d_%H%M%S")

for staged in sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file() and "__pycache__" not in p.parts):
    rel = staged.relative_to(EXPORT_DIR)
    target = ROOT / rel
    content = GENERATED + staged.read_text(encoding="utf-8")
    if target.exists() and target.read_text(encoding="utf-8") == content:
        status = "up to date"
    else:
        if target.exists():
            backup = BACKUP_DIR / rel
            backup.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(target, backup)
            status = f"replaced (old version: {backup.relative_to(ROOT)})"
        else:
            status = "created"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
    print(f"{status:80s} {rel}")


In [ ]:
# Quick check: every exported Python file compiles
import py_compile
for rel in sorted(p.relative_to(EXPORT_DIR) for p in EXPORT_DIR.rglob("*.py")):
    py_compile.compile(str(ROOT / rel), doraise=True)
    print("✅", rel)


## 10. Try the conversation

**What:** a short conversation played through the same functions the web app and the Telegram bot use: each message
is planned (follow-up, search or chat, section 7), then answered in the shape that fits it.

**Why:** to see on real data that the bot answers only what is asked: a sentence for one detail, a list for one
detail of several places, a table for a comparison, and a fixed reply for small talk. Change the messages and re-run.
The AI model writes the search summaries and the "which one…?" answers; without a model, plain facts are shown.

In [ ]:
def chat_turn(message, state):
    """One chat message, like the web app does it: plan the turn, then chat, answer a follow-up, or search."""
    plan = plan_turn(message, state["history"], state["places"], None, state["last_query"])
    if plan["action"] == "chat":
        text = chat_reply(message, state["history"])["text"]
    elif plan["action"] == "followup":
        out = answer_followup(message, state["history"], state["places"], plan["places"])
        text = f"*({out['shape']})*\n\n{out['text']}"
    else:
        results = hybrid_search(plan["query"], TOP_K)
        text = generate_answer(plan["query"], results)["text"]
        state.update(places=results.head(CONTEXT_PLACES), last_query=plan["query"])
        text += "\n\n" + "\n".join(f"{i}. {r.name} ({r.area})" for i, r in enumerate(results.head(5).itertuples(), 1))
    state["history"] += [{"role": "user", "text": message}, {"role": "assistant", "text": text}]
    return plan, text


def play(messages):
    """Shows each message, what the bot decided, and its answer (tables render as tables)."""
    state = {"history": [], "places": None, "last_query": None}
    for message in messages:
        plan, text = chat_turn(message, state)
        searched = f" → searched for *{plan['query']}*" if plan["action"] == "search" else ""
        display(Markdown(f"**🧑 {message}** &nbsp; `{plan['action']}`{searched}\n\n{text}\n\n---"))

In [ ]:
play(["hi",
      "kuy teav",
      "is the first one open now?",
      "and its phone?",
      "compare 1 and 2",
      "what are their opening hours?",
      "menu of number 2",
      "which one is cheapest?",
      "what about in Siem Reap?",
      "what's the weather today?"])

## 11. Summary and next steps

**What we built:** a bilingual (Khmer + English) food search over restaurants and online sellers:
BM25 + fuzzy typo handling → multilingual embeddings in ChromaDB → hybrid ranking with price/hours/area
filters and distance, across Phnom Penh's khans and Cambodia's provinces → optional CLIP photo search → LLM answers
grounded in the retrieved results (with a model → next model → plain-facts fallback) → a conversation that answers
follow-up questions about the places shown, in a sentence, a list or a table → a Telegram bot and a web app
(Telegram Mini App), exported as a deployable package.

**Results:** the comparison table below (also saved in `results/eval_results.csv`) appears once you add test queries
(section 2); until then the evaluation is skipped.

**Limitations**
* **Data:** the places come from OpenStreetMap (© OpenStreetMap contributors, ODbL), Overture Maps, Foodpanda and other
  sources collected by the scrapers and `data_preprocessing.ipynb`. **Facts only:** the bot repeats what the data says, so
  a rating, price, opening hours or menu stays empty unless the place's own source gave it (nothing is filled in from a
  category or a template). Most places therefore have a name, area, cuisine, link and map location but no price, hours or
  menu; the bot says so ("Our data has no hours for 1. …") and cannot check a condition such as "under $5" or "open now"
  for places without that value. Outside Phnom Penh few places have map locations, so "near me" works best in Phnom Penh. The test queries are written by you: a person must
  check the labels, and a small test set shows *trends*, not real-world quality.
* **Khmer is low-resource:** character n-grams are a rough substitute for real word segmentation, and
  multilingual models have seen far less Khmer than English.
* **LLMs can still make mistakes**, even with only retrieved results in the prompt. Always show the result cards too.
* The "open now" check depends on the stored hours being correct.

**Privacy**
* No account is needed. The bot stores locations only in memory (until `/deleteme` or a restart) and deletes photos right after the search.
* Logs contain no message text next to user ids, and secrets are never printed or logged.
* Keys stay in `.env` / host environment variables, and `.env` is in `.gitignore` and `.dockerignore`.

**Next steps (ML / DL)**
1. **Fine-tune the embedding model** on Khmer–English query/place pairs (sentence-transformers + MultipleNegativesRankingLoss).
2. **Train a dish image classifier** (e.g. fine-tune a CNN/ViT on photos of amok, lok lak, kuy teav, …).
3. **Learning-to-rank with LightGBM** (LambdaMART) on features such as bm25, embed, distance, rating, price and open_now.

In [70]:
if HAS_EVAL:
    # Final: evaluation comparison
    display(eval_table)
else:
    print("⏭️ skipped: no evaluation queries yet (see section 2)")

⏭️ skipped: no evaluation queries yet (see section 2)


In [71]:
# Final: hybrid search for the three demo queries
for q in DEMO_QUERIES:
    res = hybrid_search(q)
    print(f"🔎 {q}   (keyword method: {res.attrs['method']})")
    display(res[SHOW].round(3))

🔎 curry noodle   (keyword method: bm25)


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final
0,ovt-8bd3673f-e4c7-4d3e-a04a-dd2a0e4b08b8,Curry Zone Restaurant,restaurant,NaN,NaN,0.978,0.854,0.0,0.916
1,ovt-04f8a050-2b30-4f24-9ef4-0e94dbbee1c2,Curry Leaf India Restaurant,restaurant,NaN,NaN,0.957,0.872,0.0,0.915
2,osm-n5586787217,Khmer curry stall,,NaN,NaN,1.000,0.729,0.0,0.864
3,ovt-7bda2203-0742-436a-98a5-a84c468c0878,VN noodles,,NaN,NaN,0.686,1.000,0.0,0.843
4,ovt-7b7e38dc-063b-4b91-8356-34cd88fd564e,Universal Noodles,,NaN,NaN,0.686,0.991,0.0,0.839


🔎 carry noodle   (keyword method: bm25+typo-fix)


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final
0,osm-n4178059189,Marugame Udon,noodles,NaN,NaN,0.780,1.000,0.0,0.890
1,osm-n4586489994,Malop Chat Toul Tom Poung,noodles; local,NaN,NaN,0.780,0.845,0.0,0.812
2,ovt-7b7e38dc-063b-4b91-8356-34cd88fd564e,Universal Noodles,,NaN,NaN,0.686,0.913,0.0,0.800
3,ovt-7bda2203-0742-436a-98a5-a84c468c0878,VN noodles,,NaN,NaN,0.686,0.873,0.0,0.780
4,osm-n4711644591,Spicy Noodle,,NaN,NaN,0.686,0.838,0.0,0.762


🔎 នំបញ្ចុក   (keyword method: bm25)


,id,name,dish_en,price_usd,rating,bm25,embed,distance,final
0,osm-n4861669721,បបរ នំបញ្ចុក,,NaN,NaN,1.000,0.739,0.0,0.870
1,ovt-7bf721b4-fa12-42c6-9135-d6a8433b21d7,នំបញ្ចុកស្រុះ,restaurant,NaN,NaN,0.878,0.808,0.0,0.843
2,osm-n10292468209,នំបញ្ចុក482,asian,NaN,NaN,1.000,0.645,0.0,0.822
3,ovt-b34f1a6d-8f20-49ea-887d-7b43e4bf05f3,នំបញ្ចុក កេរម៉ែ,restaurant,NaN,NaN,0.892,0.742,0.0,0.817
4,ovt-770f2a5d-7dff-48a1-abca-7ad4e8cb12cb,នំបញ្ចុកសាច់គោ,restaurant,NaN,NaN,0.852,0.748,0.0,0.800
